# IHARQ Phase 02 / Layer 02 — FINAL Post-Restart Continuation

## Start here after the Kaggle Python-kernel restart

This notebook is **not a fresh-session restore notebook**. It assumes `/kaggle/working` survived the kernel restart and canonical Stage 18 already has `G18=PASS`.

Run the code cells from the top in order.

**Do not run the old whole-working restore cell. Do not rerun Stages 00–18.**

The sequence is:

1. post-restart disk/GPU preflight;
2. dependency **verify-only** (no `pip install`, no second restart);
3. Post-Stage18-Accepted rehydration;
4. Stage18S R1.3 install/freeze;
5. Stage18S R1.3 adaptive 12-worker-ceiling execution;
6. Stage18S analysis/closure;
7. Stage18S downstream-readiness boundary;
8. Stage18U and Stages 19–24;
9. final whole-working Hugging Face preservation.

Stage18S remains a separate **post-hoc sensitivity supplement**. Canonical Stage 18/G18 are preserved and not rewritten.


In [1]:
# =============================================================================
# IHARQ P02/L2 — POST-KERNEL-RESTART PREFLIGHT
#
# This notebook assumes:
#   * you used Kaggle "Restart & clear cell outputs", NOT Stop session;
#   * /kaggle/working was preserved;
#   * canonical Stage18 had already closed with G18 PASS;
#   * Stage18S may contain partial valid work from the interrupted run.
#
# This cell NEVER restores or deletes /kaggle/working.
# =============================================================================

from pathlib import Path
import json
import os
import subprocess

KW = Path("/kaggle/working").resolve()
RUN = KW / "iharq_p02_run"

required = {
    "whole_working_restore_receipt":
        KW / "_IHARQ_WHOLE_WORKING_RESTORE_RECEIPT.json",
    "stage18_ledger":
        RUN / "stage_ledger" / "stage_18.json",
    "G18":
        RUN / "gate_results" / "G18.json",
    "stage18_a4_completion":
        RUN / "runtime" / "analysis_inputs" / "a4_completion.json",
}

missing = [k for k,p in required.items() if not p.is_file()]
if missing:
    raise RuntimeError(
        "POST_RESTART_PREFLIGHT_REQUIRED_DISK_STATE_MISSING:"
        + json.dumps(missing)
    )

ledger18 = json.loads(required["stage18_ledger"].read_text(encoding="utf-8"))
g18 = json.loads(required["G18"].read_text(encoding="utf-8"))
a4c = json.loads(required["stage18_a4_completion"].read_text(encoding="utf-8"))

if ledger18.get("status") != "SUCCESS":
    raise RuntimeError("POST_RESTART_STAGE18_LEDGER_NOT_SUCCESS")
if g18.get("status") != "PASS":
    raise RuntimeError("POST_RESTART_G18_NOT_PASS")
if a4c.get("status") != "PASS":
    raise RuntimeError("POST_RESTART_A4_COMPLETION_NOT_PASS")

terminals = sorted(
    (RUN / "runtime" / "run_cells").glob("P02-A4-*.json")
)
if len(terminals) != 1218:
    raise RuntimeError(
        f"POST_RESTART_CANONICAL_STAGE18_TERMINALS_NOT_1218:{len(terminals)}"
    )

# Stages19–24 must not already be accepted for this continuation notebook.
already_downstream = {}
for sid in range(19, 25):
    p = RUN / "stage_ledger" / f"stage_{sid}.json"
    if p.is_file():
        try:
            d = json.loads(p.read_text(encoding="utf-8"))
            already_downstream[str(sid)] = d.get("status")
        except Exception:
            already_downstream[str(sid)] = "UNREADABLE"

bad_downstream = {
    k:v for k,v in already_downstream.items()
    if v == "SUCCESS"
}
if bad_downstream:
    raise RuntimeError(
        "POST_RESTART_DOWNSTREAM_ALREADY_ACCEPTED:"
        + json.dumps(bad_downstream, sort_keys=True)
    )

s18s_root = (
    RUN / "runtime" / "supplements"
    / "stage18S_balanced_sensitivity_R1"
)
partial_files = (
    sum(1 for p in s18s_root.rglob("*") if p.is_file())
    if s18s_root.is_dir()
    else 0
)

try:
    gpu = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=index,name,utilization.gpu,memory.used,memory.free,memory.total",
            "--format=csv,noheader,nounits",
        ],
        text=True,
        stderr=subprocess.STDOUT,
        timeout=10,
    ).strip().splitlines()
except Exception as exc:
    gpu = [f"nvidia-smi unavailable: {type(exc).__name__}:{exc}"]

report = {
    "current_python_pid": os.getpid(),
    "canonical_stage18": ledger18.get("status"),
    "G18": g18.get("status"),
    "canonical_a4_terminals": len(terminals),
    "partial_stage18S_files_present": partial_files,
    "stages19_24_success": bad_downstream,
    "gpu_state": gpu,
    "safe_next_action": "RUN_DEPENDENCY_VERIFY_ONLY",
}

print("=" * 116)
print("POST-RESTART PREFLIGHT — PASS")
print("=" * 116)
print(json.dumps(report, indent=2))
print("\nDO NOT RUN THE WHOLE-WORKING RESTORE CELL.")
print("TRUSTED MARKER:")
print("IHARQ_P02_POST_RESTART_PREFLIGHT_PASS")


POST-RESTART PREFLIGHT — PASS
{
  "current_python_pid": 3353,
  "canonical_stage18": "SUCCESS",
  "G18": "PASS",
  "canonical_a4_terminals": 1218,
  "partial_stage18S_files_present": 1368,
  "stages19_24_success": {},
  "gpu_state": [
    "0, Tesla T4, 0, 0, 14912, 15360",
    "1, Tesla T4, 0, 0, 14912, 15360"
  ],
  "safe_next_action": "RUN_DEPENDENCY_VERIFY_ONLY"
}

DO NOT RUN THE WHOLE-WORKING RESTORE CELL.
TRUSTED MARKER:
IHARQ_P02_POST_RESTART_PREFLIGHT_PASS


## 1 — Verify the existing governed Python environment

This is intentionally **verify-only**. It does not reinstall packages.

In [2]:
# =============================================================================
# IHARQ P02 — RESTORED WORKING-TREE DEPENDENCY INSTALL / VERIFY
# R4 — FULL P02 REQUIREMENTS INSTALL + PROJECT-SCOPED STRICT VERIFY; KAGGLE-WIDE pip check is diagnostic
#
# PASS A:
#   - resolve the restored canonical P02 implementation package
#   - read its authoritative requirements-kaggle.txt
#   - compare exact installed versions for diagnostic evidence
#   - ALWAYS run pip install against the COMPLETE requirements-kaggle.txt
#   - pip therefore processes every governed requirement line, not only drifted pins
#   - verify exact pins + pip dependency consistency
#   - require a KERNEL restart
#
# PASS B:
#   - after restarting ONLY the Kaggle Python kernel,
#   - rerun THIS SAME CELL
#   - DO NOT reinstall
#   - verify all exact requirement pins + pip dependency consistency
#   - verify current Stage11 R7H7 scientific stack + CUDA
#
# IMPORTANT:
#   DO NOT restart/delete the Kaggle session.
#   DO NOT clear /kaggle/working.
# =============================================================================

from __future__ import annotations

from pathlib import Path
import importlib.metadata as imd
import json
import os
import platform
import re
import subprocess
import sys

KAGGLE_WORKING = Path("/kaggle/working").resolve()

EXPECTED_PACKAGE_NAME = (
    "IHARQ_P02_L2_Kaggle_Notebook_Implementation_Package_R5"
)

RESTORE_RECEIPT = (
    KAGGLE_WORKING / "_IHARQ_WHOLE_WORKING_RESTORE_RECEIPT.json"
)

OLD_CONTINUATION_MARKER = (
    KAGGLE_WORKING / "_IHARQ_CONTINUATION_RESTORED.json"
)

RESTART_MARKER = (
    KAGGLE_WORKING
    / "_IHARQ_RESTORED_DEPENDENCY_FULL_INSTALL_R4_RESTART_REQUIRED.json"
)


def fail(code, detail=None):
    raise RuntimeError(
        str(code)
        + (f": {detail}" if detail is not None else "")
    )


def pkg_version(name):
    try:
        return imd.version(name)
    except Exception:
        return None


def normalize_name(name):
    return re.sub(r"[-_.]+", "-", str(name)).lower()


def version_equal(pkg, expected, observed):
    if observed is None:
        return False

    expected = str(expected)
    observed = str(observed)

    # Torch metadata may expose a local CUDA suffix.
    if normalize_name(pkg) == "torch":
        return observed.split("+", 1)[0] == expected.split("+", 1)[0]

    return observed == expected


def package_is_valid(root):
    root = Path(root)

    return (
        root.is_dir()
        and
        (root / "src" / "iharq" / "layer2_decoders").is_dir()
        and
        (root / "requirements-kaggle.txt").is_file()
        and
        (
            root
            / "machine_readable"
            / "p02_notebook_stage_plan_R4.yaml"
        ).is_file()
    )


def parse_requirements(path):
    """
    Return exact == pins from requirements-kaggle.txt.

    The FULL file will still be passed to pip, preserving any
    legitimate non-pin lines/options present in the governed file.
    """
    pins = {}
    raw_lines = []

    for raw in Path(path).read_text(
        encoding="utf-8",
        errors="strict",
    ).splitlines():

        s = raw.strip()

        if not s or s.startswith("#"):
            continue

        raw_lines.append(s)

        # Remove environment marker for pin parsing only.
        left = s.split(";", 1)[0].strip()

        if "==" not in left:
            continue

        name, version = left.split("==", 1)

        name = name.strip().split("[", 1)[0]
        version = version.strip()

        if name and version:
            pins[normalize_name(name)] = {
                "name": name,
                "version": version,
            }

    return pins, raw_lines


# =============================================================================
# 0. REQUIRE VERIFIED WHOLE-WORKING RESTORE
# =============================================================================

if not RESTORE_RECEIPT.is_file():
    fail(
        "WHOLE_WORKING_RESTORE_RECEIPT_MISSING",
        {
            "expected": str(RESTORE_RECEIPT),
            "action":
                "Run the ready-to-run Hugging Face whole-working restore "
                "cell first.",
        },
    )

restore_receipt = json.loads(
    RESTORE_RECEIPT.read_text(encoding="utf-8")
)

if restore_receipt.get("verification") != "PASS":
    fail(
        "WHOLE_WORKING_RESTORE_NOT_VERIFIED",
        restore_receipt,
    )

print("=" * 118)
print("IHARQ P02 — RESTORED DEPENDENCY INSTALL / VERIFY")
print("=" * 118)

print("\n[RESTORE RECEIPT]")
print(json.dumps({
    "verification": restore_receipt.get("verification"),
    "repo_id": restore_receipt.get("repo_id"),
    "revision": restore_receipt.get("revision"),
    "manifest_sha256":
        restore_receipt.get("manifest_sha256"),
    "manifested_files_verified":
        restore_receipt.get("manifested_files_verified"),
}, indent=2))


# =============================================================================
# 1. RESOLVE THE RESTORED CANONICAL PACKAGE
# =============================================================================

candidates = []

# A. Exact canonical working-tree location.
canonical = KAGGLE_WORKING / EXPECTED_PACKAGE_NAME

if package_is_valid(canonical):
    candidates.append(
        ("CANONICAL_WORKING_ROOT", canonical.resolve())
    )


# B. Previously verified IHARQ continuation marker, if restored.
if OLD_CONTINUATION_MARKER.is_file():
    try:
        marker = json.loads(
            OLD_CONTINUATION_MARKER.read_text(
                encoding="utf-8"
            )
        )

        p = marker.get("package_root")

        if p:
            p = Path(p).resolve()

            if package_is_valid(p):
                candidates.append(
                    ("RESTORED_CONTINUATION_MARKER", p)
                )

    except Exception:
        pass


# C. Controlled discovery under /kaggle/working.
#    This is only a fallback.
if not candidates:
    for q in KAGGLE_WORKING.glob(
        "**/p02_notebook_stage_plan_R4.yaml"
    ):
        if q.parent.name != "machine_readable":
            continue

        root = q.parent.parent.resolve()

        if package_is_valid(root):
            candidates.append(
                ("WORKING_TREE_DISCOVERY", root)
            )


# Deduplicate exact paths.
dedup = {}

for origin, path in candidates:
    dedup[str(path)] = (origin, path)

candidates = list(dedup.values())

if not candidates:
    fail(
        "RESTORED_P02_PACKAGE_NOT_FOUND",
        {
            "expected_canonical":
                str(canonical),
            "meaning":
                "The file restore succeeded but the governed "
                "implementation package cannot be resolved.",
        },
    )


# Prefer exact canonical root if available.
canonical_matches = [
    x
    for x in candidates
    if x[1] == canonical.resolve()
]

if canonical_matches:
    PACKAGE_ORIGIN, PACKAGE_ROOT = canonical_matches[0]

elif len(candidates) == 1:
    PACKAGE_ORIGIN, PACKAGE_ROOT = candidates[0]

else:
    fail(
        "MULTIPLE_P02_PACKAGE_ROOTS_AMBIGUOUS",
        [
            {"origin": a, "path": str(b)}
            for a, b in candidates
        ],
    )


REQUIREMENTS = PACKAGE_ROOT / "requirements-kaggle.txt"

if not REQUIREMENTS.is_file():
    fail(
        "REQUIREMENTS_KAGGLE_MISSING",
        str(REQUIREMENTS),
    )

print("\n[PACKAGE]")
print("origin       =", PACKAGE_ORIGIN)
print("package_root =", PACKAGE_ROOT)
print("requirements =", REQUIREMENTS)


# =============================================================================
# 2. PARSE THE GOVERNED REQUIREMENTS
# =============================================================================

pins, requirement_lines = parse_requirements(REQUIREMENTS)

if not pins:
    fail(
        "REQUIREMENTS_KAGGLE_HAS_NO_EXACT_PINS",
        str(REQUIREMENTS),
    )

print("\n[GOVERNED EXACT PINS]")

print(json.dumps(
    {
        v["name"]: v["version"]
        for v in pins.values()
    },
    indent=2,
    sort_keys=True,
))


# =============================================================================
# 3. EXTRA CURRENT-STAGE11 CONTRACT CHECK
#
# These versions are explicitly required by the current R7H7 continuation.
# requirements-kaggle.txt remains the installation authority.
# =============================================================================

R7H7_REQUIRED = {
    "torch": "2.13.0",
    "braindecode": "1.6.1",
    "numpy": "2.2.6",
    "scipy": "1.15.3",
    "pandas": "2.3.1",
    "scikit-learn": "1.7.1",
    "mne": "1.12.1",
}

contract_conflicts = {}

for pkg, expected in R7H7_REQUIRED.items():
    key = normalize_name(pkg)

    if key not in pins:
        contract_conflicts[pkg] = {
            "R7H7_expected": expected,
            "requirements_kaggle": "MISSING",
        }
        continue

    req_version = pins[key]["version"]

    if not version_equal(
        pkg,
        expected,
        req_version,
    ):
        contract_conflicts[pkg] = {
            "R7H7_expected": expected,
            "requirements_kaggle": req_version,
        }

if contract_conflicts:
    fail(
        "R7H7_VS_REQUIREMENTS_KAGGLE_CONFLICT",
        {
            "conflicts": contract_conflicts,
            "meaning":
                "Do not install a mixed environment. "
                "The restored requirements authority and current "
                "Stage11 contract disagree.",
        },
    )

print("\nR7H7 requirements cross-check: PASS")


# =============================================================================
# 4. PYTHON ABI EVIDENCE CHECK, WHEN RESTORED ENV SNAPSHOT EXISTS
# =============================================================================

env_candidates = []

for p in KAGGLE_WORKING.glob(
    "_IHARQ_CONTINUATION_META_*/ENVIRONMENT_SNAPSHOT.json"
):
    if p.is_file():
        env_candidates.append(p)

if env_candidates:
    env_path = sorted(
        env_candidates,
        key=lambda p: p.stat().st_mtime,
    )[-1]

    try:
        saved_env = json.loads(
            env_path.read_text(encoding="utf-8")
        )

        saved_python = str(
            saved_env.get("python") or ""
        )

        current_python = platform.python_version()

        if saved_python:
            saved_mm = ".".join(
                saved_python.split(".")[:2]
            )

            current_mm = ".".join(
                current_python.split(".")[:2]
            )

            if saved_mm != current_mm:
                fail(
                    "PYTHON_ABI_MISMATCH",
                    {
                        "saved": saved_python,
                        "current": current_python,
                        "meaning":
                            "pip cannot safely repair a different "
                            "Python major/minor ABI.",
                    },
                )

        print("\n[PYTHON ABI]")
        print("saved   =", saved_python)
        print("current =", current_python)
        print("status  = PASS")

    except RuntimeError:
        raise

    except Exception as exc:
        print(
            "\n[WARN] Could not parse optional saved "
            f"environment evidence: {exc}"
        )


# =============================================================================
# 5. COMPARE CURRENT ENVIRONMENT AGAINST EVERY EXACT PIN
# =============================================================================

observed = {}
mismatches = []

for key, spec in pins.items():
    name = spec["name"]
    expected = spec["version"]

    actual = pkg_version(name)

    observed[name] = actual

    if not version_equal(
        name,
        expected,
        actual,
    ):
        mismatches.append({
            "package": name,
            "expected": expected,
            "observed": actual,
        })


print("\n[DEPENDENCY DRIFT]")

if mismatches:
    print(json.dumps(
        {"mismatches": mismatches},
        indent=2,
    ))

else:
    print("No exact-pin drift detected.")



# =============================================================================
# CURRENT CONTINUATION MODE — VERIFY ONLY, NEVER REINSTALL
# =============================================================================
print("\n" + "=" * 118)
print("POST-RESTART DEPENDENCY VERIFY-ONLY MODE")
print("=" * 118)

if mismatches:
    raise RuntimeError(
        "POST_RESTART_DEPENDENCY_DRIFT_DETECTED__DO_NOT_CONTINUE:"
        + json.dumps(
            {
                "mismatches": mismatches,
                "action":
                    "Use the canonical full dependency installer in the historical "
                    "notebook only if these pins genuinely drifted; that path requires "
                    "another kernel restart. Do not silently continue.",
            },
            sort_keys=True,
            default=str,
        )
    )

print(
    "All governed exact pins already match. "
    "Skipping pip installation and proceeding directly to strict runtime verification."
)

# =============================================================================
# 7. PASS B — EXACT METADATA + RUNTIME IMPORT VERIFICATION
# =============================================================================

print("\n" + "=" * 118)
print("VERIFY-ONLY — CHECKING RESTORED SCIENTIFIC ENVIRONMENT")
print("=" * 118)

# Re-run global pip check for evidence after restart. This is deliberately
# NON-BLOCKING because Kaggle's unrelated preinstalled packages can conflict
# with the governed P02 versions.
pip_check_b = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "check",
    ],
    text=True,
    capture_output=True,
)

pip_check_b_output = (
    (pip_check_b.stdout or "") + (pip_check_b.stderr or "")
).strip()

if pip_check_b.returncode == 0:
    pip_check_b_status = "PASS"
    print("global pip check after restart: PASS")
else:
    pip_check_b_status = "WARN_NON_P02_GLOBAL_ENV_CONFLICTS"
    print(
        "\n[GLOBAL KAGGLE pip check AFTER RESTART — NON-BLOCKING]\n"
        "Unrelated Kaggle packages still report dependency conflicts. "
        "Continuing with strict P02 pin/import/CUDA verification.\n"
    )
    print(pip_check_b_output[-12000:])

# Verify all exact requirement pins again.
final_mismatch = []

for key, spec in pins.items():
    name = spec["name"]
    expected = spec["version"]
    actual = pkg_version(name)

    if not version_equal(
        name,
        expected,
        actual,
    ):
        final_mismatch.append({
            "package": name,
            "expected": expected,
            "observed": actual,
        })

if final_mismatch:
    fail(
        "PINNED_DEPENDENCY_VERIFICATION_FAILED",
        final_mismatch,
    )


# Import current Stage11-critical libraries.
import torch
import numpy as np
import scipy
import pandas as pd
import sklearn
import mne
import braindecode
import torchaudio


runtime = {
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "torch_cuda": str(torch.version.cuda or ""),
    "cuda_available": bool(torch.cuda.is_available()),
    "numpy": str(np.__version__),
    "scipy": str(scipy.__version__),
    "pandas": str(pd.__version__),
    "scikit-learn": str(sklearn.__version__),
    "mne": str(mne.__version__),
    "braindecode": str(
        imd.version("braindecode")
    ),
    "torchaudio": str(
        imd.version("torchaudio")
    ),
}


if not torch.cuda.is_available():
    fail(
        "CUDA_NOT_AVAILABLE_AFTER_DEPENDENCY_RESTORE",
        runtime,
    )

if torch.cuda.device_count() < 1:
    fail(
        "NO_CUDA_DEVICE_AFTER_DEPENDENCY_RESTORE"
    )


# Exact current R7H7 contract.
for pkg, expected in R7H7_REQUIRED.items():
    actual = pkg_version(pkg)

    if not version_equal(
        pkg,
        expected,
        actual,
    ):
        fail(
            "R7H7_RUNTIME_VERSION_MISMATCH",
            {
                "package": pkg,
                "expected": expected,
                "observed": actual,
            },
        )


runtime["gpu_names"] = [
    torch.cuda.get_device_name(i)
    for i in range(torch.cuda.device_count())
]


# Optional project-relevant imports if present in requirements.
optional_imports = {
    "pyriemann": "pyriemann",
    "h5py": "h5py",
    "einops": "einops",
    "moabb": "moabb",
}

optional_status = {}

for pkg, module in optional_imports.items():
    if normalize_name(pkg) not in pins:
        continue

    try:
        __import__(module)

        optional_status[pkg] = {
            "version": pkg_version(pkg),
            "import": "PASS",
        }

    except Exception as exc:
        fail(
            "OPTIONAL_GOVERNED_PACKAGE_IMPORT_FAILED",
            {
                "package": pkg,
                "module": module,
                "error":
                    f"{type(exc).__name__}: {exc}",
            },
        )


print("\n[SCIENTIFIC RUNTIME VERIFIED]")
print(json.dumps(
    {
        "runtime": runtime,
        "additional_governed_packages":
            optional_status,
        "package_root": str(PACKAGE_ROOT),
        "requirements": str(REQUIREMENTS),
        "exact_pin_count": len(pins),
        "scientific_configuration_changed":
            False,
        "global_kaggle_pip_check":
            pip_check_b_status,
        "global_kaggle_pip_check_is_p02_gate":
            False,
    },
    indent=2,
))


# Remove a stale dependency restart marker only after successful verify-only pass.
if RESTART_MARKER.is_file():
    RESTART_MARKER.unlink()


# Durable readiness receipt.
READY_RECEIPT = (
    KAGGLE_WORKING
    / "_IHARQ_RESTORED_DEPENDENCIES_VERIFIED.json"
)

READY_RECEIPT.write_text(
    json.dumps(
        {
            "status": "PASS",
            "marker":
                "IHARQ_P02_RESTORED_DEPENDENCIES_VERIFIED",
            "package_root": str(PACKAGE_ROOT),
            "requirements": str(REQUIREMENTS),
            "exact_pins": {
                v["name"]: v["version"]
                for v in pins.values()
            },
            "runtime": runtime,
            "additional_governed_packages":
                optional_status,
            "scientific_configuration_changed":
                False,
            "dependency_installation_mode":
                "POST_RESTART_VERIFY_ONLY_NO_REINSTALL",
            "global_pip_check_after_restart":
                pip_check_b_status,
            "global_pip_check_is_p02_gate":
                False,
            "global_pip_check_tail":
                pip_check_b_output[-12000:]
                if pip_check_b_output else "",
            "safe_next_action":
                "RUN_POST_STAGE18_ACCEPTED_REHYDRATION",
        },
        indent=2,
        sort_keys=True,
    ) + "\n",
    encoding="utf-8",
)


print("\n" + "=" * 118)
print("RESTORED P02 DEPENDENCIES VERIFIED")
print("=" * 118)
print("receipt =", READY_RECEIPT)

print("\nTRUSTED SUCCESS MARKER:")
print("IHARQ_P02_POST_RESTART_DEPENDENCIES_VERIFY_ONLY_PASS")

print("\nNEXT:\nRun the Post-Stage18-Accepted rehydration cell.\nDo NOT rerun earlier completed stages.")

IHARQ P02 — RESTORED DEPENDENCY INSTALL / VERIFY

[RESTORE RECEIPT]
{
  "verification": "PASS",
  "repo_id": "csthv999z/p02-phase02-final-whole-working-20260813t191036z-223f3119",
  "revision": "0f0c38f2333c252f92430ea6f604daed98981847",
  "manifest_sha256": "3c0c6b5c246f393d3170268a3a43ea4baafeda7125676e1a31c041cee9fe2e99",
  "manifested_files_verified": 17319
}

[PACKAGE]
origin       = RESTORED_CONTINUATION_MARKER
package_root = /kaggle/working/IHARQ_P02_L2_Kaggle_Notebook_Implementation_Package_R5
requirements = /kaggle/working/IHARQ_P02_L2_Kaggle_Notebook_Implementation_Package_R5/requirements-kaggle.txt

[GOVERNED EXACT PINS]
{
  "PyYAML": "6.0.2",
  "braindecode": "1.6.1",
  "h5py": "3.14.0",
  "huggingface-hub": "1.24.0",
  "jsonschema": "4.25.0",
  "mne": "1.12.1",
  "moabb": "1.5.0",
  "nbformat": "5.10.4",
  "numpy": "2.2.6",
  "pandas": "2.3.1",
  "pooch": "1.8.2",
  "pydantic": "2.11.7",
  "pyriemann": "0.12",
  "pytest": "8.4.1",
  "safetensors": "0.8.0",
  "scikit-learn"

## 2 — Rehydrate the accepted Stage-18 runtime

This reconstructs live Python objects without rerunning canonical Stage 18 and preserves any valid partial Stage18S files.

In [3]:
# =============================================================================
# IHARQ P02/L2 — POST-STAGE18-ACCEPTED FRESH-KERNEL CONTINUATION REHYDRATION
# R1 — Stage18-accepted boundary; disk-state only; NO scientific stage rerun
# =============================================================================
#
# PURPOSE
#   Reconstruct the transient Python runtime after a kernel restart when
#   Stages 13–18 are ALREADY accepted and Stages 19–24 are not yet accepted.
#
# WHY R1 CANNOT BE USED HERE
#   The SHA-pinned R6R3 rehydrator was intentionally written for the earlier
#   Stage12→13 boundary and fail-closes if any Stage13+ stage is accepted.
#   At the current boundary that guard is obsolete: 13–18 are supposed to be
#   accepted. R2 preserves the exact verified rehydrator and changes ONLY its
#   downstream-boundary guard in memory, after proving the accepted disk state.
#
# SCIENTIFIC EFFECT
#   NONE. No stage is run. No model is trained. No prediction is regenerated.
#   Stage13–18 ledgers/gates and all canonical Stage18 closure/terminal evidence are SHA-256 checked before and after.
# =============================================================================

from __future__ import annotations
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os

KAGGLE_WORKING = Path("/kaggle/working").resolve()

EXPECTED_BASE_REHYDRATOR_SHA256 = (
    "58602da3549728d8a812e83438820c1843fdc28b79b6b63e9339f512a7a4870d"
)
EXPECTED_CANONICAL_STAGE12_SOURCE_SHA256 = (
    "12c8196a581fee0571508003a90805a936664dc35418055dacb6426f51e75d75"
)
EXPECTED_ACCEPTED_DOWNSTREAM = ["13", "14", "15", "16", "17", "18"]


def _p17_fail(code, detail=None):
    raise RuntimeError(
        str(code) + (":" + str(detail) if detail is not None else "")
    )


def _p17_sha(path: Path) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def _p17_load(path: Path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def _p17_atomic_json(path: Path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(obj, indent=2, sort_keys=True, default=str) + "\n",
        encoding="utf-8",
    )
    os.replace(tmp, path)


print("=" * 112)
print("IHARQ P02 — POST-STAGE18-ACCEPTED FRESH-KERNEL REHYDRATION R1")
print("=" * 112)

# -------------------------------------------------------------------------
# 0. Require verified restore state.
# -------------------------------------------------------------------------
whole_restore = (
    KAGGLE_WORKING / "_IHARQ_WHOLE_WORKING_RESTORE_RECEIPT.json"
)
if not whole_restore.is_file():
    _p17_fail(
        "WHOLE_WORKING_RESTORE_RECEIPT_MISSING",
        whole_restore,
    )

whole = _p17_load(whole_restore)
if whole.get("verification") != "PASS":
    _p17_fail(
        "WHOLE_WORKING_RESTORE_NOT_VERIFIED",
        whole,
    )

restore_marker = (
    KAGGLE_WORKING / "_IHARQ_CONTINUATION_RESTORED.json"
)
if not restore_marker.is_file():
    _p17_fail(
        "CONTINUATION_RESTORE_MARKER_MISSING",
        restore_marker,
    )

restore = _p17_load(restore_marker)
if restore.get("verification") != "PASS":
    _p17_fail(
        "CONTINUATION_RESTORE_MARKER_NOT_VERIFIED",
        restore,
    )

PACKAGE_ROOT = Path(restore["package_root"]).resolve()
RUN_ROOT = Path(restore["run_root"]).resolve()

if not PACKAGE_ROOT.is_dir() or not RUN_ROOT.is_dir():
    _p17_fail(
        "RESTORED_PACKAGE_OR_RUN_ROOT_MISSING",
        {
            "package_root": str(PACKAGE_ROOT),
            "run_root": str(RUN_ROOT),
        },
    )

# -------------------------------------------------------------------------
# 1. Prove the disk boundary is exactly POST-STAGE18 / PRE-STAGE19.
# -------------------------------------------------------------------------
ledger_dir = RUN_ROOT / "stage_ledger"
gate_dir = RUN_ROOT / "gate_results"

for sid in ("11", "12"):
    lp = ledger_dir / f"stage_{sid}.json"
    gp = gate_dir / f"G{sid}.json"

    if not lp.is_file() or not gp.is_file():
        _p17_fail(
            "CANONICAL_PREDECESSOR_LEDGER_OR_GATE_MISSING",
            sid,
        )

    ld = _p17_load(lp)
    gd = _p17_load(gp)

    if (
        ld.get("status") != "SUCCESS"
        or gd.get("status") != "PASS"
    ):
        _p17_fail(
            "CANONICAL_PREDECESSOR_NOT_CLOSED",
            {
                "stage": sid,
                "ledger_status": ld.get("status"),
                "gate_status": gd.get("status"),
            },
        )

protected_paths = {}
protected_summary = {}

for sid in EXPECTED_ACCEPTED_DOWNSTREAM:
    lp = ledger_dir / f"stage_{sid}.json"
    gp = gate_dir / f"G{sid}.json"

    if not lp.is_file() or not gp.is_file():
        _p17_fail(
            "EXPECTED_ACCEPTED_DOWNSTREAM_EVIDENCE_MISSING",
            sid,
        )

    ld = _p17_load(lp)
    gd = _p17_load(gp)

    if (
        ld.get("status") != "SUCCESS"
        or gd.get("status") != "PASS"
    ):
        _p17_fail(
            "EXPECTED_ACCEPTED_DOWNSTREAM_NOT_CLOSED",
            {
                "stage": sid,
                "ledger_status": ld.get("status"),
                "gate_status": gd.get("status"),
            },
        )

    protected_paths[f"stage_{sid}_ledger"] = lp
    protected_paths[f"G{sid}"] = gp
    protected_summary[sid] = {
        "ledger_status": ld.get("status"),
        "gate_status": gd.get("status"),
    }

# Stage16's repaired semantic closure must still be intact.
g16 = _p17_load(gate_dir / "G16.json")

if g16.get("semantic_validation") != "PASS":
    _p17_fail(
        "G16_SEMANTIC_CLOSURE_NOT_PRESERVED",
        g16,
    )

if (
    (g16.get("stage16_runtime_repair") or {}).get("status")
    != "PASS"
):
    _p17_fail(
        "G16_RUNTIME_REPAIR_CLOSURE_NOT_PRESERVED",
        g16.get("stage16_runtime_repair"),
    )

# Stages19–24 must NOT yet be accepted.
for sid in [str(i) for i in range(19, 25)]:
    lp = ledger_dir / f"stage_{sid}.json"
    gp = gate_dir / f"G{sid}.json"

    ld = _p17_load(lp) if lp.is_file() else {}
    gd = _p17_load(gp) if gp.is_file() else {}

    if (
        ld.get("status") == "SUCCESS"
        or gd.get("status") == "PASS"
    ):
        _p17_fail(
            "STAGE19_PLUS_ALREADY_ACCEPTED_REFUSE_REHYDRATION",
            {
                "stage": sid,
                "ledger_status": ld.get("status"),
                "gate_status": gd.get("status"),
            },
        )

# Protect every canonical Stage18 A4 terminal and closure artifact as well.
_stage18_run_cells = sorted(
    (RUN_ROOT / "runtime" / "run_cells").glob("P02-A4-*.json")
)
if not _stage18_run_cells:
    # STORE root is normally RUN_ROOT/runtime; tolerate exact restored layout.
    _stage18_run_cells = sorted(
        (KAGGLE_WORKING / "iharq_p02_run" / "runtime" / "run_cells").glob("P02-A4-*.json")
    )

if len(_stage18_run_cells) != 1218:
    _p17_fail(
        "CANONICAL_STAGE18_TERMINAL_FILE_COUNT_NOT_1218",
        len(_stage18_run_cells),
    )

for _i, _p in enumerate(_stage18_run_cells):
    protected_paths[f"stage18_terminal_{_i:04d}"] = _p

_stage18_analysis_dir = _stage18_run_cells[0].parent.parent / "analysis_inputs"
for _name in (
    "a4_completion.json",
    "a4_participant_metrics.csv",
    "a4_burden_source.csv",
    "a4_role_control_participant_comparisons.csv",
    "a4_role_control_common_support.jsonl",
    "a4_role_control_statistics.json",
    "a4_c4_c5_participant_comparisons.csv",
    "a4_c4_c5_common_support.jsonl",
    "a4_c4_c5_statistics.json",
    "a4_c4_c5_comparison_artifacts.jsonl",
    "stage18_resource_constrained_anchor_budget_deviation.json",
):
    _p = _stage18_analysis_dir / _name
    if not _p.is_file():
        _p17_fail("CANONICAL_STAGE18_CLOSURE_FILE_MISSING", _p)
    protected_paths[f"stage18_closure_{_name}"] = _p

protected_hash_before = {
    k: _p17_sha(p)
    for k, p in protected_paths.items()
}

print(
    json.dumps(
        {
            "disk_boundary": "POST_STAGE18_PRE_STAGE19",
            "accepted_stages_13_18": protected_summary,
            "stage18_accepted": True,
            "scientific_stage_rerun": False,
        },
        indent=2,
    )
)

# -------------------------------------------------------------------------
# 2. Resolve exact SHA-pinned historical R6R3 rehydrator.
# -------------------------------------------------------------------------
meta_candidates = sorted(
    KAGGLE_WORKING.glob(
        "_IHARQ_CONTINUATION_META_*/CHECKPOINT_METADATA.json"
    ),
    key=lambda p: p.parent.name,
    reverse=True,
)

chosen = None

for meta_path in meta_candidates:
    try:
        meta = _p17_load(meta_path)
    except Exception:
        continue

    rel = meta.get("rehydrate_script_relpath")
    expected = str(
        meta.get("rehydrate_script_sha256") or ""
    ).lower()
    stage12_expected = str(
        meta.get("stage12_resume_script_sha256") or ""
    ).lower()

    if not rel or len(expected) != 64:
        continue

    if (
        stage12_expected
        != EXPECTED_CANONICAL_STAGE12_SOURCE_SHA256
    ):
        continue

    script = (KAGGLE_WORKING / rel).resolve()

    try:
        script.relative_to(KAGGLE_WORKING)
    except Exception:
        continue

    if (
        script.is_file()
        and _p17_sha(script) == expected
    ):
        chosen = (
            meta_path,
            meta,
            script,
            expected,
        )
        break

if chosen is None:
    _p17_fail(
        "SHA_PINNED_R6R3_REHYDRATOR_NOT_FOUND"
    )

(
    meta_path,
    continuation_meta,
    rehydrate_script,
    expected_rehydrate_sha,
) = chosen

observed_base_sha = _p17_sha(rehydrate_script)

if (
    observed_base_sha
    != EXPECTED_BASE_REHYDRATOR_SHA256
):
    _p17_fail(
        "UNEXPECTED_R6R3_REHYDRATOR_REVISION",
        {
            "expected":
                EXPECTED_BASE_REHYDRATOR_SHA256,
            "observed":
                observed_base_sha,
            "path":
                str(rehydrate_script),
        },
    )

# -------------------------------------------------------------------------
# 3. Amend ONLY the obsolete downstream guard IN MEMORY.
# -------------------------------------------------------------------------
base_source = rehydrate_script.read_text(
    encoding="utf-8"
)

old_guard = (
    'if downstream:\n'
    '    _fail("DOWNSTREAM_STAGE_ALREADY_ACCEPTED", downstream)'
)

new_guard = (
    '_POST17_ALLOWED_DOWNSTREAM = ["13", "14", "15", "16", "17", "18"]\n'
    'if downstream != _POST17_ALLOWED_DOWNSTREAM:\n'
    '    _fail("POST18_REHYDRATION_DOWNSTREAM_BOUNDARY_MISMATCH", {\n'
    '        "expected": _POST17_ALLOWED_DOWNSTREAM,\n'
    '        "observed": downstream,\n'
    '    })\n'
    'print("[POST-STAGE17 REHYDRATION] Accepted downstream boundary verified:", downstream)'
)

if base_source.count(old_guard) != 1:
    _p17_fail(
        "R6R3_DOWNSTREAM_GUARD_PATTERN_CHANGED",
        base_source.count(old_guard),
    )

patched_source = base_source.replace(
    old_guard,
    new_guard,
    1,
)

print(
    "Executing verified R6R3 rehydrator "
    "with one in-memory boundary amendment:"
)
print("  source =", rehydrate_script)
print("  sha256 =", observed_base_sha)
print(
    "  allowed accepted downstream =",
    EXPECTED_ACCEPTED_DOWNSTREAM,
)

# This reconstructs runtime objects; it DOES NOT run a stage.
exec(
    compile(
        patched_source,
        str(rehydrate_script) + "::POST17_R2",
        "exec",
    ),
    globals(),
    globals(),
)

# -------------------------------------------------------------------------
# 4. Promote live provenance identity back to canonical Stage11 R7H7.
# -------------------------------------------------------------------------
for name in (
    "SESSION",
    "RUN_STAGE",
    "STATE",
    "STORE",
    "RUNTIME_ROOT",
    "RUN_ROOT",
):
    if name not in globals():
        _p17_fail(
            "REHYDRATION_OBJECT_MISSING",
            name,
        )

r7_receipt = (
    Path(RUNTIME_ROOT)
    / "diagnostics"
    / "runtime_successor"
    / "stage11_eegnet_author_centered_R7H_10000E_P500_R1"
    / "P02_STAGE11_R7H7_TRUE_P120_PROMOTION_RECEIPT_R1.json"
)

if not r7_receipt.is_file():
    _p17_fail(
        "CANONICAL_STAGE11_R7_PROMOTION_RECEIPT_MISSING",
        r7_receipt,
    )

r7 = _p17_load(r7_receipt)

required_r7 = {
    "status": "PASS",
    "marker": "IHARQ_P02_STAGE11_R7_PROMOTED",
    "total_stage11_terminals": 120,
    "sr_probability_calibration_success": 45,
    "test_signal_selection_leakage": False,
}

for k, v in required_r7.items():
    if r7.get(k) != v:
        _p17_fail(
            "CANONICAL_STAGE11_R7_RECEIPT_MISMATCH",
            {
                "field": k,
                "expected": v,
                "observed": r7.get(k),
            },
        )

if (
    dict(r7.get("primary_terminal_counts") or {})
    != {"SUCCESS": 105}
):
    _p17_fail(
        "CANONICAL_STAGE11_PRIMARY_TERMINALS_MISMATCH"
    )

if (
    dict(r7.get("challenger_terminal_counts") or {})
    != {"SUCCESS": 15}
):
    _p17_fail(
        "CANONICAL_STAGE11_CHALLENGER_TERMINALS_MISMATCH"
    )

P02_RUNTIME_SUCCESSOR_ID = str(
    r7["runtime_successor_id"]
)
STATE["stage11_r7_successor"] = dict(r7)
STATE["runtime_successor_id"] = (
    P02_RUNTIME_SUCCESSOR_ID
)

# -------------------------------------------------------------------------
# 5. Post-rehydration invariants.
# -------------------------------------------------------------------------
if (
    not SESSION.runner.accepted("11")
    or not SESSION.runner.accepted("12")
):
    _p17_fail(
        "CANONICAL_STAGE11_OR_12_NOT_ACCEPTED_AFTER_REHYDRATION"
    )

post_acceptance = {}

for sid in EXPECTED_ACCEPTED_DOWNSTREAM:
    post_acceptance[sid] = bool(
        SESSION.runner.accepted(sid)
    )

    if not post_acceptance[sid]:
        _p17_fail(
            "DOWNSTREAM_ACCEPTANCE_LOST_AFTER_REHYDRATION",
            sid,
        )

# R10.3 registers its blocking G18 semantic validator here.
if not isinstance(
    globals().get("_SEMANTIC_VALIDATORS"),
    dict,
):
    _p17_fail(
        "SEMANTIC_VALIDATOR_REGISTRY_MISSING"
    )

run_names = set(
    getattr(RUN_STAGE, "__code__").co_names
)

if "_SEMANTIC_VALIDATORS" not in run_names:
    _p17_fail(
        "RUN_STAGE_SEMANTIC_VALIDATOR_SUPPORT_MISSING",
        sorted(run_names),
    )

# Confirm package importability now.
import iharq
import iharq.layer2_decoders.a4
import iharq.layer2_decoders.models
import iharq.layer2_decoders.checkpoints

# Canonical A4 denominator.
a4cells = list(
    STATE.get("a4cells") or []
)

if len(a4cells) != 1218:
    _p17_fail(
        "A4_PLAN_COUNT_MISMATCH_AFTER_REHYDRATION",
        len(a4cells),
    )

partial18 = [
    c["planned_run_cell_id"]
    for c in a4cells
    if STORE.terminal(c) is not None
]

if len(partial18) != 1218:
    _p17_fail(
        "ACCEPTED_STAGE18_TERMINAL_DENOMINATOR_NOT_1218",
        len(partial18),
    )

# Accepted Stage13–17 evidence must be byte-identical.
protected_hash_after = {
    k: _p17_sha(p)
    for k, p in protected_paths.items()
}

changed_protected = sorted(
    k
    for k in protected_hash_before
    if (
        protected_hash_before[k]
        != protected_hash_after.get(k)
    )
)

if changed_protected:
    _p17_fail(
        "POST18_REHYDRATION_MUTATED_ACCEPTED_DOWNSTREAM_EVIDENCE",
        changed_protected,
    )

g16_after = _p17_load(
    gate_dir / "G16.json"
)

if (
    g16_after.get("semantic_validation") != "PASS"
    or (
        g16_after.get("stage16_runtime_repair")
        or {}
    ).get("status") != "PASS"
):
    _p17_fail(
        "G16_SEMANTIC_CLOSURE_LOST_AFTER_REHYDRATION"
    )

# -------------------------------------------------------------------------
# 6. Durable non-scientific receipt.
# -------------------------------------------------------------------------
receipt = {
    "artifact_id":
        "P02-POST-STAGE18-ACCEPTED-FRESH-KERNEL-REHYDRATION-R1",
    "created_at_utc":
        datetime.now(timezone.utc).isoformat(),
    "status":
        "PASS",
    "marker":
        "IHARQ_P02_POST_STAGE18_ACCEPTED_REHYDRATED_R1",
    "boundary":
        "POST_STAGE18_PRE_STAGE19",
    "base_rehydrator_path":
        str(rehydrate_script),
    "base_rehydrator_sha256":
        observed_base_sha,
    "boundary_amendment":
        "ALLOW_EXACT_ACCEPTED_STAGES_13_18_ONLY",
    "scientific_stage_rerun":
        False,
    "scientific_configuration_changed":
        False,
    "model_training_performed":
        False,
    "prediction_regenerated":
        False,
    "accepted_stage13_18":
        post_acceptance,
    "accepted_stage18":
        True,
    "protected_stage13_18_plus_stage18_closure_byte_identical":
        True,
    "runtime_successor_id":
        str(P02_RUNTIME_SUCCESSOR_ID),
    "a4_planned_cells":
        len(a4cells),
    "existing_stage18_terminals":
        len(partial18),
    "semantic_dispatcher_available":
        True,
    "iharq_import_path":
        str(Path(iharq.__file__).resolve()),
    "next_action":
        "INSTALL_POST_STAGE18_SCIENCE_RUNTIME_THEN_STAGE18S_R1_2",
}

receipt_path = (
    Path(RUNTIME_ROOT)
    / "diagnostics"
    / "runtime_successor"
    / "post_stage18_accepted_fresh_kernel_rehydration_R1"
    / "P02_POST_STAGE18_ACCEPTED_REHYDRATION_R1.json"
)

_p17_atomic_json(
    receipt_path,
    receipt,
)

print("\n" + "=" * 112)
print("POST-STAGE18-ACCEPTED REHYDRATION R1 — PASS")
print("=" * 112)
print(
    json.dumps(
        receipt,
        indent=2,
        default=str,
    )
)

print("\nTRUSTED SUCCESS MARKER:")
print("IHARQ_P02_POST_STAGE18_ACCEPTED_REHYDRATED_R1")



# =============================================================================
# POST-STAGE18 ACCEPTED — SCIENCE RUNTIME REPLAY FOR STAGE18S ONLY
# =============================================================================
import re
import threading
import random
import copy
import gc
import time
import numpy as np
import torch

if not SESSION.runner.accepted("18"):
    raise RuntimeError("POST_STAGE18_RUNTIME_REPLAY_REQUIRES_ACCEPTED_STAGE18")

_R10_5_PLAN = list(STATE.get("a4cells") or [])
if len(_R10_5_PLAN) != 1218:
    raise RuntimeError(
        "POST_STAGE18_RUNTIME_REPLAY_A4_PLAN_COUNT_MISMATCH:"
        + str(len(_R10_5_PLAN))
    )

if not (torch.cuda.is_available() and torch.cuda.device_count() >= 2):
    raise RuntimeError("POST_STAGE18_RUNTIME_REPLAY_REQUIRES_TWO_CUDA_GPUS")

_R10_5_GPU_NAMES = [
    str(torch.cuda.get_device_name(i))
    for i in range(torch.cuda.device_count())
]

# Snapshot all canonical Stage18 protected files and all currently existing
# Stage18S supplement files before installing runtime definitions.
_POST18_RUNTIME_PROTECTED = {}
for _k, _p in protected_paths.items():
    _POST18_RUNTIME_PROTECTED[str(_p)] = _p17_sha(_p)

_POST18_S18S_ROOT = (
    Path(STORE.root)
    / "supplements"
    / "stage18S_balanced_sensitivity_R1"
)
_POST18_S18S_BEFORE = {}
if _POST18_S18S_ROOT.is_dir():
    for _p in sorted(_POST18_S18S_ROOT.rglob("*")):
        if _p.is_file():
            _POST18_S18S_BEFORE[str(_p)] = _p17_sha(_p)


# ---- Exact author-aligned Stage18 science definitions (no Stage18 execution) ----
# =============================================================================
# IHARQ P02/L2 — STAGE18 R10.3 FULLY-CORRECTED A4 REFIT SUCCESSOR
# Scientific design remains the R10 anchor-budget / single-repeat deviation.
# R10.3 adds: resumable compatible-partial execution, corrected external checkpoint routing,
# and transport-only dual-GPU scheduling of DIFFERENT independent A4 run cells.
# Completed Stage00-17 evidence is immutable; compatible Stage18 partial evidence is preserved.
# =============================================================================
from __future__ import annotations
from pathlib import Path
import copy, gc, hashlib, importlib, importlib.util, inspect, io, json, math, os, random, shutil, sys, time, threading, traceback
from collections import Counter
import numpy as np

import iharq.layer2_decoders.a4 as _r8_a4
import iharq.layer2_decoders.models as _r8_models
import iharq.layer2_decoders.metrics as _r8_metrics
import iharq.layer2_decoders.scientific as _r8_sci
import iharq.layer2_decoders.checkpoints as _r8_checkpoints
import iharq.layer2_decoders.orchestration as _r10_orch
from iharq.layer2_decoders.writers import atomic_json as _r8_pkg_atomic_json, atomic_jsonl as _r8_pkg_atomic_jsonl

R10_STAGE18_SUCCESSOR_ID = "P02-RUNTIME-SUCCESSOR-R10-STAGE18-A4-ANCHOR-BUDGET-SINGLE-REPEAT-R1"
R10_STAGE18_ARTIFACT_ID = "P02-STAGE18-R10-A4-ANCHOR-BUDGET-SINGLE-REPEAT-R1"
# Compatibility aliases retained because the author/source-aligned fitter below was developed as R8.
R8_STAGE18_SUCCESSOR_ID = R10_STAGE18_SUCCESSOR_ID
R8_STAGE18_ARTIFACT_ID = R10_STAGE18_ARTIFACT_ID
R8_ACCEPTED_STAGE11_SOURCE_SHA256 = "65a506adb6fa6ce20c2ecd11f9f72cf91341c17565d446b4258344e0701f30c2"
R8_ACCEPTED_STAGE12_SOURCE_SHA256 = "12c8196a581fee0571508003a90805a936664dc35418055dacb6426f51e75d75"
R8_EXPECTED_EXTERNAL_ADAPTER_SHA256 = "add69c0a0f8843366d6d9ed1fb16746f61d43a6d588b04f07193484e0555386a"
R8_EXPECTED_CBRAMOD_PATCH_SHA256 = "2591d7166463efe086ed77c2710768b9fa5c5df025949f5a42be1a92322f6b88"
R8_EXPECTED_CBRAMOD_CHECKPOINT_SHA256 = "a939ace9aa1e229f08391ad8bb2d197b507ae2c519a50addf087f0151b2df5c3"

R8_EEGNET_RECIPE = {
    "recipe_id": "R7H7-EEGNET-LAWHERN-SMR-SOURCE-CENTERED-P01-160HZ-10000E-TRUE-P120-R1",
    "architecture": {
        "F1": 8, "D": 2, "F2": 16, "drop_prob": 0.5,
        "kernel_length": 40, "depthwise_kernel_length": 20,
        "pool1_kernel_size": 5, "pool2_kernel_size": 10,
        "pool_mode": "mean", "conv_spatial_max_norm": 1.0,
        "final_layer_with_constraint": True, "norm_rate": 0.25, "sfreq": 160.0,
    },
    "input": {"unit_multiplier": 1.0e6, "epsilon_uv": 1.0e-6, "normalization": "FIT_ROLE_CHANNELWISE_MEAN_STD"},
    "optimizer": "Adam", "lr": 1.0e-3, "weight_decay": 0.0,
    "max_epochs": 10000, "patience": 120, "min_epochs_before_early_stop": 1,
    "effective_batch_target": 64, "batch_ladder": [64, 32, 16],
    "selection": "VALIDATION_BACC_THEN_MACRO_F1_THEN_EARLIER_EPOCH",
    "restore_best": True, "primary_augmentation": "NONE",
}
R8_ACCEPTED_STAGE11_EEGNET_RECIPE_IDS = {
    "R7H-EEGNET-LAWHERN-SMR-SOURCE-CENTERED-P01-160HZ-10000E-P500-R1",
    "R7H6-EEGNET-LAWHERN-SMR-SOURCE-CENTERED-P01-160HZ-10000E-P120-PROSPECTIVE-R1",
    "R7H7-EEGNET-LAWHERN-SMR-SOURCE-CENTERED-P01-160HZ-10000E-TRUE-P120-R1",
}
R8_EXTERNAL_RECIPE_IDS = {
    "DNN-FBCNET": "R6-FBCNET-AUTHOR-CENTERED-R1",
    "DNN-SEQ": "R6-DBCONFORMER-AUTHOR-CENTERED-R1",
    "SSL-CBRAMOD": "R6-CBRAMOD-SOURCE-CENTERED-R1",
}

R10_STAGE18_REDUCTION = {
    "protocol_deviation_id": "P02-STAGE18-RESOURCE-CONSTRAINED-ANCHOR-BUDGET-SINGLE-REPEAT-R1",
    "protocol_deviation_from_buildbook": True,
    "reason": "KAGGLE_WALL_CLOCK_CONSTRAINT_OWNER_DECISION",
    "claim_scope": "RESOURCE_CONSTRAINED_ANCHOR_BUDGET_SINGLE_REPEAT_A4_ABLATION",
    "full_buildbook_replication_equivalence": False,
    "ablation_conditions_removed": False,

    # C1/C2/C3 retain one refit repeat for every role.
    "affected_conditions": [
        "A4-C1-LONG-3P5S",
        "A4-C2-MULTI-HARD-VOTE",
        "A4-C3-MULTI-PROB-AVG",
    ],
    "repeat_reduction_scope": "ALL_REFIT_ROLES_PRESENT_IN_C1_C2_C3",
    "original_repeat_indices": [0, 1, 2, 3, 4],
    "executed_repeat_indices": [0],
    "declared_skipped_repeat_indices": [1, 2, 3, 4],

    # Expensive deep/SSL refits retain a low / middle / full-data anchor.
    "deep_roles": ["NEURAL", "SSL"],
    "deep_anchor_budget_ids": [
        "P01-L1-LOW-CAL-OFFICIAL-R2:1_PER_CLASS",
        "P01-L1-LOW-CAL-OFFICIAL-R2:8_PER_CLASS",
        "FULL_TRAIN",
    ],
    "deep_anchor_semantics": {
        "P01-L1-LOW-CAL-OFFICIAL-R2:1_PER_CLASS": "LOW_RESOURCE_EXTREME",
        "P01-L1-LOW-CAL-OFFICIAL-R2:8_PER_CLASS": "INTERMEDIATE_RESOURCE_ANCHOR",
        "FULL_TRAIN": "FULL_DATA_UPPER_BOUND",
    },

    "skip_terminal_status": "CONDITIONAL_SKIP",
    "repeat_skip_reason": "STAGE18_RESOURCE_CONSTRAINED_SINGLE_REPEAT_ALL_REFIT_ROLES",
    "deep_budget_skip_reason": "STAGE18_RESOURCE_CONSTRAINED_DEEP_ANCHOR_BUDGET_ONLY",

    "latency_measurement_repeats_original": 5,
    "latency_measurement_repeats": 1,

    # Remaining fits retain their source/author recipe.
    "per_fit_training_recipe_changed": False,
    "model_family_removed": False,
    "dataset_removed": False,
    "global_budget_identity_removed": False,
    "deep_budget_grid_reduced": True,
    "representation_removed": False,
    "statistical_comparison_definition_changed": False,

    "multi_seed_stability_claim_authorized": False,
    "dense_deep_budget_curve_claim_authorized": False,
    "anchor_budget_deep_claim_authorized": True,
}

R8_STAGE18_POLICY = {
    "artifact_id": R8_STAGE18_ARTIFACT_ID,
    "runtime_successor_id": R8_STAGE18_SUCCESSOR_ID,
    "accepted_stage11_source_sha256": R8_ACCEPTED_STAGE11_SOURCE_SHA256,
    "accepted_stage12_source_sha256": R8_ACCEPTED_STAGE12_SOURCE_SHA256,
    "scope": "STAGE18_A4_AUTHOR_SOURCE_REFIT_WITH_ANCHOR_BUDGET_SINGLE_REPEAT_REDUCTION",
    "protocol_deviation": R10_STAGE18_REDUCTION,
    "deep_budget_grid_reduced": True,
    "dense_deep_budget_curve_claim_authorized": False,
    "anchor_budget_deep_claim_authorized": True,
    "protocol_deviation_from_buildbook": True,
    "full_buildbook_replication_equivalence": False,
    "ablation_conditions_removed": False,
    "completed_A0_artifacts_mutated": False,
    "new_hyperparameter_grid": False,
    "test_outcome_influence": "PROHIBITED",
    "representative_selection": "UNCHANGED_VALIDATION_ONLY",
    "class_weight_policy": "FREEZE_MATCHING_ACCEPTED_A0_SELECTION",
    "test_signal_load": "AFTER_VALIDATION_SELECTED_CHECKPOINT_SEALED_AND_ROUNDTRIP_PASS",
    "a4_input_lengths": {"multi_view_raw_samples_160Hz": 320, "long_raw_samples_160Hz": 560},
    "cbramod_a4_long": "INPUT_INCOMPATIBLE_FAIL_CLOSED_NO_PAD_NO_CROP",
    "eegnet_recipe": R8_EEGNET_RECIPE,
    "external_recipe_ids": R8_EXTERNAL_RECIPE_IDS,
}
R8_STAGE18_POLICY_SHA256 = hashlib.sha256(json.dumps(R8_STAGE18_POLICY, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def _r8_fail(code, detail=None):
    raise RuntimeError(str(code) + (" : " + str(detail) if detail is not None else ""))


# R10.3 dual-GPU transport primitives.
# Scientific jobs remain single-GPU.  Each worker thread owns one CUDA device.
_R10_3_GPU_TLS = threading.local()
_R10_3_MODEL_INIT_LOCK = threading.RLock()
_R10_3_DATA_IO_LOCK = threading.RLock()

def _r10_3_gpu_index():
    return int(getattr(_R10_3_GPU_TLS, "gpu_id", 0))

def _r10_3_device_string():
    import torch
    if not torch.cuda.is_available():
        return "cpu"
    idx = _r10_3_gpu_index()
    if idx < 0 or idx >= torch.cuda.device_count():
        idx = 0
    return f"cuda:{idx}"

def _r8_seed(seed):
    """
    Deterministic, device-local seed setter for R10.3.

    Unlike torch.manual_seed()/cuda.manual_seed_all(), this does not reset the
    RNG stream of the *other* GPU while an independent ablation fit is running.
    Calls that can affect CPU-side model construction are additionally enclosed
    by _R10_3_MODEL_INIT_LOCK in the fitters below.
    """
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed % (2**32 - 1))
    import torch
    torch.random.default_generator.manual_seed(seed)
    if torch.cuda.is_available():
        idx = _r10_3_gpu_index()
        idx = min(max(0, idx), torch.cuda.device_count() - 1)
        with torch.cuda.device(idx):
            torch.cuda.manual_seed(seed)
    try:
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    except Exception:
        pass


def _r8_load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def _r8_sha(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda:f.read(8*1024*1024), b""): h.update(b)
    return h.hexdigest()

# -----------------------------------------------------------------------------
# 1. Prove the corrected predecessor science exists before installing R10.
# -----------------------------------------------------------------------------
_r7_receipt = Path(RUNTIME_ROOT) / "diagnostics" / "runtime_successor" / "stage11_eegnet_author_centered_R7H_10000E_P500_R1" / "P02_STAGE11_R7H7_TRUE_P120_PROMOTION_RECEIPT_R1.json"
if not _r7_receipt.is_file(): _r8_fail("R10_STAGE11_R7_RECEIPT_MISSING", _r7_receipt)
_r7 = _r8_load_json(_r7_receipt)
if _r7.get("status") != "PASS" or _r7.get("marker") != "IHARQ_P02_STAGE11_R7_PROMOTED" or _r7.get("test_signal_selection_leakage") is not False:
    _r8_fail("R10_STAGE11_R7_RECEIPT_INVALID", _r7)

_r6_candidates = sorted((Path(RUNTIME_ROOT) / "diagnostics" / "runtime_successor").rglob("P02_STAGE12_R6R5_FINAL_RECEIPT_R1.json"))
if not _r6_candidates: _r8_fail("R10_STAGE12_R6_FINAL_RECEIPT_MISSING")
_r6 = _r8_load_json(_r6_candidates[-1])
if _r6.get("status") != "PASS" or int(_r6.get("successful_models", -1)) != 135 or _r6.get("test_set_used_for_hyperparameter_selection") is not False:
    _r8_fail("R10_STAGE12_R6_FINAL_RECEIPT_INVALID", _r6)
if not SESSION.runner.accepted("11") or not SESSION.runner.accepted("12"):
    _r8_fail("R10_REQUIRES_ACCEPTED_STAGE11_AND_STAGE12")

# Stage18 R10.3 is scientifically identical to the already-authorized R10
# anchor-budget/single-repeat policy.  Therefore compatible partial R10 evidence
# is resumable and MUST NOT be discarded merely to install transport fixes.
_existing_a4 = []
for _c in STATE.get("a4cells") or []:
    _t = STORE.terminal(_c)
    if _t is not None:
        _existing_a4.append(_c["planned_run_cell_id"])

# POST-STAGE18 runtime-definition replay: canonical Stage18 is intentionally accepted.
if False:
    pass

_r10_partial_archive_manifest = None
_r10_3_resume_authorized = False
_r10_3_resume_policy_path = (
    Path(RUNTIME_ROOT)
    / "diagnostics" / "runtime_successor"
    / "P02_STAGE18_R10_ANCHOR_BUDGET_POLICY_R1.json"
)

if _existing_a4:
    if not _r10_3_resume_policy_path.is_file():
        _r8_fail(
            "R10_3_PARTIAL_STAGE18_EXISTS_WITHOUT_R10_POLICY_RECEIPT",
            {
                "terminals": len(_existing_a4),
                "expected_policy_receipt": str(_r10_3_resume_policy_path),
            },
        )
    _old_policy = _r8_load_json(_r10_3_resume_policy_path)
    if (
        _old_policy.get("policy_sha256") != R8_STAGE18_POLICY_SHA256
        or _old_policy.get("status") != "INSTALLED_PRE_RESULT"
    ):
        _r8_fail(
            "R10_3_PARTIAL_STAGE18_POLICY_MISMATCH",
            {
                "expected_policy_sha256": R8_STAGE18_POLICY_SHA256,
                "observed": _old_policy.get("policy_sha256"),
                "status": _old_policy.get("status"),
            },
        )

    # Successful author/source deep caches may be reused only under the exact
    # same R10 policy hash.  Known fail-closed CBraMod LONG receipts are also
    # compatible.  Any mixed-success cache from another policy is rejected.
    _mixed_success = []
    for _mp in sorted((Path(STORE.root) / "diagnostics").glob("A4MEM-*.json")):
        try:
            _md = _r8_load_json(_mp)
        except Exception:
            continue
        if _md.get("branch") not in {
            "DNN-EEGNET", "DNN-FBCNET", "DNN-SEQ", "SSL-CBRAMOD"
        }:
            continue
        if _md.get("status") == "SUCCESS" and _md.get("r8_policy_sha256") != R8_STAGE18_POLICY_SHA256:
            _mixed_success.append(str(_mp))
    if _mixed_success:
        _r8_fail(
            "R10_3_MIXED_POLICY_SUCCESS_CACHE_REFUSE_RESUME",
            _mixed_success[:20],
        )

    # Verify every resource-deviation terminal that already exists still states
    # the same owner-authorized R10 deviation.  Ordinary SUCCESS / legitimate
    # INPUT_INCOMPATIBLE terminals need no synthetic rewrite.
    _bad_reduced = []
    for _c in (STATE.get("a4cells") or []):
        _t = STORE.terminal(_c)
        if _t is None:
            continue
        _reason = str(_t.get("reason") or "")
        if _reason in {
            R10_STAGE18_REDUCTION["repeat_skip_reason"],
            R10_STAGE18_REDUCTION["deep_budget_skip_reason"],
        }:
            if (
                _t.get("terminal_status") != R10_STAGE18_REDUCTION["skip_terminal_status"]
                or _t.get("protocol_deviation_id")
                != R10_STAGE18_REDUCTION["protocol_deviation_id"]
            ):
                _bad_reduced.append({
                    "run_cell_id": _c["planned_run_cell_id"],
                    "terminal": _t,
                })
    if _bad_reduced:
        _r8_fail("R10_3_INCOMPATIBLE_RESOURCE_SKIP_TERMINALS", _bad_reduced[:20])

    _r10_3_resume_authorized = True
    print(
        f"[R10.3 RESUME] Preserving {len(_existing_a4)} compatible Stage18 "
        "run-cell terminals under the unchanged R10 scientific policy."
    )

_r8_resume_authorized = _r10_3_resume_authorized

# -----------------------------------------------------------------------------
# 2. Find and verify the persisted R6 CBraMod source-centered amplitude patch.
# -----------------------------------------------------------------------------
_r8_cb_patch = None
_cb_binding = copy.deepcopy((STATE.get("implementation_bindings") or {}).get("SSL-CBRAMOD") or {})
_cb_cfg = copy.deepcopy(_cb_binding.get("plugin_config") or {})
_cands = []
if _cb_cfg.get("r6_adapter_patch_file"):
    _cands.append(Path(_cb_cfg["r6_adapter_patch_file"]))
_cands += list(Path(RUNTIME_ROOT).rglob("iharq_stage12_r6_adapter_patch.py"))
for _p in _cands:
    try:
        if _p.is_file() and _r8_sha(_p) == R8_EXPECTED_CBRAMOD_PATCH_SHA256:
            _r8_cb_patch = _p.resolve(); break
    except Exception:
        pass
if _r8_cb_patch is None:
    _r8_fail("R8_VERIFIED_CBRAMOD_R6_PATCH_NOT_FOUND", R8_EXPECTED_CBRAMOD_PATCH_SHA256)
if str(_r8_cb_patch.parent) not in sys.path: sys.path.insert(0, str(_r8_cb_patch.parent))
importlib.invalidate_caches()
sys.modules.pop("iharq_stage12_r6_adapter_patch", None)
_r8_cb_mod = importlib.import_module("iharq_stage12_r6_adapter_patch")
if _r8_sha(Path(_r8_cb_mod.__file__).resolve()) != R8_EXPECTED_CBRAMOD_PATCH_SHA256:
    _r8_fail("R8_CBRAMOD_PATCH_IMPORT_SHA_MISMATCH")


# -----------------------------------------------------------------------------
# 2B. R10.3 corrected external-first checkpoint route.
# -----------------------------------------------------------------------------
# Promoted FBCNet / DBConformer / CBraMod adapters expose a governed external
# export/reload interface. They must never fall through to the generic torch
# `_build()` state-dict route. This reproduces the already-accepted Stage-12
# external-first persistence principle and updates captured package references.
if not hasattr(_r8_checkpoints, "_external_roundtrip"):
    _r8_fail("R10_3_EXTERNAL_CHECKPOINT_ROUNDTRIP_HELPER_MISSING")

_r10_3_cb_cls = getattr(_r8_cb_mod, "IHARQCBraModR6ScaledAdapter", None)
if _r10_3_cb_cls is None:
    _r8_fail("R10_3_CBRAMOD_R6_SCALED_ADAPTER_CLASS_MISSING")
for _method in (
    "export_iharq_checkpoint_bytes",
    "reload_iharq_checkpoint_bytes",
):
    if not callable(getattr(_r10_3_cb_cls, _method, None)):
        _r8_fail(
            "R10_3_CBRAMOD_GOVERNED_CHECKPOINT_SURFACE_MISSING",
            _method,
        )

# Preserve the package-native fallback exactly once. If an earlier current-
# session R10.2 hotfix exists, recover the pre-hotfix package-native function.
if not hasattr(_r8_checkpoints, "_iharq_r10_3_base_save_roundtrip"):
    _candidate = getattr(
        _r8_checkpoints,
        "_iharq_r10_2_pre_external_first_save_roundtrip",
        None,
    )
    if _candidate is None:
        _candidate = _r8_checkpoints.save_roundtrip
    _r8_checkpoints._iharq_r10_3_base_save_roundtrip = _candidate

_R10_3_BASE_SAVE_ROUNDTRIP = (
    _r8_checkpoints._iharq_r10_3_base_save_roundtrip
)
_r10_3_old_save_roundtrip = _r8_checkpoints.save_roundtrip

def R10_3_external_first_save_roundtrip(model, X, path):
    # Exact corrected Stage-12 routing principle: any qualified outer adapter
    # carrying `.plugin` MUST use the governed external checkpoint interface.
    # `_external_roundtrip` fail-closes if export/reload is unavailable.
    if getattr(model, "plugin", None) is not None:
        return _r8_checkpoints._external_roundtrip(
            model, X, Path(path)
        )
    return _R10_3_BASE_SAVE_ROUNDTRIP(model, X, path)

R10_3_external_first_save_roundtrip.__iharq_external_first__ = True
_r8_checkpoints.save_roundtrip = R10_3_external_first_save_roundtrip

# Update package modules that captured the old function with `from ... import`.
# This makes the correction robust even if a future internal call does not use
# `_r8_checkpoints.save_roundtrip` dynamically.
_r10_3_checkpoint_reference_updates = []
for _mn, _mod in list(sys.modules.items()):
    if not str(_mn).startswith("iharq.layer2_decoders") or _mod is None:
        continue
    try:
        _items = list(vars(_mod).items())
    except Exception:
        continue
    for _attr, _value in _items:
        if (
            _value is _r10_3_old_save_roundtrip
            and not str(_attr).startswith("_iharq_")
        ):
            setattr(
                _mod,
                _attr,
                R10_3_external_first_save_roundtrip,
            )
            _r10_3_checkpoint_reference_updates.append(
                f"{_mn}.{_attr}"
            )

if hasattr(_r8_a4, "save_roundtrip"):
    _r8_a4.save_roundtrip = R10_3_external_first_save_roundtrip
globals()["save_roundtrip"] = R10_3_external_first_save_roundtrip

# No-science route probe: proves that an external wrapper is actually persisted
# and reloaded through EXTERNAL_GOVERNED_INTERFACE before Stage18 resumes.
class _R10_3_DummyPlugin:
    def __init__(self, payload=b"IHARQ-R10.3-EXTERNAL-CHECKPOINT-PROBE"):
        self.payload = bytes(payload)
    def predict(self, X):
        return np.zeros(len(X), dtype=np.int64)
    def export_iharq_checkpoint_bytes(self):
        return self.payload
    def reload_iharq_checkpoint_bytes(self, payload):
        if bytes(payload) != self.payload:
            raise RuntimeError("R10_3_DUMMY_PAYLOAD_MISMATCH")
        return _R10_3_DummyPlugin(payload)

class _R10_3_DummyWrapper:
    def __init__(self):
        self.plugin = _R10_3_DummyPlugin()

_r10_3_checkpoint_probe_dir = (
    Path(RUNTIME_ROOT)
    / "diagnostics" / "runtime_successor"
    / "stage18_r10_3_checkpoint_route_probe"
)
_r10_3_checkpoint_probe_dir.mkdir(parents=True, exist_ok=True)
R10_3_CHECKPOINT_ROUTE_PROBE = R10_3_external_first_save_roundtrip(
    _R10_3_DummyWrapper(),
    np.zeros((2, 1, 4), dtype=np.float32),
    _r10_3_checkpoint_probe_dir / "dummy.external.chk",
)
if (
    R10_3_CHECKPOINT_ROUTE_PROBE.get("status") != "PASS"
    or R10_3_CHECKPOINT_ROUTE_PROBE.get("checkpoint_format")
       != "EXTERNAL_GOVERNED_INTERFACE"
):
    _r8_fail(
        "R10_3_EXTERNAL_CHECKPOINT_ROUTE_PROBE_FAILED",
        R10_3_CHECKPOINT_ROUTE_PROBE,
    )

# POST-STAGE18 definition replay: canonical member evidence is immutable.
_r10_3_archived_failed = []

# Preserve the post-rehydration admitted factory and original A4 member fitter exactly once.
if not hasattr(_r8_a4, "_iharq_r8_parent_make_adapter"):
    _r8_a4._iharq_r8_parent_make_adapter = _r8_a4.make_adapter
if not hasattr(_r8_a4, "_iharq_r8_parent_fit_member"):
    _r8_a4._iharq_r8_parent_fit_member = _r8_a4._fit_member
_R8_PARENT_MAKE_ADAPTER = _r8_a4._iharq_r8_parent_make_adapter
_R8_PARENT_FIT_MEMBER = _r8_a4._iharq_r8_parent_fit_member

# -----------------------------------------------------------------------------
# 3. EEGNet: exact accepted R7H7 recipe, but remove the A0-only 480-sample guard.
# -----------------------------------------------------------------------------
import torch

class _R8EEGInputWrapper(torch.nn.Module):
    def __init__(self, base_model, n_chans, n_times):
        super().__init__()
        self.base_model=base_model; self.n_chans=int(n_chans); self.n_times=int(n_times)
        self.eps_uv=float(R8_EEGNET_RECIPE["input"]["epsilon_uv"])
        self.register_buffer("iharq_unit_multiplier", torch.tensor(float(R8_EEGNET_RECIPE["input"]["unit_multiplier"]), dtype=torch.float32))
        self.register_buffer("iharq_channel_mean_uv", torch.zeros(1,self.n_chans,1,dtype=torch.float32))
        self.register_buffer("iharq_channel_std_uv", torch.ones(1,self.n_chans,1,dtype=torch.float32))
    def fit_input_statistics(self, X):
        x=np.asarray(X,dtype=np.float32)
        if x.ndim!=3 or x.shape[1]!=self.n_chans or x.shape[-1]!=self.n_times:
            _r8_fail("R8_EEGNET_NORMALIZATION_FIT_SHAPE_INVALID", {"expected":[None,self.n_chans,self.n_times],"observed":list(x.shape)})
        uv=x.astype(np.float64,copy=False)*float(self.iharq_unit_multiplier.item())
        mean=uv.mean(axis=(0,2)); std=uv.std(axis=(0,2),ddof=0)
        if not np.isfinite(mean).all() or not np.isfinite(std).all(): _r8_fail("R8_EEGNET_NORMALIZATION_NONFINITE")
        std=np.maximum(std,self.eps_uv)
        with torch.no_grad():
            self.iharq_channel_mean_uv.copy_(torch.from_numpy(mean.astype(np.float32)).view(1,self.n_chans,1))
            self.iharq_channel_std_uv.copy_(torch.from_numpy(std.astype(np.float32)).view(1,self.n_chans,1))
        return {"source_unit":"V","unit_multiplier":float(self.iharq_unit_multiplier.item()),"fit_samples":int(len(x)),"channel_count":self.n_chans,"n_times":self.n_times,"test_statistics_used":False}
    def forward(self,x):
        x=x*self.iharq_unit_multiplier
        x=(x-self.iharq_channel_mean_uv)/self.iharq_channel_std_uv
        return self.base_model(x)

class R8AuthorCenteredEEGNetAdapter(_r8_models.Adapter):
    supports_class_weights=True
    score_type="SOFTMAX_PROBABILITY"
    resolved_variant="EEGNet-R7H7-LAWHERN-SMR-A4-EXACT-LENGTH-R8"
    def __init__(self,seed,input_samples,n_chans):
        self.seed=int(seed); self.input_samples=int(input_samples); self.n_chans=int(n_chans); self.device="cpu"; self.model=None
        self.branch="DNN-EEGNET"; self.variant="EEGNet"; self.actual_batch_size=None; self.gradient_accumulation=None; self.input_statistics=None
    def admission(self):
        try:
            from braindecode import models as bd_models
            cls=getattr(bd_models,"EEGNet"); sig=inspect.signature(cls)
            required={"n_chans","n_outputs","n_times","F1","D","F2","kernel_length","depthwise_kernel_length","pool1_kernel_size","pool2_kernel_size","conv_spatial_max_norm","drop_prob","final_layer_with_constraint","norm_rate","sfreq"}
            missing=sorted(required-set(sig.parameters))
            return {"status":"DEPENDENCY_BLOCKED","reason":"BRAINCDECODE_EEGNET_SIGNATURE_INCOMPATIBLE","missing_parameters":missing} if missing else {"status":"ADMITTED","resolved_variant":self.resolved_variant}
        except Exception as exc:
            return {"status":"DEPENDENCY_BLOCKED","reason":f"{type(exc).__name__}:{str(exc)[:300]}"}
    def _build(self):
        from braindecode import models as bd_models
        _r8_seed(self.seed); a=R8_EEGNET_RECIPE["architecture"]
        base=bd_models.EEGNet(n_chans=self.n_chans,n_outputs=2,n_times=self.input_samples,sfreq=float(a["sfreq"]),F1=int(a["F1"]),D=int(a["D"]),F2=int(a["F2"]),kernel_length=int(a["kernel_length"]),depthwise_kernel_length=int(a["depthwise_kernel_length"]),pool1_kernel_size=int(a["pool1_kernel_size"]),pool2_kernel_size=int(a["pool2_kernel_size"]),pool_mode=str(a["pool_mode"]),conv_spatial_max_norm=float(a["conv_spatial_max_norm"]),drop_prob=float(a["drop_prob"]),final_layer_with_constraint=bool(a["final_layer_with_constraint"]),norm_rate=float(a["norm_rate"]))
        return _R8EEGInputWrapper(base,self.n_chans,self.input_samples)
    def scores(self,X,device=None,batch_size=128):
        d=str(device or self.device); x=np.asarray(X,dtype=np.float32)
        if x.ndim!=3 or x.shape[1]!=self.n_chans or x.shape[-1]!=self.input_samples: _r8_fail("R8_EEGNET_SCORE_INPUT_SHAPE_INVALID",list(x.shape))
        self.model.eval(); out=[]
        with torch.inference_mode():
            for start in range(0,len(x),int(batch_size)):
                z=self.model(torch.from_numpy(x[start:start+int(batch_size)]).to(d))
                if z.ndim!=2 or z.shape[1]!=2 or not torch.isfinite(z).all(): _r8_fail("R8_EEGNET_LOGITS_INVALID")
                out.append(torch.softmax(z,dim=1).detach().cpu().numpy())
        p=np.concatenate(out,axis=0) if out else np.empty((0,2),dtype=np.float32)
        if len(p) and (not np.isfinite(p).all() or np.max(np.abs(p.sum(1)-1.0))>1e-5): _r8_fail("R8_EEGNET_PROBABILITY_INVALID")
        return p
    def predict(self,X,device=None,**kw): return np.argmax(self.scores(X,device=device),axis=1)
    def fit(self,X,y,**kw):
        X=np.asarray(X,dtype=np.float32); y=np.asarray(y,dtype=np.int64); Xv=np.asarray(kw.get("X_val"),dtype=np.float32); yv=np.asarray(kw.get("y_val"),dtype=np.int64)
        if set(np.unique(y).tolist())!={0,1} or set(np.unique(yv).tolist())!={0,1}: _r8_fail("R8_EEGNET_BINARY_ROLE_REQUIRED")
        device=str(kw.get("device") or ("cuda" if torch.cuda.is_available() else "cpu")); requested=int(kw.get("batch_size") or 64)
        ladder=[b for b in R8_EEGNET_RECIPE["batch_ladder"] if b<=requested] or [16]
        class_weights=kw.get("class_weights"); last_oom=None
        for bs in ladder:
            try:
                with _R10_3_MODEL_INIT_LOCK:
                    _r8_seed(self.seed)
                    self.model = self._build()
                    self.input_statistics = self.model.fit_input_statistics(X)
                    self.model = self.model.to(device)
                self.device = device
                cw=None if class_weights is None else torch.tensor(class_weights,dtype=torch.float32,device=device)
                loss_fn=torch.nn.CrossEntropyLoss(weight=cw); opt=torch.optim.Adam(self.model.parameters(),lr=1e-3,weight_decay=0.0); accum=max(1,int(math.ceil(64/int(bs))))
                best_state=None; best_metrics=None; best_epoch=None; bad=0; epochs_completed=0
                ds=torch.utils.data.TensorDataset(torch.from_numpy(X),torch.from_numpy(y))
                for epoch in range(1,int(R8_EEGNET_RECIPE["max_epochs"])+1):
                    gen=torch.Generator(); gen.manual_seed(self.seed+epoch)
                    loader=torch.utils.data.DataLoader(ds,batch_size=int(bs),shuffle=True,generator=gen,num_workers=0,pin_memory=False,drop_last=False)
                    self.model.train(); opt.zero_grad(set_to_none=True); pending=0
                    for xb,yb in loader:
                        xb=xb.to(device); yb=yb.to(device); z=self.model(xb); loss=loss_fn(z,yb)
                        if not torch.isfinite(loss): _r8_fail("R8_EEGNET_NONFINITE_TRAIN_LOSS")
                        (loss/accum).backward(); pending+=1
                        if pending%accum==0: opt.step(); opt.zero_grad(set_to_none=True)
                    if pending%accum: opt.step(); opt.zero_grad(set_to_none=True)
                    scores=self.scores(Xv,device=device,batch_size=128); pred=np.argmax(scores,axis=1); vm=_r8_metrics.evaluate(yv,pred,scores,self.score_type)
                    key=(float(vm["BACC"]),float(vm["F1_MACRO"]),-epoch); old=(-np.inf,-np.inf,-10**9) if best_metrics is None else (float(best_metrics["BACC"]),float(best_metrics["F1_MACRO"]),-int(best_epoch))
                    if key>old:
                        best_metrics=dict(vm); best_epoch=epoch; best_state={k:v.detach().cpu().clone() for k,v in self.model.state_dict().items()}; bad=0
                    else: bad+=1
                    epochs_completed=epoch
                    if bad>=int(R8_EEGNET_RECIPE["patience"]): break
                if best_state is None: _r8_fail("R8_EEGNET_NO_BEST_STATE")
                self.model.load_state_dict(best_state,strict=True); self.actual_batch_size=int(bs); self.gradient_accumulation=int(accum)
                self.r8_training_provenance={"recipe_id":R8_EEGNET_RECIPE["recipe_id"],"policy_sha256":R8_STAGE18_POLICY_SHA256,"best_epoch":int(best_epoch),"epochs_completed":int(epochs_completed),"best_validation":best_metrics,"class_weights":class_weights,"test_set_used_for_selection":False}
                return self
            except RuntimeError as exc:
                if "out of memory" not in str(exc).lower(): raise
                last_oom=f"{type(exc).__name__}:{str(exc)[:300]}"; self.model=None; gc.collect();
                if torch.cuda.is_available(): torch.cuda.empty_cache()
        raise ResourceWarning("R8_EEGNET_RESOURCE_BLOCKED_ALL_AUTHOR_BATCHES:"+str(last_oom))

# -----------------------------------------------------------------------------
# 4. External family construction/training: exact accepted R6 recipes, A4 length.
# -----------------------------------------------------------------------------
def _r8_replace_outer_plugin(outer, plugin):
    object.__setattr__(outer,"plugin",plugin)
    if hasattr(outer,"resolved_variant"): outer.resolved_variant=getattr(plugin,"resolved_variant",outer.resolved_variant)
    return outer


def _r8_make_adapter(branch,seed,input_samples,n_chans,params):
    branch=str(branch)
    if branch=="DNN-EEGNET": return R8AuthorCenteredEEGNetAdapter(seed,input_samples,n_chans)
    outer=_R8_PARENT_MAKE_ADAPTER(branch,seed,input_samples,n_chans,params)
    if branch!="SSL-CBRAMOD": return outer
    old=getattr(outer,"plugin",None)
    if old is None: _r8_fail("R8_CBRAMOD_PROMOTED_PLUGIN_MISSING")
    cfg=copy.deepcopy(getattr(old,"config",{}) or {})
    cfg.update({"iharq_seed":int(seed),"seed":int(seed),"expected_n_chans":int(n_chans),"expected_n_times":int(input_samples),"runtime_successor_id":R8_STAGE18_SUCCESSOR_ID,"input_scale":1.0e4,"preparation_cache_max_gib":0.0})
    p=_r8_cb_mod.build_cbramod_r6(getattr(old,"checkpoint_path",None),cfg); p.seed=int(seed)
    return _r8_replace_outer_plugin(outer,p)


def _r8_head_parameter(name,branch):
    n=str(name).lower(); return any(k in n for k in (("final_layer","classifier","head") if branch=="SSL-CBRAMOD" else ("classifier","head","lastlayer","final_layer","fc")))


def _r8_eval_external(plugin,X,y,device,batch_size=128):
    y=np.asarray(y,dtype=int)
    if set(np.unique(y).tolist())!={0,1}: _r8_fail("R8_EXTERNAL_BINARY_EVAL_ROLE_REQUIRED")
    plugin.model.eval(); probs=[]
    with torch.no_grad():
        for start in range(0,len(X),int(batch_size)):
            xb=torch.as_tensor(X[start:start+int(batch_size)],dtype=torch.float32,device=device); z=plugin._forward_logits(xb)
            if isinstance(z,(tuple,list)): z=z[-1]
            if z.ndim!=2 or z.shape[1]!=2 or not torch.isfinite(z).all(): _r8_fail("R8_EXTERNAL_LOGITS_INVALID")
            probs.append(torch.softmax(z,dim=1).detach().cpu().numpy())
    pr=np.concatenate(probs,axis=0) if probs else np.empty((0,2),dtype=np.float32); yp=np.argmax(pr,axis=1).astype(int) if len(pr) else np.empty((0,),dtype=int)
    return yp,pr,_r8_metrics.evaluate(y,yp,pr,getattr(plugin,"score_type","SOFTMAX_PROBABILITY"))


def _r8_eval_external_adaptive(plugin,X,y,device):
    # Evaluation batch is transport-only; preserve scientific batch semantics while avoiding avoidable OOM.
    last=None
    for bs in (256,128,64,32,16,8):
        try:
            return _r8_eval_external(plugin,X,y,device,batch_size=min(bs,max(1,len(X))))
        except RuntimeError as exc:
            if "out of memory" not in str(exc).lower(): raise
            last=exc
            if torch.cuda.is_available(): torch.cuda.empty_cache()
    raise ResourceWarning("R8_EXTERNAL_VALIDATION_EVAL_RESOURCE_BLOCKED:"+str(last)[:300])


def _r8_train_external(outer,X,y,Xv,yv,recipe,class_weights,seed,device):
    plugin=outer.plugin; raw_n_times=int(X.shape[-1]); n_chans=int(X.shape[1]); branch=str(recipe["branch"])
    # Preserve the original source-aligned ordering exactly: seed -> model-local
    # preparation -> model construction.  Only this short CPU/init segment is
    # serialized; the expensive epoch loop runs concurrently across GPUs.
    with _R10_3_MODEL_INIT_LOCK:
        _r8_seed(seed)
        Xp=np.asarray(plugin._prepare_and_validate(np.asarray(X,dtype=np.float32)),dtype=np.float32)
        Xvp=np.asarray(plugin._prepare_and_validate(np.asarray(Xv,dtype=np.float32)),dtype=np.float32)
        plugin.seed=int(seed)
        plugin.raw_shape=(n_chans,raw_n_times)
        plugin.prepared_shape=tuple(map(int,Xp.shape[1:]))
        plugin.model=plugin._build_model(n_chans,raw_n_times).to(device)
    plugin.device=str(device)
    model=plugin.model; bs=int(recipe["batch_size"]); cw=None if class_weights is None else torch.tensor(class_weights,dtype=torch.float32,device=device)
    loss_fn=torch.nn.CrossEntropyLoss(weight=cw,label_smoothing=float(recipe.get("label_smoothing",0.0)))
    if branch=="SSL-CBRAMOD":
        body=[]; head=[]
        for name,p in model.named_parameters(): (head if _r8_head_parameter(name,branch) else body).append(p)
        if not body or not head: _r8_fail("R8_CBRAMOD_HEAD_BODY_SPLIT_FAILED")
        opt=torch.optim.AdamW([{"params":body,"lr":float(recipe["body_lr"])},{"params":head,"lr":float(recipe["head_lr"])}],lr=float(recipe["body_lr"]),weight_decay=float(recipe.get("weight_decay",0.0)))
    else:
        opt=torch.optim.Adam(model.parameters(),lr=float(recipe["lr"]),weight_decay=float(recipe.get("weight_decay",0.0)))
    ds=torch.utils.data.TensorDataset(torch.from_numpy(Xp),torch.from_numpy(np.asarray(y,dtype=np.int64))); steps=max(1,int(math.ceil(len(ds)/bs)))
    sched=None
    if str(recipe.get("scheduler","constant")).lower()=="cosine": sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=max(1,int(recipe["max_epochs"])*steps),eta_min=float(recipe.get("eta_min",1e-6)))
    best_bacc=-np.inf; best_state=None; best_metrics=None; best_epoch=0; bad=0; started=time.time(); history=[]
    for epoch in range(1,int(recipe["max_epochs"])+1):
        gen=torch.Generator(); gen.manual_seed(int(seed)+epoch); loader=torch.utils.data.DataLoader(ds,batch_size=bs,shuffle=True,generator=gen,num_workers=0,pin_memory=True,drop_last=False)
        model.train(); losses=[]
        for xb,yb in loader:
            xb=xb.to(device,non_blocking=True); yb=yb.to(device,non_blocking=True); opt.zero_grad(set_to_none=True); z=plugin._forward_logits(xb)
            if isinstance(z,(tuple,list)): z=z[-1]
            loss=loss_fn(z,yb)
            if not torch.isfinite(loss): _r8_fail("R8_EXTERNAL_NONFINITE_TRAIN_LOSS",{"branch":branch,"epoch":epoch})
            loss.backward(); gcval=recipe.get("grad_clip")
            if gcval is not None and float(gcval)>0: torch.nn.utils.clip_grad_norm_(model.parameters(),float(gcval))
            opt.step();
            if sched is not None: sched.step()
            losses.append(float(loss.detach().cpu().item()))
        _,_,vm=_r8_eval_external_adaptive(plugin,Xvp,yv,device)
        if float(vm["BACC"])>float(best_bacc):
            best_bacc=float(vm["BACC"]); best_epoch=int(epoch); best_metrics=copy.deepcopy(vm); best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else: bad+=1
        history.append({"epoch":epoch,"validation":vm,"best_epoch":best_epoch,"bad_epochs":bad})
        if bad>=int(recipe["patience"]): break
    if best_state is None: _r8_fail("R8_EXTERNAL_NO_BEST_STATE",branch)
    model.load_state_dict(best_state,strict=True); plugin.model.eval(); plugin.device=str(device); plugin.actual_batch_size=bs; plugin.gradient_accumulation=1
    return {"recipe_id":recipe["recipe_id"],"best_epoch":best_epoch,"epochs_completed":len(history),"best_validation":best_metrics,"elapsed_seconds":float(time.time()-started),"prepared_train_shape":list(Xp.shape),"prepared_validation_shape":list(Xvp.shape),"test_set_used_for_selection":False}

# -----------------------------------------------------------------------------
# 5. R8 A4 deep-member fitter. Classical/Riemannian stay exactly package-native.
# -----------------------------------------------------------------------------
def _r8_release_model(obj):
    try:
        obj.model = None
    except Exception:
        try:
            obj.plugin.model = None
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        idx = min(max(0, _r10_3_gpu_index()), torch.cuda.device_count() - 1)
        with torch.cuda.device(idx):
            torch.cuda.empty_cache()


def _r8_a0_science(c,store,branch):
    t=_r8_a4.a0_term(store,c["dataset_id"],branch,c["budget_id"],c["model_repeat_index"])
    if not t or t.get("terminal_status")!="SUCCESS": _r8_fail("R8_A4_MATCHING_A0_SUCCESS_MISSING",{"branch":branch,"cell":c["planned_run_cell_id"]})
    metric=_r8_load_json(Path(store.root)/t["metric_source"])
    return t,metric


def R8_A4_fit_member(c,store,core,a4,b,member,schema_path,config_sha,freeze,fixture=False,implementation_bindings=None,runtime_overrides=None):
    if fixture or b not in {"DNN-EEGNET","DNN-FBCNET","DNN-SEQ","SSL-CBRAMOD"}:
        return _R8_PARENT_FIT_MEMBER(c,store,core,a4,b,member,schema_path,config_sha,freeze,fixture,implementation_bindings,runtime_overrides)
    ds=c["dataset_id"]; budget=c["budget_id"]; rep=c["model_repeat_index"]; key=f'{ds}__{budget}__{b}__{rep}__{member or "LONG"}'; meta=Path(store.root)/"diagnostics"/f"A4MEM-{key}.json"; rowsf=Path(store.root)/"diagnostics"/f"A4MEM-{key}.jsonl"
    if meta.exists() and rowsf.exists():
        old=_r8_load_json(meta)
        if old.get("config_sha256")==config_sha and old.get("r8_policy_sha256")==R8_STAGE18_POLICY_SHA256 and old.get("status")=="SUCCESS":
            return {**old,"rows":[json.loads(x) for x in rowsf.read_text(encoding="utf-8").splitlines() if x.strip()]}
    # Existing generic caches are deliberately not reused under R8.
    if meta.exists() and _r8_load_json(meta).get("r8_policy_sha256")!=R8_STAGE18_POLICY_SHA256:
        print("[R8] predecessor A4MEM cache ignored and replaced:",meta.name)

    # Existing authority: CBraMod A4_LONG 560@160 -> 700@200 remains fail-closed.
    if b=="SSL-CBRAMOD" and member is None:
        out={"status":"INPUT_INCOMPATIBLE","reason":"CBRAMOD_A4_LONG_INCOMPATIBLE_FAIL_CLOSED_NO_PAD_NO_CROP","config_sha256":config_sha,"branch":b,"member":"LONG","r8_policy_sha256":R8_STAGE18_POLICY_SHA256,"scientific_training_performed":False}
        _r8_pkg_atomic_json(meta,out); return out

    try:
        t,metric=_r8_a0_science(c,store,b)
        class_policy=t.get("class_weight_policy",metric.get("class_weight_policy")); class_weights=t.get("class_weights",metric.get("class_weights"))
        if class_policy is None: _r8_fail("R8_A4_A0_CLASS_WEIGHT_POLICY_MISSING",{"branch":b,"cell":c["planned_run_cell_id"]})
        # Exact A4 fit/validation signals. Test is intentionally NOT loaded yet.
        # Dataset reads/cache mutation are serialized briefly; training remains parallel.
        with _R10_3_DATA_IO_LOCK:
            tr=_r8_a4._train(a4,core,ds,budget,freeze)
            va=a4.rows(dataset_id=ds,role="validation")
            X,y,_=a4.load_rows(tr,member)
            Xv,yv,_=a4.load_rows(va,member)
        X=np.asarray(X,dtype=np.float32); y=np.asarray(y,dtype=np.int64); Xv=np.asarray(Xv,dtype=np.float32); yv=np.asarray(yv,dtype=np.int64)
        p=copy.deepcopy(metric.get("selected_params") or t.get("selected_params") or {})
        pp={**dict((implementation_bindings or {}).get(b,{}) or {}),**p,**dict(runtime_overrides or {})}
        with _R10_3_MODEL_INIT_LOCK:
            m=_r8_make_adapter(b,int(c["seed_id"]),X.shape[-1],X.shape[1],pp)
        if hasattr(m,"admission"):
            adm=m.admission()
            if adm.get("status")!="ADMITTED":
                out={"status":adm.get("status","DEPENDENCY_BLOCKED"),"reason":adm,"config_sha256":config_sha,"branch":b,"member":member or "LONG","r8_policy_sha256":R8_STAGE18_POLICY_SHA256}; _r8_pkg_atomic_json(meta,out); return out
        device=_r10_3_device_string()
        if device=="cpu" and b in {"DNN-FBCNET","DNN-SEQ","SSL-CBRAMOD"}:
            raise ResourceWarning("R8_EXTERNAL_CPU_FULL_REFIT_NOT_PREAUTHORIZED_BY_RESOURCE_POLICY")
        if b=="DNN-EEGNET":
            if p.get("recipe_id") and p.get("recipe_id") not in R8_ACCEPTED_STAGE11_EEGNET_RECIPE_IDS:
                _r8_fail("R8_EEGNET_A0_RECIPE_ID_OUTSIDE_ACCEPTED_STAGE11_LINEAGE",p.get("recipe_id"))
            # Stage18 is a NEW fit, so it uses the active R7H7 TRUE-P120 successor recipe;
            # grandfathered P500/FLOOR500 A0 artifacts remain immutable and valid references.
            m.fit(X,y,batch_size=64,effective_batch_target=64,device=device,X_val=Xv,y_val=yv,class_weights=class_weights,patience=120,restore_best=True)
            train_prov=copy.deepcopy(m.r8_training_provenance); recipe_id=R8_EEGNET_RECIPE["recipe_id"]
        else:
            recipe=copy.deepcopy(p)
            expected=R8_EXTERNAL_RECIPE_IDS[b]
            if recipe.get("recipe_id")!=expected: _r8_fail("R8_EXTERNAL_A0_RECIPE_ID_CHANGED",{"branch":b,"expected":expected,"observed":recipe.get("recipe_id")})
            recipe.update({"branch":b,"dataset_id":ds,"budget_id":str(budget)})
            train_prov=_r8_train_external(m,X,y,Xv,yv,recipe,class_weights,int(c["seed_id"]),device); recipe_id=expected

        # Seal the validation-selected model BEFORE first A4 test-signal load.
        # Round-trip reconstruction may instantiate a fresh model, so keep that
        # CPU-side construction under the same deterministic initialization lock.
        with _R10_3_MODEL_INIT_LOCK:
            chk=_r8_checkpoints.save_roundtrip(
                m,
                Xv[:min(4,len(Xv))],
                Path(store.root)/"checkpoints"/f"A4MEM-R8-{key}.pkl",
            )
        if chk.get("status")!="PASS": _r8_fail("R8_A4_CHECKPOINT_ROUNDTRIP_FAILED",chk)

        # First test-signal load for this member occurs only after checkpoint seal.
        with _R10_3_DATA_IO_LOCK:
            te=a4.rows(dataset_id=ds,role="test")
            Xt,yt,rows=a4.load_rows(te,member)
        Xt=np.asarray(Xt,dtype=np.float32); yt=np.asarray(yt,dtype=np.int64)
        times=[]
        for _ in range(int(R10_STAGE18_REDUCTION["latency_measurement_repeats"])):
            if torch.cuda.is_available(): torch.cuda.synchronize(torch.device(device))
            t0=time.perf_counter(); _=m.predict(Xt[:1]); _=m.scores(Xt[:1]);
            if torch.cuda.is_available(): torch.cuda.synchronize(torch.device(device))
            times.append(time.perf_counter()-t0)
        pred=np.asarray(m.predict(Xt),dtype=int); scores=m.scores(Xt)
        src=[]
        for i,(r,yp) in enumerate(zip(rows,pred)):
            src.append({"dataset_id":ds,"event_id":r["event_id"],"window_id":r["window_id"],"window_record_id":r["window_record_id"],"subject_id":r["subject_id"],"session_id":r["session_id"],"split_record_id":r["split_record_id"],"role":r["role"],"y_true":int(yt[i]),"y_pred":int(yp),"score_vector":None if scores is None else np.asarray(scores[i]).tolist(),"score_type":m.score_type})
        _r8_pkg_atomic_jsonl(rowsf,src)
        out={"status":"SUCCESS","config_sha256":config_sha,"branch":b,"member":member or "LONG","checkpoint_sha256":chk["checkpoint_sha256"],"model_id":f"{b}:A4-R8:{key}","score_type":m.score_type,"checkpoint_format":chk.get("checkpoint_format"),"execution_device":device,"latency_summary":{"batch1_latency_median_s":float(np.median(times)),"batch1_latency_p95_s":float(np.quantile(times,.95)),"repeats":len(times)},"r8_policy_sha256":R8_STAGE18_POLICY_SHA256,"r8_runtime_successor_id":R8_STAGE18_SUCCESSOR_ID,"recipe_id":recipe_id,"class_weight_policy":class_policy,"class_weights":class_weights,"class_weight_frozen_from_a0":True,"matching_a0_checkpoint_sha256":t.get("checkpoint_sha256"),"test_loaded_after_checkpoint_sealed":True,"test_set_used_for_selection":False,"raw_train_shape":list(X.shape),"raw_validation_shape":list(Xv.shape),"raw_test_shape":list(Xt.shape),"training_provenance":train_prov}
        _r8_pkg_atomic_json(meta,out)
        _r8_release_model(m)
        return {**out,"rows":src}
    except (ImportError,ModuleNotFoundError) as exc:
        if "m" in locals(): _r8_release_model(m)
        out={"status":"DEPENDENCY_BLOCKED","reason":f"{type(exc).__name__}:{str(exc)[:300]}","config_sha256":config_sha,"branch":b,"member":member or "LONG","r8_policy_sha256":R8_STAGE18_POLICY_SHA256}; _r8_pkg_atomic_json(meta,out); return out
    except ResourceWarning as exc:
        if "m" in locals(): _r8_release_model(m)
        out={"status":"RESOURCE_BLOCKED","reason":str(exc)[:500],"config_sha256":config_sha,"branch":b,"member":member or "LONG","r8_policy_sha256":R8_STAGE18_POLICY_SHA256}; _r8_pkg_atomic_json(meta,out); return out
    except Exception as exc:
        if "m" in locals(): _r8_release_model(m)
        status=_r8_sci._classify_exception(exc)
        out={"status":status,"reason":f"{type(exc).__name__}:{str(exc)[:500]}","config_sha256":config_sha,"branch":b,"member":member or "LONG","r8_policy_sha256":R8_STAGE18_POLICY_SHA256}; _r8_pkg_atomic_json(meta,out); return out

_r8_a4._fit_member = R8_A4_fit_member



# -----------------------------------------------------------------------------
# 6. Owner-authorized R10 refit-replication and deep-budget-density reduction.
# -----------------------------------------------------------------------------
# The frozen A4 run-cell plan is NOT rewritten. Every planned cell still obtains
# a terminal. Resource-constrained omissions are explicit CONDITIONAL_SKIP
# evidence and can never impersonate successful refits.
if not hasattr(_r10_orch, "_iharq_r10_parent_execute_role"):
    # If an earlier Stage18 successor was installed in this kernel, recover the
    # package-native parent saved by that successor rather than wrapping a
    # previously reduced execute_role.
    if hasattr(_r10_orch, "_iharq_r9_parent_execute_role"):
        _r10_orch._iharq_r10_parent_execute_role = _r10_orch._iharq_r9_parent_execute_role
    else:
        _r10_orch._iharq_r10_parent_execute_role = _r10_orch.execute_role
_R10_PARENT_EXECUTE_ROLE = _r10_orch._iharq_r10_parent_execute_role

_R10_CONDITIONS = set(R10_STAGE18_REDUCTION["affected_conditions"])
_R10_DEEP_ROLES = set(R10_STAGE18_REDUCTION["deep_roles"])
_R10_ANCHOR_BUDGETS = set(R10_STAGE18_REDUCTION["deep_anchor_budget_ids"])
_R10_EXEC_REPEATS = set(R10_STAGE18_REDUCTION["executed_repeat_indices"])

def _r10_reduction_reason(c):
    """Return an explicit R10 resource-deviation reason, or None if executable."""
    if str(c.get("condition_id")) not in _R10_CONDITIONS:
        return None

    rep = int(c.get("model_repeat_index", 0))
    if rep not in _R10_EXEC_REPEATS:
        return R10_STAGE18_REDUCTION["repeat_skip_reason"]

    if (
        str(c.get("role_id")) in _R10_DEEP_ROLES
        and str(c.get("budget_id")) not in _R10_ANCHOR_BUDGETS
    ):
        return R10_STAGE18_REDUCTION["deep_budget_skip_reason"]

    return None

def _r10_reduced_refit_cell(c):
    return _r10_reduction_reason(c) is not None

def _r10_executable_refit_cell(c):
    return (
        str(c.get("condition_id")) in _R10_CONDITIONS
        and _r10_reduction_reason(c) is None
    )

def R10_A4_execute_role(
    c, store, core, a4, reps, schema_path, config_sha, freeze, data_contract,
    fixture=False, implementation_bindings=None, runtime_overrides=None
):
    old = store.terminal(c)
    if old:
        return old

    reason = _r10_reduction_reason(c)
    if reason is not None:
        dimension = (
            "REPEAT_DEPTH"
            if reason == R10_STAGE18_REDUCTION["repeat_skip_reason"]
            else "DEEP_BUDGET_DENSITY"
        )
        return store.write_terminal(
            c,
            R10_STAGE18_REDUCTION["skip_terminal_status"],
            reason=reason,
            reduction_dimension=dimension,
            protocol_deviation_id=R10_STAGE18_REDUCTION["protocol_deviation_id"],
            protocol_deviation_from_buildbook=True,
            full_buildbook_replication_equivalence=False,
            intended_original_repeat_indices=R10_STAGE18_REDUCTION["original_repeat_indices"],
            executed_repeat_indices=R10_STAGE18_REDUCTION["executed_repeat_indices"],
            deep_anchor_budget_ids=R10_STAGE18_REDUCTION["deep_anchor_budget_ids"],
            omitted_repeat_index=int(c.get("model_repeat_index", 0)),
            omitted_budget_id=(
                str(c.get("budget_id"))
                if dimension == "DEEP_BUDGET_DENSITY"
                else None
            ),
            scientific_training_performed=False,
            prediction_regenerated=False,
            test_signal_loaded=False,
            ablation_condition_removed=False,
            dataset_removed=False,
            model_family_removed=False,
            reporting_scope=R10_STAGE18_REDUCTION["claim_scope"],
        )

    return _R10_PARENT_EXECUTE_ROLE(
        c, store, core, a4, reps, schema_path, config_sha, freeze, data_contract,
        fixture, implementation_bindings, runtime_overrides
    )

# Fail fast if the package interface changes again.
_r10_expected_execute_role_params = [
    "c", "store", "core", "a4", "reps", "schema_path", "config_sha",
    "freeze", "data_contract", "fixture", "implementation_bindings",
    "runtime_overrides",
]
_r10_parent_execute_role_params = list(
    inspect.signature(_R10_PARENT_EXECUTE_ROLE).parameters
)
_r10_wrapper_execute_role_params = list(
    inspect.signature(R10_A4_execute_role).parameters
)
if _r10_parent_execute_role_params != _r10_expected_execute_role_params:
    _r8_fail(
        "R10_PARENT_EXECUTE_ROLE_SIGNATURE_CHANGED",
        {
            "expected": _r10_expected_execute_role_params,
            "observed": _r10_parent_execute_role_params,
        },
    )
if _r10_wrapper_execute_role_params != _r10_expected_execute_role_params:
    _r8_fail(
        "R10_WRAPPER_EXECUTE_ROLE_SIGNATURE_INVALID",
        {
            "expected": _r10_expected_execute_role_params,
            "observed": _r10_wrapper_execute_role_params,
        },
    )

_r10_orch.execute_role = R10_A4_execute_role

# -----------------------------------------------------------------------------
# 6B. R10.3 transport-only dual-GPU scheduler.
#
# GPU0 executes the current deep A4 run cell.
# GPU1 prefetches ONE DIFFERENT future deep A4 run cell from a different
# (dataset,budget,role,repeat) cache group.  A single model/run cell is NEVER
# split across GPUs.  Same-group cells are serialized to prevent A4MEM cache
# collisions (especially C2/C3, which intentionally share member caches).
# -----------------------------------------------------------------------------
_R10_3_GPU_NAMES = []
try:
    _R10_3_GPU_NAMES = [
        str(torch.cuda.get_device_name(i))
        for i in range(torch.cuda.device_count())
    ]
except Exception:
    _R10_3_GPU_NAMES = []

_R10_3_DUAL_GPU_AVAILABLE = (
    torch.cuda.is_available()
    and torch.cuda.device_count() >= 2
    and len(_R10_3_GPU_NAMES) >= 2
    and _R10_3_GPU_NAMES[0] == _R10_3_GPU_NAMES[1]
)

def _r10_3_available_ram_gib():
    try:
        import psutil
        return float(psutil.virtual_memory().available) / (1024.0 ** 3)
    except Exception:
        try:
            vals = {}
            for line in Path("/proc/meminfo").read_text().splitlines():
                if ":" in line:
                    k, v = line.split(":", 1)
                    vals[k] = v.strip()
            kb = float(vals.get("MemAvailable", "0 kB").split()[0])
            return kb / (1024.0 ** 2)
        except Exception:
            return None

def _r10_3_gpu1_resource_ok():
    if not _R10_3_DUAL_GPU_AVAILABLE:
        return False
    ram = _r10_3_available_ram_gib()
    if ram is not None and ram < 10.0:
        return False
    try:
        with torch.cuda.device(1):
            free_b, total_b = torch.cuda.mem_get_info(1)
        if float(free_b) / (1024.0 ** 3) < 6.0:
            return False
    except Exception:
        pass
    return True

_R10_3_PREFETCH_ENABLED = bool(_R10_3_DUAL_GPU_AVAILABLE)
_R10_3_BG_LOCK = threading.RLock()
_R10_3_BG = {
    "thread": None,
    "cell_id": None,
    "group": None,
    "result": None,
    "error": None,
    "traceback": None,
    "done": None,
}
_R10_3_PLAN = list(STATE.get("a4cells") or [])
_R10_3_PLAN_INDEX = {
    str(c["planned_run_cell_id"]): i
    for i, c in enumerate(_R10_3_PLAN)
}
_R10_3_TRANSPORT_ROOT = (
    Path(RUNTIME_ROOT)
    / "diagnostics" / "runtime_successor"
    / "stage18_r10_3_dual_gpu_transport"
)
_R10_3_TRANSPORT_ROOT.mkdir(parents=True, exist_ok=True)

def _r10_3_group(c):
    return (
        str(c.get("dataset_id")),
        str(c.get("budget_id")),
        str(c.get("role_id")),
        int(c.get("model_repeat_index", 0)),
    )

def _r10_3_is_deep_executable(c):
    return (
        str(c.get("condition_id")) in _R10_CONDITIONS
        and str(c.get("role_id")) in _R10_DEEP_ROLES
        and _r10_reduction_reason(c) is None
    )

def _r10_3_transport_write(c, gpu_id, mode, phase, extra=None):
    rid = str(c["planned_run_cell_id"])
    payload = {
        "artifact_id": "P02-STAGE18-R10-3-DUAL-GPU-TRANSPORT-R1",
        "planned_run_cell_id": rid,
        "gpu_id": int(gpu_id),
        "gpu_name": (
            _R10_3_GPU_NAMES[int(gpu_id)]
            if int(gpu_id) < len(_R10_3_GPU_NAMES)
            else None
        ),
        "mode": str(mode),
        "phase": str(phase),
        "single_model_multi_gpu": False,
        "same_run_cell_on_multiple_gpus": False,
        "scientific_configuration_changed": False,
        "policy_sha256": R8_STAGE18_POLICY_SHA256,
    }
    if extra:
        payload.update(copy.deepcopy(extra))
    _r8_pkg_atomic_json(
        _R10_3_TRANSPORT_ROOT / f"{rid}.json",
        payload,
    )

def _r10_3_reap_background(block=False):
    global _R10_3_BG
    with _R10_3_BG_LOCK:
        th = _R10_3_BG.get("thread")
        done = _R10_3_BG.get("done")
    if th is None:
        return None
    if block:
        th.join()
    elif done is not None and not done.is_set():
        return None

    with _R10_3_BG_LOCK:
        err = _R10_3_BG.get("error")
        tb = _R10_3_BG.get("traceback")
        result = _R10_3_BG.get("result")
        cid = _R10_3_BG.get("cell_id")
        _R10_3_BG = {
            "thread": None, "cell_id": None, "group": None,
            "result": None, "error": None, "traceback": None, "done": None,
        }
    if err is not None:
        raise RuntimeError(
            "R10_3_GPU1_BACKGROUND_CELL_FAILED:"
            + json.dumps(
                {"cell_id": cid, "error": err, "traceback": tb},
                sort_keys=True,
            )
        )
    return result

def _r10_3_background_active():
    with _R10_3_BG_LOCK:
        th = _R10_3_BG.get("thread")
        done = _R10_3_BG.get("done")
        return bool(th is not None and done is not None and not done.is_set())

def _r10_3_background_group():
    with _R10_3_BG_LOCK:
        return _R10_3_BG.get("group")

def _r10_3_background_cell_id():
    with _R10_3_BG_LOCK:
        return _R10_3_BG.get("cell_id")

def _r10_3_choose_future(current):
    if not _R10_3_PREFETCH_ENABLED or not _r10_3_gpu1_resource_ok():
        return None
    cur_i = _R10_3_PLAN_INDEX.get(str(current["planned_run_cell_id"]), -1)
    cur_group = _r10_3_group(current)
    for cand in _R10_3_PLAN[cur_i + 1:]:
        if not _r10_3_is_deep_executable(cand):
            continue
        if _r10_3_group(cand) == cur_group:
            continue
        if STORE.terminal(cand) is not None:
            continue
        # Do not schedule a C3 cell when the same group's C2/C1 can seed the
        # identical member cache first; the first eligible condition is enough.
        return cand
    return None

def _r10_3_launch_background(
    c, store, core, a4, reps, schema_path, config_sha, freeze,
    data_contract, fixture, implementation_bindings, runtime_overrides
):
    global _R10_3_BG
    _r10_3_reap_background(block=False)
    if _r10_3_background_active():
        return False

    done = threading.Event()
    group = _r10_3_group(c)
    rid = str(c["planned_run_cell_id"])

    # Representative selection is dataset+budget specific.  Recompute the
    # validation-selected mapping for the FUTURE cell; never reuse the current
    # cell's `reps` object across a dataset/budget boundary.
    future_reps = _r8_a4.select_reps(
        store,
        c["dataset_id"],
        str(c["budget_id"]),
    )[0]

    def _worker():
        _R10_3_GPU_TLS.gpu_id = 1
        try:
            with torch.cuda.device(1):
                _r10_3_transport_write(c, 1, "GPU1_FUTURE_ABLATION_PREFETCH", "STARTED")
                out = _R10_PARENT_EXECUTE_ROLE(
                    c, store, core, a4, future_reps, schema_path, config_sha, freeze,
                    data_contract, fixture, implementation_bindings, runtime_overrides
                )
                _r10_3_transport_write(
                    c, 1, "GPU1_FUTURE_ABLATION_PREFETCH", "COMPLETED",
                    {"terminal_status": None if out is None else out.get("terminal_status")},
                )
                with _R10_3_BG_LOCK:
                    _R10_3_BG["result"] = out
        except BaseException as exc:
            with _R10_3_BG_LOCK:
                _R10_3_BG["error"] = f"{type(exc).__name__}:{str(exc)[:1000]}"
                _R10_3_BG["traceback"] = traceback.format_exc()[-8000:]
        finally:
            done.set()

    th = threading.Thread(
        target=_worker,
        name=f"IHARQ-Stage18-GPU1-{rid}",
        daemon=True,
    )
    with _R10_3_BG_LOCK:
        _R10_3_BG = {
            "thread": th,
            "cell_id": rid,
            "group": group,
            "result": None,
            "error": None,
            "traceback": None,
            "done": done,
        }
    th.start()
    return True

def _r10_3_wait_background():
    return _r10_3_reap_background(block=True)

def _r10_3_pause_background_scheduling():
    global _R10_3_PREFETCH_ENABLED
    _R10_3_PREFETCH_ENABLED = False

def _r10_3_enable_background_scheduling():
    global _R10_3_PREFETCH_ENABLED
    _R10_3_PREFETCH_ENABLED = bool(_R10_3_DUAL_GPU_AVAILABLE)

def R10_3_A4_execute_role(
    c, store, core, a4, reps, schema_path, config_sha, freeze, data_contract,
    fixture=False, implementation_bindings=None, runtime_overrides=None
):
    # Existing durable evidence is authoritative and immediately reusable.
    old = store.terminal(c)
    if old:
        return old

    # Resource-reduction terminals are deterministic bookkeeping and may be
    # written while GPU1 is busy because they touch no scientific data/model.
    reason = _r10_reduction_reason(c)
    if reason is not None:
        dimension = (
            "REPEAT_DEPTH"
            if reason == R10_STAGE18_REDUCTION["repeat_skip_reason"]
            else "DEEP_BUDGET_DENSITY"
        )
        return store.write_terminal(
            c,
            R10_STAGE18_REDUCTION["skip_terminal_status"],
            reason=reason,
            reduction_dimension=dimension,
            protocol_deviation_id=R10_STAGE18_REDUCTION["protocol_deviation_id"],
            protocol_deviation_from_buildbook=True,
            full_buildbook_replication_equivalence=False,
            intended_original_repeat_indices=R10_STAGE18_REDUCTION["original_repeat_indices"],
            executed_repeat_indices=R10_STAGE18_REDUCTION["executed_repeat_indices"],
            deep_anchor_budget_ids=R10_STAGE18_REDUCTION["deep_anchor_budget_ids"],
            omitted_repeat_index=int(c.get("model_repeat_index", 0)),
            omitted_budget_id=(
                str(c.get("budget_id"))
                if dimension == "DEEP_BUDGET_DENSITY"
                else None
            ),
            scientific_training_performed=False,
            prediction_regenerated=False,
            test_signal_loaded=False,
            ablation_condition_removed=False,
            dataset_removed=False,
            model_family_removed=False,
            reporting_scope=R10_STAGE18_REDUCTION["claim_scope"],
        )

    deep = _r10_3_is_deep_executable(c)

    if deep:
        # Never allow two conditions from the same A4MEM cache group to execute
        # concurrently across GPUs.
        bg_group = _r10_3_background_group()
        if bg_group is not None and bg_group == _r10_3_group(c):
            _r10_3_wait_background()
            old = store.terminal(c)
            if old:
                return old

        _r10_3_reap_background(block=False)

        # If GPU1 is idle, prefetch a different future ablation group.
        if not _r10_3_background_active():
            future = _r10_3_choose_future(c)
            if future is not None:
                _r10_3_launch_background(
                    future, store, core, a4, reps, schema_path, config_sha, freeze,
                    data_contract, fixture, implementation_bindings, runtime_overrides
                )

        _R10_3_GPU_TLS.gpu_id = 0
        if torch.cuda.is_available():
            with torch.cuda.device(0):
                _r10_3_transport_write(c, 0, "GPU0_CURRENT_ABLATION", "STARTED")
                out = _R10_PARENT_EXECUTE_ROLE(
                    c, store, core, a4, reps, schema_path, config_sha, freeze,
                    data_contract, fixture, implementation_bindings, runtime_overrides
                )
                _r10_3_transport_write(
                    c, 0, "GPU0_CURRENT_ABLATION", "COMPLETED",
                    {"terminal_status": None if out is None else out.get("terminal_status")},
                )
                return out

        return _R10_PARENT_EXECUTE_ROLE(
            c, store, core, a4, reps, schema_path, config_sha, freeze,
            data_contract, fixture, implementation_bindings, runtime_overrides
        )

    # Package-native classical/Riemannian/C0/C4/C5 work is kept out of
    # concurrent access to the shared A4 object.  Finish any GPU1 deep prefetch
    # first, then execute this inexpensive/non-deep cell normally.
    if _r10_3_background_active():
        _r10_3_wait_background()

    _R10_3_GPU_TLS.gpu_id = 0
    return _R10_PARENT_EXECUTE_ROLE(
        c, store, core, a4, reps, schema_path, config_sha, freeze,
        data_contract, fixture, implementation_bindings, runtime_overrides
    )

_r10_3_wrapper_params = list(
    inspect.signature(R10_3_A4_execute_role).parameters
)
if _r10_3_wrapper_params != _r10_expected_execute_role_params:
    _r8_fail(
        "R10_3_WRAPPER_EXECUTE_ROLE_SIGNATURE_INVALID",
        {
            "expected": _r10_expected_execute_role_params,
            "observed": _r10_3_wrapper_params,
        },
    )

_r10_orch.execute_role = R10_3_A4_execute_role


# -----------------------------------------------------------------------------
# 7. R10 semantic validator + explicit protocol-deviation evidence.
# -----------------------------------------------------------------------------
def R10_STAGE18_SEMANTIC_AUDIT():
    issues = []
    deep = []
    cells = list(STATE.get("a4cells") or [])

    refit_cells = [
        c for c in cells if str(c.get("condition_id")) in _R10_CONDITIONS
    ]
    reduced_cells = [c for c in refit_cells if _r10_reduced_refit_cell(c)]
    executable_refit_cells = [c for c in refit_cells if _r10_executable_refit_cell(c)]
    executable_deep_cells = [
        c for c in executable_refit_cells
        if str(c.get("role_id")) in _R10_DEEP_ROLES
    ]

    repeat_reduced = [
        c for c in reduced_cells
        if _r10_reduction_reason(c) == R10_STAGE18_REDUCTION["repeat_skip_reason"]
    ]
    budget_reduced = [
        c for c in reduced_cells
        if _r10_reduction_reason(c) == R10_STAGE18_REDUCTION["deep_budget_skip_reason"]
    ]

    # Every original planned A4 cell remains in the denominator.
    missing_terminals = [
        c["planned_run_cell_id"] for c in cells if STORE.terminal(c) is None
    ]
    if missing_terminals:
        issues.append({
            "kind": "A4_PLANNED_TERMINALS_MISSING",
            "count": len(missing_terminals),
            "examples": missing_terminals[:20],
        })

    # Every R10 omission must be transparently represented as the exact
    # owner-authorized conditional skip.
    bad_reduced = []
    for c in reduced_cells:
        expected_reason = _r10_reduction_reason(c)
        t = STORE.terminal(c)
        if (
            not t
            or t.get("terminal_status") != R10_STAGE18_REDUCTION["skip_terminal_status"]
            or t.get("reason") != expected_reason
            or t.get("protocol_deviation_id") != R10_STAGE18_REDUCTION["protocol_deviation_id"]
            or t.get("scientific_training_performed") is not False
            or t.get("ablation_condition_removed") is not False
        ):
            bad_reduced.append({
                "run_cell_id": c["planned_run_cell_id"],
                "expected_reason": expected_reason,
                "terminal": t,
            })
    if bad_reduced:
        issues.append({
            "kind": "R10_DECLARED_REDUCTIONS_NOT_EXPLICITLY_ACCOUNTED",
            "count": len(bad_reduced),
            "examples": bad_reduced[:10],
        })

    # Cells retained by the reduced design must never be skipped *because of*
    # R10. Independent scientific/dependency/resource terminal states remain
    # legitimate and are preserved as negative evidence.
    wrongly_reduced = []
    reduction_reasons = {
        R10_STAGE18_REDUCTION["repeat_skip_reason"],
        R10_STAGE18_REDUCTION["deep_budget_skip_reason"],
    }
    for c in executable_refit_cells:
        t = STORE.terminal(c)
        if t and (
            t.get("reason") in reduction_reasons
            or t.get("protocol_deviation_id") == R10_STAGE18_REDUCTION["protocol_deviation_id"]
        ):
            wrongly_reduced.append(c["planned_run_cell_id"])
    if wrongly_reduced:
        issues.append({
            "kind": "R10_EXECUTABLE_REFIT_CELL_INCORRECTLY_REDUCED",
            "count": len(wrongly_reduced),
            "examples": wrongly_reduced[:20],
        })

    # Verify the intended three anchor budgets actually exist in the frozen plan.
    observed_budgets = sorted({str(c.get("budget_id")) for c in cells})
    missing_anchor_budgets = sorted(_R10_ANCHOR_BUDGETS - set(observed_budgets))
    if missing_anchor_budgets:
        issues.append({
            "kind": "R10_DEEP_ANCHOR_BUDGET_MISSING_FROM_FROZEN_PLAN",
            "missing": missing_anchor_budgets,
            "observed": observed_budgets,
        })

    # For each dataset and each deep role, retain all three C1/C2/C3 conditions
    # at every anchor budget under MR00. This is the minimum semantic core.
    datasets = sorted({str(c.get("dataset_id")) for c in cells})
    missing_semantic_core = []
    for ds in datasets:
        for role in sorted(_R10_DEEP_ROLES):
            for budget in sorted(_R10_ANCHOR_BUDGETS):
                for cond in sorted(_R10_CONDITIONS):
                    candidates = [
                        c for c in cells
                        if str(c.get("dataset_id")) == ds
                        and str(c.get("role_id")) == role
                        and str(c.get("budget_id")) == budget
                        and str(c.get("condition_id")) == cond
                        and int(c.get("model_repeat_index", 0)) in _R10_EXEC_REPEATS
                    ]
                    if not candidates:
                        missing_semantic_core.append({
                            "dataset_id": ds,
                            "role_id": role,
                            "budget_id": budget,
                            "condition_id": cond,
                        })
    if missing_semantic_core:
        issues.append({
            "kind": "R10_DEEP_ANCHOR_SEMANTIC_CORE_PLAN_INCOMPLETE",
            "count": len(missing_semantic_core),
            "examples": missing_semantic_core[:20],
        })

    # Author/source-aligned provenance remains mandatory for every deep member
    # that actually executes under R10.
    for p in sorted((Path(STORE.root) / "diagnostics").glob("A4MEM-*.json")):
        d = _r8_load_json(p)
        if d.get("branch") not in {"DNN-EEGNET", "DNN-FBCNET", "DNN-SEQ", "SSL-CBRAMOD"}:
            continue
        deep.append(d)

        if d.get("r8_policy_sha256") != R8_STAGE18_POLICY_SHA256:
            issues.append({
                "kind": "DEEP_A4_MEMBER_NOT_R10_POLICY_ALIGNED",
                "path": str(p),
            })
            continue

        if (
            d.get("branch") == "SSL-CBRAMOD"
            and d.get("member") == "LONG"
            and d.get("status") == "SUCCESS"
        ):
            issues.append({
                "kind": "FORBIDDEN_CBRAMOD_A4_LONG_SUCCESS",
                "path": str(p),
            })

        if d.get("status") == "SUCCESS":
            if d.get("class_weight_frozen_from_a0") is not True:
                issues.append({
                    "kind": "A4_CLASS_WEIGHT_NOT_FROZEN_FROM_A0",
                    "path": str(p),
                })
            if (
                d.get("test_loaded_after_checkpoint_sealed") is not True
                or d.get("test_set_used_for_selection") is not False
            ):
                issues.append({
                    "kind": "A4_TEST_ISOLATION_PROVENANCE_INVALID",
                    "path": str(p),
                })
            if d.get("recipe_id") not in {
                R8_EEGNET_RECIPE["recipe_id"],
                *R8_EXTERNAL_RECIPE_IDS.values(),
            }:
                issues.append({
                    "kind": "A4_RECIPE_ID_NOT_ACCEPTED_AUTHOR_SOURCE_RECIPE",
                    "path": str(p),
                    "recipe_id": d.get("recipe_id"),
                })

    completion_path = Path(STORE.root) / "analysis_inputs" / "a4_completion.json"
    completion = None
    if not completion_path.is_file():
        issues.append({"kind": "A4_COMPLETION_ARTIFACT_MISSING"})
    else:
        completion = _r8_load_json(completion_path)
        if int(completion.get("planned", -1)) != len(cells):
            issues.append({
                "kind": "A4_COMPLETION_PLANNED_CENSUS_MISMATCH",
                "observed": completion.get("planned"),
                "expected": len(cells),
            })
        if int(completion.get("role_control_incomplete", -1)) != 0:
            issues.append({
                "kind": "A4_ROLE_CONTROL_CLOSURE_INCOMPLETE",
                "observed": completion.get("role_control_incomplete"),
            })
        if int(completion.get("c4_c5_incomplete", -1)) != 0:
            issues.append({
                "kind": "A4_C4_C5_CLOSURE_INCOMPLETE",
                "observed": completion.get("c4_c5_incomplete"),
            })
        if completion.get("burden_source_complete") is not True:
            issues.append({"kind": "A4_BURDEN_SOURCE_INCOMPLETE"})

    conditions_present = sorted({str(c.get("condition_id")) for c in cells})
    required_conditions = sorted({
        "A4-C0-CORE",
        "A4-C1-LONG-3P5S",
        "A4-C2-MULTI-HARD-VOTE",
        "A4-C3-MULTI-PROB-AVG",
        "A4-C4-MODEL-HARD-VOTE",
        "A4-C5-MODEL-PROB-AVG",
    })
    if conditions_present != required_conditions:
        issues.append({
            "kind": "A4_CONDITION_CENSUS_CHANGED",
            "observed": conditions_present,
            "expected": required_conditions,
        })

    # Transparent workload accounting. C2/C3 share member-fit caches, so the
    # unique deep member estimate is grouped by dataset/budget/role/repeat and
    # multiplied by four members (LONG + 3 views).
    full_deep_groups = {
        (
            str(c["dataset_id"]),
            str(c["budget_id"]),
            str(c["role_id"]),
            int(c["model_repeat_index"]),
        )
        for c in refit_cells
        if str(c.get("role_id")) in _R10_DEEP_ROLES
    }
    exec_deep_groups = {
        (
            str(c["dataset_id"]),
            str(c["budget_id"]),
            str(c["role_id"]),
            int(c["model_repeat_index"]),
        )
        for c in executable_deep_cells
    }

    original_unique_deep_member_fits = len(full_deep_groups) * 4
    reduced_unique_deep_member_fits = len(exec_deep_groups) * 4

    terminal_counts = Counter(
        str(STORE.terminal(c).get("terminal_status"))
        for c in cells
        if STORE.terminal(c) is not None
    )

    reduction_reason_counts = Counter(
        str(_r10_reduction_reason(c)) for c in reduced_cells
    )

    deviation = {
        "artifact_id": "P02-STAGE18-R10-ANCHOR-BUDGET-SINGLE-REPEAT-DEVIATION-R1",
        "status": "PASS" if not issues else "FAIL",
        "protocol_deviation_from_buildbook": True,
        "protocol_deviation_id": R10_STAGE18_REDUCTION["protocol_deviation_id"],
        "reason": R10_STAGE18_REDUCTION["reason"],
        "claim_scope": R10_STAGE18_REDUCTION["claim_scope"],
        "full_buildbook_replication_equivalence": False,
        "ablation_conditions_removed": False,
        "model_family_removed": False,
        "dataset_removed": False,
        "representation_removed": False,

        "datasets_preserved": datasets,
        "budgets_in_frozen_plan": observed_budgets,
        "conditions_preserved": conditions_present,
        "refit_roles_present": sorted({str(c.get("role_id")) for c in refit_cells}),
        "deep_roles": sorted(_R10_DEEP_ROLES),

        "original_repeat_indices": R10_STAGE18_REDUCTION["original_repeat_indices"],
        "executed_repeat_indices": R10_STAGE18_REDUCTION["executed_repeat_indices"],
        "declared_skipped_repeat_indices": R10_STAGE18_REDUCTION["declared_skipped_repeat_indices"],

        "deep_anchor_budget_ids": R10_STAGE18_REDUCTION["deep_anchor_budget_ids"],
        "deep_anchor_semantics": R10_STAGE18_REDUCTION["deep_anchor_semantics"],
        "deep_budget_grid_reduced": True,

        "planned_a4_cells": len(cells),
        "refit_cells_in_frozen_plan": len(refit_cells),
        "declared_reduced_run_cells": len(reduced_cells),
        "repeat_reduced_run_cells": len(repeat_reduced),
        "deep_budget_reduced_run_cells": len(budget_reduced),
        "reduction_reason_counts": dict(sorted(reduction_reason_counts.items())),
        "executable_refit_run_cells": len(executable_refit_cells),
        "executable_deep_refit_run_cells": len(executable_deep_cells),

        "original_unique_deep_member_fits_estimate": original_unique_deep_member_fits,
        "reduced_unique_deep_member_fits_estimate": reduced_unique_deep_member_fits,
        "deep_member_fit_reduction_fraction": (
            None if original_unique_deep_member_fits == 0
            else 1.0 - (
                reduced_unique_deep_member_fits
                / original_unique_deep_member_fits
            )
        ),

        "latency_measurement_repeats": R10_STAGE18_REDUCTION["latency_measurement_repeats"],
        "per_fit_training_recipe_changed": False,
        "representative_selection_changed": False,
        "test_isolation_changed": False,
        "checkpoint_roundtrip_changed": False,
        "statistical_comparison_definition_changed": False,

        "multi_seed_stability_claim_authorized": False,
        "dense_deep_budget_curve_claim_authorized": False,
        "anchor_budget_deep_claim_authorized": True,

        "terminal_counts": dict(sorted(terminal_counts.items())),
        "issues": issues,
    }

    deviation_path = (
        Path(STORE.root)
        / "analysis_inputs"
        / "stage18_resource_constrained_anchor_budget_deviation.json"
    )
    _r8_pkg_atomic_json(deviation_path, deviation)

    # Propagate the limitation into downstream source-of-truth artifacts without
    # deleting their original scientific fields.
    annotation = {
        "protocol_deviation_from_buildbook": True,
        "protocol_deviation_id": R10_STAGE18_REDUCTION["protocol_deviation_id"],
        "full_buildbook_replication_equivalence": False,
        "resource_constrained_single_repeat": True,
        "deep_budget_grid_reduced": True,
        "deep_anchor_budget_ids": R10_STAGE18_REDUCTION["deep_anchor_budget_ids"],
        "multi_seed_stability_claim_authorized": False,
        "dense_deep_budget_curve_claim_authorized": False,
        "anchor_budget_deep_claim_authorized": True,
        "declared_reduced_run_cells": len(reduced_cells),
        "protocol_deviation_source": str(
            deviation_path.relative_to(Path(STORE.root))
        ),
    }

    if completion is not None:
        completion.update(annotation)
        _r8_pkg_atomic_json(completion_path, completion)

    for _name in ("a4_role_control_statistics.json", "a4_c4_c5_statistics.json"):
        _p = Path(STORE.root) / "analysis_inputs" / _name
        if _p.is_file():
            _d = _r8_load_json(_p)
            _d.update(annotation)
            _r8_pkg_atomic_json(_p, _d)

    report = {
        "artifact_id": R8_STAGE18_ARTIFACT_ID,
        "status": "PASS" if not issues else "FAIL",
        "runtime_successor_id": R8_STAGE18_SUCCESSOR_ID,
        "policy_sha256": R8_STAGE18_POLICY_SHA256,

        "protocol_deviation_from_buildbook": True,
        "protocol_deviation_id": R10_STAGE18_REDUCTION["protocol_deviation_id"],
        "full_buildbook_replication_equivalence": False,
        "ablation_conditions_removed": False,

        "planned_a4_cells": len(cells),
        "declared_reduced_run_cells": len(reduced_cells),
        "repeat_reduced_run_cells": len(repeat_reduced),
        "deep_budget_reduced_run_cells": len(budget_reduced),
        "executable_refit_run_cells": len(executable_refit_cells),
        "executable_deep_refit_run_cells": len(executable_deep_cells),

        "deep_anchor_budget_ids": R10_STAGE18_REDUCTION["deep_anchor_budget_ids"],
        "original_unique_deep_member_fits_estimate": original_unique_deep_member_fits,
        "reduced_unique_deep_member_fits_estimate": reduced_unique_deep_member_fits,
        "deep_member_receipts": len(deep),

        "completed_A0_artifacts_mutated": False,
        "new_hyperparameter_grid": False,
        "per_fit_training_recipe_changed": False,
        "test_outcome_influence": "PROHIBITED",
        "multi_seed_stability_claim_authorized": False,
        "dense_deep_budget_curve_claim_authorized": False,
        "anchor_budget_deep_claim_authorized": True,

        "cbramod_a4_long_rule": "INPUT_INCOMPATIBLE_FAIL_CLOSED_NO_PAD_NO_CROP",
        "deviation_receipt": str(deviation_path.relative_to(Path(STORE.root))),
        "issues": issues,
    }

    _r8_pkg_atomic_json(
        Path(RUNTIME_ROOT)
        / "diagnostics" / "runtime_successor"
        / "G18_R10_ANCHOR_BUDGET_A4_SEMANTIC_AUDIT.json",
        report,
    )
    return report


# -----------------------------------------------------------------------------
# 7B. R10.3 transport audit wraps the unchanged R10 scientific semantic audit.
# -----------------------------------------------------------------------------
_R10_3_BASE_SEMANTIC_AUDIT = R10_STAGE18_SEMANTIC_AUDIT

def R10_3_STAGE18_SEMANTIC_AUDIT():
    report = _R10_3_BASE_SEMANTIC_AUDIT()
    issues = list(report.get("issues") or [])

    # At closure no background ablation may still be in flight.
    try:
        _r10_3_wait_background()
    except Exception as exc:
        issues.append({
            "kind": "R10_3_GPU1_BACKGROUND_NOT_CLEAN_AT_CLOSURE",
            "error": f"{type(exc).__name__}:{str(exc)[:1000]}",
        })

    transport = []
    for p in sorted(_R10_3_TRANSPORT_ROOT.glob("*.json")):
        try:
            d = _r8_load_json(p)
        except Exception as exc:
            issues.append({
                "kind": "R10_3_TRANSPORT_RECEIPT_UNREADABLE",
                "path": str(p),
                "error": f"{type(exc).__name__}:{str(exc)[:300]}",
            })
            continue
        transport.append(d)
        if d.get("single_model_multi_gpu") is not False:
            issues.append({
                "kind": "R10_3_FORBIDDEN_SINGLE_MODEL_MULTI_GPU",
                "path": str(p),
            })
        if d.get("policy_sha256") != R8_STAGE18_POLICY_SHA256:
            issues.append({
                "kind": "R10_3_TRANSPORT_POLICY_HASH_MISMATCH",
                "path": str(p),
            })

    # New R10.3 successful external members must prove the governed external
    # checkpoint route.  Pre-R10.3 compatible successes are grandfathered only
    # because they already passed their earlier round-trip and are preserved for
    # resume; they do not acquire synthetic R10.3 provenance.
    for p in sorted((Path(STORE.root) / "diagnostics").glob("A4MEM-*.json")):
        try:
            d = _r8_load_json(p)
        except Exception:
            continue
        if d.get("status") != "SUCCESS":
            continue
        if d.get("execution_device") is None:
            continue
        if d.get("branch") in {"DNN-FBCNET", "DNN-SEQ", "SSL-CBRAMOD"}:
            if d.get("checkpoint_format") != "EXTERNAL_GOVERNED_INTERFACE":
                issues.append({
                    "kind": "R10_3_EXTERNAL_MEMBER_NOT_EXTERNAL_CHECKPOINT_ROUTE",
                    "path": str(p),
                    "branch": d.get("branch"),
                    "checkpoint_format": d.get("checkpoint_format"),
                })

    by_gpu = Counter(str(d.get("gpu_id")) for d in transport if d.get("phase") == "COMPLETED")
    gpu1_completed = int(by_gpu.get("1", 0))
    gpu0_completed = int(by_gpu.get("0", 0))

    # GPU1 use is a transport optimization, not a scientific acceptance gate.
    # If RAM/GPU-memory guards prevent safe overlap, Stage18 falls back to the
    # single-GPU execution path without weakening or changing the experiment.
    gpu1_transport_used = bool(gpu1_completed > 0)

    if (
        R10_3_CHECKPOINT_ROUTE_PROBE.get("status") != "PASS"
        or R10_3_CHECKPOINT_ROUTE_PROBE.get("checkpoint_format")
           != "EXTERNAL_GOVERNED_INTERFACE"
    ):
        issues.append({
            "kind": "R10_3_EXTERNAL_CHECKPOINT_ROUTE_PROBE_NOT_PASS",
            "probe": R10_3_CHECKPOINT_ROUTE_PROBE,
        })

    report.update({
        "status": "PASS" if not issues else "FAIL",
        "issues": issues,
        "execution_transport_revision": "R10.3_DUAL_GPU_DIFFERENT_ABLATION_CELLS",
        "dual_gpu_available": bool(_R10_3_DUAL_GPU_AVAILABLE),
        "gpu_names": _R10_3_GPU_NAMES,
        "single_model_multi_gpu": False,
        "gpu0_completed_ablation_cells": gpu0_completed,
        "gpu1_completed_ablation_cells": gpu1_completed,
        "gpu1_transport_used": gpu1_transport_used,
        "dual_gpu_fallback_is_non_scientific": True,
        "compatible_partial_terminals_resumed": len(_existing_a4),
        "external_first_checkpoint_route": True,
        "checkpoint_route_probe": R10_3_CHECKPOINT_ROUTE_PROBE,
        "checkpoint_reference_updates":
            sorted(set(_r10_3_checkpoint_reference_updates)),
        "failed_attempt_files_archived_before_retry":
            len(_r10_3_archived_failed),
        "scientific_policy_sha256_unchanged": R8_STAGE18_POLICY_SHA256,
    })

    _r8_pkg_atomic_json(
        Path(RUNTIME_ROOT)
        / "diagnostics" / "runtime_successor"
        / "G18_R10_3_DUAL_GPU_SEMANTIC_TRANSPORT_AUDIT.json",
        report,
    )
    return report

R10_STAGE18_SEMANTIC_AUDIT = R10_3_STAGE18_SEMANTIC_AUDIT

# Compatibility alias for notebook code that still references the older symbol.
R8_STAGE18_SEMANTIC_AUDIT = R10_3_STAGE18_SEMANTIC_AUDIT
_SEMANTIC_VALIDATORS["18"] = R10_3_STAGE18_SEMANTIC_AUDIT


# -----------------------------------------------------------------------------
# 8. Successor annotation + durable pre-result policy.
# -----------------------------------------------------------------------------
if "_IHARQ_STAGE18_BASE_ANNOTATE_R10" not in globals():
    _IHARQ_STAGE18_BASE_ANNOTATE_R10 = (
        globals().get("_IHARQ_STAGE18_BASE_ANNOTATE_R9")
        or globals().get("_R8_PREV_ANNOTATE")
        or _annotate_stage_successor
    )
_R10_PREV_ANNOTATE = _IHARQ_STAGE18_BASE_ANNOTATE_R10

def _r8_annotate_stage_successor(sid, semantic=None):
    ledger = _R10_PREV_ANNOTATE(sid, semantic)
    if str(sid) == "18":
        lp = Path(RUN_ROOT) / "stage_ledger" / "stage_18.json"
        d = _r8_load_json(lp)
        d.setdefault("runtime_successor", {})
        d["runtime_successor"].update({
            "id": R8_STAGE18_SUCCESSOR_ID,
            "scientific_configuration_changed": True,
            "scientific_change_scope":
                "A4_AUTHOR_SOURCE_REFIT_PLUS_SINGLE_REPEAT_PLUS_DEEP_ANCHOR_BUDGET_REDUCTION;R10_3_TRANSPORT_FIXES",
            "policy_sha256": R8_STAGE18_POLICY_SHA256,
            "completed_A0_artifacts_mutated": False,
            "test_outcome_influence": "PROHIBITED",
            "protocol_deviation_from_buildbook": True,
            "protocol_deviation_id": R10_STAGE18_REDUCTION["protocol_deviation_id"],
            "full_buildbook_replication_equivalence": False,
            "ablation_conditions_removed": False,
            "refit_repeat_indices_executed": R10_STAGE18_REDUCTION["executed_repeat_indices"],
            "deep_anchor_budget_ids": R10_STAGE18_REDUCTION["deep_anchor_budget_ids"],
            "deep_budget_grid_reduced": True,
            "multi_seed_stability_claim_authorized": False,
            "dense_deep_budget_curve_claim_authorized": False,
            "anchor_budget_deep_claim_authorized": True,
            "execution_transport_revision": "R10.3_DUAL_GPU_DIFFERENT_ABLATION_CELLS",
            "single_model_multi_gpu": False,
            "external_first_checkpoint_route": True,
        })
        _r8_pkg_atomic_json(lp, d)

        gp = Path(RUN_ROOT) / "gate_results" / "G18.json"
        if gp.is_file():
            g = _r8_load_json(gp)
            g.update({
                "runtime_successor_id": R8_STAGE18_SUCCESSOR_ID,
                "r8_author_source_aligned_A4_refit": True,
                "r10_resource_constrained_anchor_budget_single_repeat": True,
                "policy_sha256": R8_STAGE18_POLICY_SHA256,
                "protocol_deviation_from_buildbook": True,
                "protocol_deviation_id": R10_STAGE18_REDUCTION["protocol_deviation_id"],
                "full_buildbook_replication_equivalence": False,
                "ablation_conditions_removed": False,
                "deep_budget_grid_reduced": True,
                "deep_anchor_budget_ids": R10_STAGE18_REDUCTION["deep_anchor_budget_ids"],
                "multi_seed_stability_claim_authorized": False,
                "dense_deep_budget_curve_claim_authorized": False,
                "anchor_budget_deep_claim_authorized": True,
                "execution_transport_revision": "R10.3_DUAL_GPU_DIFFERENT_ABLATION_CELLS",
                "single_model_multi_gpu": False,
                "external_first_checkpoint_route": True,
            })
            _r8_pkg_atomic_json(gp, g)
        return d
    return ledger

_annotate_stage_successor = _r8_annotate_stage_successor

P02_RUNTIME_SUCCESSOR_ID = R8_STAGE18_SUCCESSOR_ID
STATE["runtime_successor_id"] = R8_STAGE18_SUCCESSOR_ID
STATE["stage18_r8_author_alignment_policy"] = copy.deepcopy(R8_STAGE18_POLICY)
STATE["stage18_r10_resource_constrained_policy"] = copy.deepcopy(R8_STAGE18_POLICY)
STATE["stage18_r10_3_execution_transport"] = {
    "revision": "R10.3_DUAL_GPU_DIFFERENT_ABLATION_CELLS",
    "dual_gpu_available": bool(_R10_3_DUAL_GPU_AVAILABLE),
    "gpu_names": list(_R10_3_GPU_NAMES),
    "single_model_multi_gpu": False,
    "external_first_checkpoint_route": True,
    "scientific_policy_sha256": R8_STAGE18_POLICY_SHA256,
}


# ---- Exact accepted R10.5 low-calibration resolver ----
# -----------------------------------------------------------------------------
print("\n[2/8] Installing canonical low-cal budget resolver at A0/A4 runtime boundary...")

import iharq.layer2_decoders.data as _r10_5_data

_R10_5_CANONICAL_LOW_CAL_RE = re.compile(
    r"^P01-L1-LOW-CAL-OFFICIAL-R2:([0-9]+)_PER_CLASS$"
)

_R10_5_FREEZE = SESSION.ctx.state["runtime_freeze"]
_R10_5_ALLOWED_BUDGETS = tuple(
    sorted(
        int(x)
        for x in _R10_5_FREEZE["budgets"]["per_class"]
    )
)
_R10_5_EXPECTED_ALLOWED_BUDGETS = (1, 2, 4, 8, 16, 32)

if _R10_5_ALLOWED_BUDGETS != _R10_5_EXPECTED_ALLOWED_BUDGETS:
    raise RuntimeError(
        "R10_5_FROZEN_LOW_CAL_BUDGET_SET_MISMATCH:"
        + repr(_R10_5_ALLOWED_BUDGETS)
    )

def _r10_5_resolve_per_class_budget(budget):
    text = str(budget)

    if text == "FULL_TRAIN":
        return None

    m = _R10_5_CANONICAL_LOW_CAL_RE.fullmatch(text)
    if m is not None:
        value = int(m.group(1))
    elif text.isdigit():
        # Kept only for fixture/backward compatibility.
        value = int(text)
    else:
        raise ValueError(
            "R10_5_UNRECOGNIZED_LOW_CAL_BUDGET_ID:"
            + text
        )

    if value not in _R10_5_ALLOWED_BUDGETS:
        raise ValueError(
            "R10_5_LOW_CAL_BUDGET_NOT_IN_FROZEN_AUTHORITY:"
            f"{text}:{value}"
        )

    return value

_R10_5_CORE = SESSION.ctx.state["core"]
_R10_5_A4 = SESSION.ctx.state["a4"]
_R10_5_BUDGET_SEED = int(
    _R10_5_FREEZE["budgets"]["seed"]
)

# Compute the immutable memberships once and share them read-only. This removes
# repeated membership recomputation without changing membership identity.
_R10_5_FROZEN_MEMBERSHIPS = (
    _r10_5_data.frozen_budget_memberships(
        _R10_5_CORE,
        budgets=_R10_5_ALLOWED_BUDGETS,
        seed=_R10_5_BUDGET_SEED,
    )
)

def _r10_5_budget_member_ids(ds, budget):
    value = _r10_5_resolve_per_class_budget(budget)
    if value is None:
        return None

    key = (
        f"{ds}:budget-{value}-seed-{_R10_5_BUDGET_SEED}"
    )
    if key not in _R10_5_FROZEN_MEMBERSHIPS:
        raise RuntimeError(
            "R10_5_FROZEN_BUDGET_MEMBERSHIP_KEY_MISSING:"
            + key
        )
    return _R10_5_FROZEN_MEMBERSHIPS[key]

def _r10_5_scientific_train_rows(
    core_obj,
    ds,
    budget,
    freeze_obj,
):
    ids = _r10_5_budget_member_ids(ds, budget)
    if ids is None:
        return core_obj.rows(
            dataset_id=ds,
            role="train",
        )
    return [
        r
        for r in core_obj.rows(
            dataset_id=ds,
            role="calibration",
        )
        if r["event_id"] in ids
    ]

def _r10_5_a4_train_rows(
    a4_obj,
    core_obj,
    ds,
    budget,
    freeze_obj,
):
    ids = _r10_5_budget_member_ids(ds, budget)
    if ids is None:
        return a4_obj.rows(
            dataset_id=ds,
            role="train",
        )
    return [
        r
        for r in a4_obj.rows(
            dataset_id=ds,
            role="calibration",
        )
        if r["event_id"] in ids
    ]

# Persist original package references exactly once for provenance/debugging.
if not hasattr(_r8_sci, "_iharq_r10_5_parent_train_rows"):
    _r8_sci._iharq_r10_5_parent_train_rows = _r8_sci.train_rows
if not hasattr(_r8_a4, "_iharq_r10_5_parent_train"):
    _r8_a4._iharq_r10_5_parent_train = _r8_a4._train

_r8_sci.train_rows = _r10_5_scientific_train_rows
_r8_a4._train = _r10_5_a4_train_rows

# Full regression-test of the already-accepted Stage09 parser boundary before
# touching any Stage18 failure evidence.
_R10_5_BUDGET_PARSER_TESTS = {}
for _b in [
    *[
        f"P01-L1-LOW-CAL-OFFICIAL-R2:{n}_PER_CLASS"
        for n in _R10_5_ALLOWED_BUDGETS
    ],
    "FULL_TRAIN",
]:
    _R10_5_BUDGET_PARSER_TESTS[_b] = (
        _r10_5_resolve_per_class_budget(_b)
    )

# Must fail closed on an unauthorized canonical budget.
try:
    _r10_5_resolve_per_class_budget(
        "P01-L1-LOW-CAL-OFFICIAL-R2:3_PER_CLASS"
    )
except ValueError:
    _R10_5_UNAUTHORIZED_BUDGET_REJECTION = "PASS"
else:
    raise RuntimeError(
        "R10_5_UNAUTHORIZED_BUDGET_WAS_NOT_REJECTED"
    )

_R10_5_MEMBERSHIP_TESTS = []
_R10_5_DATASETS = sorted(
    {
        str(c["dataset_id"])
        for c in _R10_5_PLAN
    }
)

for _ds in _R10_5_DATASETS:
    # FULL_TRAIN must remain exactly the frozen train role in both A0 and A4.
    _full_core = _r8_sci.train_rows(
        _R10_5_CORE,
        _ds,
        "FULL_TRAIN",
        _R10_5_FREEZE,
    )
    _expected_full_core = _R10_5_CORE.rows(
        dataset_id=_ds,
        role="train",
    )
    if [
        r["window_record_id"]
        for r in _full_core
    ] != [
        r["window_record_id"]
        for r in _expected_full_core
    ]:
        raise RuntimeError(
            "R10_5_FULL_TRAIN_CORE_ROW_IDENTITY_CHANGED:"
            + _ds
        )

    _full_a4 = _r8_a4._train(
        _R10_5_A4,
        _R10_5_CORE,
        _ds,
        "FULL_TRAIN",
        _R10_5_FREEZE,
    )
    _expected_full_a4 = _R10_5_A4.rows(
        dataset_id=_ds,
        role="train",
    )
    if [
        r["window_record_id"]
        for r in _full_a4
    ] != [
        r["window_record_id"]
        for r in _expected_full_a4
    ]:
        raise RuntimeError(
            "R10_5_FULL_TRAIN_A4_ROW_IDENTITY_CHANGED:"
            + _ds
        )

    for _n in _R10_5_ALLOWED_BUDGETS:
        _canonical = (
            f"P01-L1-LOW-CAL-OFFICIAL-R2:{_n}_PER_CLASS"
        )
        _key = (
            f"{_ds}:budget-{_n}-seed-{_R10_5_BUDGET_SEED}"
        )
        if _key not in _R10_5_FROZEN_MEMBERSHIPS:
            raise RuntimeError(
                "R10_5_MEMBERSHIP_KEY_MISSING:"
                + _key
            )

        _expected_ids = set(
            _R10_5_FROZEN_MEMBERSHIPS[_key]
        )

        _observed_core = _r8_sci.train_rows(
            _R10_5_CORE,
            _ds,
            _canonical,
            _R10_5_FREEZE,
        )
        _observed_core_ids = {
            r["event_id"]
            for r in _observed_core
        }
        if _observed_core_ids != _expected_ids:
            raise RuntimeError(
                "R10_5_LOW_CAL_CORE_MEMBERSHIP_CHANGED:"
                f"{_ds}:{_n}"
            )
        if len(_observed_core) != 2 * int(_n):
            raise RuntimeError(
                "R10_5_LOW_CAL_CORE_CARDINALITY_MISMATCH:"
                f"{_ds}:{_n}:{len(_observed_core)}"
            )

        _label_counts = Counter(
            r["label"]
            for r in _observed_core
        )
        if _label_counts != Counter({
            "left_hand": int(_n),
            "right_hand": int(_n),
        }):
            raise RuntimeError(
                "R10_5_LOW_CAL_CLASS_BALANCE_MISMATCH:"
                f"{_ds}:{_n}:{dict(_label_counts)}"
            )
        if any(
            r["role"] != "calibration"
            for r in _observed_core
        ):
            raise RuntimeError(
                "R10_5_LOW_CAL_ROLE_FIREWALL_VIOLATION:"
                f"{_ds}:{_n}"
            )

        _observed_a4 = _r8_a4._train(
            _R10_5_A4,
            _R10_5_CORE,
            _ds,
            _canonical,
            _R10_5_FREEZE,
        )
        _observed_a4_ids = {
            r["event_id"]
            for r in _observed_a4
        }
        if _observed_a4_ids != _expected_ids:
            raise RuntimeError(
                "R10_5_A4_LOW_CAL_PARENT_MEMBERSHIP_CHANGED:"
                f"{_ds}:{_n}"
            )
        if len(_observed_a4) != 2 * int(_n):
            raise RuntimeError(
                "R10_5_A4_LOW_CAL_CARDINALITY_MISMATCH:"
                f"{_ds}:{_n}:{len(_observed_a4)}"
            )
        if any(
            r["role"] != "calibration"
            for r in _observed_a4
        ):
            raise RuntimeError(
                "R10_5_A4_LOW_CAL_ROLE_FIREWALL_VIOLATION:"
                f"{_ds}:{_n}"
            )

        _R10_5_MEMBERSHIP_TESTS.append({
            "dataset_id": _ds,
            "budget_id": _canonical,
            "per_class": int(_n),
            "core_rows": len(_observed_core),
            "a4_rows": len(_observed_a4),
            "event_membership": "EXACT_MATCH",
            "class_balance": dict(_label_counts),
            "role": "calibration",
            "status": "PASS",
        })

if len(_R10_5_MEMBERSHIP_TESTS) != (
    len(_R10_5_DATASETS)
    * len(_R10_5_ALLOWED_BUDGETS)
):
    raise RuntimeError(
        "R10_5_MEMBERSHIP_TEST_COUNT_MISMATCH:"
        + str(len(_R10_5_MEMBERSHIP_TESTS))
    )

print(
    "  canonical parser tests:",
    json.dumps(
        _R10_5_BUDGET_PARSER_TESTS,
        sort_keys=True,
    ),
)
print(
    "  unauthorized budget rejection:",
    _R10_5_UNAUTHORIZED_BUDGET_REJECTION,
)
print(
    "  exact membership/cardinality/class-balance tests:",
    len(_R10_5_MEMBERSHIP_TESTS),
)

# -----------------------------------------------------------------------------

# ---- Exact accepted R10.5 same-GPU RNG science patches ----
# 4. Same-GPU concurrency: dedicated streams + virtualized CUDA RNG state.
# -----------------------------------------------------------------------------
print("\n[4/8] Installing same-GPU concurrent-fit RNG isolation...")

_R10_5_GPU_RNG_LOCKS = {
    i: threading.RLock()
    for i in range(
        min(2, torch.cuda.device_count())
    )
}

def _r10_5_device_index(device=None):
    if device is None:
        return int(
            getattr(
                _R10_3_GPU_TLS,
                "gpu_id",
                0,
            )
        )

    text = str(device)
    if text == "cuda":
        return int(
            getattr(
                _R10_3_GPU_TLS,
                "gpu_id",
                0,
            )
        )
    if text.startswith("cuda:"):
        return int(text.split(":", 1)[1])
    return 0

# Replace the R10 seed setter with a lock-aware equivalent. CPU-side calls are
# already enclosed by the model-init lock; only CUDA generator mutation needs
# an additional per-device lock under multi-slot execution.
def _r8_seed(seed):
    seed = int(seed)

    random.seed(seed)
    np.random.seed(
        seed % (2**32 - 1)
    )
    torch.random.default_generator.manual_seed(
        seed
    )

    if torch.cuda.is_available():
        idx = int(
            getattr(
                _R10_3_GPU_TLS,
                "gpu_id",
                0,
            )
        )
        idx = min(
            max(0, idx),
            torch.cuda.device_count() - 1,
        )
        lock = _R10_5_GPU_RNG_LOCKS.get(
            idx,
            threading.RLock(),
        )
        with lock:
            with torch.cuda.device(idx):
                torch.cuda.manual_seed(seed)

    try:
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    except Exception:
        pass

def _r10_5_init_virtual_cuda_rng(seed, device):
    if not torch.cuda.is_available():
        return None

    idx = _r10_5_device_index(device)
    lock = _R10_5_GPU_RNG_LOCKS[idx]

    with lock:
        with torch.cuda.device(idx):
            previous = torch.cuda.get_rng_state(idx)
            torch.cuda.manual_seed(int(seed))
            local = torch.cuda.get_rng_state(idx)
            torch.cuda.set_rng_state(
                previous,
                idx,
            )

    return local

def _r10_5_rng_forward(
    rng_state,
    device,
    fn,
):
    """
    Execute one stochastic training forward using the calling fit's virtual
    CUDA RNG state.

    CUDA random kernels reserve their Philox seed/offset at launch; the global
    device generator is restored immediately after launch bookkeeping, while
    the actual kernels remain on the worker's dedicated CUDA stream.
    """
    if (
        rng_state is None
        or not torch.cuda.is_available()
    ):
        return fn(), rng_state

    idx = _r10_5_device_index(device)
    lock = _R10_5_GPU_RNG_LOCKS[idx]

    with lock:
        with torch.cuda.device(idx):
            previous = torch.cuda.get_rng_state(idx)
            torch.cuda.set_rng_state(
                rng_state,
                idx,
            )
            out = fn()
            next_state = torch.cuda.get_rng_state(idx)
            torch.cuda.set_rng_state(
                previous,
                idx,
            )

    return out, next_state

# Patch EEGNet fit with the SAME R7H7 recipe, adding only virtual RNG handling
# around the stochastic training forward.
def _r10_5_eegnet_fit(self, X, y, **kw):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)
    Xv = np.asarray(
        kw.get("X_val"),
        dtype=np.float32,
    )
    yv = np.asarray(
        kw.get("y_val"),
        dtype=np.int64,
    )

    if (
        set(np.unique(y).tolist()) != {0, 1}
        or set(np.unique(yv).tolist()) != {0, 1}
    ):
        _r8_fail(
            "R8_EEGNET_BINARY_ROLE_REQUIRED"
        )

    device = str(
        kw.get("device")
        or (
            f"cuda:{_r10_5_device_index()}"
            if torch.cuda.is_available()
            else "cpu"
        )
    )
    requested = int(
        kw.get("batch_size") or 64
    )
    ladder = [
        b
        for b in R8_EEGNET_RECIPE["batch_ladder"]
        if b <= requested
    ] or [16]

    class_weights = kw.get("class_weights")
    last_oom = None

    for bs in ladder:
        try:
            with _R10_3_MODEL_INIT_LOCK:
                _r8_seed(self.seed)
                self.model = self._build()
                self.input_statistics = (
                    self.model.fit_input_statistics(X)
                )
                self.model = self.model.to(device)

            self.device = device

            rng_state = _r10_5_init_virtual_cuda_rng(
                self.seed,
                device,
            )

            cw = (
                None
                if class_weights is None
                else torch.tensor(
                    class_weights,
                    dtype=torch.float32,
                    device=device,
                )
            )

            loss_fn = torch.nn.CrossEntropyLoss(
                weight=cw
            )
            opt = torch.optim.Adam(
                self.model.parameters(),
                lr=1e-3,
                weight_decay=0.0,
            )
            accum = max(
                1,
                int(
                    math.ceil(
                        64 / int(bs)
                    )
                ),
            )

            best_state = None
            best_metrics = None
            best_epoch = None
            bad = 0
            epochs_completed = 0

            ds = torch.utils.data.TensorDataset(
                torch.from_numpy(X),
                torch.from_numpy(y),
            )

            for epoch in range(
                1,
                int(
                    R8_EEGNET_RECIPE[
                        "max_epochs"
                    ]
                )
                + 1,
            ):
                gen = torch.Generator()
                gen.manual_seed(
                    self.seed + epoch
                )

                loader = (
                    torch.utils.data.DataLoader(
                        ds,
                        batch_size=int(bs),
                        shuffle=True,
                        generator=gen,
                        num_workers=0,
                        pin_memory=False,
                        drop_last=False,
                    )
                )

                self.model.train()
                opt.zero_grad(
                    set_to_none=True
                )
                pending = 0

                for xb, yb in loader:
                    xb = xb.to(device)
                    yb = yb.to(device)

                    z, rng_state = (
                        _r10_5_rng_forward(
                            rng_state,
                            device,
                            lambda xb=xb:
                                self.model(xb),
                        )
                    )

                    loss = loss_fn(z, yb)
                    if not torch.isfinite(loss):
                        _r8_fail(
                            "R8_EEGNET_NONFINITE_TRAIN_LOSS"
                        )

                    (loss / accum).backward()
                    pending += 1

                    if pending % accum == 0:
                        opt.step()
                        opt.zero_grad(
                            set_to_none=True
                        )

                if pending % accum:
                    opt.step()
                    opt.zero_grad(
                        set_to_none=True
                    )

                scores = self.scores(
                    Xv,
                    device=device,
                    batch_size=128,
                )
                pred = np.argmax(
                    scores,
                    axis=1,
                )
                vm = _r8_metrics.evaluate(
                    yv,
                    pred,
                    scores,
                    self.score_type,
                )

                key = (
                    float(vm["BACC"]),
                    float(vm["F1_MACRO"]),
                    -epoch,
                )
                old = (
                    (-np.inf, -np.inf, -10**9)
                    if best_metrics is None
                    else (
                        float(
                            best_metrics["BACC"]
                        ),
                        float(
                            best_metrics["F1_MACRO"]
                        ),
                        -int(best_epoch),
                    )
                )

                if key > old:
                    best_metrics = dict(vm)
                    best_epoch = epoch
                    best_state = {
                        k:
                            v.detach()
                            .cpu()
                            .clone()
                        for k, v
                        in self.model.state_dict().items()
                    }
                    bad = 0
                else:
                    bad += 1

                epochs_completed = epoch
                if bad >= int(
                    R8_EEGNET_RECIPE[
                        "patience"
                    ]
                ):
                    break

            if best_state is None:
                _r8_fail(
                    "R8_EEGNET_NO_BEST_STATE"
                )

            self.model.load_state_dict(
                best_state,
                strict=True,
            )
            self.actual_batch_size = int(bs)
            self.gradient_accumulation = int(
                accum
            )
            self.r8_training_provenance = {
                "recipe_id":
                    R8_EEGNET_RECIPE[
                        "recipe_id"
                    ],
                "policy_sha256":
                    R8_STAGE18_POLICY_SHA256,
                "best_epoch":
                    int(best_epoch),
                "epochs_completed":
                    int(epochs_completed),
                "best_validation":
                    best_metrics,
                "class_weights":
                    class_weights,
                "test_set_used_for_selection":
                    False,
                "same_gpu_virtual_cuda_rng":
                    True,
            }
            return self

        except RuntimeError as exc:
            if (
                "out of memory"
                not in str(exc).lower()
            ):
                raise

            last_oom = (
                f"{type(exc).__name__}:"
                f"{str(exc)[:300]}"
            )
            self.model = None
            gc.collect()

            if torch.cuda.is_available():
                idx = _r10_5_device_index(
                    device
                )
                with torch.cuda.device(idx):
                    torch.cuda.empty_cache()

    raise ResourceWarning(
        "R8_EEGNET_RESOURCE_BLOCKED_ALL_AUTHOR_BATCHES:"
        + str(last_oom)
    )

R8AuthorCenteredEEGNetAdapter.fit = (
    _r10_5_eegnet_fit
)

# Patch external training with the SAME accepted R6 recipe semantics; only the
# stochastic training forward gets virtual RNG isolation.
def _r8_train_external(
    outer,
    X,
    y,
    Xv,
    yv,
    recipe,
    class_weights,
    seed,
    device,
):
    plugin = outer.plugin
    raw_n_times = int(X.shape[-1])
    n_chans = int(X.shape[1])
    branch = str(recipe["branch"])

    with _R10_3_MODEL_INIT_LOCK:
        _r8_seed(seed)

        Xp = np.asarray(
            plugin._prepare_and_validate(
                np.asarray(
                    X,
                    dtype=np.float32,
                )
            ),
            dtype=np.float32,
        )
        Xvp = np.asarray(
            plugin._prepare_and_validate(
                np.asarray(
                    Xv,
                    dtype=np.float32,
                )
            ),
            dtype=np.float32,
        )

        plugin.seed = int(seed)
        plugin.raw_shape = (
            n_chans,
            raw_n_times,
        )
        plugin.prepared_shape = tuple(
            map(
                int,
                Xp.shape[1:],
            )
        )

        plugin.model = (
            plugin._build_model(
                n_chans,
                raw_n_times,
            )
            .to(device)
        )

    plugin.device = str(device)
    model = plugin.model
    bs = int(recipe["batch_size"])

    rng_state = _r10_5_init_virtual_cuda_rng(
        seed,
        device,
    )

    cw = (
        None
        if class_weights is None
        else torch.tensor(
            class_weights,
            dtype=torch.float32,
            device=device,
        )
    )

    loss_fn = torch.nn.CrossEntropyLoss(
        weight=cw,
        label_smoothing=float(
            recipe.get(
                "label_smoothing",
                0.0,
            )
        ),
    )

    if branch == "SSL-CBRAMOD":
        body = []
        head = []

        for name, p in model.named_parameters():
            (
                head
                if _r8_head_parameter(
                    name,
                    branch,
                )
                else body
            ).append(p)

        if not body or not head:
            _r8_fail(
                "R8_CBRAMOD_HEAD_BODY_SPLIT_FAILED"
            )

        opt = torch.optim.AdamW(
            [
                {
                    "params": body,
                    "lr": float(
                        recipe["body_lr"]
                    ),
                },
                {
                    "params": head,
                    "lr": float(
                        recipe["head_lr"]
                    ),
                },
            ],
            lr=float(
                recipe["body_lr"]
            ),
            weight_decay=float(
                recipe.get(
                    "weight_decay",
                    0.0,
                )
            ),
        )
    else:
        opt = torch.optim.Adam(
            model.parameters(),
            lr=float(recipe["lr"]),
            weight_decay=float(
                recipe.get(
                    "weight_decay",
                    0.0,
                )
            ),
        )

    ds = torch.utils.data.TensorDataset(
        torch.from_numpy(Xp),
        torch.from_numpy(
            np.asarray(
                y,
                dtype=np.int64,
            )
        ),
    )

    steps = max(
        1,
        int(
            math.ceil(
                len(ds) / bs
            )
        ),
    )

    sched = None
    if (
        str(
            recipe.get(
                "scheduler",
                "constant",
            )
        ).lower()
        == "cosine"
    ):
        sched = (
            torch.optim.lr_scheduler
            .CosineAnnealingLR(
                opt,
                T_max=max(
                    1,
                    int(
                        recipe[
                            "max_epochs"
                        ]
                    )
                    * steps,
                ),
                eta_min=float(
                    recipe.get(
                        "eta_min",
                        1e-6,
                    )
                ),
            )
        )

    best_bacc = -np.inf
    best_state = None
    best_metrics = None
    best_epoch = 0
    bad = 0
    started = time.time()
    history = []

    for epoch in range(
        1,
        int(recipe["max_epochs"]) + 1,
    ):
        gen = torch.Generator()
        gen.manual_seed(
            int(seed) + epoch
        )

        loader = torch.utils.data.DataLoader(
            ds,
            batch_size=bs,
            shuffle=True,
            generator=gen,
            num_workers=0,
            pin_memory=True,
            drop_last=False,
        )

        model.train()
        losses = []

        for xb, yb in loader:
            xb = xb.to(
                device,
                non_blocking=True,
            )
            yb = yb.to(
                device,
                non_blocking=True,
            )
            opt.zero_grad(
                set_to_none=True
            )

            z, rng_state = (
                _r10_5_rng_forward(
                    rng_state,
                    device,
                    lambda xb=xb:
                        plugin._forward_logits(
                            xb
                        ),
                )
            )

            if isinstance(
                z,
                (tuple, list),
            ):
                z = z[-1]

            loss = loss_fn(
                z,
                yb,
            )

            if not torch.isfinite(loss):
                _r8_fail(
                    "R8_EXTERNAL_NONFINITE_TRAIN_LOSS",
                    {
                        "branch": branch,
                        "epoch": epoch,
                    },
                )

            loss.backward()

            gcval = recipe.get(
                "grad_clip"
            )
            if (
                gcval is not None
                and float(gcval) > 0
            ):
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    float(gcval),
                )

            opt.step()

            if sched is not None:
                sched.step()

            losses.append(
                float(
                    loss.detach()
                    .cpu()
                    .item()
                )
            )

        _, _, vm = (
            _r8_eval_external_adaptive(
                plugin,
                Xvp,
                yv,
                device,
            )
        )

        if float(vm["BACC"]) > float(
            best_bacc
        ):
            best_bacc = float(
                vm["BACC"]
            )
            best_epoch = int(epoch)
            best_metrics = copy.deepcopy(vm)
            best_state = {
                k:
                    v.detach()
                    .cpu()
                    .clone()
                for k, v
                in model.state_dict().items()
            }
            bad = 0
        else:
            bad += 1

        history.append({
            "epoch": epoch,
            "validation": vm,
            "best_epoch": best_epoch,
            "bad_epochs": bad,
        })

        if bad >= int(
            recipe["patience"]
        ):
            break

    if best_state is None:
        _r8_fail(
            "R8_EXTERNAL_NO_BEST_STATE",
            branch,
        )

    model.load_state_dict(
        best_state,
        strict=True,
    )
    plugin.model.eval()
    plugin.device = str(device)
    plugin.actual_batch_size = bs
    plugin.gradient_accumulation = 1

    return {
        "recipe_id":
            recipe["recipe_id"],
        "best_epoch":
            best_epoch,
        "epochs_completed":
            len(history),
        "best_validation":
            best_metrics,
        "elapsed_seconds":
            float(
                time.time() - started
            ),
        "prepared_train_shape":
            list(Xp.shape),
        "prepared_validation_shape":
            list(Xvp.shape),
        "test_set_used_for_selection":
            False,
        "same_gpu_virtual_cuda_rng":
            True,
    }

print("  CUDA RNG virtualization: INSTALLED")



# =============================================================================
# POST-RUNTIME-REPLAY IMMUTABILITY / CONTRACT CHECKS
# =============================================================================
_required_stage18s_symbols = [
    "_R10_PARENT_EXECUTE_ROLE",
    "_R10_3_GPU_TLS",
    "_R10_3_DATA_IO_LOCK",
    "_R10_3_MODEL_INIT_LOCK",
    "_R10_5_GPU_RNG_LOCKS",
    "_r10_5_rng_forward",
    "_r8_a4",
    "_r8_pkg_atomic_json",
    "_r8_pkg_atomic_jsonl",
    "R8_STAGE18_POLICY_SHA256",
    "R8_STAGE18_SUCCESSOR_ID",
]
_missing = [x for x in _required_stage18s_symbols if x not in globals()]
if _missing:
    raise RuntimeError(
        "POST_STAGE18_RUNTIME_REPLAY_REQUIRED_SYMBOLS_MISSING:"
        + ",".join(_missing)
    )

_changed = []
for _p_str, _sha in _POST18_RUNTIME_PROTECTED.items():
    _p = Path(_p_str)
    if not _p.is_file() or _p17_sha(_p) != _sha:
        _changed.append(_p_str)
if _changed:
    raise RuntimeError(
        "POST_STAGE18_RUNTIME_REPLAY_MUTATED_CANONICAL_STAGE18:"
        + json.dumps(_changed[:30])
    )

# Existing partial Stage18S evidence must also remain byte-identical.
_s18s_changed = []
for _p_str, _sha in _POST18_S18S_BEFORE.items():
    _p = Path(_p_str)
    if not _p.is_file() or _p17_sha(_p) != _sha:
        _s18s_changed.append(_p_str)
if _s18s_changed:
    raise RuntimeError(
        "POST_STAGE18_RUNTIME_REPLAY_MUTATED_PARTIAL_STAGE18S:"
        + json.dumps(_s18s_changed[:30])
    )

# No scientific execution has occurred.
_POST18_RUNTIME_RECEIPT = {
    "artifact_id": "P02-POST-STAGE18-ACCEPTED-STAGE18S-RUNTIME-REPLAY-R1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "PASS",
    "canonical_stage18_accepted": True,
    "canonical_stage18_terminal_count": 1218,
    "canonical_stage18_mutated": False,
    "partial_stage18S_files_preserved": len(_POST18_S18S_BEFORE),
    "partial_stage18S_mutated": False,
    "scientific_stage_rerun": False,
    "model_training_performed": False,
    "prediction_regenerated": False,
    "stage18_policy_sha256": R8_STAGE18_POLICY_SHA256,
    "low_cal_budget_parser_restored": True,
    "same_gpu_virtual_cuda_rng_restored": True,
    "next_action": "RUN_STAGE18S_R1_2_INSTALL_FREEZE_THEN_EXECUTION",
}
_POST18_RUNTIME_RECEIPT_PATH = (
    Path(RUNTIME_ROOT)
    / "diagnostics"
    / "runtime_successor"
    / "post_stage18_accepted_stage18S_runtime_replay_R1"
    / "P02_POST_STAGE18_STAGE18S_RUNTIME_REPLAY_R1.json"
)
_p17_atomic_json(
    _POST18_RUNTIME_RECEIPT_PATH,
    _POST18_RUNTIME_RECEIPT,
)

print("\n" + "=" * 112)
print("POST-STAGE18 ACCEPTED + PARTIAL-STAGE18S REHYDRATION — PASS")
print("=" * 112)
print(json.dumps(_POST18_RUNTIME_RECEIPT, indent=2, default=str))
print("\nNEXT:")
print("  1. Run Stage18S R1.2 INSTALL/FREEZE")
print("  2. Run Stage18S R1.2 12-WORKER EXECUTION")
print("  3. Run Stage18S R1.2 ANALYZE/CLOSE")
print("\nTRUSTED FINAL MARKER:")
print("IHARQ_P02_POST_STAGE18_ACCEPTED_STAGE18S_RUNTIME_READY_R1")


IHARQ P02 — POST-STAGE18-ACCEPTED FRESH-KERNEL REHYDRATION R1
{
  "disk_boundary": "POST_STAGE18_PRE_STAGE19",
  "accepted_stages_13_18": {
    "13": {
      "ledger_status": "SUCCESS",
      "gate_status": "PASS"
    },
    "14": {
      "ledger_status": "SUCCESS",
      "gate_status": "PASS"
    },
    "15": {
      "ledger_status": "SUCCESS",
      "gate_status": "PASS"
    },
    "16": {
      "ledger_status": "SUCCESS",
      "gate_status": "PASS"
    },
    "17": {
      "ledger_status": "SUCCESS",
      "gate_status": "PASS"
    },
    "18": {
      "ledger_status": "SUCCESS",
      "gate_status": "PASS"
    }
  },
  "stage18_accepted": true,
  "scientific_stage_rerun": false
}
Executing verified R6R3 rehydrator with one in-memory boundary amendment:
  source = /kaggle/working/_IHARQ_CONTINUATION_META_20260812T005832Z/resume_scripts/IHARQ_P02_R6R3_REHYDRATE_EXACT.py
  sha256 = 58602da3549728d8a812e83438820c1843fdc28b79b6b63e9339f512a7a4870d
  allowed accepted downstream = ['13',

## 3 — Stage 18S R1.3 balanced sensitivity continuation

Scientific plan: additional MR01/MR02 at 1/8/FULL plus MR00 at 4/16/32 for deep C1/C2/C3. Transport ceiling: six independent slots per T4, twelve total, with adaptive resource admission/backoff.

In [4]:
# =============================================================================
# IHARQ P02/L2 — STAGE 18S BALANCED SENSITIVITY CONTINUATION R1 — INSTALL/FREEZE
#
# CURRENT-LIVE-SESSION FIRST:
#   Run this AFTER canonical Stage18 R10.5.1 has G18 PASS, in the SAME live
#   Kaggle kernel in which R10.5.1 is installed.
#
# Canonical Stage18 is immutable. This creates a separate supplemental Store.
# =============================================================================

from __future__ import annotations

from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import copy
import hashlib
import json
import os
import shutil

from iharq.layer2_decoders.store import Store

print("=" * 118)
print("STAGE 18S — BALANCED SENSITIVITY CONTINUATION R1 — INSTALL / SCIENTIFIC FREEZE")
print("=" * 118)

S18S_ID = "P02-STAGE18S-BALANCED-SENSITIVITY-R1"
S18S_REVISION = "R1-THREE-REPEAT-ANCHORS-PLUS-4-16-32-MR00"
S18S_TRANSPORT_REVISION = "R1.3-SIX-SLOTS-PER-GPU-ADAPTIVE-RAMPED-BACKOFF"
S18S_EXPECTED_CELLS = 162
S18S_EXPECTED_GROUPS = 54
S18S_EXPECTED_MEMBER_RECEIPTS = 216
S18S_EXPECTED_TERMINALS = {
    "SUCCESS": 135,
    "INPUT_INCOMPATIBLE": 27,
}
S18S_EXPECTED_MEMBER_STATUSES = {
    "SUCCESS": 189,
    "INPUT_INCOMPATIBLE": 27,
}

# -----------------------------------------------------------------------------
# 0. Require the live R10.5 scientific/runtime layer. Do NOT reconstruct a
#    different runtime silently after canonical Stage18 has already closed.
# -----------------------------------------------------------------------------
_required = [
    "SESSION", "STATE", "STORE", "RUN_ROOT", "RUNTIME_ROOT",
    "_R10_PARENT_EXECUTE_ROLE", "_R10_3_GPU_TLS",
    "_R10_5_GPU_RNG_LOCKS", "_r10_5_rng_forward",
    "_R10_3_DATA_IO_LOCK", "_R10_3_MODEL_INIT_LOCK",
    "_r8_a4", "_r8_pkg_atomic_json", "_r8_pkg_atomic_jsonl",
    "R8_STAGE18_POLICY_SHA256", "R8_STAGE18_SUCCESSOR_ID",
]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError(
        "STAGE18S_REQUIRES_SAME_LIVE_R10_5_KERNEL:"
        + ",".join(_missing)
    )

if not SESSION.runner.accepted("18"):
    raise RuntimeError("STAGE18S_REQUIRES_ACCEPTED_CANONICAL_STAGE18")

RUN_ROOT = Path(RUN_ROOT).resolve()
RUNTIME_ROOT = Path(RUNTIME_ROOT).resolve()
STORE_ROOT = Path(STORE.root).resolve()

_g18_path = RUN_ROOT / "gate_results" / "G18.json"
_s18_ledger_path = RUN_ROOT / "stage_ledger" / "stage_18.json"
_a4_completion_path = STORE_ROOT / "analysis_inputs" / "a4_completion.json"

for _p in (_g18_path, _s18_ledger_path, _a4_completion_path):
    if not _p.is_file():
        raise RuntimeError("STAGE18S_CANONICAL_EVIDENCE_MISSING:" + str(_p))

_g18 = json.loads(_g18_path.read_text(encoding="utf-8"))
_s18_ledger = json.loads(_s18_ledger_path.read_text(encoding="utf-8"))
_a4_completion = json.loads(_a4_completion_path.read_text(encoding="utf-8"))

if _g18.get("status") != "PASS":
    raise RuntimeError("STAGE18S_REQUIRES_G18_PASS")
if _s18_ledger.get("status") != "SUCCESS":
    raise RuntimeError("STAGE18S_REQUIRES_STAGE18_LEDGER_SUCCESS")
if _a4_completion.get("status") != "PASS":
    raise RuntimeError("STAGE18S_REQUIRES_A4_COMPLETION_PASS")

# The supplement should precede canonical downstream finalization in this
# notebook so the supplemental files travel with the final runtime bundle.
_downstream_accepted = {}
for _sid in ("18U", "19", "20", "21", "22", "23", "24"):
    try:
        _downstream_accepted[_sid] = bool(SESSION.runner.accepted(_sid))
    except Exception:
        _downstream_accepted[_sid] = False

if any(_downstream_accepted.values()):
    raise RuntimeError(
        "STAGE18S_MUST_RUN_BEFORE_18U_AND_STAGES19_24:"
        + json.dumps(_downstream_accepted, sort_keys=True)
    )

# -----------------------------------------------------------------------------
# 1. Protect canonical Stage18 evidence by hashing every canonical A4 terminal
#    plus the Stage18/G18/closure files. The execution and analysis cells check
#    that all hashes remain byte-identical.
# -----------------------------------------------------------------------------
def _s18s_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

S18S_CANONICAL_PROTECTED_HASHES = {}

for _p in (_g18_path, _s18_ledger_path, _a4_completion_path):
    S18S_CANONICAL_PROTECTED_HASHES[str(_p)] = _s18s_sha256(_p)

_canonical_a4_terminals = sorted(
    (STORE_ROOT / "run_cells").glob("P02-A4-*.json")
)
if len(_canonical_a4_terminals) != 1218:
    raise RuntimeError(
        "STAGE18S_EXPECTED_1218_CANONICAL_A4_TERMINALS:"
        + str(len(_canonical_a4_terminals))
    )

for _p in _canonical_a4_terminals:
    S18S_CANONICAL_PROTECTED_HASHES[str(_p)] = _s18s_sha256(_p)

def S18S_ASSERT_CANONICAL_IMMUTABLE():
    changed = []
    missing = []
    for p_str, expected in S18S_CANONICAL_PROTECTED_HASHES.items():
        p = Path(p_str)
        if not p.is_file():
            missing.append(p_str)
            continue
        observed = _s18s_sha256(p)
        if observed != expected:
            changed.append({
                "path": p_str,
                "expected": expected,
                "observed": observed,
            })
    if missing or changed:
        raise RuntimeError(
            "STAGE18S_CANONICAL_STAGE18_MUTATED:"
            + json.dumps(
                {"missing": missing[:20], "changed": changed[:20]},
                sort_keys=True,
            )
        )
    return True

# -----------------------------------------------------------------------------
# 2. Construct the fixed supplemental plan from the already-frozen canonical
#    1218 A4 plan. No new dataset/model/condition identity is invented.
# -----------------------------------------------------------------------------
S18S_ANCHOR_BUDGETS = {
    "P01-L1-LOW-CAL-OFFICIAL-R2:1_PER_CLASS",
    "P01-L1-LOW-CAL-OFFICIAL-R2:8_PER_CLASS",
    "FULL_TRAIN",
}
S18S_PROBE_BUDGETS = {
    "P01-L1-LOW-CAL-OFFICIAL-R2:4_PER_CLASS",
    "P01-L1-LOW-CAL-OFFICIAL-R2:16_PER_CLASS",
    "P01-L1-LOW-CAL-OFFICIAL-R2:32_PER_CLASS",
}
S18S_CONDITIONS = {
    "A4-C1-LONG-3P5S",
    "A4-C2-MULTI-HARD-VOTE",
    "A4-C3-MULTI-PROB-AVG",
}
S18S_ROLES = {"NEURAL", "SSL"}

_canonical_plan = list(STATE.get("a4cells") or [])
if len(_canonical_plan) != 1218:
    raise RuntimeError(
        "STAGE18S_CANONICAL_A4_PLAN_COUNT_MISMATCH:"
        + str(len(_canonical_plan))
    )

S18S_PLAN = []

for c in _canonical_plan:
    if str(c.get("condition_id")) not in S18S_CONDITIONS:
        continue
    if str(c.get("role_id")) not in S18S_ROLES:
        continue

    budget = str(c.get("budget_id"))
    rep = int(c.get("model_repeat_index", 0))

    selected = (
        (budget in S18S_ANCHOR_BUDGETS and rep in {1, 2})
        or
        (budget in S18S_PROBE_BUDGETS and rep == 0)
    )
    if not selected:
        continue

    x = copy.deepcopy(c)
    parent_id = str(x["planned_run_cell_id"])
    if not parent_id.startswith("P02-A4-"):
        raise RuntimeError(
            "STAGE18S_UNEXPECTED_PARENT_RUN_CELL_ID:" + parent_id
        )

    x["supplement_parent_run_cell_id"] = parent_id
    x["planned_run_cell_id"] = parent_id.replace(
        "P02-A4-",
        "P02-A4S-R1-",
        1,
    )
    x["supplement_id"] = S18S_ID
    S18S_PLAN.append(x)

S18S_PLAN = sorted(
    S18S_PLAN,
    key=lambda c: str(c["planned_run_cell_id"]),
)

if len(S18S_PLAN) != S18S_EXPECTED_CELLS:
    raise RuntimeError(
        f"STAGE18S_PLAN_COUNT_MISMATCH:{len(S18S_PLAN)}"
    )

# Stable plan identity uses scientific/run-cell identity only, not incidental
# Python dictionary layout.
_plan_identity = []
for c in S18S_PLAN:
    _plan_identity.append({
        "dataset_id": c.get("dataset_id"),
        "budget_id": c.get("budget_id"),
        "role_id": c.get("role_id"),
        "model_repeat_index": c.get("model_repeat_index"),
        "condition_id": c.get("condition_id"),
        "seed_id": c.get("seed_id"),
        "branch_slot": c.get("branch_slot"),
        "ablation_id": c.get("ablation_id"),
        "parent_run_cell_id": c.get("supplement_parent_run_cell_id"),
        "supplement_run_cell_id": c.get("planned_run_cell_id"),
    })

S18S_PLAN_HASH = hashlib.sha256(
    json.dumps(
        _plan_identity,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")
).hexdigest()

def _s18s_group(c):
    return (
        str(c["dataset_id"]),
        str(c["budget_id"]),
        str(c["role_id"]),
        int(c["model_repeat_index"]),
    )

S18S_GROUPS = sorted({_s18s_group(c) for c in S18S_PLAN})
if len(S18S_GROUPS) != S18S_EXPECTED_GROUPS:
    raise RuntimeError(
        f"STAGE18S_GROUP_COUNT_MISMATCH:{len(S18S_GROUPS)}"
    )

_group_sizes = Counter(_s18s_group(c) for c in S18S_PLAN)
if set(_group_sizes.values()) != {3}:
    raise RuntimeError(
        "STAGE18S_GROUP_MUST_HAVE_EXACTLY_C1_C2_C3:"
        + json.dumps(dict(_group_sizes), default=str)
    )

# -----------------------------------------------------------------------------
# 3. Separate supplemental Store. Canonical A0 terminal/metric files are hard-
#    linked/copy-seeded read-only so package-native select_reps/a0_term semantics
#    remain exact while supplemental A4 outputs get new identities.
# -----------------------------------------------------------------------------
S18S_ROOT = (
    STORE_ROOT
    / "supplements"
    / "stage18S_balanced_sensitivity_R1"
)
S18S_ROOT.mkdir(parents=True, exist_ok=True)

S18S_STORE = Store(
    S18S_ROOT,
    config_sha256=SESSION.ctx.state["config_hash"],
)

def _link_or_copy(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        # Existing seed must remain byte-identical.
        if _s18s_sha256(src) != _s18s_sha256(dst):
            raise RuntimeError(
                "STAGE18S_EXISTING_A0_SEED_HASH_MISMATCH:"
                + str(dst)
            )
        return "EXISTING_IDENTICAL"
    try:
        os.link(src, dst)
        return "HARDLINK"
    except Exception:
        shutil.copy2(src, dst)
        return "COPY"

_seed_modes = Counter()
_seed_files = 0

for src_dir_name in ("run_cells", "metrics"):
    src_dir = STORE_ROOT / src_dir_name
    dst_dir = S18S_ROOT / src_dir_name

    for p in sorted(src_dir.glob("P02-A0-*.json")):
        mode = _link_or_copy(
            p,
            dst_dir / p.name,
        )
        _seed_modes[mode] += 1
        _seed_files += 1

if _seed_files < 1000:
    # Expected A0 seed surface is substantially larger; fail closed rather than
    # silently allowing representative selection against a partial A0 set.
    raise RuntimeError(
        "STAGE18S_A0_SEED_SURFACE_UNEXPECTEDLY_SMALL:"
        + str(_seed_files)
    )

# Verify every supplement group resolves a validation-selected representative.
_rep_check = []
for group in S18S_GROUPS:
    ds, budget, role, rep = group
    reps, prov = _r8_a4.select_reps(
        S18S_STORE,
        ds,
        budget,
    )
    selected = reps.get(role)
    if not selected:
        raise RuntimeError(
            "STAGE18S_VALIDATION_REPRESENTATIVE_MISSING:"
            + repr(group)
        )
    _rep_check.append({
        "dataset_id": ds,
        "budget_id": budget,
        "role_id": role,
        "model_repeat_index": rep,
        "selected_branch": selected,
        "test_outcome_influence": "PROHIBITED",
    })

# -----------------------------------------------------------------------------
# 4. Freeze/registration receipts.
# -----------------------------------------------------------------------------
S18S_ANALYSIS_DIR = S18S_ROOT / "analysis"
S18S_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

S18S_RUNTIME_ANALYSIS_DIR = STORE_ROOT / "analysis_inputs"
S18S_RUNTIME_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

S18S_FREEZE = {
    "artifact_id": "P02-STAGE18S-BALANCED-SENSITIVITY-SCIENTIFIC-FREEZE-R1",
    "supplement_id": S18S_ID,
    "revision": S18S_REVISION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "FROZEN_BEFORE_SUPPLEMENT_EXECUTION",
    "canonical_stage18_status": _s18_ledger.get("status"),
    "canonical_g18_status": _g18.get("status"),
    "canonical_stage18_immutable": True,
    "canonical_a4_terminal_files_protected": 1218,
    "stage18_policy_sha256": R8_STAGE18_POLICY_SHA256,
    "plan_sha256": S18S_PLAN_HASH,
    "planned_cells": len(S18S_PLAN),
    "planned_groups": len(S18S_GROUPS),
    "scope": {
        "datasets": sorted({str(c["dataset_id"]) for c in S18S_PLAN}),
        "roles": sorted(S18S_ROLES),
        "conditions": sorted(S18S_CONDITIONS),
        "anchor_repeat_extension": {
            "budgets": sorted(S18S_ANCHOR_BUDGETS),
            "additional_repeats": [1, 2],
            "combined_with_existing_canonical_repeat": [0],
            "resulting_descriptive_repeats": [0, 1, 2],
        },
        "intermediate_budget_probe": {
            "budgets": sorted(S18S_PROBE_BUDGETS),
            "repeat": [0],
        },
        "intentionally_not_executed": {
            "budget": "P01-L1-LOW-CAL-OFFICIAL-R2:2_PER_CLASS",
            "reason": "BOUNDED_COMPUTE_BALANCED_SENSITIVITY_DESIGN",
        },
    },
    "claim_scope": {
        "post_hoc_sensitivity_evidence": True,
        "retroactive_confirmatory_claim": False,
        "full_buildbook_replication_equivalence": False,
        "five_repeat_stability_claim": False,
        "three_repeat_anchor_sensitivity_descriptive": True,
        "six_point_mr00_budget_sensitivity_descriptive": True,
        "test_outcome_used_for_model_or_checkpoint_selection": False,
        "note": (
            "Supplement scope was chosen after canonical Stage18 results existed; "
            "therefore it is sensitivity evidence, not a pre-Stage18 confirmatory plan."
        ),
    },
    "scientific_training_policy": {
        "same_stage18_model_recipes": True,
        "same_class_weight_policy": True,
        "same_validation_selection": True,
        "same_restore_best": True,
        "same_checkpoint_before_test_firewall": True,
        "same_cbramod_long_fail_closed": True,
        "hyperparameter_search_added": False,
    },
    "transport": {
        "max_slots_per_gpu": 6,
        "max_workers": 12,
        "single_model_multi_gpu": False,
        "same_cache_group_on_multiple_workers": False,
        "same_gpu_virtual_cuda_rng": True,
        "adaptive_resource_guards": True,
    },
    "expected_terminal_counts": S18S_EXPECTED_TERMINALS,
    "expected_member_receipts": S18S_EXPECTED_MEMBER_RECEIPTS,
    "expected_member_status_counts": S18S_EXPECTED_MEMBER_STATUSES,
}

_freeze_path = (
    S18S_RUNTIME_ANALYSIS_DIR
    / "stage18S_balanced_sensitivity_R1_preexecution_freeze.json"
)
if _freeze_path.exists():
    old = json.loads(_freeze_path.read_text(encoding="utf-8"))
    # A previous R1.1 installer may already have frozen the SAME scientific
    # supplement with a lower transport ceiling. Scientific identity must be
    # identical; transport is allowed to have a superseding receipt.
    for key in (
        "supplement_id",
        "revision",
        "stage18_policy_sha256",
        "plan_sha256",
        "planned_cells",
        "planned_groups",
    ):
        if old.get(key) != S18S_FREEZE.get(key):
            raise RuntimeError(
                "STAGE18S_EXISTING_FREEZE_CONFLICT:"
                + key
            )
    _scientific_freeze = old
else:
    _r8_pkg_atomic_json(
        _freeze_path,
        S18S_FREEZE,
    )
    _scientific_freeze = S18S_FREEZE

_local_freeze_path = S18S_ANALYSIS_DIR / "preexecution_freeze.json"
if _local_freeze_path.exists():
    _local_old = json.loads(_local_freeze_path.read_text(encoding="utf-8"))
    for key in (
        "supplement_id",
        "revision",
        "stage18_policy_sha256",
        "plan_sha256",
        "planned_cells",
        "planned_groups",
    ):
        if _local_old.get(key) != _scientific_freeze.get(key):
            raise RuntimeError(
                "STAGE18S_LOCAL_FREEZE_CONFLICT:" + key
            )
else:
    _r8_pkg_atomic_json(
        _local_freeze_path,
        _scientific_freeze,
    )

S18S_TRANSPORT_SUCCESSOR = {
    "artifact_id": "P02-STAGE18S-R1-TRANSPORT-SUCCESSOR-R1.3",
    "supplement_id": S18S_ID,
    "transport_revision": S18S_TRANSPORT_REVISION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "INSTALLED",
    "scientific_plan_sha256": S18S_PLAN_HASH,
    "scientific_policy_changed": False,
    "canonical_stage18_changed": False,
    "max_slots_per_gpu": 6,
    "max_workers": 12,
    "single_model_multi_gpu": False,
    "adaptive_vram_ram_admission": True,
    "automatic_resource_backoff": True,
    "automatic_retry_of_resource_blocked_supplement_cells": True,
}
_r8_pkg_atomic_json(
    S18S_ANALYSIS_DIR / "transport_successor_R1_3.json",
    S18S_TRANSPORT_SUCCESSOR,
)
_r8_pkg_atomic_json(
    S18S_RUNTIME_ANALYSIS_DIR
    / "stage18S_R1_transport_successor_R1_3.json",
    S18S_TRANSPORT_SUCCESSOR,
)
_r8_pkg_atomic_json(
    S18S_ANALYSIS_DIR / "supplement_plan.json",
    {
        "supplement_id": S18S_ID,
        "plan_sha256": S18S_PLAN_HASH,
        "rows": S18S_PLAN,
    },
)
_r8_pkg_atomic_json(
    S18S_ANALYSIS_DIR / "validation_selected_representatives.json",
    {
        "supplement_id": S18S_ID,
        "rows": _rep_check,
    },
)

S18S_ASSERT_CANONICAL_IMMUTABLE()

print("\nSupplement plan frozen:")
print(json.dumps({
    "supplement_id": S18S_ID,
    "plan_sha256": S18S_PLAN_HASH,
    "cells": len(S18S_PLAN),
    "groups": len(S18S_GROUPS),
    "anchor_budgets": sorted(S18S_ANCHOR_BUDGETS),
    "anchor_new_repeats": [1, 2],
    "probe_budgets": sorted(S18S_PROBE_BUDGETS),
    "probe_repeat": [0],
    "expected_member_receipts": S18S_EXPECTED_MEMBER_RECEIPTS,
    "a0_seed_files": _seed_files,
    "a0_seed_modes": dict(sorted(_seed_modes.items())),
    "canonical_stage18_protected": True,
    "transport_revision": S18S_TRANSPORT_REVISION,
    "max_slots_per_gpu": 6,
    "max_workers": 12,
}, indent=2))

print("\nTRUSTED INSTALL MARKER:")
print("IHARQ_P02_STAGE18S_BALANCED_SENSITIVITY_R1_3_12WORKER_INSTALLED")


STAGE 18S — BALANCED SENSITIVITY CONTINUATION R1 — INSTALL / SCIENTIFIC FREEZE

Supplement plan frozen:
{
  "supplement_id": "P02-STAGE18S-BALANCED-SENSITIVITY-R1",
  "plan_sha256": "543d896dcc4ff0f66045ead0d8705e24bb4750d1611d15a401538ecaef6bc7c1",
  "cells": 162,
  "groups": 54,
  "anchor_budgets": [
    "FULL_TRAIN",
    "P01-L1-LOW-CAL-OFFICIAL-R2:1_PER_CLASS",
    "P01-L1-LOW-CAL-OFFICIAL-R2:8_PER_CLASS"
  ],
  "anchor_new_repeats": [
    1,
    2
  ],
  "probe_budgets": [
    "P01-L1-LOW-CAL-OFFICIAL-R2:16_PER_CLASS",
    "P01-L1-LOW-CAL-OFFICIAL-R2:32_PER_CLASS",
    "P01-L1-LOW-CAL-OFFICIAL-R2:4_PER_CLASS"
  ],
  "probe_repeat": [
    0
  ],
  "expected_member_receipts": 216,
  "a0_seed_files": 1335,
  "a0_seed_modes": {
    "EXISTING_IDENTICAL": 1335
  },
  "canonical_stage18_protected": true,
  "transport_revision": "R1.3-SIX-SLOTS-PER-GPU-ADAPTIVE-RAMPED-BACKOFF",
  "max_slots_per_gpu": 6,
  "max_workers": 12
}

TRUSTED INSTALL MARKER:
IHARQ_P02_STAGE18S_BALANCED_SENSITIVITY

In [5]:
# =============================================================================
# STAGE 18S R1.3 — TWELVE-WORKER-CEILING / 6-SLOTS-PER-GPU ADAPTIVE SUPPLEMENT EXECUTION
#
# Resumable. Reuses compatible successful supplemental cells/member receipts.
# Retryable supplement failures are archived inside the supplement root only.
# Canonical Stage18 is never edited.
# =============================================================================

from pathlib import Path
from collections import Counter
from datetime import timedelta, datetime, timezone
import copy
import gc
import json
import queue
import shutil
import subprocess
import threading
import time
import traceback

import torch

print("=" * 118)
print("STAGE 18S R1.3 — BALANCED SENSITIVITY — 12-WORKER CEILING / ADAPTIVE ADMISSION")
print("=" * 118)

_required = [
    "S18S_ID", "S18S_PLAN", "S18S_GROUPS", "S18S_STORE", "S18S_ROOT",
    "S18S_EXPECTED_CELLS", "S18S_EXPECTED_GROUPS",
    "S18S_EXPECTED_MEMBER_RECEIPTS",
    "S18S_EXPECTED_TERMINALS", "S18S_EXPECTED_MEMBER_STATUSES",
    "S18S_ASSERT_CANONICAL_IMMUTABLE",
]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError(
        "STAGE18S_EXECUTION_REQUIRES_INSTALLER_CELL:"
        + ",".join(_missing)
    )

S18S_ASSERT_CANONICAL_IMMUTABLE()

if torch.cuda.device_count() < 2:
    raise RuntimeError("STAGE18S_REQUESTED_TRANSPORT_REQUIRES_TWO_CUDA_GPUS")

S18S_GPU_NAMES = [
    str(torch.cuda.get_device_name(i))
    for i in range(torch.cuda.device_count())
]
print("GPUs:", S18S_GPU_NAMES[:2])

# -----------------------------------------------------------------------------
# 1. Retry cleanup inside supplement Store only.
# -----------------------------------------------------------------------------
S18S_RETRYABLE_MEMBER_STATUSES = {
    "FAILED",
    "RESOURCE_BLOCKED",
    "DEPENDENCY_BLOCKED",
    "CHECKPOINT_BLOCKED",
    "INVALID",
}
S18S_RETRYABLE_TERMINALS = {
    "FAILED",
    "RESOURCE_BLOCKED",
    "DEPENDENCY_BLOCKED",
    "CHECKPOINT_BLOCKED",
    "INVALID",
}

_recovery_stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
S18S_RECOVERY_ROOT = (
    S18S_ROOT
    / "diagnostics"
    / "recovery"
    / _recovery_stamp
)
S18S_RECOVERY_ROOT.mkdir(parents=True, exist_ok=True)

_archived_member_files = []
_archived_terminal_files = []

def _archive_then_remove(p, category):
    p = Path(p)
    if not p.is_file():
        return
    rel = p.relative_to(S18S_ROOT)
    dst = S18S_RECOVERY_ROOT / category / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, dst)
    p.unlink()
    return str(rel)

# Member receipts.
for p in sorted((S18S_ROOT / "diagnostics").glob("A4MEM-*.json")):
    d = None
    try:
        d = json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        pass
    if not isinstance(d, dict):
        continue

    status = str(d.get("status"))
    if status not in S18S_RETRYABLE_MEMBER_STATUSES:
        continue

    key = p.stem[len("A4MEM-"):]
    related = [
        p,
        p.with_suffix(".jsonl"),
        S18S_ROOT / "checkpoints" / f"A4MEM-R8-{key}.pkl",
        S18S_ROOT / "checkpoints" / f"A4MEM-{key}.pkl",
    ]
    for q in related:
        rel = _archive_then_remove(q, "member_retry")
        if rel:
            _archived_member_files.append(rel)

# Run-cell retryable terminals and associated output payloads.
for c in S18S_PLAN:
    t = S18S_STORE.terminal(c)
    if not t:
        continue
    if str(t.get("terminal_status")) not in S18S_RETRYABLE_TERMINALS:
        continue

    rid = str(c["planned_run_cell_id"])
    candidates = [
        S18S_ROOT / "run_cells" / f"{rid}.json",
        S18S_ROOT / "metrics" / f"{rid}.json",
        S18S_ROOT / "raw_outputs" / f"{rid}.jsonl",
        S18S_ROOT / "failures" / f"{rid}.json",
    ]

    for base in (S18S_ROOT / "records", S18S_ROOT / "manifests" / "record_partitions"):
        if base.is_dir():
            candidates.extend(base.rglob(f"{rid}.json"))
            candidates.extend(base.rglob(f"{rid}.jsonl"))

    for q in candidates:
        rel = _archive_then_remove(q, "terminal_retry")
        if rel:
            _archived_terminal_files.append(rel)

print("\nRetry preparation:")
print("  archived retryable member files :", len(_archived_member_files))
print("  archived retryable run-cell files:", len(_archived_terminal_files))

# -----------------------------------------------------------------------------
# 2. Group the 162 cells into 54 cache-groups. C1/C2/C3 for one group always
#    stay on one worker so C2/C3 shared member caches cannot collide.
# -----------------------------------------------------------------------------
S18S_PLAN_INDEX = {
    str(c["planned_run_cell_id"]): i
    for i, c in enumerate(S18S_PLAN)
}
S18S_GROUP_CELLS = {}

for c in S18S_PLAN:
    g = (
        str(c["dataset_id"]),
        str(c["budget_id"]),
        str(c["role_id"]),
        int(c["model_repeat_index"]),
    )
    S18S_GROUP_CELLS.setdefault(g, []).append(c)

_condition_rank = {
    "A4-C1-LONG-3P5S": 1,
    "A4-C2-MULTI-HARD-VOTE": 2,
    "A4-C3-MULTI-PROB-AVG": 3,
}
for g in list(S18S_GROUP_CELLS):
    S18S_GROUP_CELLS[g] = sorted(
        S18S_GROUP_CELLS[g],
        key=lambda c: _condition_rank[str(c["condition_id"])],
    )

if len(S18S_GROUP_CELLS) != S18S_EXPECTED_GROUPS:
    raise RuntimeError("STAGE18S_GROUP_RECONSTRUCTION_MISMATCH")

# Longest-processing-time-ish ordering to reduce the final straggler tail.
def _budget_weight(budget):
    b = str(budget)
    if b == "FULL_TRAIN":
        return 3.0
    if b.endswith(":32_PER_CLASS"):
        return 2.5
    if b.endswith(":16_PER_CLASS"):
        return 2.1
    if b.endswith(":8_PER_CLASS"):
        return 1.8
    if b.endswith(":4_PER_CLASS"):
        return 1.45
    if b.endswith(":1_PER_CLASS"):
        return 1.0
    return 1.0

def _group_weight(g):
    ds, budget, role, rep = g
    role_weight = 1.18 if role == "SSL" else 1.0
    # Extra repeat does not change scientific recipe; tiny tie-breaker only.
    repeat_weight = 1.0 + 0.01 * int(rep)
    return _budget_weight(budget) * role_weight * repeat_weight

S18S_GROUP_ORDER = sorted(
    S18S_GROUP_CELLS,
    key=lambda g: (-_group_weight(g), repr(g)),
)

# -----------------------------------------------------------------------------
# 3. Twelve adaptive workers: 6 slots per T4.
# -----------------------------------------------------------------------------
S18S_SLOTS_PER_GPU = 6
S18S_WORKER_SPECS = [
    (0, 0), (1, 0),
    (0, 1), (1, 1),
    (0, 2), (1, 2),
    (0, 3), (1, 3),
    (0, 4), (1, 4),
    (0, 5), (1, 5),
]

S18S_LOCK = threading.RLock()
S18S_GPU_ADMISSION_LOCKS = {
    0: threading.RLock(),
    1: threading.RLock(),
}
# Dynamic per-GPU ceiling. A resource/OOM event lowers only the affected GPU's
# ceiling; scientific cells are retried rather than converted into final
# resource failures.
S18S_GPU_SLOT_CAP = {0: 6, 1: 6}
S18S_GPU_BACKOFF_EVENTS = Counter()
S18S_STOP = threading.Event()
S18S_QUEUE = queue.Queue()
S18S_WORKERS = []
S18S_ACTIVE = {}
S18S_WORKER_ERRORS = {}
S18S_GROUP_STATUS = {}

S18S_TRANSPORT_ROOT = (
    S18S_ROOT
    / "diagnostics"
    / "six_worker_transport"
)
S18S_TRANSPORT_ROOT.mkdir(parents=True, exist_ok=True)

def _s18s_available_ram_gib():
    try:
        import psutil
        return float(psutil.virtual_memory().available) / (1024.0 ** 3)
    except Exception:
        return None

def _s18s_resource_ok(gpu_id, slot_id):
    gpu_id = int(gpu_id)
    slot_id = int(slot_id)

    with S18S_LOCK:
        cap = int(S18S_GPU_SLOT_CAP[gpu_id])
    if slot_id >= cap:
        return False

    ram = _s18s_available_ram_gib()

    # Higher numbered slots require progressively more global RAM headroom.
    # These are transport guards only; scientific batch sizes are untouched.
    ram_floor = {
        0: 5.5,
        1: 6.5,
        2: 7.5,
        3: 9.0,
        4: 11.0,
        5: 13.0,
    }[slot_id]

    if ram is not None and ram < ram_floor:
        return False

    try:
        with torch.cuda.device(gpu_id):
            free_b, total_b = torch.cuda.mem_get_info(gpu_id)
        free_gib = float(free_b) / (1024.0 ** 3)

        # Later slots are admitted only if substantial free VRAM remains.
        # This makes 6/GPU the ceiling rather than an unsafe unconditional load.
        vram_floor = {
            0: 2.0,
            1: 2.5,
            2: 3.0,
            3: 3.8,
            4: 4.8,
            5: 5.8,
        }[slot_id]

        if free_gib < vram_floor:
            return False
    except Exception:
        return slot_id == 0

    return True


def _s18s_lower_gpu_cap(gpu_id, reason):
    gpu_id = int(gpu_id)
    with S18S_LOCK:
        old = int(S18S_GPU_SLOT_CAP[gpu_id])
        new = max(1, old - 1)
        S18S_GPU_SLOT_CAP[gpu_id] = new
        S18S_GPU_BACKOFF_EVENTS[(gpu_id, str(reason))] += 1
    print(
        f"[Stage18S R1.2 BACKOFF] GPU{gpu_id} slot cap {old} -> {new}"
        f" due to {reason}",
        flush=True,
    )
    return new


def _s18s_remove_retryable_terminal(c):
    """Remove only supplement-local retryable run-cell outputs before retry."""
    rid = str(c["planned_run_cell_id"])
    candidates = [
        S18S_ROOT / "run_cells" / f"{rid}.json",
        S18S_ROOT / "metrics" / f"{rid}.json",
        S18S_ROOT / "raw_outputs" / f"{rid}.jsonl",
        S18S_ROOT / "failures" / f"{rid}.json",
    ]
    for base in (
        S18S_ROOT / "records",
        S18S_ROOT / "manifests" / "record_partitions",
    ):
        if base.is_dir():
            candidates.extend(base.rglob(f"{rid}.json"))
            candidates.extend(base.rglob(f"{rid}.jsonl"))

    retry_dir = (
        S18S_RECOVERY_ROOT
        / "automatic_resource_retry"
        / rid
    )
    for p in candidates:
        p = Path(p)
        if not p.is_file():
            continue
        rel = p.relative_to(S18S_ROOT)
        dst = retry_dir / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dst)
        p.unlink()


def _s18s_is_resource_error(exc):
    s = f"{type(exc).__name__}:{exc}".lower()
    return any(
        token in s
        for token in (
            "out of memory",
            "cuda error: out of memory",
            "resource_blocked",
            "cublas_status_alloc_failed",
        )
    )


def _s18s_output_is_transport_resource_failure(out):
    if not isinstance(out, dict):
        return False
    status = str(out.get("terminal_status") or "")
    reason = " ".join(
        str(out.get(k) or "")
        for k in ("reason", "error", "detail", "message")
    ).lower()

    if status == "RESOURCE_BLOCKED":
        return True

    if status == "FAILED" and any(
        token in reason
        for token in (
            "out of memory",
            "cuda",
            "cublas_status_alloc_failed",
            "resource",
        )
    ):
        return True

    return False

def _s18s_group_complete(g):
    return all(
        S18S_STORE.terminal(c) is not None
        for c in S18S_GROUP_CELLS[g]
    )

def _s18s_transport_write(c, gpu_id, slot_id, phase, extra=None):
    payload = {
        "artifact_id": "P02-STAGE18S-R1-SIX-WORKER-TRANSPORT",
        "supplement_id": S18S_ID,
        "planned_run_cell_id": str(c["planned_run_cell_id"]),
        "parent_canonical_run_cell_id": str(c["supplement_parent_run_cell_id"]),
        "gpu_id": int(gpu_id),
        "slot_id": int(slot_id),
        "gpu_name": S18S_GPU_NAMES[int(gpu_id)],
        "phase": str(phase),
        "scheduler_revision": "STAGE18S_R1_3_SIX_SLOTS_PER_GPU_ADAPTIVE_RAMPED",
        "dedicated_cuda_stream": True,
        "virtualized_per_fit_cuda_rng": True,
        "single_model_multi_gpu": False,
        "same_cache_group_on_multiple_workers": False,
        "scientific_recipe_changed": False,
        "stage18_policy_sha256": R8_STAGE18_POLICY_SHA256,
    }
    if extra:
        payload.update(copy.deepcopy(extra))

    d = (
        S18S_TRANSPORT_ROOT
        / f"gpu{gpu_id}_slot{slot_id}"
    )
    d.mkdir(parents=True, exist_ok=True)
    _r8_pkg_atomic_json(
        d / f"{c['planned_run_cell_id']}.json",
        payload,
    )

# Reset/resume queue.
for g in S18S_GROUP_ORDER:
    if not _s18s_group_complete(g):
        S18S_QUEUE.put(g)

print("\nScheduler:")
print("  slots per GPU :", S18S_SLOTS_PER_GPU)
print("  max workers   :", len(S18S_WORKER_SPECS))
print("  groups total  :", len(S18S_GROUP_ORDER))
print("  groups queued :", S18S_QUEUE.qsize())

def _s18s_worker(gpu_id, slot_id):
    worker_id = f"GPU{gpu_id}-SLOT{slot_id}"
    _R10_3_GPU_TLS.gpu_id = int(gpu_id)

    try:
        with torch.cuda.device(int(gpu_id)):
            stream = torch.cuda.Stream(device=int(gpu_id))

            while not S18S_STOP.is_set():
                with S18S_GPU_ADMISSION_LOCKS[int(gpu_id)]:
                    if not _s18s_resource_ok(gpu_id, slot_id):
                        _admitted = False
                    else:
                        _admitted = True

                if not _admitted:
                    time.sleep(1.5)
                    continue

                try:
                    group = S18S_QUEUE.get_nowait()
                except queue.Empty:
                    break

                try:
                    if _s18s_group_complete(group):
                        continue

                    with S18S_LOCK:
                        S18S_ACTIVE[worker_id] = {
                            "gpu_id": int(gpu_id),
                            "slot_id": int(slot_id),
                            "group": list(group),
                            "cell": None,
                        }
                        S18S_GROUP_STATUS[group] = "RUNNING_" + worker_id

                    ds, budget, role, rep = group
                    reps, selection_prov = _r8_a4.select_reps(
                        S18S_STORE,
                        ds,
                        budget,
                    )

                    if not reps.get(role):
                        raise RuntimeError(
                            "STAGE18S_REPRESENTATIVE_MISSING:"
                            + repr(group)
                        )

                    for c in S18S_GROUP_CELLS[group]:
                        if S18S_STOP.is_set():
                            break

                        old = S18S_STORE.terminal(c)
                        if old is not None:
                            continue

                        rid = str(c["planned_run_cell_id"])
                        with S18S_LOCK:
                            S18S_ACTIVE[worker_id]["cell"] = rid

                        _s18s_transport_write(
                            c,
                            gpu_id,
                            slot_id,
                            "STARTED",
                            {
                                "group": list(group),
                                "selected_branch": reps.get(role),
                            },
                        )

                        ctx = SESSION.ctx
                        state = ctx.state
                        overrides = {
                            "batch_size": int(
                                state.get(
                                    "resource_profile",
                                    {},
                                ).get(
                                    "recommended_neural_batch_size",
                                    16 if ctx.fixture else 64,
                                )
                            )
                        }

                        with torch.cuda.stream(stream):
                            out = _R10_PARENT_EXECUTE_ROLE(
                                c,
                                S18S_STORE,
                                state["core"],
                                state["a4"],
                                reps,
                                state["schema"],
                                state["config_hash"],
                                state["runtime_freeze"],
                                state["config"]["data"],
                                ctx.fixture,
                                state.get("implementation_bindings"),
                                overrides,
                            )

                        stream.synchronize()

                        _out_status = (
                            None if out is None
                            else str(out.get("terminal_status"))
                        )
                        if _s18s_output_is_transport_resource_failure(out):
                            # GPU oversubscription is transport evidence, not a
                            # scientific negative result. Preserve a retry trace,
                            # remove only supplement-local retryable output, lower
                            # the affected GPU ceiling, and requeue the group.
                            _s18s_transport_write(
                                c,
                                gpu_id,
                                slot_id,
                                "RESOURCE_BACKOFF_RETRY",
                                {
                                    "group": list(group),
                                    "terminal_status": _out_status,
                                    "reason": (
                                        None if out is None
                                        else out.get("reason")
                                    ),
                                },
                            )
                            _s18s_remove_retryable_terminal(c)

                            # Remove only resource/OOM deep-member receipts from
                            # the supplemental store so the package may refit them.
                            for _mp in sorted(
                                (S18S_ROOT / "diagnostics").glob("A4MEM-*.json")
                            ):
                                try:
                                    _md = json.loads(
                                        _mp.read_text(encoding="utf-8")
                                    )
                                except Exception:
                                    continue

                                _ms = str(_md.get("status") or "")
                                _mr = " ".join(
                                    str(_md.get(k) or "")
                                    for k in ("reason", "error", "detail")
                                ).lower()

                                if (
                                    _ms in {
                                        "FAILED",
                                        "RESOURCE_BLOCKED",
                                        "CHECKPOINT_BLOCKED",
                                    }
                                    and any(
                                        token in _mr
                                        for token in (
                                            "out of memory",
                                            "cuda",
                                            "cublas_status_alloc_failed",
                                            "resource",
                                        )
                                    )
                                ):
                                    _mretry = (
                                        S18S_RECOVERY_ROOT
                                        / "automatic_resource_retry_members"
                                        / _mp.name
                                    )
                                    _mretry.parent.mkdir(
                                        parents=True,
                                        exist_ok=True,
                                    )
                                    shutil.copy2(_mp, _mretry)
                                    _mp.unlink()
                                    _rowp = _mp.with_suffix(".jsonl")
                                    if _rowp.is_file():
                                        _row_dst = _mretry.with_suffix(".jsonl")
                                        shutil.copy2(_rowp, _row_dst)
                                        _rowp.unlink()

                            _s18s_lower_gpu_cap(
                                gpu_id,
                                "RESOURCE_OR_OOM",
                            )
                            try:
                                with torch.cuda.device(int(gpu_id)):
                                    torch.cuda.empty_cache()
                            except Exception:
                                pass
                            raise ResourceWarning(
                                "STAGE18S_AUTO_RETRY_RESOURCE_OR_OOM"
                            )

                        _s18s_transport_write(
                            c,
                            gpu_id,
                            slot_id,
                            "COMPLETED",
                            {
                                "group": list(group),
                                "selected_branch": reps.get(role),
                                "terminal_status": (
                                    None if out is None
                                    else out.get("terminal_status")
                                ),
                            },
                        )

                    with S18S_LOCK:
                        S18S_GROUP_STATUS[group] = (
                            "COMPLETE"
                            if _s18s_group_complete(group)
                            else "PARTIAL_STOP"
                        )

                except BaseException as exc:
                    # Automatic transport backoff/retry for oversubscription.
                    if (
                        isinstance(exc, ResourceWarning)
                        or _s18s_is_resource_error(exc)
                    ):
                        if "c" in locals():
                            try:
                                _s18s_remove_retryable_terminal(c)
                            except Exception:
                                pass
                        _s18s_lower_gpu_cap(
                            gpu_id,
                            type(exc).__name__,
                        )
                        try:
                            with torch.cuda.device(int(gpu_id)):
                                torch.cuda.empty_cache()
                        except Exception:
                            pass
                        gc.collect()

                        with S18S_LOCK:
                            S18S_GROUP_STATUS[group] = "REQUEUED_AFTER_RESOURCE_BACKOFF"

                        if not _s18s_group_complete(group):
                            S18S_QUEUE.put(group)

                        time.sleep(2.0)
                    else:
                        err = (
                            f"{type(exc).__name__}:"
                            f"{str(exc)[:1800]}"
                        )
                        with S18S_LOCK:
                            S18S_WORKER_ERRORS[worker_id] = {
                                "error": err,
                                "traceback": traceback.format_exc()[-12000:],
                                "group": list(group) if "group" in locals() else None,
                            }
                            if "group" in locals():
                                S18S_GROUP_STATUS[group] = "ERROR"
                        S18S_STOP.set()
                finally:
                    with S18S_LOCK:
                        S18S_ACTIVE.pop(worker_id, None)
                    S18S_QUEUE.task_done()

                if S18S_STOP.is_set():
                    break

    except BaseException as exc:
        with S18S_LOCK:
            S18S_WORKER_ERRORS[worker_id] = {
                "error": (
                    f"{type(exc).__name__}:"
                    f"{str(exc)[:1800]}"
                ),
                "traceback": traceback.format_exc()[-12000:],
            }
        S18S_STOP.set()

def _s18s_raise_worker_errors():
    with S18S_LOCK:
        errors = copy.deepcopy(S18S_WORKER_ERRORS)
    if errors:
        raise RuntimeError(
            "STAGE18S_WORKER_POOL_FAILED:"
            + json.dumps(
                errors,
                sort_keys=True,
                default=str,
            )
        )

def _s18s_snapshot():
    terms = [
        (c, S18S_STORE.terminal(c))
        for c in S18S_PLAN
        if S18S_STORE.terminal(c) is not None
    ]
    tc = Counter(
        str(t.get("terminal_status"))
        for _, t in terms
    )

    members = []
    for p in sorted((S18S_ROOT / "diagnostics").glob("A4MEM-*.json")):
        try:
            d = json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            continue
        if d.get("r8_policy_sha256") == R8_STAGE18_POLICY_SHA256:
            members.append(d)

    mc = Counter(str(d.get("status")) for d in members)

    with S18S_LOCK:
        active = copy.deepcopy(S18S_ACTIVE)
        errors = copy.deepcopy(S18S_WORKER_ERRORS)

    return {
        "terminals": len(terms),
        "terminal_counts": dict(sorted(tc.items())),
        "member_receipts": len(members),
        "member_counts": dict(sorted(mc.items())),
        "active_workers": active,
        "active_worker_count": len(active),
        "worker_errors": errors,
        "queue_size": S18S_QUEUE.qsize(),
    }

def _s18s_gpu_snapshot():
    try:
        out = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=index,name,utilization.gpu,memory.used,memory.free,memory.total",
                "--format=csv,noheader,nounits",
            ],
            text=True,
            stderr=subprocess.DEVNULL,
            timeout=5,
        )
        return [x.strip() for x in out.splitlines() if x.strip()]
    except Exception:
        return []

# -----------------------------------------------------------------------------
# 4. Launch six workers + progress monitor.
# -----------------------------------------------------------------------------
S18S_STOP.clear()
S18S_WORKER_ERRORS.clear()
S18S_ACTIVE.clear()
S18S_WORKERS = []

# Launch the same slot index on both GPUs, then allow allocations to become
# visible before admitting the next pair. This prevents all twelve threads
# from simultaneously observing the same pre-allocation free-VRAM value.
for _slot_id in range(S18S_SLOTS_PER_GPU):
    for _gpu_id in (0, 1):
        w = threading.Thread(
            target=_s18s_worker,
            args=(_gpu_id, _slot_id),
            name=f"IHARQ-STAGE18S-GPU{_gpu_id}-SLOT{_slot_id}",
            daemon=True,
        )
        S18S_WORKERS.append(w)
        w.start()

    # Higher slots are intentionally ramped more slowly. Training that is
    # already admitted continues concurrently; only *new* slot admission waits.
    _pair_ramp_seconds = {
        0: 3.0,
        1: 4.0,
        2: 5.0,
        3: 5.0,
        4: 5.0,
        5: 0.0,
    }[_slot_id]

    if _pair_ramp_seconds > 0:
        time.sleep(_pair_ramp_seconds)

_start = time.time()
_last_print = 0.0

while any(w.is_alive() for w in S18S_WORKERS):
    for w in S18S_WORKERS:
        w.join(timeout=0.5)

    # If a worker has failed, stop NEW claims immediately but keep joining the
    # already-running atomic CUDA fits so the cell never exits with orphan
    # Stage18S GPU threads still mutating supplement evidence in the background.
    with S18S_LOCK:
        _pool_has_error = bool(S18S_WORKER_ERRORS)
    if _pool_has_error:
        S18S_STOP.set()

    now = time.time()
    if now - _last_print >= 30.0:
        _last_print = now
        snap = _s18s_snapshot()
        by_gpu = Counter(
            str(x["gpu_id"])
            for x in snap["active_workers"].values()
        )
        print(
            "[Stage18S]"
            f" elapsed={timedelta(seconds=int(now-_start))}"
            f" | terminals={snap['terminals']}/{S18S_EXPECTED_CELLS}"
            f" | members={snap['member_receipts']}/{S18S_EXPECTED_MEMBER_RECEIPTS}"
            f" | active={snap['active_worker_count']}/12"
            f" | active_by_gpu={dict(sorted(by_gpu.items()))}"
            f" | queue={snap['queue_size']}",
            flush=True,
        )
        print(
            "    terminal_counts="
            + json.dumps(snap["terminal_counts"], sort_keys=True)
            + " | member_counts="
            + json.dumps(snap["member_counts"], sort_keys=True),
            flush=True,
        )
        print(
            "    workers="
            + json.dumps(
                snap["active_workers"],
                sort_keys=True,
                default=str,
            ),
            flush=True,
        )
        gpu_lines = _s18s_gpu_snapshot()
        if gpu_lines:
            print(
                "    nvidia-smi="
                + " || ".join(gpu_lines[:2]),
                flush=True,
            )

for w in S18S_WORKERS:
    w.join()

_s18s_raise_worker_errors()
elapsed = time.time() - _start

# -----------------------------------------------------------------------------
# 5. Strict supplemental closure.
# -----------------------------------------------------------------------------
final = _s18s_snapshot()

if final["terminals"] != S18S_EXPECTED_CELLS:
    raise RuntimeError(
        "STAGE18S_TERMINAL_CLOSURE_INCOMPLETE:"
        + json.dumps(final, default=str)
    )

if final["terminal_counts"] != dict(sorted(S18S_EXPECTED_TERMINALS.items())):
    raise RuntimeError(
        "STAGE18S_TERMINAL_STATUS_COUNTS_UNEXPECTED:"
        + json.dumps(
            {
                "expected": S18S_EXPECTED_TERMINALS,
                "observed": final["terminal_counts"],
            },
            sort_keys=True,
        )
    )

if final["member_receipts"] != S18S_EXPECTED_MEMBER_RECEIPTS:
    raise RuntimeError(
        "STAGE18S_MEMBER_RECEIPT_COUNT_UNEXPECTED:"
        + str(final["member_receipts"])
    )

if final["member_counts"] != dict(sorted(S18S_EXPECTED_MEMBER_STATUSES.items())):
    raise RuntimeError(
        "STAGE18S_MEMBER_STATUS_COUNTS_UNEXPECTED:"
        + json.dumps(
            {
                "expected": S18S_EXPECTED_MEMBER_STATUSES,
                "observed": final["member_counts"],
            },
            sort_keys=True,
        )
    )

# Deep scientific provenance checks.
_success_member_issues = []
_incompat_issues = []

for p in sorted((S18S_ROOT / "diagnostics").glob("A4MEM-*.json")):
    d = json.loads(p.read_text(encoding="utf-8"))
    if d.get("r8_policy_sha256") != R8_STAGE18_POLICY_SHA256:
        continue

    status = str(d.get("status"))

    if status == "SUCCESS":
        if d.get("test_loaded_after_checkpoint_sealed") is not True:
            _success_member_issues.append([p.name, "TEST_FIREWALL_FALSE"])
        if d.get("test_set_used_for_selection") is not False:
            _success_member_issues.append([p.name, "TEST_SELECTION_NOT_FALSE"])
        if not d.get("recipe_id"):
            _success_member_issues.append([p.name, "RECIPE_ID_MISSING"])
        if not d.get("checkpoint_sha256"):
            _success_member_issues.append([p.name, "CHECKPOINT_SHA_MISSING"])
        if d.get("class_weight_frozen_from_a0") is not True:
            _success_member_issues.append([p.name, "CLASS_WEIGHT_NOT_FROZEN_FROM_A0"])

    elif status == "INPUT_INCOMPATIBLE":
        if (
            d.get("branch") != "SSL-CBRAMOD"
            or d.get("member") != "LONG"
            or d.get("reason")
            != "CBRAMOD_A4_LONG_INCOMPATIBLE_FAIL_CLOSED_NO_PAD_NO_CROP"
        ):
            _incompat_issues.append([p.name, d])

if _success_member_issues:
    raise RuntimeError(
        "STAGE18S_SUCCESS_MEMBER_PROVENANCE_INVALID:"
        + json.dumps(_success_member_issues[:30], default=str)
    )
if _incompat_issues:
    raise RuntimeError(
        "STAGE18S_INPUT_INCOMPATIBLE_NOT_EXPECTED_CBRAMOD_LONG:"
        + json.dumps(_incompat_issues[:30], default=str)
    )

S18S_ASSERT_CANONICAL_IMMUTABLE()

execution_receipt = {
    "artifact_id": "P02-STAGE18S-BALANCED-SENSITIVITY-EXECUTION-R1",
    "supplement_id": S18S_ID,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "PASS",
    "elapsed_seconds": elapsed,
    "planned_cells": S18S_EXPECTED_CELLS,
    "planned_groups": S18S_EXPECTED_GROUPS,
    "terminal_counts": final["terminal_counts"],
    "member_receipts": final["member_receipts"],
    "member_status_counts": final["member_counts"],
    "retry_archives": {
        "member_files": len(_archived_member_files),
        "run_cell_files": len(_archived_terminal_files),
    },
    "transport": {
        "max_slots_per_gpu": 6,
        "max_workers": 12,
        "single_model_multi_gpu": False,
        "same_cache_group_on_multiple_workers": False,
        "same_gpu_virtual_cuda_rng": True,
        "adaptive_resource_guards": True,
        "automatic_resource_backoff": True,
        "final_gpu_slot_caps": dict(S18S_GPU_SLOT_CAP),
        "gpu_backoff_events": {
            f"{k[0]}:{k[1]}": v
            for k, v in S18S_GPU_BACKOFF_EVENTS.items()
        },
    },
    "canonical_stage18_byte_identity_preserved": True,
    "scientific_recipe_changed": False,
    "test_outcome_used_for_selection": False,
}

_r8_pkg_atomic_json(
    S18S_ROOT / "analysis" / "execution_receipt.json",
    execution_receipt,
)

print("\n" + "=" * 118)
print("STAGE 18S EXECUTION CLOSED")
print("=" * 118)
print(json.dumps(execution_receipt, indent=2, default=str))
print("\nTRUSTED EXECUTION MARKER:")
print("IHARQ_P02_STAGE18S_BALANCED_SENSITIVITY_R1_3_12WORKER_EXECUTION_PASS")


STAGE 18S R1.3 — BALANCED SENSITIVITY — 12-WORKER CEILING / ADAPTIVE ADMISSION
GPUs: ['Tesla T4', 'Tesla T4']

Retry preparation:
  archived retryable member files : 0
  archived retryable run-cell files: 0

Scheduler:
  slots per GPU : 6
  max workers   : 12
  groups total  : 54
  groups queued : 54
Loading weights from local directory
Loading weights from local directory
Loading weights from local directory
Loading weights from local directory
Loading weights from local directory
[Stage18S] elapsed=0:00:06 | terminals=6/162 | members=8/216 | active=12/12 | active_by_gpu={'0': 6, '1': 6} | queue=42
    terminal_counts={"INPUT_INCOMPATIBLE": 6} | member_counts={"INPUT_INCOMPATIBLE": 6, "SUCCESS": 2}
    workers={"GPU0-SLOT0": {"cell": "P02-A4S-R1-BNCI2014_001-SSL-A4-C2-MULTI-HARD-VOTE-FULL_TRAIN-MR02", "gpu_id": 0, "group": ["BNCI2014_001", "FULL_TRAIN", "SSL", 2], "slot_id": 0}, "GPU0-SLOT1": {"cell": "P02-A4S-R1-BNCI2014_001-SSL-A4-C2-MULTI-HARD-VOTE-FULL_TRAIN-MR01", "gpu_id": 0, "g

In [6]:
# =============================================================================
# STAGE 18S R1.3 — SUPPLEMENT ANALYSIS / CLOSURE / PROJECT REGISTRATION
#
# Combines:
#   canonical Stage18 MR00 at 1 / 8 / FULL
#   supplement MR01/MR02 at 1 / 8 / FULL
#   supplement MR00 at 4 / 16 / 32
#
# Produces:
#   * three-repeat anchor stability tables
#   * six-point MR00 budget-sensitivity tables
#   * participant-level descriptive paired effects
#   * training-provenance summaries
#   * a separate supplement gate/manifest
#
# Canonical Stage18/G18 remain unchanged.
# =============================================================================

from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import csv
import json
import math
import statistics

import numpy as np
import pandas as pd

print("=" * 118)
print("STAGE 18S R1.3 — ANALYSIS / CLOSURE")
print("=" * 118)

_required = [
    "S18S_ID", "S18S_PLAN", "S18S_STORE", "S18S_ROOT",
    "S18S_EXPECTED_TERMINALS", "S18S_EXPECTED_MEMBER_STATUSES",
    "S18S_ASSERT_CANONICAL_IMMUTABLE",
]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError(
        "STAGE18S_ANALYSIS_REQUIRES_EXECUTION_CONTEXT:"
        + ",".join(_missing)
    )

S18S_ASSERT_CANONICAL_IMMUTABLE()

# Require execution receipt PASS.
_exec_receipt_path = S18S_ROOT / "analysis" / "execution_receipt.json"
if not _exec_receipt_path.is_file():
    raise RuntimeError("STAGE18S_EXECUTION_RECEIPT_MISSING")

_exec_receipt = json.loads(
    _exec_receipt_path.read_text(encoding="utf-8")
)
if _exec_receipt.get("status") != "PASS":
    raise RuntimeError("STAGE18S_EXECUTION_RECEIPT_NOT_PASS")

COND_SHORT = {
    "A4-C1-LONG-3P5S": "C1",
    "A4-C2-MULTI-HARD-VOTE": "C2",
    "A4-C3-MULTI-PROB-AVG": "C3",
}
BUDGET_SHORT = {
    "P01-L1-LOW-CAL-OFFICIAL-R2:1_PER_CLASS": "1",
    "P01-L1-LOW-CAL-OFFICIAL-R2:4_PER_CLASS": "4",
    "P01-L1-LOW-CAL-OFFICIAL-R2:8_PER_CLASS": "8",
    "P01-L1-LOW-CAL-OFFICIAL-R2:16_PER_CLASS": "16",
    "P01-L1-LOW-CAL-OFFICIAL-R2:32_PER_CLASS": "32",
    "FULL_TRAIN": "FULL",
}
BUDGET_RANK = {
    "1": 1,
    "4": 4,
    "8": 8,
    "16": 16,
    "32": 32,
    "FULL": 999,
}

def _load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def _metric_for(store, c):
    t = store.terminal(c)
    if not t or t.get("terminal_status") != "SUCCESS":
        return None, t
    rel = t.get("metric_source")
    if not rel:
        return None, t
    p = Path(store.root) / rel
    if not p.is_file():
        return None, t
    return _load_json(p), t

def _metric_values(d):
    if not isinstance(d, dict):
        return {}
    m = d.get("metrics")
    if not isinstance(m, dict):
        m = d
    out = {}
    for k in ("BACC", "F1_MACRO", "ACC", "ROC_AUC"):
        try:
            out[k] = float(m[k])
        except Exception:
            out[k] = np.nan
    return out

# Canonical lookup.
canonical_plan = list(STATE.get("a4cells") or [])

def _key(c):
    return (
        str(c.get("dataset_id")),
        str(c.get("budget_id")),
        int(c.get("model_repeat_index", 0)),
        str(c.get("role_id")),
        str(c.get("condition_id")),
    )

canonical_lookup = {_key(c): c for c in canonical_plan}
supp_lookup = {_key(c): c for c in S18S_PLAN}

# Exact canonical C0 baselines.
def _c0_cell(ds, budget, rep, role):
    return canonical_lookup.get(
        (
            str(ds),
            str(budget),
            int(rep),
            str(role),
            "A4-C0-CORE",
        )
    )

# -----------------------------------------------------------------------------
# 1. Build combined effect rows.
# -----------------------------------------------------------------------------
effect_rows = []

def _append_effect(source, store, c):
    d, t = _metric_for(store, c)
    if d is None:
        return

    ds = str(c["dataset_id"])
    budget = str(c["budget_id"])
    rep = int(c["model_repeat_index"])
    role = str(c["role_id"])
    cond = str(c["condition_id"])

    c0 = _c0_cell(ds, budget, rep, role)
    if c0 is None:
        raise RuntimeError(
            "STAGE18S_MATCHING_CANONICAL_C0_CELL_MISSING:"
            + repr((ds, budget, rep, role))
        )

    c0d, c0t = _metric_for(STORE, c0)
    if c0d is None:
        raise RuntimeError(
            "STAGE18S_MATCHING_CANONICAL_C0_METRIC_MISSING:"
            + str(c0["planned_run_cell_id"])
        )

    mv = _metric_values(d)
    bv = _metric_values(c0d)

    effect_rows.append({
        "source": source,
        "dataset_id": ds,
        "budget_id": budget,
        "budget_short": BUDGET_SHORT.get(budget, budget),
        "model_repeat_index": rep,
        "role_id": role,
        "condition_id": cond,
        "condition_short": COND_SHORT[cond],
        "resolved_branch": (
            d.get("validation_selected_branch")
            or t.get("resolved_branch")
        ),
        "a4_run_cell_id": str(c["planned_run_cell_id"]),
        "c0_run_cell_id": str(c0["planned_run_cell_id"]),
        "BACC_C0": bv.get("BACC"),
        "BACC_A4": mv.get("BACC"),
        "delta_BACC": mv.get("BACC") - bv.get("BACC"),
        "F1_C0": bv.get("F1_MACRO"),
        "F1_A4": mv.get("F1_MACRO"),
        "delta_F1": mv.get("F1_MACRO") - bv.get("F1_MACRO"),
    })

# Canonical MR00 anchors.
for c in canonical_plan:
    if str(c.get("condition_id")) not in COND_SHORT:
        continue
    if str(c.get("role_id")) not in {"NEURAL", "SSL"}:
        continue
    if str(c.get("budget_id")) not in S18S_ANCHOR_BUDGETS:
        continue
    if int(c.get("model_repeat_index", 0)) != 0:
        continue

    t = STORE.terminal(c)
    if t and t.get("terminal_status") == "SUCCESS":
        _append_effect("CANONICAL_STAGE18_MR00", STORE, c)

# Supplement successful cells.
for c in S18S_PLAN:
    t = S18S_STORE.terminal(c)
    if t and t.get("terminal_status") == "SUCCESS":
        _append_effect("STAGE18S_SUPPLEMENT", S18S_STORE, c)

effects = pd.DataFrame(effect_rows)

# Canonical MR00 anchors contribute 45 successful deep effects:
#   NEURAL C1/C2/C3 = 27, SSL C2/C3 = 18.
# The supplement contributes 135 successful effects, for 180 combined rows.
expected_combined = 180
if len(effects) != expected_combined:
    raise RuntimeError(
        "STAGE18S_COMBINED_EFFECT_ROW_COUNT_UNEXPECTED:"
        + json.dumps(
            {
                "expected": expected_combined,
                "observed": len(effects),
            }
        )
    )

# -----------------------------------------------------------------------------
# 2. Three-repeat stability at 1 / 8 / FULL.
# -----------------------------------------------------------------------------
anchors = effects[
    effects["budget_id"].isin(S18S_ANCHOR_BUDGETS)
    & effects["model_repeat_index"].isin([0, 1, 2])
].copy()

anchor_summary_rows = []

for key, g in anchors.groupby(
    [
        "dataset_id",
        "role_id",
        "condition_id",
        "budget_id",
    ],
    dropna=False,
):
    ds, role, cond, budget = key
    g = g.sort_values("model_repeat_index")
    deltas = g["delta_BACC"].to_numpy(dtype=float)

    if len(deltas) != 3:
        raise RuntimeError(
            "STAGE18S_ANCHOR_TRAJECTORY_NOT_THREE_REPEATS:"
            + repr(key)
            + ":"
            + str(len(deltas))
        )

    tol = 1e-12
    pos = np.all(deltas > tol)
    neg = np.all(deltas < -tol)
    zeroish = np.all(np.abs(deltas) <= tol)

    if pos:
        sign_state = "ALL_POSITIVE"
    elif neg:
        sign_state = "ALL_NEGATIVE"
    elif zeroish:
        sign_state = "ALL_ZERO"
    else:
        sign_state = "MIXED"

    sd = float(np.std(deltas, ddof=1))
    mean = float(np.mean(deltas))

    anchor_summary_rows.append({
        "dataset_id": ds,
        "role_id": role,
        "condition_id": cond,
        "condition_short": COND_SHORT[cond],
        "budget_id": budget,
        "budget_short": BUDGET_SHORT[budget],
        "n_repeats": 3,
        "repeat_indices": [0, 1, 2],
        "delta_BACC_mean": mean,
        "delta_BACC_median": float(np.median(deltas)),
        "delta_BACC_sd": sd,
        "delta_BACC_min": float(np.min(deltas)),
        "delta_BACC_max": float(np.max(deltas)),
        "delta_BACC_range": float(np.max(deltas) - np.min(deltas)),
        "sign_consistency": sign_state,
        "abs_mean_over_repeat_sd": (
            None
            if sd <= 1e-12
            else abs(mean) / sd
        ),
        "repeat0": float(deltas[0]),
        "repeat1": float(deltas[1]),
        "repeat2": float(deltas[2]),
    })

anchor_summary = pd.DataFrame(anchor_summary_rows)

if len(anchor_summary) != 45:
    raise RuntimeError(
        "STAGE18S_EXPECTED_45_THREE_REPEAT_TRAJECTORIES:"
        + str(len(anchor_summary))
    )

# -----------------------------------------------------------------------------
# 3. Six-point MR00 budget sensitivity: 1, 4, 8, 16, 32, FULL.
# -----------------------------------------------------------------------------
mr00 = effects[
    effects["model_repeat_index"] == 0
    & effects["budget_short"].isin(BUDGET_RANK)
].copy()

budget_summary_rows = []

for key, g in mr00.groupby(
    ["dataset_id", "role_id", "condition_id"],
    dropna=False,
):
    ds, role, cond = key
    g = g.copy()
    g["budget_rank"] = g["budget_short"].map(BUDGET_RANK)
    g = g.sort_values("budget_rank")

    if len(g) != 6:
        raise RuntimeError(
            "STAGE18S_MR00_TRAJECTORY_NOT_SIX_BUDGETS:"
            + repr(key)
            + ":"
            + str(len(g))
        )

    deltas = g["delta_BACC"].to_numpy(dtype=float)
    signs = np.sign(deltas)
    # Ignore exact zeros when counting direction changes.
    nz = [int(s) for s in signs if int(s) != 0]
    sign_flips = sum(
        a != b
        for a, b in zip(nz, nz[1:])
    )

    peak_i = int(np.argmax(deltas))
    trough_i = int(np.argmin(deltas))

    budget_summary_rows.append({
        "dataset_id": ds,
        "role_id": role,
        "condition_id": cond,
        "condition_short": COND_SHORT[cond],
        "n_budgets": 6,
        "budget_sequence": "1|4|8|16|32|FULL",
        "delta_sequence": "|".join(
            f"{x:.6f}" for x in deltas
        ),
        "sign_flip_count": int(sign_flips),
        "delta_BACC_mean": float(np.mean(deltas)),
        "delta_BACC_median": float(np.median(deltas)),
        "delta_BACC_min": float(np.min(deltas)),
        "delta_BACC_max": float(np.max(deltas)),
        "peak_budget": str(g.iloc[peak_i]["budget_short"]),
        "peak_delta_BACC": float(deltas[peak_i]),
        "trough_budget": str(g.iloc[trough_i]["budget_short"]),
        "trough_delta_BACC": float(deltas[trough_i]),
        "non_monotonic_direction": bool(sign_flips > 0),
    })

budget_summary = pd.DataFrame(budget_summary_rows)

if len(budget_summary) != 15:
    raise RuntimeError(
        "STAGE18S_EXPECTED_15_SIX_BUDGET_TRAJECTORIES:"
        + str(len(budget_summary))
    )

# -----------------------------------------------------------------------------
# 4. Participant-level descriptive paired BACC differences.
# -----------------------------------------------------------------------------
core_test_truth = {}
for ds in sorted(set(effects["dataset_id"])):
    core_test_truth[ds] = {}
    for r in SESSION.ctx.state["core"].rows(
        dataset_id=ds,
        role="test",
    ):
        core_test_truth[ds][str(r["event_id"])] = {
            "dataset_id": ds,
            "subject_id": str(r["subject_id"]),
            "y_true": int(str(r["label"]) == "right_hand"),
        }

def _read_jsonl(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def _bacc(y_true, y_pred):
    recalls = []
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    for cls in (0, 1):
        mask = y_true == cls
        if not np.any(mask):
            return np.nan
        recalls.append(float(np.mean(y_pred[mask] == cls)))
    return float(np.mean(recalls))

participant_rows = []

for _, er in effects.iterrows():
    # Locate A4 source rows.
    a4_store = (
        STORE
        if er["source"] == "CANONICAL_STAGE18_MR00"
        else S18S_STORE
    )
    a4_cell = (
        canonical_lookup[
            (
                er["dataset_id"],
                er["budget_id"],
                int(er["model_repeat_index"]),
                er["role_id"],
                er["condition_id"],
            )
        ]
        if er["source"] == "CANONICAL_STAGE18_MR00"
        else supp_lookup[
            (
                er["dataset_id"],
                er["budget_id"],
                int(er["model_repeat_index"]),
                er["role_id"],
                er["condition_id"],
            )
        ]
    )
    a4_t = a4_store.terminal(a4_cell)
    a4_rows = _read_jsonl(
        Path(a4_store.root) / a4_t["source_rows"]
    )

    c0 = _c0_cell(
        er["dataset_id"],
        er["budget_id"],
        int(er["model_repeat_index"]),
        er["role_id"],
    )
    c0_t = STORE.terminal(c0)
    c0_rows = _read_jsonl(
        STORE_ROOT / c0_t["prediction_partition"]
    )

    a4_map = {
        str(r["event_id"]): r
        for r in a4_rows
    }
    c0_map = {
        str(r["source_event_id"]): r
        for r in c0_rows
    }
    ds_truth = core_test_truth[er["dataset_id"]]
    common = sorted(
        set(a4_map)
        & set(c0_map)
        & set(ds_truth)
    )

    by_subject = defaultdict(list)
    for eid in common:
        truth = ds_truth[eid]
        by_subject[truth["subject_id"]].append(eid)

    for subject_id, eids in by_subject.items():
        yt = [ds_truth[e]["y_true"] for e in eids]
        yp0 = [int(c0_map[e]["y_pred"]) for e in eids]
        yp4 = [int(a4_map[e]["y_pred"]) for e in eids]

        b0 = _bacc(yt, yp0)
        b4 = _bacc(yt, yp4)
        if not np.isfinite(b0) or not np.isfinite(b4):
            continue

        participant_rows.append({
            "source": er["source"],
            "dataset_id": er["dataset_id"],
            "budget_id": er["budget_id"],
            "budget_short": er["budget_short"],
            "model_repeat_index": int(er["model_repeat_index"]),
            "role_id": er["role_id"],
            "condition_id": er["condition_id"],
            "condition_short": er["condition_short"],
            "subject_id": subject_id,
            "matched_events": len(eids),
            "BACC_C0": b0,
            "BACC_A4": b4,
            "delta_BACC": b4 - b0,
        })

participant_effects = pd.DataFrame(participant_rows)

# -----------------------------------------------------------------------------
# 5. Training provenance audit.
# -----------------------------------------------------------------------------
member_rows = []
member_issues = []

for p in sorted((S18S_ROOT / "diagnostics").glob("A4MEM-*.json")):
    d = _load_json(p)
    if d.get("r8_policy_sha256") != R8_STAGE18_POLICY_SHA256:
        continue

    prov = d.get("training_provenance")
    if not isinstance(prov, dict):
        prov = {}

    member_rows.append({
        "file": p.name,
        "status": d.get("status"),
        "branch": d.get("branch"),
        "member": d.get("member"),
        "recipe_id": d.get("recipe_id"),
        "execution_device": d.get("execution_device"),
        "epochs_completed": prov.get("epochs_completed"),
        "best_epoch": prov.get("best_epoch"),
        "test_loaded_after_checkpoint_sealed": d.get("test_loaded_after_checkpoint_sealed"),
        "test_set_used_for_selection": d.get("test_set_used_for_selection"),
        "class_weight_frozen_from_a0": d.get("class_weight_frozen_from_a0"),
    })

    if d.get("status") == "SUCCESS":
        if d.get("test_loaded_after_checkpoint_sealed") is not True:
            member_issues.append([p.name, "TEST_FIREWALL"])
        if d.get("test_set_used_for_selection") is not False:
            member_issues.append([p.name, "TEST_SELECTION"])
        if d.get("class_weight_frozen_from_a0") is not True:
            member_issues.append([p.name, "CLASS_WEIGHT"])
        if not d.get("recipe_id"):
            member_issues.append([p.name, "RECIPE"])

member_df = pd.DataFrame(member_rows)

if member_issues:
    raise RuntimeError(
        "STAGE18S_MEMBER_PROVENANCE_ANALYSIS_ISSUES:"
        + json.dumps(member_issues[:30])
    )

training_summary_rows = []
for branch, g in member_df[member_df["status"] == "SUCCESS"].groupby("branch"):
    ep = pd.to_numeric(g["epochs_completed"], errors="coerce").dropna()
    be = pd.to_numeric(g["best_epoch"], errors="coerce").dropna()
    training_summary_rows.append({
        "branch": branch,
        "successful_member_receipts": len(g),
        "recipe_ids": "|".join(sorted(set(g["recipe_id"].dropna().astype(str)))),
        "epochs_min": None if not len(ep) else float(ep.min()),
        "epochs_median": None if not len(ep) else float(ep.median()),
        "epochs_max": None if not len(ep) else float(ep.max()),
        "best_epoch_median": None if not len(be) else float(be.median()),
    })

training_summary = pd.DataFrame(training_summary_rows)

# -----------------------------------------------------------------------------
# 6. Decision-support summaries — descriptive only.
# -----------------------------------------------------------------------------
anchor_sign_counts = Counter(anchor_summary["sign_consistency"].tolist())
budget_flip_count = int((budget_summary["sign_flip_count"] > 0).sum())

anchor_sd = pd.to_numeric(
    anchor_summary["delta_BACC_sd"],
    errors="coerce",
).dropna()

decision_support = {
    "artifact_id": "P02-STAGE18S-R1-DECISION-SUPPORT-SUMMARY",
    "supplement_id": S18S_ID,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "PASS",
    "claim_scope": "POST_HOC_SENSITIVITY_DESCRIPTIVE",
    "three_repeat_anchor_trajectories": len(anchor_summary),
    "anchor_sign_consistency_counts": dict(sorted(anchor_sign_counts.items())),
    "median_repeat_sd_delta_BACC": (
        None
        if not len(anchor_sd)
        else float(anchor_sd.median())
    ),
    "p90_repeat_sd_delta_BACC": (
        None
        if not len(anchor_sd)
        else float(anchor_sd.quantile(0.90))
    ),
    "six_budget_mr00_trajectories": len(budget_summary),
    "mr00_trajectories_with_sign_flip": budget_flip_count,
    "mr00_sign_flip_fraction": (
        budget_flip_count / len(budget_summary)
        if len(budget_summary)
        else None
    ),
    "participant_effect_rows": len(participant_effects),
    "training_provenance_issues": 0,
    "interpretation_rules": {
        "three_repeat_results_are_descriptive_not_five_repeat_confirmatory": True,
        "budget_probe_is_post_hoc": True,
        "do_not_merge_into_canonical_G18": True,
        "do_not_claim_full_buildbook_equivalence": True,
    },
}

# -----------------------------------------------------------------------------
# 7. Persist analysis to both supplement root and canonical runtime analysis
#    inputs. These are new supplemental files only; no canonical Stage18 file is
#    overwritten.
# -----------------------------------------------------------------------------
analysis_dir = S18S_ROOT / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)
runtime_analysis = STORE_ROOT / "analysis_inputs"

effects.to_csv(
    analysis_dir / "combined_cell_effects.csv",
    index=False,
)
anchor_summary.to_csv(
    analysis_dir / "three_repeat_anchor_stability.csv",
    index=False,
)
budget_summary.to_csv(
    analysis_dir / "six_budget_mr00_sensitivity.csv",
    index=False,
)
participant_effects.to_csv(
    analysis_dir / "participant_paired_effects.csv",
    index=False,
)
member_df.to_csv(
    analysis_dir / "member_training_provenance.csv",
    index=False,
)
training_summary.to_csv(
    analysis_dir / "training_provenance_summary.csv",
    index=False,
)
_r8_pkg_atomic_json(
    analysis_dir / "decision_support_summary.json",
    decision_support,
)

# Project-visible copies with unique names.
for src_name, dst_name in [
    ("combined_cell_effects.csv", "stage18S_R1_combined_cell_effects.csv"),
    ("three_repeat_anchor_stability.csv", "stage18S_R1_three_repeat_anchor_stability.csv"),
    ("six_budget_mr00_sensitivity.csv", "stage18S_R1_six_budget_mr00_sensitivity.csv"),
    ("participant_paired_effects.csv", "stage18S_R1_participant_paired_effects.csv"),
    ("member_training_provenance.csv", "stage18S_R1_member_training_provenance.csv"),
    ("training_provenance_summary.csv", "stage18S_R1_training_provenance_summary.csv"),
]:
    srcp = analysis_dir / src_name
    dstp = runtime_analysis / dst_name
    # New supplement paths only; overwrite is allowed only if byte-identical.
    if dstp.exists():
        if _s18s_sha256(srcp) != _s18s_sha256(dstp):
            raise RuntimeError(
                "STAGE18S_RUNTIME_ANALYSIS_EXISTING_CONFLICT:"
                + dst_name
            )
    else:
        shutil.copy2(srcp, dstp)

decision_path = (
    runtime_analysis
    / "stage18S_R1_decision_support_summary.json"
)
if decision_path.exists():
    old = _load_json(decision_path)
    # Result reruns are allowed only when they close to identical key identity;
    # replace timestamped detail only in supplement root, not here.
    for k in (
        "supplement_id",
        "status",
        "claim_scope",
        "three_repeat_anchor_trajectories",
        "six_budget_mr00_trajectories",
    ):
        if old.get(k) != decision_support.get(k):
            raise RuntimeError(
                "STAGE18S_DECISION_SUMMARY_CONFLICT:" + k
            )
else:
    _r8_pkg_atomic_json(
        decision_path,
        decision_support,
    )

supplement_gate = {
    "artifact_id": "P02-STAGE18S-R1-SUPPLEMENT-GATE",
    "supplement_id": S18S_ID,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "PASS",
    "canonical_stage18_modified": False,
    "canonical_G18_modified": False,
    "execution_receipt_status": _exec_receipt.get("status"),
    "planned_cells": len(S18S_PLAN),
    "terminal_counts": _exec_receipt.get("terminal_counts"),
    "member_status_counts": _exec_receipt.get("member_status_counts"),
    "three_repeat_anchor_trajectories": len(anchor_summary),
    "six_budget_mr00_trajectories": len(budget_summary),
    "participant_effect_rows": len(participant_effects),
    "training_provenance_issues": 0,
    "claim_scope": "POST_HOC_SENSITIVITY_DESCRIPTIVE",
    "ready_for_downstream_preservation": True,
}
_r8_pkg_atomic_json(
    analysis_dir / "supplement_gate.json",
    supplement_gate,
)

# Explicit handoff note.
handoff = {
    "artifact_id": "P02-STAGE18S-R1-RUNTIME-EVIDENCE-HANDOFF",
    "supplement_id": S18S_ID,
    "status": "PASS",
    "canonical_stage18_remains_source_of_G18": True,
    "supplement_is_additional_sensitivity_evidence": True,
    "analysis_inputs": [
        "analysis_inputs/stage18S_R1_combined_cell_effects.csv",
        "analysis_inputs/stage18S_R1_three_repeat_anchor_stability.csv",
        "analysis_inputs/stage18S_R1_six_budget_mr00_sensitivity.csv",
        "analysis_inputs/stage18S_R1_participant_paired_effects.csv",
        "analysis_inputs/stage18S_R1_member_training_provenance.csv",
        "analysis_inputs/stage18S_R1_training_provenance_summary.csv",
        "analysis_inputs/stage18S_R1_decision_support_summary.json",
        "analysis_inputs/stage18S_balanced_sensitivity_R1_preexecution_freeze.json",
    ],
    "reporting_limits": {
        "post_hoc": True,
        "not_five_repeat_confirmatory": True,
        "not_full_buildbook_equivalent": True,
        "do_not_overwrite_original_stage18_protocol_deviation": True,
    },
}

handoff_path = (
    STORE_ROOT
    / "handoffs"
    / "stage18S_R1_sensitivity_evidence.json"
)
_r8_pkg_atomic_json(
    handoff_path,
    handoff,
)

S18S_ASSERT_CANONICAL_IMMUTABLE()

print("\nTHREE-REPEAT ANCHOR STABILITY")
print("--------------------------------")
print(
    anchor_summary[
        [
            "dataset_id", "role_id", "condition_short", "budget_short",
            "delta_BACC_mean", "delta_BACC_sd", "sign_consistency",
            "repeat0", "repeat1", "repeat2",
        ]
    ].round(4).to_string(index=False)
)

print("\nSIX-BUDGET MR00 SENSITIVITY")
print("--------------------------------")
print(
    budget_summary[
        [
            "dataset_id", "role_id", "condition_short",
            "sign_flip_count", "delta_BACC_mean",
            "peak_budget", "peak_delta_BACC",
            "trough_budget", "trough_delta_BACC",
        ]
    ].round(4).to_string(index=False)
)

print("\nTRAINING PROVENANCE")
print("--------------------------------")
print(training_summary.round(2).to_string(index=False))

print("\nDECISION SUPPORT")
print("--------------------------------")
print(json.dumps(decision_support, indent=2, default=str))

print("\nSUPPLEMENT GATE")
print("--------------------------------")
print(json.dumps(supplement_gate, indent=2, default=str))

print("\nTRUSTED CLOSURE MARKER:")
print("IHARQ_P02_STAGE18S_BALANCED_SENSITIVITY_R1_3_SUPPLEMENT_PASS")


STAGE 18S R1.3 — ANALYSIS / CLOSURE

THREE-REPEAT ANCHOR STABILITY
--------------------------------
  dataset_id role_id condition_short budget_short  delta_BACC_mean  delta_BACC_sd sign_consistency  repeat0  repeat1  repeat2
BNCI2014_001  NEURAL              C1         FULL           0.0012         0.0414            MIXED  -0.0451   0.0347   0.0139
BNCI2014_001  NEURAL              C1            1          -0.0058         0.0348            MIXED   0.0069  -0.0451   0.0208
BNCI2014_001  NEURAL              C1            8          -0.0486         0.0451     ALL_NEGATIVE  -0.0035  -0.0486  -0.0938
BNCI2014_001  NEURAL              C2         FULL          -0.0174         0.0159            MIXED  -0.0313   0.0000  -0.0208
BNCI2014_001  NEURAL              C2            1          -0.0162         0.0315            MIXED  -0.0035  -0.0521   0.0069
BNCI2014_001  NEURAL              C2            8          -0.0208         0.0331            MIXED   0.0104  -0.0174  -0.0556
BNCI2014_001  NEUR

In [7]:
# =============================================================================
# STAGE 18S R1.3 — DOWNSTREAM READINESS BOUNDARY
# =============================================================================

from pathlib import Path
import json

_sroot = (
    Path(STORE.root)
    / "supplements"
    / "stage18S_balanced_sensitivity_R1"
)
_gate = _sroot / "analysis" / "supplement_gate.json"
_exec = _sroot / "analysis" / "execution_receipt.json"

if not _gate.is_file() or not _exec.is_file():
    raise RuntimeError("STAGE18S_DOWNSTREAM_READINESS_FILES_MISSING")

gd = json.loads(_gate.read_text(encoding="utf-8"))
ed = json.loads(_exec.read_text(encoding="utf-8"))

if gd.get("status") != "PASS":
    raise RuntimeError("STAGE18S_SUPPLEMENT_GATE_NOT_PASS")
if ed.get("status") != "PASS":
    raise RuntimeError("STAGE18S_EXECUTION_RECEIPT_NOT_PASS")

if not SESSION.runner.accepted("18"):
    raise RuntimeError("CANONICAL_STAGE18_NO_LONGER_ACCEPTED")

print("=" * 110)
print("STAGE18S R1.3 DOWNSTREAM READINESS — PASS")
print("=" * 110)
print(json.dumps({
    "canonical_stage18": "ACCEPTED_IMMUTABLE",
    "supplement_gate": gd.get("status"),
    "supplement_claim_scope": gd.get("claim_scope"),
    "ready_for_stage18U_then_19_24": True,
}, indent=2))
print("\nIHARQ_P02_STAGE18S_R1_3_DOWNSTREAM_READY")


STAGE18S R1.3 DOWNSTREAM READINESS — PASS
{
  "canonical_stage18": "ACCEPTED_IMMUTABLE",
  "supplement_gate": "PASS",
  "supplement_claim_scope": "POST_HOC_SENSITIVITY_DESCRIPTIVE",
  "ready_for_stage18U_then_19_24": true
}

IHARQ_P02_STAGE18S_R1_3_DOWNSTREAM_READY


### Optional interruption behavior

If you manually interrupt Stage18S, do **not** immediately start another execution cell while old worker threads are still using CUDA. The scheduler is resumable from durable supplement evidence after the workers exit or after a kernel restart followed by the same rehydration sequence.

## Stage 18U — Same-notebook additional full-execution ablation dispatcher; current extra set empty

**Gate:** `G18U`  
**Inputs:** Stage05 frozen unlock matrix  
**Outputs:** executed extra ablation records or NOT_AUTHORIZED rows

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [11]:
# =============================================================================
# IHARQ P02/L2 — STAGE 18U WITH LIVE PROGRESS HEARTBEAT
#
# Scientific execution is UNCHANGED:
#     result = RUN_STAGE("18U")
#
# The only addition is a read-only monitor thread that reports elapsed time,
# ledger/gate state, and whether Stage18U has been accepted.
# =============================================================================

from pathlib import Path
import json
import threading
import time
from datetime import timedelta

STAGE_ID = "18U"
HEARTBEAT_SECONDS = 5.0

_run_root = Path(
    globals().get("RUN_ROOT", "/kaggle/working/iharq_p02_run")
).resolve()

_ledger_path = _run_root / "stage_ledger" / "stage_18U.json"
_gate_path = _run_root / "gate_results" / "G18U.json"

if "RUN_STAGE" not in globals():
    raise RuntimeError("RUN_STAGE_NOT_AVAILABLE__REHYDRATION_REQUIRED")

if "SESSION" not in globals():
    raise RuntimeError("SESSION_NOT_AVAILABLE__REHYDRATION_REQUIRED")


def _read_json_if_complete(path):
    try:
        if not path.is_file():
            return None
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        # Atomic replacement should normally make this unnecessary, but the
        # progress monitor must never interfere with the scientific stage.
        return None


_stop = threading.Event()
_started = time.monotonic()


def _progress_monitor():
    last_signature = None

    while not _stop.wait(HEARTBEAT_SECONDS):
        elapsed = str(
            timedelta(seconds=int(time.monotonic() - _started))
        )

        ledger = _read_json_if_complete(_ledger_path)
        gate = _read_json_if_complete(_gate_path)

        ledger_status = (
            ledger.get("status")
            if isinstance(ledger, dict)
            else "NOT_WRITTEN_YET"
        )
        gate_status = (
            gate.get("status")
            if isinstance(gate, dict)
            else "NOT_WRITTEN_YET"
        )

        try:
            accepted = bool(SESSION.runner.accepted(STAGE_ID))
        except Exception:
            accepted = False

        signature = (
            ledger_status,
            gate_status,
            accepted,
        )

        # Always print a heartbeat, even if Stage18U has not yet committed an
        # intermediate artifact. This makes silent work visibly alive.
        print(
            f"[Stage18U] elapsed={elapsed} | "
            f"ledger={ledger_status} | "
            f"G18U={gate_status} | "
            f"accepted={accepted}",
            flush=True,
        )

        if signature != last_signature:
            if isinstance(ledger, dict):
                # Print useful stage-level fields without dumping huge payloads.
                summary = {
                    k: ledger.get(k)
                    for k in (
                        "stage_id",
                        "status",
                        "planned",
                        "executed",
                        "success",
                        "failed",
                        "not_authorized",
                        "conditional_skip",
                    )
                    if k in ledger
                }
                if summary:
                    print(
                        "    ledger_update="
                        + json.dumps(summary, sort_keys=True, default=str),
                        flush=True,
                    )

            if isinstance(gate, dict):
                summary = {
                    k: gate.get(k)
                    for k in (
                        "gate_id",
                        "status",
                        "decision",
                        "reason",
                    )
                    if k in gate
                }
                if summary:
                    print(
                        "    gate_update="
                        + json.dumps(summary, sort_keys=True, default=str),
                        flush=True,
                    )

            last_signature = signature


print("=" * 110)
print("STAGE 18U — LIVE PROGRESS")
print("=" * 110)
print("run_root =", _run_root)
print("ledger   =", _ledger_path)
print("gate     =", _gate_path)
print("heartbeat_seconds =", HEARTBEAT_SECONDS)

try:
    print(
        "accepted_before =",
        bool(SESSION.runner.accepted(STAGE_ID)),
        flush=True,
    )
except Exception:
    print("accepted_before = UNKNOWN", flush=True)

print(
    "\nNOTE: the frozen Stage18U extra set is currently described as empty. "
    "If so, there may be no meaningful N/M training progress; the heartbeat "
    "will still show that the dispatcher is alive until it commits the ledger "
    "and G18U closure.\n",
    flush=True,
)

_monitor = threading.Thread(
    target=_progress_monitor,
    name="IHARQ-STAGE18U-PROGRESS-MONITOR",
    daemon=True,
)
_monitor.start()

_stage_exc = None

try:
    # -------------------------------------------------------------------------
    # EXACT SCIENTIFIC/DISPATCH EXECUTION — unchanged from the original cell.
    # -------------------------------------------------------------------------
    result = RUN_STAGE("18U")

except BaseException as exc:
    _stage_exc = exc

finally:
    _stop.set()
    _monitor.join(timeout=2.0)

elapsed = str(timedelta(seconds=int(time.monotonic() - _started)))
ledger = _read_json_if_complete(_ledger_path)
gate = _read_json_if_complete(_gate_path)

print("\n" + "=" * 110)
print("STAGE 18U — FINAL STATUS")
print("=" * 110)
print("elapsed =", elapsed)

try:
    accepted_after = bool(SESSION.runner.accepted(STAGE_ID))
except Exception:
    accepted_after = False

print("accepted_after =", accepted_after)

if isinstance(ledger, dict):
    print("\nSTAGE LEDGER:")
    print(json.dumps(ledger, indent=2, default=str))
else:
    print("\nSTAGE LEDGER: not present")

if isinstance(gate, dict):
    print("\nG18U:")
    print(json.dumps(gate, indent=2, default=str))
else:
    print("\nG18U: not present")

if _stage_exc is not None:
    print(
        "\nSTAGE18U TERMINATED WITH EXCEPTION:",
        type(_stage_exc).__name__,
        str(_stage_exc),
        flush=True,
    )
    raise _stage_exc

print("\nRUN_STAGE('18U') RETURN:")
try:
    print(json.dumps(result, indent=2, default=str))
except Exception:
    print(repr(result))

if not accepted_after:
    raise RuntimeError("STAGE18U_RETURNED_BUT_WAS_NOT_ACCEPTED")

print("\nTRUSTED PROGRESS-WRAPPER MARKER:")
print("IHARQ_P02_STAGE18U_PROGRESS_WRAPPER_COMPLETE")


STAGE 18U — LIVE PROGRESS
run_root = /kaggle/working/iharq_p02_run
ledger   = /kaggle/working/iharq_p02_run/stage_ledger/stage_18U.json
gate     = /kaggle/working/iharq_p02_run/gate_results/G18U.json
heartbeat_seconds = 5.0
accepted_before = True

NOTE: the frozen Stage18U extra set is currently described as empty. If so, there may be no meaningful N/M training progress; the heartbeat will still show that the dispatcher is alive until it commits the ledger and G18U closure.


STAGE 18U — FINAL STATUS
elapsed = 0:00:00
accepted_after = True

STAGE LEDGER:
{
  "stage_id": "18U",
  "status": "SUCCESS",
  "attempt_id": "6c35912e38f7",
  "inputs": {},
  "input_hashes": {},
  "outputs": {
    "status": "PASS",
    "decision": "NO_ADDITIONAL_FULL_EXECUTION_ABLATION_UNLOCKED",
    "authority_check": true,
    "contract_check": true,
    "artifact": "stage_artifacts/18U_unlock.json"
  },
  "output_hashes": {
    "status": "727e9bd17b077ba162e13ba877135cc6fbc6b30f83da0b34b3e20233f1cc885f",
    "

## Stage 19 — Failure/missingness/negative-result accounting

**Gate:** `G19`  
**Inputs:** all branch ledgers  
**Outputs:** FailureCaseIndex + NegativeResultNote + diagnostics

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [13]:
# =============================================================================
# IHARQ P02/L2 — STAGE 19 WITH LIVE PROGRESS HEARTBEAT
# Failure/missingness/negative-result accounting
#
# Scientific/governed execution is UNCHANGED:
#     result = RUN_STAGE("19")
#
# The monitor is READ-ONLY. It reports elapsed time, ledger/gate state,
# acceptance, and any newly observed ledger counters while RUN_STAGE executes.
# =============================================================================

from pathlib import Path
import json
import threading
import time
from datetime import timedelta

STAGE_ID = "19"
GATE_ID = "G19"
HEARTBEAT_SECONDS = 5.0

_run_root = Path(
    globals().get("RUN_ROOT", "/kaggle/working/iharq_p02_run")
).resolve()

_ledger_path = _run_root / "stage_ledger" / f"stage_{STAGE_ID}.json"
_gate_path = _run_root / "gate_results" / f"{GATE_ID}.json"

if "RUN_STAGE" not in globals():
    raise RuntimeError("RUN_STAGE_NOT_AVAILABLE__REHYDRATION_REQUIRED")
if "SESSION" not in globals():
    raise RuntimeError("SESSION_NOT_AVAILABLE__REHYDRATION_REQUIRED")


def _read_json(path):
    try:
        if not path.is_file():
            return None
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        # Monitoring must never interfere with governed execution.
        return None


def _accepted():
    try:
        obj = SESSION.runner.accepted(STAGE_ID)
        return bool(obj), obj
    except Exception:
        return False, None


def _compact(d):
    """Return useful progress-like scalar fields without dumping large payloads."""
    if not isinstance(d, dict):
        return {}

    preferred = (
        "stage_id", "status", "planned", "total", "expected",
        "processed", "completed", "executed", "written",
        "success", "successful", "failed", "failure",
        "skipped", "conditional_skip", "dependency_blocked",
        "resource_blocked", "not_authorized", "records",
        "rows", "artifacts", "files", "missing", "negative_results",
        "gate_id", "decision", "reason",
    )

    out = {}
    for k in preferred:
        if k in d and isinstance(d[k], (str, int, float, bool, type(None))):
            out[k] = d[k]

    # Also surface other simple numeric counters that look progress-related.
    for k, v in d.items():
        lk = str(k).lower()
        if (
            k not in out
            and isinstance(v, (int, float))
            and any(tok in lk for tok in (
                "count", "total", "done", "complete", "processed",
                "success", "fail", "skip", "missing", "record",
                "row", "artifact", "file",
            ))
        ):
            out[k] = v

    return out


_stop = threading.Event()
_started = time.monotonic()


def _progress_monitor():
    last_ledger = None
    last_gate = None
    last_accept = None

    # Immediate heartbeat, then every HEARTBEAT_SECONDS.
    while not _stop.is_set():
        elapsed = str(
            timedelta(seconds=int(time.monotonic() - _started))
        )

        ledger = _read_json(_ledger_path)
        gate = _read_json(_gate_path)
        accepted, _ = _accepted()

        ledger_status = (
            ledger.get("status")
            if isinstance(ledger, dict)
            else "NOT_WRITTEN_YET"
        )
        gate_status = (
            gate.get("status")
            if isinstance(gate, dict)
            else "NOT_WRITTEN_YET"
        )

        print(
            f"[Stage {STAGE_ID}] elapsed={elapsed} | "
            f"ledger={ledger_status} | "
            f"{GATE_ID}={gate_status} | "
            f"accepted={accepted}",
            flush=True,
        )

        ledger_compact = _compact(ledger)
        gate_compact = _compact(gate)

        if ledger_compact and ledger_compact != last_ledger:
            print(
                "    ledger_update="
                + json.dumps(
                    ledger_compact,
                    sort_keys=True,
                    default=str,
                ),
                flush=True,
            )
            last_ledger = ledger_compact

        if gate_compact and gate_compact != last_gate:
            print(
                "    gate_update="
                + json.dumps(
                    gate_compact,
                    sort_keys=True,
                    default=str,
                ),
                flush=True,
            )
            last_gate = gate_compact

        if accepted != last_accept:
            print(
                f"    accepted_update={accepted}",
                flush=True,
            )
            last_accept = accepted

        if _stop.wait(HEARTBEAT_SECONDS):
            break


print("=" * 112)
print(f"STAGE {STAGE_ID} — LIVE PROGRESS")
print("=" * 112)
print("description =", "Failure/missingness/negative-result accounting")
print("run_root    =", _run_root)
print("ledger      =", _ledger_path)
print("gate        =", _gate_path)
print("heartbeat   =", HEARTBEAT_SECONDS, "seconds")

accepted_before, _accepted_obj_before = _accepted()
print("accepted_before =", accepted_before)
print(
    "\nThe heartbeat remains visible even if this stage commits its governed "
    "ledger only near the end. No scientific/runtime behavior is changed.\n",
    flush=True,
)

_monitor = threading.Thread(
    target=_progress_monitor,
    name=f"IHARQ-STAGE{STAGE_ID}-PROGRESS-MONITOR",
    daemon=True,
)
_monitor.start()

_stage_exc = None

try:
    # -------------------------------------------------------------------------
    # EXACT GOVERNED EXECUTION — unchanged from the original notebook.
    # -------------------------------------------------------------------------
    result = RUN_STAGE(STAGE_ID)

except BaseException as exc:
    _stage_exc = exc

finally:
    _stop.set()
    _monitor.join(timeout=2.0)

elapsed = str(timedelta(seconds=int(time.monotonic() - _started)))
ledger = _read_json(_ledger_path)
gate = _read_json(_gate_path)
accepted_after, _accepted_obj_after = _accepted()

print("\n" + "=" * 112)
print(f"STAGE {STAGE_ID} — FINAL STATUS")
print("=" * 112)
print("elapsed        =", elapsed)
print("accepted_after =", accepted_after)

if isinstance(ledger, dict):
    print("\nSTAGE LEDGER:")
    print(json.dumps(ledger, indent=2, default=str))
else:
    print("\nSTAGE LEDGER: not present")

if isinstance(gate, dict):
    print(f"\n{GATE_ID}:")
    print(json.dumps(gate, indent=2, default=str))
else:
    print(f"\n{GATE_ID}: not present")

if _stage_exc is not None:
    print(
        f"\nSTAGE {STAGE_ID} TERMINATED WITH EXCEPTION:",
        type(_stage_exc).__name__,
        str(_stage_exc),
        flush=True,
    )
    raise _stage_exc

print(f"\nRUN_STAGE('{STAGE_ID}') RETURN:")
try:
    print(json.dumps(result, indent=2, default=str))
except Exception:
    print(repr(result))

if not accepted_after:
    raise RuntimeError(
        f"STAGE{STAGE_ID}_RETURNED_BUT_WAS_NOT_ACCEPTED"
    )

print("\nTRUSTED PROGRESS-WRAPPER MARKER:")
print(f"IHARQ_P02_STAGE{STAGE_ID}_PROGRESS_WRAPPER_COMPLETE")


STAGE 19 — LIVE PROGRESS
description = Failure/missingness/negative-result accounting
run_root    = /kaggle/working/iharq_p02_run
ledger      = /kaggle/working/iharq_p02_run/stage_ledger/stage_19.json
gate        = /kaggle/working/iharq_p02_run/gate_results/G19.json
heartbeat   = 5.0 seconds
accepted_before = True

The heartbeat remains visible even if this stage commits its governed ledger only near the end. No scientific/runtime behavior is changed.

[Stage 19] elapsed=0:00:00 | ledger=SUCCESS | G19=PASS | accepted=True
    ledger_update={"stage_id": "19", "status": "SUCCESS"}
    gate_update={"gate_id": "G19", "stage_id": "19", "status": "PASS"}
    accepted_update=True

STAGE 19 — FINAL STATUS
elapsed        = 0:00:00
accepted_after = True

STAGE LEDGER:
{
  "stage_id": "19",
  "status": "SUCCESS",
  "attempt_id": "243091602665",
  "inputs": {},
  "input_hashes": {},
  "outputs": {
    "status": "PASS",
    "failure_evidence_aggregation": "COMPLETE",
    "failures": 648,
    "negat

## Stage 20 — Downstream-readiness validation

**Gate:** `G20`  
**Inputs:** all records  
**Outputs:** Layer2ReadinessReport

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [14]:
# =============================================================================
# IHARQ P02/L2 — STAGE 20 WITH LIVE PROGRESS HEARTBEAT
# Downstream-readiness validation
#
# Scientific/governed execution is UNCHANGED:
#     result = RUN_STAGE("20")
#
# The monitor is READ-ONLY. It reports elapsed time, ledger/gate state,
# acceptance, and any newly observed ledger counters while RUN_STAGE executes.
# =============================================================================

from pathlib import Path
import json
import threading
import time
from datetime import timedelta

STAGE_ID = "20"
GATE_ID = "G20"
HEARTBEAT_SECONDS = 5.0

_run_root = Path(
    globals().get("RUN_ROOT", "/kaggle/working/iharq_p02_run")
).resolve()

_ledger_path = _run_root / "stage_ledger" / f"stage_{STAGE_ID}.json"
_gate_path = _run_root / "gate_results" / f"{GATE_ID}.json"

if "RUN_STAGE" not in globals():
    raise RuntimeError("RUN_STAGE_NOT_AVAILABLE__REHYDRATION_REQUIRED")
if "SESSION" not in globals():
    raise RuntimeError("SESSION_NOT_AVAILABLE__REHYDRATION_REQUIRED")


def _read_json(path):
    try:
        if not path.is_file():
            return None
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        # Monitoring must never interfere with governed execution.
        return None


def _accepted():
    try:
        obj = SESSION.runner.accepted(STAGE_ID)
        return bool(obj), obj
    except Exception:
        return False, None


def _compact(d):
    """Return useful progress-like scalar fields without dumping large payloads."""
    if not isinstance(d, dict):
        return {}

    preferred = (
        "stage_id", "status", "planned", "total", "expected",
        "processed", "completed", "executed", "written",
        "success", "successful", "failed", "failure",
        "skipped", "conditional_skip", "dependency_blocked",
        "resource_blocked", "not_authorized", "records",
        "rows", "artifacts", "files", "missing", "negative_results",
        "gate_id", "decision", "reason",
    )

    out = {}
    for k in preferred:
        if k in d and isinstance(d[k], (str, int, float, bool, type(None))):
            out[k] = d[k]

    # Also surface other simple numeric counters that look progress-related.
    for k, v in d.items():
        lk = str(k).lower()
        if (
            k not in out
            and isinstance(v, (int, float))
            and any(tok in lk for tok in (
                "count", "total", "done", "complete", "processed",
                "success", "fail", "skip", "missing", "record",
                "row", "artifact", "file",
            ))
        ):
            out[k] = v

    return out


_stop = threading.Event()
_started = time.monotonic()


def _progress_monitor():
    last_ledger = None
    last_gate = None
    last_accept = None

    # Immediate heartbeat, then every HEARTBEAT_SECONDS.
    while not _stop.is_set():
        elapsed = str(
            timedelta(seconds=int(time.monotonic() - _started))
        )

        ledger = _read_json(_ledger_path)
        gate = _read_json(_gate_path)
        accepted, _ = _accepted()

        ledger_status = (
            ledger.get("status")
            if isinstance(ledger, dict)
            else "NOT_WRITTEN_YET"
        )
        gate_status = (
            gate.get("status")
            if isinstance(gate, dict)
            else "NOT_WRITTEN_YET"
        )

        print(
            f"[Stage {STAGE_ID}] elapsed={elapsed} | "
            f"ledger={ledger_status} | "
            f"{GATE_ID}={gate_status} | "
            f"accepted={accepted}",
            flush=True,
        )

        ledger_compact = _compact(ledger)
        gate_compact = _compact(gate)

        if ledger_compact and ledger_compact != last_ledger:
            print(
                "    ledger_update="
                + json.dumps(
                    ledger_compact,
                    sort_keys=True,
                    default=str,
                ),
                flush=True,
            )
            last_ledger = ledger_compact

        if gate_compact and gate_compact != last_gate:
            print(
                "    gate_update="
                + json.dumps(
                    gate_compact,
                    sort_keys=True,
                    default=str,
                ),
                flush=True,
            )
            last_gate = gate_compact

        if accepted != last_accept:
            print(
                f"    accepted_update={accepted}",
                flush=True,
            )
            last_accept = accepted

        if _stop.wait(HEARTBEAT_SECONDS):
            break


print("=" * 112)
print(f"STAGE {STAGE_ID} — LIVE PROGRESS")
print("=" * 112)
print("description =", "Downstream-readiness validation")
print("run_root    =", _run_root)
print("ledger      =", _ledger_path)
print("gate        =", _gate_path)
print("heartbeat   =", HEARTBEAT_SECONDS, "seconds")

accepted_before, _accepted_obj_before = _accepted()
print("accepted_before =", accepted_before)
print(
    "\nThe heartbeat remains visible even if this stage commits its governed "
    "ledger only near the end. No scientific/runtime behavior is changed.\n",
    flush=True,
)

_monitor = threading.Thread(
    target=_progress_monitor,
    name=f"IHARQ-STAGE{STAGE_ID}-PROGRESS-MONITOR",
    daemon=True,
)
_monitor.start()

_stage_exc = None

try:
    # -------------------------------------------------------------------------
    # EXACT GOVERNED EXECUTION — unchanged from the original notebook.
    # -------------------------------------------------------------------------
    result = RUN_STAGE(STAGE_ID)

except BaseException as exc:
    _stage_exc = exc

finally:
    _stop.set()
    _monitor.join(timeout=2.0)

elapsed = str(timedelta(seconds=int(time.monotonic() - _started)))
ledger = _read_json(_ledger_path)
gate = _read_json(_gate_path)
accepted_after, _accepted_obj_after = _accepted()

print("\n" + "=" * 112)
print(f"STAGE {STAGE_ID} — FINAL STATUS")
print("=" * 112)
print("elapsed        =", elapsed)
print("accepted_after =", accepted_after)

if isinstance(ledger, dict):
    print("\nSTAGE LEDGER:")
    print(json.dumps(ledger, indent=2, default=str))
else:
    print("\nSTAGE LEDGER: not present")

if isinstance(gate, dict):
    print(f"\n{GATE_ID}:")
    print(json.dumps(gate, indent=2, default=str))
else:
    print(f"\n{GATE_ID}: not present")

if _stage_exc is not None:
    print(
        f"\nSTAGE {STAGE_ID} TERMINATED WITH EXCEPTION:",
        type(_stage_exc).__name__,
        str(_stage_exc),
        flush=True,
    )
    raise _stage_exc

print(f"\nRUN_STAGE('{STAGE_ID}') RETURN:")
try:
    print(json.dumps(result, indent=2, default=str))
except Exception:
    print(repr(result))

if not accepted_after:
    raise RuntimeError(
        f"STAGE{STAGE_ID}_RETURNED_BUT_WAS_NOT_ACCEPTED"
    )

print("\nTRUSTED PROGRESS-WRAPPER MARKER:")
print(f"IHARQ_P02_STAGE{STAGE_ID}_PROGRESS_WRAPPER_COMPLETE")


STAGE 20 — LIVE PROGRESS
description = Downstream-readiness validation
run_root    = /kaggle/working/iharq_p02_run
ledger      = /kaggle/working/iharq_p02_run/stage_ledger/stage_20.json
gate        = /kaggle/working/iharq_p02_run/gate_results/G20.json
heartbeat   = 5.0 seconds
accepted_before = False

The heartbeat remains visible even if this stage commits its governed ledger only near the end. No scientific/runtime behavior is changed.

[Stage 20] elapsed=0:00:00 | ledger=NOT_WRITTEN_YET | G20=NOT_WRITTEN_YET | accepted=False
    accepted_update=False

STAGE 20 — FINAL STATUS
elapsed        = 0:00:00
accepted_after = True

STAGE LEDGER:
{
  "stage_id": "20",
  "status": "SUCCESS",
  "attempt_id": "b5a19c3412a2",
  "inputs": {},
  "input_hashes": {},
  "outputs": {
    "status": "PASS",
    "runtime_evidence_checked": true,
    "prediction_partitions": 1014,
    "a0": true,
    "a4": true,
    "c4c5": true,
    "p03": {
      "status": "PASS",
      "missing_prediction_fields": [],
  

## Stage 21 — Figure-source and table-source datasets

**Gate:** `G21`  
**Inputs:** validated evidence  
**Outputs:** analysis/L10 source tables

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [15]:
# =============================================================================
# IHARQ P02/L2 — STAGE 21 WITH LIVE PROGRESS HEARTBEAT
# Figure-source and table-source datasets
#
# Scientific/governed execution is UNCHANGED:
#     result = RUN_STAGE("21")
#
# The monitor is READ-ONLY. It reports elapsed time, ledger/gate state,
# acceptance, and any newly observed ledger counters while RUN_STAGE executes.
# =============================================================================

from pathlib import Path
import json
import threading
import time
from datetime import timedelta

STAGE_ID = "21"
GATE_ID = "G21"
HEARTBEAT_SECONDS = 5.0

_run_root = Path(
    globals().get("RUN_ROOT", "/kaggle/working/iharq_p02_run")
).resolve()

_ledger_path = _run_root / "stage_ledger" / f"stage_{STAGE_ID}.json"
_gate_path = _run_root / "gate_results" / f"{GATE_ID}.json"

if "RUN_STAGE" not in globals():
    raise RuntimeError("RUN_STAGE_NOT_AVAILABLE__REHYDRATION_REQUIRED")
if "SESSION" not in globals():
    raise RuntimeError("SESSION_NOT_AVAILABLE__REHYDRATION_REQUIRED")


def _read_json(path):
    try:
        if not path.is_file():
            return None
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        # Monitoring must never interfere with governed execution.
        return None


def _accepted():
    try:
        obj = SESSION.runner.accepted(STAGE_ID)
        return bool(obj), obj
    except Exception:
        return False, None


def _compact(d):
    """Return useful progress-like scalar fields without dumping large payloads."""
    if not isinstance(d, dict):
        return {}

    preferred = (
        "stage_id", "status", "planned", "total", "expected",
        "processed", "completed", "executed", "written",
        "success", "successful", "failed", "failure",
        "skipped", "conditional_skip", "dependency_blocked",
        "resource_blocked", "not_authorized", "records",
        "rows", "artifacts", "files", "missing", "negative_results",
        "gate_id", "decision", "reason",
    )

    out = {}
    for k in preferred:
        if k in d and isinstance(d[k], (str, int, float, bool, type(None))):
            out[k] = d[k]

    # Also surface other simple numeric counters that look progress-related.
    for k, v in d.items():
        lk = str(k).lower()
        if (
            k not in out
            and isinstance(v, (int, float))
            and any(tok in lk for tok in (
                "count", "total", "done", "complete", "processed",
                "success", "fail", "skip", "missing", "record",
                "row", "artifact", "file",
            ))
        ):
            out[k] = v

    return out


_stop = threading.Event()
_started = time.monotonic()


def _progress_monitor():
    last_ledger = None
    last_gate = None
    last_accept = None

    # Immediate heartbeat, then every HEARTBEAT_SECONDS.
    while not _stop.is_set():
        elapsed = str(
            timedelta(seconds=int(time.monotonic() - _started))
        )

        ledger = _read_json(_ledger_path)
        gate = _read_json(_gate_path)
        accepted, _ = _accepted()

        ledger_status = (
            ledger.get("status")
            if isinstance(ledger, dict)
            else "NOT_WRITTEN_YET"
        )
        gate_status = (
            gate.get("status")
            if isinstance(gate, dict)
            else "NOT_WRITTEN_YET"
        )

        print(
            f"[Stage {STAGE_ID}] elapsed={elapsed} | "
            f"ledger={ledger_status} | "
            f"{GATE_ID}={gate_status} | "
            f"accepted={accepted}",
            flush=True,
        )

        ledger_compact = _compact(ledger)
        gate_compact = _compact(gate)

        if ledger_compact and ledger_compact != last_ledger:
            print(
                "    ledger_update="
                + json.dumps(
                    ledger_compact,
                    sort_keys=True,
                    default=str,
                ),
                flush=True,
            )
            last_ledger = ledger_compact

        if gate_compact and gate_compact != last_gate:
            print(
                "    gate_update="
                + json.dumps(
                    gate_compact,
                    sort_keys=True,
                    default=str,
                ),
                flush=True,
            )
            last_gate = gate_compact

        if accepted != last_accept:
            print(
                f"    accepted_update={accepted}",
                flush=True,
            )
            last_accept = accepted

        if _stop.wait(HEARTBEAT_SECONDS):
            break


print("=" * 112)
print(f"STAGE {STAGE_ID} — LIVE PROGRESS")
print("=" * 112)
print("description =", "Figure-source and table-source datasets")
print("run_root    =", _run_root)
print("ledger      =", _ledger_path)
print("gate        =", _gate_path)
print("heartbeat   =", HEARTBEAT_SECONDS, "seconds")

accepted_before, _accepted_obj_before = _accepted()
print("accepted_before =", accepted_before)
print(
    "\nThe heartbeat remains visible even if this stage commits its governed "
    "ledger only near the end. No scientific/runtime behavior is changed.\n",
    flush=True,
)

_monitor = threading.Thread(
    target=_progress_monitor,
    name=f"IHARQ-STAGE{STAGE_ID}-PROGRESS-MONITOR",
    daemon=True,
)
_monitor.start()

_stage_exc = None

try:
    # -------------------------------------------------------------------------
    # EXACT GOVERNED EXECUTION — unchanged from the original notebook.
    # -------------------------------------------------------------------------
    result = RUN_STAGE(STAGE_ID)

except BaseException as exc:
    _stage_exc = exc

finally:
    _stop.set()
    _monitor.join(timeout=2.0)

elapsed = str(timedelta(seconds=int(time.monotonic() - _started)))
ledger = _read_json(_ledger_path)
gate = _read_json(_gate_path)
accepted_after, _accepted_obj_after = _accepted()

print("\n" + "=" * 112)
print(f"STAGE {STAGE_ID} — FINAL STATUS")
print("=" * 112)
print("elapsed        =", elapsed)
print("accepted_after =", accepted_after)

if isinstance(ledger, dict):
    print("\nSTAGE LEDGER:")
    print(json.dumps(ledger, indent=2, default=str))
else:
    print("\nSTAGE LEDGER: not present")

if isinstance(gate, dict):
    print(f"\n{GATE_ID}:")
    print(json.dumps(gate, indent=2, default=str))
else:
    print(f"\n{GATE_ID}: not present")

if _stage_exc is not None:
    print(
        f"\nSTAGE {STAGE_ID} TERMINATED WITH EXCEPTION:",
        type(_stage_exc).__name__,
        str(_stage_exc),
        flush=True,
    )
    raise _stage_exc

print(f"\nRUN_STAGE('{STAGE_ID}') RETURN:")
try:
    print(json.dumps(result, indent=2, default=str))
except Exception:
    print(repr(result))

if not accepted_after:
    raise RuntimeError(
        f"STAGE{STAGE_ID}_RETURNED_BUT_WAS_NOT_ACCEPTED"
    )

print("\nTRUSTED PROGRESS-WRAPPER MARKER:")
print(f"IHARQ_P02_STAGE{STAGE_ID}_PROGRESS_WRAPPER_COMPLETE")


STAGE 21 — LIVE PROGRESS
description = Figure-source and table-source datasets
run_root    = /kaggle/working/iharq_p02_run
ledger      = /kaggle/working/iharq_p02_run/stage_ledger/stage_21.json
gate        = /kaggle/working/iharq_p02_run/gate_results/G21.json
heartbeat   = 5.0 seconds
accepted_before = False

The heartbeat remains visible even if this stage commits its governed ledger only near the end. No scientific/runtime behavior is changed.

[Stage 21] elapsed=0:00:00 | ledger=NOT_WRITTEN_YET | G21=NOT_WRITTEN_YET | accepted=False
    accepted_update=False

STAGE 21 — FINAL STATUS
elapsed        = 0:00:00
accepted_after = True

STAGE LEDGER:
{
  "stage_id": "21",
  "status": "SUCCESS",
  "attempt_id": "3d6646e74d1e",
  "inputs": {},
  "input_hashes": {},
  "outputs": {
    "status": "PASS",
    "source_families": 18,
    "c4_c5_export": "PASS",
    "artifact": "stage_artifacts/21_sources.json"
  },
  "output_hashes": {
    "status": "727e9bd17b077ba162e13ba877135cc6fbc6b30f83da0b3

## Stage 22 — Protocol/Analysis/Layer0/EvidenceMap/Layer10/P03 handoffs

**Gate:** `G22`  
**Inputs:** validated bundle  
**Outputs:** governed handoff packets

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [16]:
# =============================================================================
# IHARQ P02/L2 — STAGE 22 WITH LIVE PROGRESS HEARTBEAT
# Protocol/Analysis/Layer0/EvidenceMap/Layer10/P03 handoffs
#
# Scientific/governed execution is UNCHANGED:
#     result = RUN_STAGE("22")
#
# The monitor is READ-ONLY. It reports elapsed time, ledger/gate state,
# acceptance, and any newly observed ledger counters while RUN_STAGE executes.
# =============================================================================

from pathlib import Path
import json
import threading
import time
from datetime import timedelta

STAGE_ID = "22"
GATE_ID = "G22"
HEARTBEAT_SECONDS = 5.0

_run_root = Path(
    globals().get("RUN_ROOT", "/kaggle/working/iharq_p02_run")
).resolve()

_ledger_path = _run_root / "stage_ledger" / f"stage_{STAGE_ID}.json"
_gate_path = _run_root / "gate_results" / f"{GATE_ID}.json"

if "RUN_STAGE" not in globals():
    raise RuntimeError("RUN_STAGE_NOT_AVAILABLE__REHYDRATION_REQUIRED")
if "SESSION" not in globals():
    raise RuntimeError("SESSION_NOT_AVAILABLE__REHYDRATION_REQUIRED")


def _read_json(path):
    try:
        if not path.is_file():
            return None
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        # Monitoring must never interfere with governed execution.
        return None


def _accepted():
    try:
        obj = SESSION.runner.accepted(STAGE_ID)
        return bool(obj), obj
    except Exception:
        return False, None


def _compact(d):
    """Return useful progress-like scalar fields without dumping large payloads."""
    if not isinstance(d, dict):
        return {}

    preferred = (
        "stage_id", "status", "planned", "total", "expected",
        "processed", "completed", "executed", "written",
        "success", "successful", "failed", "failure",
        "skipped", "conditional_skip", "dependency_blocked",
        "resource_blocked", "not_authorized", "records",
        "rows", "artifacts", "files", "missing", "negative_results",
        "gate_id", "decision", "reason",
    )

    out = {}
    for k in preferred:
        if k in d and isinstance(d[k], (str, int, float, bool, type(None))):
            out[k] = d[k]

    # Also surface other simple numeric counters that look progress-related.
    for k, v in d.items():
        lk = str(k).lower()
        if (
            k not in out
            and isinstance(v, (int, float))
            and any(tok in lk for tok in (
                "count", "total", "done", "complete", "processed",
                "success", "fail", "skip", "missing", "record",
                "row", "artifact", "file",
            ))
        ):
            out[k] = v

    return out


_stop = threading.Event()
_started = time.monotonic()


def _progress_monitor():
    last_ledger = None
    last_gate = None
    last_accept = None

    # Immediate heartbeat, then every HEARTBEAT_SECONDS.
    while not _stop.is_set():
        elapsed = str(
            timedelta(seconds=int(time.monotonic() - _started))
        )

        ledger = _read_json(_ledger_path)
        gate = _read_json(_gate_path)
        accepted, _ = _accepted()

        ledger_status = (
            ledger.get("status")
            if isinstance(ledger, dict)
            else "NOT_WRITTEN_YET"
        )
        gate_status = (
            gate.get("status")
            if isinstance(gate, dict)
            else "NOT_WRITTEN_YET"
        )

        print(
            f"[Stage {STAGE_ID}] elapsed={elapsed} | "
            f"ledger={ledger_status} | "
            f"{GATE_ID}={gate_status} | "
            f"accepted={accepted}",
            flush=True,
        )

        ledger_compact = _compact(ledger)
        gate_compact = _compact(gate)

        if ledger_compact and ledger_compact != last_ledger:
            print(
                "    ledger_update="
                + json.dumps(
                    ledger_compact,
                    sort_keys=True,
                    default=str,
                ),
                flush=True,
            )
            last_ledger = ledger_compact

        if gate_compact and gate_compact != last_gate:
            print(
                "    gate_update="
                + json.dumps(
                    gate_compact,
                    sort_keys=True,
                    default=str,
                ),
                flush=True,
            )
            last_gate = gate_compact

        if accepted != last_accept:
            print(
                f"    accepted_update={accepted}",
                flush=True,
            )
            last_accept = accepted

        if _stop.wait(HEARTBEAT_SECONDS):
            break


print("=" * 112)
print(f"STAGE {STAGE_ID} — LIVE PROGRESS")
print("=" * 112)
print("description =", "Protocol/Analysis/Layer0/EvidenceMap/Layer10/P03 handoffs")
print("run_root    =", _run_root)
print("ledger      =", _ledger_path)
print("gate        =", _gate_path)
print("heartbeat   =", HEARTBEAT_SECONDS, "seconds")

accepted_before, _accepted_obj_before = _accepted()
print("accepted_before =", accepted_before)
print(
    "\nThe heartbeat remains visible even if this stage commits its governed "
    "ledger only near the end. No scientific/runtime behavior is changed.\n",
    flush=True,
)

_monitor = threading.Thread(
    target=_progress_monitor,
    name=f"IHARQ-STAGE{STAGE_ID}-PROGRESS-MONITOR",
    daemon=True,
)
_monitor.start()

_stage_exc = None

try:
    # -------------------------------------------------------------------------
    # EXACT GOVERNED EXECUTION — unchanged from the original notebook.
    # -------------------------------------------------------------------------
    result = RUN_STAGE(STAGE_ID)

except BaseException as exc:
    _stage_exc = exc

finally:
    _stop.set()
    _monitor.join(timeout=2.0)

elapsed = str(timedelta(seconds=int(time.monotonic() - _started)))
ledger = _read_json(_ledger_path)
gate = _read_json(_gate_path)
accepted_after, _accepted_obj_after = _accepted()

print("\n" + "=" * 112)
print(f"STAGE {STAGE_ID} — FINAL STATUS")
print("=" * 112)
print("elapsed        =", elapsed)
print("accepted_after =", accepted_after)

if isinstance(ledger, dict):
    print("\nSTAGE LEDGER:")
    print(json.dumps(ledger, indent=2, default=str))
else:
    print("\nSTAGE LEDGER: not present")

if isinstance(gate, dict):
    print(f"\n{GATE_ID}:")
    print(json.dumps(gate, indent=2, default=str))
else:
    print(f"\n{GATE_ID}: not present")

if _stage_exc is not None:
    print(
        f"\nSTAGE {STAGE_ID} TERMINATED WITH EXCEPTION:",
        type(_stage_exc).__name__,
        str(_stage_exc),
        flush=True,
    )
    raise _stage_exc

print(f"\nRUN_STAGE('{STAGE_ID}') RETURN:")
try:
    print(json.dumps(result, indent=2, default=str))
except Exception:
    print(repr(result))

if not accepted_after:
    raise RuntimeError(
        f"STAGE{STAGE_ID}_RETURNED_BUT_WAS_NOT_ACCEPTED"
    )

print("\nTRUSTED PROGRESS-WRAPPER MARKER:")
print(f"IHARQ_P02_STAGE{STAGE_ID}_PROGRESS_WRAPPER_COMPLETE")


STAGE 22 — LIVE PROGRESS
description = Protocol/Analysis/Layer0/EvidenceMap/Layer10/P03 handoffs
run_root    = /kaggle/working/iharq_p02_run
ledger      = /kaggle/working/iharq_p02_run/stage_ledger/stage_22.json
gate        = /kaggle/working/iharq_p02_run/gate_results/G22.json
heartbeat   = 5.0 seconds
accepted_before = False

The heartbeat remains visible even if this stage commits its governed ledger only near the end. No scientific/runtime behavior is changed.

[Stage 22] elapsed=0:00:00 | ledger=NOT_WRITTEN_YET | G22=NOT_WRITTEN_YET | accepted=False
    accepted_update=False

STAGE 22 — FINAL STATUS
elapsed        = 0:00:00
accepted_after = True

STAGE LEDGER:
{
  "stage_id": "22",
  "status": "SUCCESS",
  "attempt_id": "6bad6ea9c5ff",
  "inputs": {},
  "input_hashes": {},
  "outputs": {
    "status": "PASS",
    "handoffs": {
      "protocol_v1_handoff": "protocol_v1_handoff.yaml",
      "phase_analysis_handoff": "phase_analysis_handoff.yaml",
      "layer0_handoff": "layer0_hando

## Stage 23 — Gate matrix and evidence-sufficiency evaluation

**Gate:** `G23`  
**Inputs:** all validation  
**Outputs:** gate_decision + insufficiency route

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [17]:
# =============================================================================
# IHARQ P02/L2 — STAGE 23 WITH LIVE PROGRESS HEARTBEAT
# Gate matrix and evidence-sufficiency evaluation
#
# Scientific/governed execution is UNCHANGED:
#     result = RUN_STAGE("23")
#
# The monitor is READ-ONLY. It reports elapsed time, ledger/gate state,
# acceptance, and any newly observed ledger counters while RUN_STAGE executes.
# =============================================================================

from pathlib import Path
import json
import threading
import time
from datetime import timedelta

STAGE_ID = "23"
GATE_ID = "G23"
HEARTBEAT_SECONDS = 5.0

_run_root = Path(
    globals().get("RUN_ROOT", "/kaggle/working/iharq_p02_run")
).resolve()

_ledger_path = _run_root / "stage_ledger" / f"stage_{STAGE_ID}.json"
_gate_path = _run_root / "gate_results" / f"{GATE_ID}.json"

if "RUN_STAGE" not in globals():
    raise RuntimeError("RUN_STAGE_NOT_AVAILABLE__REHYDRATION_REQUIRED")
if "SESSION" not in globals():
    raise RuntimeError("SESSION_NOT_AVAILABLE__REHYDRATION_REQUIRED")


def _read_json(path):
    try:
        if not path.is_file():
            return None
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        # Monitoring must never interfere with governed execution.
        return None


def _accepted():
    try:
        obj = SESSION.runner.accepted(STAGE_ID)
        return bool(obj), obj
    except Exception:
        return False, None


def _compact(d):
    """Return useful progress-like scalar fields without dumping large payloads."""
    if not isinstance(d, dict):
        return {}

    preferred = (
        "stage_id", "status", "planned", "total", "expected",
        "processed", "completed", "executed", "written",
        "success", "successful", "failed", "failure",
        "skipped", "conditional_skip", "dependency_blocked",
        "resource_blocked", "not_authorized", "records",
        "rows", "artifacts", "files", "missing", "negative_results",
        "gate_id", "decision", "reason",
    )

    out = {}
    for k in preferred:
        if k in d and isinstance(d[k], (str, int, float, bool, type(None))):
            out[k] = d[k]

    # Also surface other simple numeric counters that look progress-related.
    for k, v in d.items():
        lk = str(k).lower()
        if (
            k not in out
            and isinstance(v, (int, float))
            and any(tok in lk for tok in (
                "count", "total", "done", "complete", "processed",
                "success", "fail", "skip", "missing", "record",
                "row", "artifact", "file",
            ))
        ):
            out[k] = v

    return out


_stop = threading.Event()
_started = time.monotonic()


def _progress_monitor():
    last_ledger = None
    last_gate = None
    last_accept = None

    # Immediate heartbeat, then every HEARTBEAT_SECONDS.
    while not _stop.is_set():
        elapsed = str(
            timedelta(seconds=int(time.monotonic() - _started))
        )

        ledger = _read_json(_ledger_path)
        gate = _read_json(_gate_path)
        accepted, _ = _accepted()

        ledger_status = (
            ledger.get("status")
            if isinstance(ledger, dict)
            else "NOT_WRITTEN_YET"
        )
        gate_status = (
            gate.get("status")
            if isinstance(gate, dict)
            else "NOT_WRITTEN_YET"
        )

        print(
            f"[Stage {STAGE_ID}] elapsed={elapsed} | "
            f"ledger={ledger_status} | "
            f"{GATE_ID}={gate_status} | "
            f"accepted={accepted}",
            flush=True,
        )

        ledger_compact = _compact(ledger)
        gate_compact = _compact(gate)

        if ledger_compact and ledger_compact != last_ledger:
            print(
                "    ledger_update="
                + json.dumps(
                    ledger_compact,
                    sort_keys=True,
                    default=str,
                ),
                flush=True,
            )
            last_ledger = ledger_compact

        if gate_compact and gate_compact != last_gate:
            print(
                "    gate_update="
                + json.dumps(
                    gate_compact,
                    sort_keys=True,
                    default=str,
                ),
                flush=True,
            )
            last_gate = gate_compact

        if accepted != last_accept:
            print(
                f"    accepted_update={accepted}",
                flush=True,
            )
            last_accept = accepted

        if _stop.wait(HEARTBEAT_SECONDS):
            break


print("=" * 112)
print(f"STAGE {STAGE_ID} — LIVE PROGRESS")
print("=" * 112)
print("description =", "Gate matrix and evidence-sufficiency evaluation")
print("run_root    =", _run_root)
print("ledger      =", _ledger_path)
print("gate        =", _gate_path)
print("heartbeat   =", HEARTBEAT_SECONDS, "seconds")

accepted_before, _accepted_obj_before = _accepted()
print("accepted_before =", accepted_before)
print(
    "\nThe heartbeat remains visible even if this stage commits its governed "
    "ledger only near the end. No scientific/runtime behavior is changed.\n",
    flush=True,
)

_monitor = threading.Thread(
    target=_progress_monitor,
    name=f"IHARQ-STAGE{STAGE_ID}-PROGRESS-MONITOR",
    daemon=True,
)
_monitor.start()

_stage_exc = None

try:
    # -------------------------------------------------------------------------
    # EXACT GOVERNED EXECUTION — unchanged from the original notebook.
    # -------------------------------------------------------------------------
    result = RUN_STAGE(STAGE_ID)

except BaseException as exc:
    _stage_exc = exc

finally:
    _stop.set()
    _monitor.join(timeout=2.0)

elapsed = str(timedelta(seconds=int(time.monotonic() - _started)))
ledger = _read_json(_ledger_path)
gate = _read_json(_gate_path)
accepted_after, _accepted_obj_after = _accepted()

print("\n" + "=" * 112)
print(f"STAGE {STAGE_ID} — FINAL STATUS")
print("=" * 112)
print("elapsed        =", elapsed)
print("accepted_after =", accepted_after)

if isinstance(ledger, dict):
    print("\nSTAGE LEDGER:")
    print(json.dumps(ledger, indent=2, default=str))
else:
    print("\nSTAGE LEDGER: not present")

if isinstance(gate, dict):
    print(f"\n{GATE_ID}:")
    print(json.dumps(gate, indent=2, default=str))
else:
    print(f"\n{GATE_ID}: not present")

if _stage_exc is not None:
    print(
        f"\nSTAGE {STAGE_ID} TERMINATED WITH EXCEPTION:",
        type(_stage_exc).__name__,
        str(_stage_exc),
        flush=True,
    )
    raise _stage_exc

print(f"\nRUN_STAGE('{STAGE_ID}') RETURN:")
try:
    print(json.dumps(result, indent=2, default=str))
except Exception:
    print(repr(result))

if not accepted_after:
    raise RuntimeError(
        f"STAGE{STAGE_ID}_RETURNED_BUT_WAS_NOT_ACCEPTED"
    )

print("\nTRUSTED PROGRESS-WRAPPER MARKER:")
print(f"IHARQ_P02_STAGE{STAGE_ID}_PROGRESS_WRAPPER_COMPLETE")


STAGE 23 — LIVE PROGRESS
description = Gate matrix and evidence-sufficiency evaluation
run_root    = /kaggle/working/iharq_p02_run
ledger      = /kaggle/working/iharq_p02_run/stage_ledger/stage_23.json
gate        = /kaggle/working/iharq_p02_run/gate_results/G23.json
heartbeat   = 5.0 seconds
accepted_before = False

The heartbeat remains visible even if this stage commits its governed ledger only near the end. No scientific/runtime behavior is changed.

[Stage 23] elapsed=0:00:00 | ledger=NOT_WRITTEN_YET | G23=NOT_WRITTEN_YET | accepted=False
    accepted_update=False
[Stage 23] elapsed=0:00:05 | ledger=NOT_WRITTEN_YET | G23=NOT_WRITTEN_YET | accepted=False
[Stage 23] elapsed=0:00:10 | ledger=NOT_WRITTEN_YET | G23=NOT_WRITTEN_YET | accepted=False
[Stage 23] elapsed=0:00:15 | ledger=NOT_WRITTEN_YET | G23=NOT_WRITTEN_YET | accepted=False
[Stage 23] elapsed=0:00:20 | ledger=NOT_WRITTEN_YET | G23=NOT_WRITTEN_YET | accepted=False
[Stage 23] elapsed=0:00:25 | ledger=NOT_WRITTEN_YET | G23=NO

In [19]:
# =============================================================================
# IHARQ P02/L2 — PRE-STAGE24 STAGE18 + STAGE18S CONSOLIDATION / RELEASE READINESS
# Revision: R1.1 — artifact-path resolver fix
#
# PURPOSE
# -------
# 1) Keep canonical Stage18/G18 immutable.
# 2) Register the completed Stage18S post-hoc sensitivity supplement as an
#    explicit member of the Stage18 *release/evidence family*.
# 3) Inventory and hash every Stage18S output, all Stage18/Stage18S checkpoints,
#    CSV/JSON/JSONL/Parquet/tabular outputs, handoffs, receipts, and provenance.
# 4) Verify all expected pre-G24 scientific/evidence surfaces before Stage24.
# 5) Write fail-closed release-readiness manifests that the replacement Stage24
#    cell MUST consume.
#
# IMPORTANT
# ---------
# This DOES NOT rewrite:
#   * stage_ledger/stage_18.json
#   * gate_results/G18.json
#   * canonical Stage18 run-cell/metric/checkpoint evidence
#
# Stage18S remains:
#   POST_HOC_SENSITIVITY_DESCRIPTIVE
#
# It is consolidated only for preservation/release/downstream traceability.
# =============================================================================

from __future__ import annotations

from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import csv
import hashlib
import json
import os
import re
import shutil
import sys

# -----------------------------------------------------------------------------
# Frozen expectations from the completed run.
# -----------------------------------------------------------------------------
EXPECTED_CANONICAL_A4_TERMINALS = 1218

EXPECTED_S18S_PLANNED_CELLS = 162
EXPECTED_S18S_TERMINAL_COUNTS = {
    "SUCCESS": 135,
    "INPUT_INCOMPATIBLE": 27,
}
EXPECTED_S18S_MEMBER_RECEIPTS = 216
EXPECTED_S18S_MEMBER_STATUS_COUNTS = {
    "SUCCESS": 189,
    "INPUT_INCOMPATIBLE": 27,
}

EXPECTED_S18S_ANALYSIS_FILES = [
    "stage18S_R1_combined_cell_effects.csv",
    "stage18S_R1_three_repeat_anchor_stability.csv",
    "stage18S_R1_six_budget_mr00_sensitivity.csv",
    "stage18S_R1_participant_paired_effects.csv",
    "stage18S_R1_member_training_provenance.csv",
    "stage18S_R1_training_provenance_summary.csv",
    "stage18S_R1_decision_support_summary.json",
    "stage18S_balanced_sensitivity_R1_preexecution_freeze.json",
]

EXPECTED_POST18_STAGE_ARTIFACTS = [
    "stage_artifacts/18U_unlock.json",
    "stage_artifacts/19_failure.json",
    "stage_artifacts/20_readiness.json",
    "stage_artifacts/21_sources.json",
    "stage_artifacts/22_handoffs.json",
    "stage_artifacts/23_sufficiency.json",
]

EXPECTED_HANDOFF_BASENAMES = [
    "protocol_v1_handoff.yaml",
    "phase_analysis_handoff.yaml",
    "layer0_handoff.yaml",
    "evidence_map_handoff.yaml",
    "layer10_source_handoff.yaml",
    "p03_handoff.yaml",
    "stage18S_R1_sensitivity_evidence.json",
]

CHECKPOINT_SUFFIXES = {
    ".pt", ".pth", ".pkl", ".ckpt", ".safetensors",
    ".joblib", ".onnx",
}
TABULAR_SUFFIXES = {
    ".csv", ".json", ".jsonl", ".parquet", ".feather", ".tsv",
}

# -----------------------------------------------------------------------------
# Paths.
# -----------------------------------------------------------------------------
KW = Path("/kaggle/working").resolve()

def _resolve_run_root():
    candidates = []
    if "RUN_ROOT" in globals():
        try:
            candidates.append(Path(RUN_ROOT).resolve())
        except Exception:
            pass

    if "STORE" in globals():
        try:
            sr = Path(STORE.root).resolve()
            # STORE.root is expected to be <RUN_ROOT>/runtime.
            candidates.extend([sr.parent, sr])
        except Exception:
            pass

    candidates.extend([
        KW / "iharq_p02_run",
        KW / "IHARQ_P02_RUN",
    ])

    for p in candidates:
        if (p / "stage_ledger").is_dir() and (p / "gate_results").is_dir():
            return p

    raise RuntimeError(
        "PRE_G24_RUN_ROOT_NOT_RESOLVED:"
        + json.dumps([str(p) for p in candidates])
    )

RUN_ROOT = _resolve_run_root()

if "STORE" in globals():
    try:
        STORE_ROOT = Path(STORE.root).resolve()
    except Exception:
        STORE_ROOT = RUN_ROOT / "runtime"
else:
    STORE_ROOT = RUN_ROOT / "runtime"

if not STORE_ROOT.is_dir():
    raise RuntimeError(f"PRE_G24_STORE_ROOT_MISSING:{STORE_ROOT}")

S18S_ROOT = (
    STORE_ROOT
    / "supplements"
    / "stage18S_balanced_sensitivity_R1"
)

if not S18S_ROOT.is_dir():
    raise RuntimeError(f"PRE_G24_STAGE18S_ROOT_MISSING:{S18S_ROOT}")

CONSOLIDATED_ROOT = STORE_ROOT / "stage18_consolidated_release_R1"
CONSOLIDATED_ANALYSIS = CONSOLIDATED_ROOT / "analysis"
CONSOLIDATED_MANIFESTS = CONSOLIDATED_ROOT / "manifests"
CONSOLIDATED_POINTERS = CONSOLIDATED_ROOT / "heavy_artifact_candidates"
CONSOLIDATED_ANALYSIS.mkdir(parents=True, exist_ok=True)
CONSOLIDATED_MANIFESTS.mkdir(parents=True, exist_ok=True)
CONSOLIDATED_POINTERS.mkdir(parents=True, exist_ok=True)

RUNTIME_ANALYSIS = STORE_ROOT / "analysis_inputs"
RUNTIME_HANDOFFS = STORE_ROOT / "handoffs"
RUNTIME_ANALYSIS.mkdir(parents=True, exist_ok=True)
RUNTIME_HANDOFFS.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# Helpers.
# -----------------------------------------------------------------------------
def _utc():
    return datetime.now(timezone.utc).isoformat()

def _sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def _atomic_json(path: Path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(obj, indent=2, sort_keys=True, default=str) + "\n",
        encoding="utf-8",
    )
    os.replace(tmp, path)

def _atomic_jsonl(path: Path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(
                json.dumps(row, sort_keys=True, default=str)
                + "\n"
            )
    os.replace(tmp, path)

def _load_json(path: Path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def _accepted(stage_id: str) -> bool:
    if "SESSION" not in globals():
        return False
    try:
        return bool(SESSION.runner.accepted(str(stage_id)))
    except Exception:
        return False

def _file_row(path: Path, *, logical_role: str, root: Path | None = None):
    path = Path(path)
    rel = (
        path.relative_to(root).as_posix()
        if root is not None and path.is_relative_to(root)
        else str(path)
    )
    return {
        "path": rel,
        "absolute_source_path": str(path),
        "logical_role": logical_role,
        "bytes": path.stat().st_size,
        "sha256": _sha256_file(path),
        "suffix": path.suffix.lower(),
    }

def _find_basename(root: Path, basename: str):
    hits = sorted(
        p for p in root.rglob(basename)
        if p.is_file()
    )
    return hits

def _resolve_runtime_artifact_path(
    rel_or_abs,
    *,
    logical_name: str,
    require_unique_fallback: bool = True,
):
    """
    Resolve an artifact path emitted by a governed stage.

    IMPORTANT:
    Stage ledger paths such as:
        records/Layer2ReadinessReport/...jsonl
    are Store-relative paths, not necessarily RUN_ROOT-relative paths.

    We therefore resolve against the authoritative runtime/store roots first,
    while retaining RUN_ROOT compatibility for stage_artifacts, ledgers, gates,
    logs, and older package revisions.
    """
    raw = str(rel_or_abs or "").strip()
    if not raw:
        raise RuntimeError(
            f"PRE_G24_EMPTY_ARTIFACT_PATH:{logical_name}"
        )

    p = Path(raw)
    if p.is_absolute():
        if p.is_file():
            return p.resolve()
        raise RuntimeError(
            f"PRE_G24_ABSOLUTE_ARTIFACT_MISSING:{logical_name}:{p}"
        )

    # Ordered by the actual ownership contract:
    #   STORE_ROOT owns records/metrics/predictions/etc.
    #   RUN_ROOT owns stage_artifacts/ledgers/gates/logs.
    candidates = []

    def _add(base):
        try:
            cand = (Path(base).resolve() / p).resolve()
        except Exception:
            return
        if cand not in candidates:
            candidates.append(cand)

    _add(STORE_ROOT)

    if "STORE" in globals():
        try:
            _add(Path(STORE.root))
        except Exception:
            pass

    _add(RUN_ROOT / "runtime")
    _add(RUN_ROOT)

    existing = [x for x in candidates if x.is_file()]

    if len(existing) == 1:
        return existing[0]

    if len(existing) > 1:
        # Duplicate physical paths via equivalent roots are harmless.
        unique_real = []
        for x in existing:
            rx = x.resolve()
            if rx not in unique_real:
                unique_real.append(rx)

        if len(unique_real) == 1:
            return unique_real[0]

        # If duplicates have identical bytes, prefer STORE_ROOT ownership.
        sigs = {
            (x.stat().st_size, _sha256_file(x))
            for x in unique_real
        }
        if len(sigs) == 1:
            for preferred_base in (STORE_ROOT, RUN_ROOT / "runtime", RUN_ROOT):
                for x in unique_real:
                    try:
                        x.relative_to(Path(preferred_base).resolve())
                        return x
                    except Exception:
                        pass

        raise RuntimeError(
            "PRE_G24_ARTIFACT_PATH_AMBIGUOUS:"
            + json.dumps({
                "logical_name": logical_name,
                "ledger_path": raw,
                "candidates": [str(x) for x in unique_real],
            }, sort_keys=True)
        )

    # Last-resort bounded suffix search. This is validation-only; it never
    # fabricates, copies, or moves the evidence.
    basename = p.name
    suffix = p.as_posix().lstrip("./")
    fallback = []

    for root in (STORE_ROOT, RUN_ROOT):
        if not Path(root).is_dir():
            continue
        for hit in Path(root).rglob(basename):
            if not hit.is_file():
                continue
            hp = hit.as_posix()
            if hp.endswith(suffix):
                rh = hit.resolve()
                if rh not in fallback:
                    fallback.append(rh)

    if len(fallback) == 1:
        return fallback[0]

    if fallback and not require_unique_fallback:
        return fallback[0]

    raise RuntimeError(
        "PRE_G24_ARTIFACT_NOT_RESOLVED:"
        + json.dumps({
            "logical_name": logical_name,
            "ledger_path": raw,
            "checked": [str(x) for x in candidates],
            "suffix_matches": [str(x) for x in fallback],
        }, sort_keys=True)
    )


def _validate_nonempty_jsonl(path: Path, *, logical_name: str):
    """
    Validate that an expected JSONL evidence record is a real, parseable,
    non-empty governed output. This is stronger than a bare path-exists check.
    """
    path = Path(path)
    rows = 0
    byte_lines = 0

    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                continue
            byte_lines += len(line.encode("utf-8"))
            try:
                obj = json.loads(line)
            except Exception as exc:
                raise RuntimeError(
                    f"PRE_G24_JSONL_PARSE_FAILURE:{logical_name}:"
                    f"{path}:line={line_no}:{type(exc).__name__}:{exc}"
                )
            if not isinstance(obj, dict):
                raise RuntimeError(
                    f"PRE_G24_JSONL_ROW_NOT_OBJECT:{logical_name}:"
                    f"{path}:line={line_no}"
                )
            rows += 1

    if rows < 1:
        raise RuntimeError(
            f"PRE_G24_JSONL_EMPTY:{logical_name}:{path}"
        )

    return {
        "path": str(path),
        "rows": rows,
        "bytes": path.stat().st_size,
        "sha256": _sha256_file(path),
    }

def _unique_by_path(rows):
    out = {}
    for r in rows:
        out[r["absolute_source_path"]] = r
    return sorted(out.values(), key=lambda x: x["absolute_source_path"])

def _is_s18s_output(path: Path):
    # A0 files in the supplement Store were read-only seed inputs and are not
    # Stage18S-produced evidence.
    name = path.name
    if (
        "/run_cells/" in path.as_posix()
        and name.startswith("P02-A0-")
    ):
        return False
    if (
        "/metrics/" in path.as_posix()
        and name.startswith("P02-A0-")
    ):
        return False
    return True

def _text_secret_hits(path: Path):
    # Fail-closed scan only for bounded text-like files.
    if path.stat().st_size > 8 * 1024 * 1024:
        return []
    if path.suffix.lower() not in {
        ".txt", ".md", ".json", ".jsonl", ".yaml", ".yml",
        ".csv", ".tsv", ".py", ".toml", ".ini", ".cfg",
    }:
        return []
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return []

    patterns = {
        "HF_TOKEN": r"\bhf_[A-Za-z0-9]{20,}\b",
        "GITHUB_PAT": r"\bgithub_pat_[A-Za-z0-9_]{20,}\b",
        "GITHUB_TOKEN": r"\bgh[pousr]_[A-Za-z0-9]{30,}\b",
        "AWS_ACCESS_KEY": r"\bAKIA[0-9A-Z]{16}\b",
    }
    hits = []
    for kind, pat in patterns.items():
        if re.search(pat, text):
            hits.append(kind)
    return hits

# -----------------------------------------------------------------------------
# 1. Accepted-stage / gate closure.
# -----------------------------------------------------------------------------
# Complete Build-Book stage sequence before Stage24.
required_stages = [
    "00", "01", "02", "03", "04", "05", "06", "07",
    "08", "09", "10", "11", "12", "13", "14", "15",
    "16", "17", "18", "18U", "19", "20", "21", "22", "23",
]

stage_status = {}
for sid in required_stages:
    ledger_path = RUN_ROOT / "stage_ledger" / f"stage_{sid}.json"
    if not ledger_path.is_file():
        raise RuntimeError(f"PRE_G24_LEDGER_MISSING:{ledger_path}")
    ledger = _load_json(ledger_path)

    if ledger.get("status") != "SUCCESS":
        raise RuntimeError(
            f"PRE_G24_STAGE_NOT_SUCCESS:{sid}:{ledger.get('status')}"
        )

    if not _accepted(sid):
        raise RuntimeError(f"PRE_G24_STAGE_NOT_ACCEPTED:{sid}")

    stage_status[sid] = {
        "status": ledger.get("status"),
        "attempt_id": ledger.get("attempt_id"),
        "ledger_sha256": _sha256_file(ledger_path),
    }

required_gates = [
    "G00", "G01", "G02", "G03", "G04", "G05", "G06", "G07",
    "G08", "G09", "G10", "G11", "G12", "G13", "G14", "G15",
    "G16", "G17", "G18", "G18U", "G19", "G20", "G21", "G22", "G23",
]
gate_status = {}
for gid in required_gates:
    gp = RUN_ROOT / "gate_results" / f"{gid}.json"
    if not gp.is_file():
        raise RuntimeError(f"PRE_G24_GATE_MISSING:{gp}")
    gd = _load_json(gp)
    if gd.get("status") != "PASS":
        raise RuntimeError(
            f"PRE_G24_GATE_NOT_PASS:{gid}:{gd.get('status')}"
        )
    gate_status[gid] = {
        "status": gd.get("status"),
        "sha256": _sha256_file(gp),
    }

# Stage23 evidence sufficiency must be explicit PASS + missing=[].
s23 = _load_json(RUN_ROOT / "stage_ledger" / "stage_23.json")
suff = (s23.get("outputs") or {}).get("evidence_sufficiency") or {}
if suff.get("status") != "PASS" or list(suff.get("missing") or []):
    raise RuntimeError(
        "PRE_G24_STAGE23_EVIDENCE_SUFFICIENCY_NOT_CLOSED:"
        + json.dumps(suff, sort_keys=True)
    )

s23_state = (s23.get("outputs") or {}).get("state") or {}
required_s23_state_flags = [
    "a0_complete",
    "a4_complete",
    "a4_role_controls_complete",
    "c4_c5_complete",
    "a4_burden_complete",
    "failure_evidence",
    "training_policy_challenger_complete",
    "figure_table_sources",
    "handoffs",
    "p03_complete",
    "readiness_record",
    "security_pass",
]
bad_s23_flags = {
    k: s23_state.get(k)
    for k in required_s23_state_flags
    if s23_state.get(k) is not True
}
if bad_s23_flags:
    raise RuntimeError(
        "PRE_G24_STAGE23_REQUIRED_STATE_NOT_COMPLETE:"
        + json.dumps(bad_s23_flags, sort_keys=True)
    )

# -----------------------------------------------------------------------------
# 2. Canonical Stage18 remains closed and untouched.
# -----------------------------------------------------------------------------
a4_completion = STORE_ROOT / "analysis_inputs" / "a4_completion.json"
if not a4_completion.is_file():
    raise RuntimeError(f"PRE_G24_A4_COMPLETION_MISSING:{a4_completion}")
a4c = _load_json(a4_completion)
if a4c.get("status") != "PASS":
    raise RuntimeError("PRE_G24_A4_COMPLETION_NOT_PASS")

canonical_a4_terminals = sorted(
    (STORE_ROOT / "run_cells").glob("P02-A4-*.json")
)
if len(canonical_a4_terminals) != EXPECTED_CANONICAL_A4_TERMINALS:
    raise RuntimeError(
        "PRE_G24_CANONICAL_A4_TERMINAL_COUNT_MISMATCH:"
        f"{len(canonical_a4_terminals)}"
    )

canonical_stage18_protected_rows = [
    _file_row(
        RUN_ROOT / "stage_ledger" / "stage_18.json",
        logical_role="CANONICAL_STAGE18_LEDGER",
        root=RUN_ROOT,
    ),
    _file_row(
        RUN_ROOT / "gate_results" / "G18.json",
        logical_role="CANONICAL_G18",
        root=RUN_ROOT,
    ),
    _file_row(
        a4_completion,
        logical_role="CANONICAL_STAGE18_A4_COMPLETION",
        root=RUN_ROOT,
    ),
]

# -----------------------------------------------------------------------------
# 3. Stage18S exact closure.
# -----------------------------------------------------------------------------
s18s_exec_path = S18S_ROOT / "analysis" / "execution_receipt.json"
s18s_gate_path = S18S_ROOT / "analysis" / "supplement_gate.json"

if not s18s_exec_path.is_file():
    raise RuntimeError(f"PRE_G24_S18S_EXECUTION_RECEIPT_MISSING:{s18s_exec_path}")
if not s18s_gate_path.is_file():
    raise RuntimeError(f"PRE_G24_S18S_GATE_MISSING:{s18s_gate_path}")

s18s_exec = _load_json(s18s_exec_path)
s18s_gate = _load_json(s18s_gate_path)

if s18s_exec.get("status") != "PASS":
    raise RuntimeError("PRE_G24_S18S_EXECUTION_NOT_PASS")
if s18s_gate.get("status") != "PASS":
    raise RuntimeError("PRE_G24_S18S_GATE_NOT_PASS")
if s18s_gate.get("canonical_stage18_modified") is not False:
    raise RuntimeError("PRE_G24_S18S_CANONICAL_STAGE18_WAS_MARKED_MODIFIED")
if s18s_gate.get("canonical_G18_modified") is not False:
    raise RuntimeError("PRE_G24_S18S_CANONICAL_G18_WAS_MARKED_MODIFIED")
if s18s_gate.get("ready_for_downstream_preservation") is not True:
    raise RuntimeError("PRE_G24_S18S_NOT_READY_FOR_DOWNSTREAM_PRESERVATION")
if s18s_gate.get("claim_scope") != "POST_HOC_SENSITIVITY_DESCRIPTIVE":
    raise RuntimeError("PRE_G24_S18S_CLAIM_SCOPE_DRIFT")

if int(s18s_exec.get("planned_cells", -1)) != EXPECTED_S18S_PLANNED_CELLS:
    raise RuntimeError("PRE_G24_S18S_PLANNED_CELL_COUNT_DRIFT")

if dict(s18s_exec.get("terminal_counts") or {}) != EXPECTED_S18S_TERMINAL_COUNTS:
    raise RuntimeError(
        "PRE_G24_S18S_TERMINAL_COUNTS_DRIFT:"
        + json.dumps(s18s_exec.get("terminal_counts"), sort_keys=True)
    )

if int(s18s_exec.get("member_receipts", -1)) != EXPECTED_S18S_MEMBER_RECEIPTS:
    raise RuntimeError("PRE_G24_S18S_MEMBER_RECEIPT_COUNT_DRIFT")

if (
    dict(s18s_exec.get("member_status_counts") or {})
    != EXPECTED_S18S_MEMBER_STATUS_COUNTS
):
    raise RuntimeError(
        "PRE_G24_S18S_MEMBER_STATUS_COUNTS_DRIFT:"
        + json.dumps(s18s_exec.get("member_status_counts"), sort_keys=True)
    )

if int(s18s_gate.get("training_provenance_issues", -1)) != 0:
    raise RuntimeError("PRE_G24_S18S_TRAINING_PROVENANCE_ISSUES_NONZERO")

# Exact project-visible analysis copies.
missing_analysis = []
analysis_rows = []
for name in EXPECTED_S18S_ANALYSIS_FILES:
    p = RUNTIME_ANALYSIS / name
    if not p.is_file():
        missing_analysis.append(str(p))
    else:
        analysis_rows.append(
            _file_row(
                p,
                logical_role="STAGE18S_PROJECT_VISIBLE_ANALYSIS",
                root=RUN_ROOT,
            )
        )

if missing_analysis:
    raise RuntimeError(
        "PRE_G24_S18S_PROJECT_VISIBLE_ANALYSIS_MISSING:"
        + json.dumps(missing_analysis)
    )

# Explicit Stage18S handoff.
s18s_handoff = RUNTIME_HANDOFFS / "stage18S_R1_sensitivity_evidence.json"
if not s18s_handoff.is_file():
    raise RuntimeError(f"PRE_G24_S18S_HANDOFF_MISSING:{s18s_handoff}")
s18s_handoff_obj = _load_json(s18s_handoff)
if s18s_handoff_obj.get("status") != "PASS":
    raise RuntimeError("PRE_G24_S18S_HANDOFF_NOT_PASS")
if s18s_handoff_obj.get("canonical_stage18_remains_source_of_G18") is not True:
    raise RuntimeError("PRE_G24_S18S_HANDOFF_G18_BOUNDARY_DRIFT")

# -----------------------------------------------------------------------------
# 4. Expected downstream stage artifacts and handoffs.
# -----------------------------------------------------------------------------
missing_post18 = [
    rel for rel in EXPECTED_POST18_STAGE_ARTIFACTS
    if not (RUN_ROOT / rel).is_file()
]
if missing_post18:
    raise RuntimeError(
        "PRE_G24_EXPECTED_POST18_STAGE_ARTIFACTS_MISSING:"
        + json.dumps(missing_post18)
    )

handoff_hits = {}
missing_handoffs = []
for basename in EXPECTED_HANDOFF_BASENAMES:
    hits = _find_basename(RUN_ROOT, basename)
    if not hits:
        missing_handoffs.append(basename)
    else:
        handoff_hits[basename] = [
            str(p.relative_to(RUN_ROOT))
            for p in hits
        ]

if missing_handoffs:
    raise RuntimeError(
        "PRE_G24_EXPECTED_HANDOFFS_MISSING:"
        + json.dumps(missing_handoffs)
    )

# Stage20 readiness record path must resolve.
#
# IMPORTANT: Stage20 writes this as a Store-relative path:
#   records/Layer2ReadinessReport/...
# It is therefore resolved through STORE_ROOT/runtime ownership rather than
# blindly prepending RUN_ROOT.
s20 = _load_json(RUN_ROOT / "stage_ledger" / "stage_20.json")
readiness_rel = (s20.get("outputs") or {}).get("readiness_record")
if not readiness_rel:
    raise RuntimeError("PRE_G24_STAGE20_READINESS_RECORD_PATH_MISSING")

readiness_path = _resolve_runtime_artifact_path(
    readiness_rel,
    logical_name="STAGE20_LAYER2_READINESS_REPORT",
)
readiness_record_validation = _validate_nonempty_jsonl(
    readiness_path,
    logical_name="STAGE20_LAYER2_READINESS_REPORT",
)

print(
    "Stage20 readiness record resolved =",
    readiness_record_validation["path"],
)
print(
    "Stage20 readiness record rows =",
    readiness_record_validation["rows"],
)
print(
    "Stage20 readiness record sha256 =",
    readiness_record_validation["sha256"],
)

# -----------------------------------------------------------------------------
# 5. Complete Stage18S-produced inventory + Stage18/18S checkpoint/tabular index.
# -----------------------------------------------------------------------------
s18s_rows = []
for p in sorted(S18S_ROOT.rglob("*")):
    if not p.is_file():
        continue
    if p.name.endswith(".tmp"):
        continue
    if not _is_s18s_output(p):
        continue
    s18s_rows.append(
        _file_row(
            p,
            logical_role="STAGE18S_SUPPLEMENT_OUTPUT",
            root=RUN_ROOT,
        )
    )

# All checkpoint-like files under the canonical run root.
checkpoint_rows = []
for p in sorted(RUN_ROOT.rglob("*")):
    if not p.is_file():
        continue
    if p.suffix.lower() in CHECKPOINT_SUFFIXES:
        checkpoint_rows.append(
            _file_row(
                p,
                logical_role=(
                    "STAGE18S_OR_CANONICAL_CHECKPOINT"
                    if "stage18S_balanced_sensitivity_R1" in p.as_posix()
                    else "CANONICAL_OR_UPSTREAM_CHECKPOINT"
                ),
                root=RUN_ROOT,
            )
        )

# Every tabular/structured result surface under Store + Stage artifacts/handoffs.
tabular_rows = []
candidate_roots = [
    STORE_ROOT,
    RUN_ROOT / "stage_artifacts",
    RUN_ROOT / "gate_results",
    RUN_ROOT / "stage_ledger",
]
for root in candidate_roots:
    if not root.exists():
        continue
    for p in sorted(root.rglob("*")):
        if p.is_file() and p.suffix.lower() in TABULAR_SUFFIXES:
            tabular_rows.append(
                _file_row(
                    p,
                    logical_role="STRUCTURED_OR_TABULAR_PHASE_OUTPUT",
                    root=RUN_ROOT,
                )
            )

checkpoint_rows = _unique_by_path(checkpoint_rows)
tabular_rows = _unique_by_path(tabular_rows)

if not checkpoint_rows:
    raise RuntimeError("PRE_G24_NO_CHECKPOINT_FILES_DISCOVERED")
if not tabular_rows:
    raise RuntimeError("PRE_G24_NO_STRUCTURED_TABULAR_FILES_DISCOVERED")

# -----------------------------------------------------------------------------
# 6. Secret scan of the release-relevant result surfaces.
#    Never scan in-memory HF token values; no token is required in this cell.
# -----------------------------------------------------------------------------
secret_hits = []
for row in _unique_by_path(
    s18s_rows
    + analysis_rows
    + tabular_rows
    + canonical_stage18_protected_rows
):
    p = Path(row["absolute_source_path"])
    for kind in _text_secret_hits(p):
        secret_hits.append({
            "path": str(p),
            "kind": kind,
        })

if secret_hits:
    raise RuntimeError(
        "PRE_G24_SECRET_SCAN_FAILED:"
        + json.dumps(secret_hits[:20], sort_keys=True)
    )

# -----------------------------------------------------------------------------
# 7. Write immutable release-family indices.
# -----------------------------------------------------------------------------
s18s_manifest_path = (
    CONSOLIDATED_MANIFESTS
    / "stage18S_complete_output_manifest.jsonl"
)
checkpoint_manifest_path = (
    CONSOLIDATED_MANIFESTS
    / "stage18_and_stage18S_checkpoint_manifest.jsonl"
)
tabular_manifest_path = (
    CONSOLIDATED_MANIFESTS
    / "stage18_and_phase_structured_output_manifest.jsonl"
)

_atomic_jsonl(s18s_manifest_path, s18s_rows)
_atomic_jsonl(checkpoint_manifest_path, checkpoint_rows)
_atomic_jsonl(tabular_manifest_path, tabular_rows)

# Human-readable compact CSV index for quick inspection.
csv_index_path = (
    CONSOLIDATED_MANIFESTS
    / "stage18_release_family_quick_index.csv"
)
with csv_index_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "category", "path", "bytes", "sha256", "logical_role"
        ],
    )
    writer.writeheader()
    for category, rows in (
        ("stage18S_output", s18s_rows),
        ("checkpoint", checkpoint_rows),
        ("structured_output", tabular_rows),
    ):
        for r in rows:
            writer.writerow({
                "category": category,
                "path": r["path"],
                "bytes": r["bytes"],
                "sha256": r["sha256"],
                "logical_role": r["logical_role"],
            })

stage18_release_family = {
    "artifact_id": "P02-STAGE18-CONSOLIDATED-RELEASE-FAMILY-R1",
    "created_at_utc": _utc(),
    "status": "PASS",
    "phase_id": "P02",
    "canonical_stage18": {
        "status": "SUCCESS",
        "G18": "PASS",
        "canonical_a4_terminals": len(canonical_a4_terminals),
        "source_of_canonical_G18": True,
        "mutated_by_this_cell": False,
        "protected_files": canonical_stage18_protected_rows,
    },
    "stage18S": {
        "supplement_id": s18s_gate.get("supplement_id"),
        "status": s18s_gate.get("status"),
        "claim_scope": s18s_gate.get("claim_scope"),
        "post_hoc": True,
        "supplement_is_additional_sensitivity_evidence": True,
        "canonical_stage18_modified": False,
        "canonical_G18_modified": False,
        "planned_cells": EXPECTED_S18S_PLANNED_CELLS,
        "terminal_counts": EXPECTED_S18S_TERMINAL_COUNTS,
        "member_receipts": EXPECTED_S18S_MEMBER_RECEIPTS,
        "member_status_counts": EXPECTED_S18S_MEMBER_STATUS_COUNTS,
        "complete_output_files": len(s18s_rows),
        "complete_output_bytes": sum(r["bytes"] for r in s18s_rows),
        "complete_output_manifest": str(
            s18s_manifest_path.relative_to(RUN_ROOT)
        ),
        "project_visible_analysis_files": [
            r["path"] for r in analysis_rows
        ],
        "handoff": str(s18s_handoff.relative_to(RUN_ROOT)),
    },
    "checkpoints": {
        "files": len(checkpoint_rows),
        "bytes": sum(r["bytes"] for r in checkpoint_rows),
        "manifest": str(
            checkpoint_manifest_path.relative_to(RUN_ROOT)
        ),
        "release_policy":
            "BYTES_MAY_BE_EXTERNALIZED_TO_HF; EXACT_HASHED_POINTER_REQUIRED",
    },
    "structured_outputs": {
        "files": len(tabular_rows),
        "bytes": sum(r["bytes"] for r in tabular_rows),
        "manifest": str(
            tabular_manifest_path.relative_to(RUN_ROOT)
        ),
    },
    "interpretation_boundary": {
        "canonical_stage18_and_stage18S_are_not_retroactively_merged":
            True,
        "stage18S_is_grouped_into_stage18_release_family_for_preservation":
            True,
        "stage18S_must_be_visible_in_final_release":
            True,
        "stage18S_must_not_be_presented_as_prespecified_five_repeat_confirmatory":
            True,
    },
}

stage18_release_family_path = (
    RUN_ROOT
    / "stage_artifacts"
    / "18_stage18S_consolidated_release_family_R1.json"
)
_atomic_json(stage18_release_family_path, stage18_release_family)

# Project-visible analysis index.
analysis_index = {
    "artifact_id": "P02-STAGE18-RELEASE-ANALYSIS-INDEX-R1",
    "created_at_utc": _utc(),
    "status": "PASS",
    "canonical_stage18_source_of_G18": True,
    "stage18S_registered_as_post_hoc_sensitivity": True,
    "stage18_release_family_artifact":
        str(stage18_release_family_path.relative_to(RUN_ROOT)),
    "stage18S_analysis_inputs": [
        r["path"] for r in analysis_rows
    ],
    "stage18S_output_manifest":
        str(s18s_manifest_path.relative_to(RUN_ROOT)),
    "checkpoint_manifest":
        str(checkpoint_manifest_path.relative_to(RUN_ROOT)),
    "structured_output_manifest":
        str(tabular_manifest_path.relative_to(RUN_ROOT)),
}
analysis_index_path = (
    RUNTIME_ANALYSIS
    / "stage18_consolidated_release_index_R1.json"
)
_atomic_json(analysis_index_path, analysis_index)

# Explicit handoff for the replacement Stage24 release builder.
release_handoff = {
    "artifact_id": "P02-STAGE18-AND-STAGE18S-RELEASE-HANDOFF-R1",
    "created_at_utc": _utc(),
    "status": "PASS",
    "consumer": "STAGE24_RELEASE_FINALIZER_R1",
    "canonical_G18_unchanged": True,
    "canonical_stage18_unchanged": True,
    "stage18S_preservation_required": True,
    "stage18S_claim_scope": "POST_HOC_SENSITIVITY_DESCRIPTIVE",
    "required_manifests": {
        "release_family": str(
            stage18_release_family_path.relative_to(RUN_ROOT)
        ),
        "stage18S_outputs": str(
            s18s_manifest_path.relative_to(RUN_ROOT)
        ),
        "checkpoints": str(
            checkpoint_manifest_path.relative_to(RUN_ROOT)
        ),
        "structured_outputs": str(
            tabular_manifest_path.relative_to(RUN_ROOT)
        ),
        "quick_index": str(
            csv_index_path.relative_to(RUN_ROOT)
        ),
    },
}
release_handoff_path = (
    RUNTIME_HANDOFFS
    / "stage18_and_stage18S_release_handoff_R1.json"
)
_atomic_json(release_handoff_path, release_handoff)

# -----------------------------------------------------------------------------
# 8. P01 cumulative source discovery for the post-G24 cumulative project state.
#    We verify availability now so Stage24 does not discover it too late.
# -----------------------------------------------------------------------------
def _discover_p01_candidates():
    dirs = []
    zips = []

    roots = [KW]
    if "PACKAGE_ROOT" in globals():
        try:
            pr = Path(PACKAGE_ROOT).resolve()
            roots.extend([pr, pr.parent])
        except Exception:
            pass

    # Bounded direct + recursive discovery. The working snapshot contains only
    # project files (tens of thousands, not millions), so recursive manifest
    # discovery is safer than assuming a particular extraction depth.
    for root in list(dict.fromkeys(roots)):
        if not root.exists():
            continue

        direct = [
            root,
            root / "IHARQ_Cumulative_GitHub_Ready_Through_P01_R1",
        ]
        for d in direct:
            if (
                (d / "CURRENT_CUMULATIVE_REPOSITORY_MANIFEST.json").is_file()
                and (d / "CURRENT_PROJECT_STATUS.json").is_file()
            ):
                dirs.append(d.resolve())

        try:
            for manifest in root.rglob(
                "CURRENT_CUMULATIVE_REPOSITORY_MANIFEST.json"
            ):
                d = manifest.parent
                if (d / "CURRENT_PROJECT_STATUS.json").is_file():
                    dirs.append(d.resolve())
                if len(dirs) >= 20:
                    break
        except Exception:
            pass

        for pat in (
            "*Cumulative*P01*.zip",
            "*Through_P01*.zip",
            "*Phase_01*Final*.zip",
        ):
            try:
                for p in root.rglob(pat):
                    if p.is_file():
                        zips.append(p.resolve())
                    if len(zips) >= 20:
                        break
            except Exception:
                pass

    return (
        sorted(set(dirs)),
        sorted(set(zips)),
    )

p01_dirs, p01_zips = _discover_p01_candidates()

if not p01_dirs and not p01_zips:
    raise RuntimeError(
        "PRE_G24_P01_CUMULATIVE_SOURCE_NOT_FOUND__"
        "REPLACEMENT_STAGE24_REQUIRES_PRIOR_PROJECT_STATE_FOR_CUMULATIVE_ZIP"
    )

# -----------------------------------------------------------------------------
# 9. Disk/resource readiness.
# -----------------------------------------------------------------------------
du = shutil.disk_usage(KW)
free_gib = du.free / (1024 ** 3)

if free_gib < 3.0:
    raise RuntimeError(
        f"PRE_G24_DISK_FREE_TOO_LOW_FOR_RELEASE_PACKAGING:{free_gib:.3f}GiB"
    )

# -----------------------------------------------------------------------------
# 10. Final pre-G24 receipt.
# -----------------------------------------------------------------------------
pre_g24_receipt = {
    "artifact_id": "P02-PRE-G24-STAGE18S-CONSOLIDATION-READINESS-R1",
    "created_at_utc": _utc(),
    "status": "PASS",
    "run_root": str(RUN_ROOT),
    "store_root": str(STORE_ROOT),
    "accepted_stages": stage_status,
    "passed_gates": gate_status,
    "stage23_evidence_sufficiency": suff,
    "stage23_required_state": s23_state,
    "canonical_stage18": {
        "a4_completion_status": a4c.get("status"),
        "a4_terminal_files": len(canonical_a4_terminals),
        "G18_unchanged": True,
    },
    "stage18S": {
        "execution_status": s18s_exec.get("status"),
        "supplement_gate_status": s18s_gate.get("status"),
        "claim_scope": s18s_gate.get("claim_scope"),
        "planned_cells": s18s_exec.get("planned_cells"),
        "terminal_counts": s18s_exec.get("terminal_counts"),
        "member_receipts": s18s_exec.get("member_receipts"),
        "member_status_counts": s18s_exec.get("member_status_counts"),
        "complete_output_files": len(s18s_rows),
        "complete_output_bytes": sum(r["bytes"] for r in s18s_rows),
        "project_visible_analysis_files": len(analysis_rows),
    },
    "stage20_readiness_record": readiness_record_validation,
    "phase_release_inventory": {
        "checkpoint_files": len(checkpoint_rows),
        "checkpoint_bytes": sum(r["bytes"] for r in checkpoint_rows),
        "structured_output_files": len(tabular_rows),
        "structured_output_bytes": sum(r["bytes"] for r in tabular_rows),
        "expected_stage_artifacts_present":
            len(EXPECTED_POST18_STAGE_ARTIFACTS),
        "expected_handoff_families_present":
            len(EXPECTED_HANDOFF_BASENAMES),
    },
    "secret_scan": {
        "status": "PASS",
        "hits": [],
    },
    "p01_cumulative_source": {
        "directory_candidates": [str(p) for p in p01_dirs],
        "zip_candidates": [str(p) for p in p01_zips],
        "available": True,
    },
    "disk": {
        "free_gib": round(free_gib, 3),
    },
    "release_family_artifact":
        str(stage18_release_family_path.relative_to(RUN_ROOT)),
    "release_handoff":
        str(release_handoff_path.relative_to(RUN_ROOT)),
    "next_action":
        "RUN_REPLACEMENT_STAGE24_GOVERNED_FINALIZATION_AND_RELEASE_R1",
}

PRE_G24_RECEIPT_PATH = (
    RUN_ROOT
    / "stage_artifacts"
    / "23B_pre_stage24_release_readiness_R1.json"
)
_atomic_json(PRE_G24_RECEIPT_PATH, pre_g24_receipt)

# Byte identity recheck for protected canonical Stage18 files after all writes.
for row in canonical_stage18_protected_rows:
    p = Path(row["absolute_source_path"])
    now = _sha256_file(p)
    if now != row["sha256"]:
        raise RuntimeError(
            "PRE_G24_CANONICAL_STAGE18_BYTE_IDENTITY_CHANGED:"
            + str(p)
        )

print("=" * 120)
print("PRE-STAGE24 STAGE18 + STAGE18S CONSOLIDATION / RELEASE READINESS — PASS")
print("=" * 120)
print(json.dumps({
    "canonical_stage18_a4_terminals":
        len(canonical_a4_terminals),
    "stage18S_outputs_indexed":
        len(s18s_rows),
    "stage18S_terminal_counts":
        s18s_exec.get("terminal_counts"),
    "stage18S_member_receipts":
        s18s_exec.get("member_receipts"),
    "checkpoint_files_indexed":
        len(checkpoint_rows),
    "structured_output_files_indexed":
        len(tabular_rows),
    "p01_cumulative_directory_candidates":
        len(p01_dirs),
    "p01_cumulative_zip_candidates":
        len(p01_zips),
    "disk_free_gib":
        round(free_gib, 3),
    "canonical_G18_modified":
        False,
    "stage20_readiness_record":
        readiness_record_validation["path"],
    "stage20_readiness_rows":
        readiness_record_validation["rows"],
    "ready_for_replacement_stage24":
        True,
    "receipt":
        str(PRE_G24_RECEIPT_PATH),
}, indent=2))

print("\nTRUSTED MARKER:")
print("IHARQ_P02_PRE_G24_STAGE18_STAGE18S_CONSOLIDATION_READY_R1")


Stage20 readiness record resolved = /kaggle/working/iharq_p02_run/runtime/records/Layer2ReadinessReport/dataset=ALL/branch=READINESS/budget=ALL/P02-L2-READINESS-P03.jsonl
Stage20 readiness record rows = 1
Stage20 readiness record sha256 = 1df39fb42d39e9d0b7dcee948bb938b17ba7350d581d3fbc0a9c2c34501d79a2
PRE-STAGE24 STAGE18 + STAGE18S CONSOLIDATION / RELEASE READINESS — PASS
{
  "canonical_stage18_a4_terminals": 1218,
  "stage18S_outputs_indexed": 1743,
  "stage18S_terminal_counts": {
    "INPUT_INCOMPATIBLE": 27,
    "SUCCESS": 135
  },
  "stage18S_member_receipts": 216,
  "checkpoint_files_indexed": 1281,
  "structured_output_files_indexed": 15149,
  "p01_cumulative_directory_candidates": 0,
  "p01_cumulative_zip_candidates": 1,
  "disk_free_gib": 3.733,
  "canonical_G18_modified": false,
  "stage20_readiness_record": "/kaggle/working/iharq_p02_run/runtime/records/Layer2ReadinessReport/dataset=ALL/branch=READINESS/budget=ALL/P02-L2-READINESS-P03.jsonl",
  "stage20_readiness_rows": 1,
 

## Stage 24 — Final execution bundle/checksums/secret scan/export

**Gate:** `G24`  
**Inputs:** all outputs  
**Outputs:** immutable P02 bundle

This cell invokes the governed production handler through the single `NotebookSession` dispatcher. It does not redefine the scientific design.


In [23]:
# =============================================================================
# IHARQ P02/L2 — STAGE 24 LOW-DISK TRANSPORT SUCCESSOR R2.2 — MAX-3GiB PACKAGE TRANSPORT
# EMERGENCY CLEANUP + MAX-3GiB PACKAGE UPLOAD + CREATE/UPLOAD/DELETE + GITHUB-READY PACKAGE ON HF
#
# WHY THIS EXISTS
# ---------------
# The governed Stage24 handler completed far enough to attempt final bundle
# transport, but the original monolithic ZIP finalizer exhausted Kaggle disk:
#
#     OSError: [Errno 28] No space left on device
#
# This cell fixes ONLY the transport/release layer. It does not rerun training,
# alter metrics, rewrite Stage18/G18, or replace the accepted Stage11/12 science.
#
# CORE STRATEGY
# -------------
# 1. Remove ONLY failed/derived Stage24 transport scratch created by the failed
#    attempt, especially the multi-GiB *.tmp ZIP that consumed the disk.
# 2. Preserve every canonical runtime/scientific file and every accepted ledger.
# 3. If Stage24/G24 is already accepted (expected after the observed failure),
#    DO NOT rerun Stage24 science. If it is not accepted, run the exact Stage24
#    HANDLER through StageRunner without invoking the old monolithic ZIP finalizer.
# 4. Reproduce the original finalizer's manifest/checksum/security preparation,
#    but replace its one-huge-local-ZIP transport with:
#
#       a) heavy/checkpoint/raw artifacts -> HF directly from source files;
#       b) remaining execution-bundle files -> bounded ZIP parts;
#          create one part -> validate -> upload -> DELETE LOCAL PART;
#       c) exact multipart manifest + per-file SHA-256 pointers;
#       d) P00+P01 + P02 lightweight cumulative/GitHub-ready package;
#       e) upload the GitHub-ready package to the SAME HF repo too;
#       f) upload cumulative project-state package to HF too;
#       g) delete uploaded local release ZIPs/parts immediately.
#
# 5. Keep only small local receipts/pointers so Kaggle disk remains healthy.
#
# GOVERNANCE
# ----------
# - Canonical Stage18/G18 remains untouched.
# - Stage18S remains POST_HOC_SENSITIVITY_DESCRIPTIVE and is explicitly
#   preserved in the release family/manifests.
# - Corrected/accepted Stage11 and Stage12 remain the canonical upstream
#   evidence consumed by downstream stages.
# - GitHub-ready is an intermediate curated derivative. This cell uploads the
#   GitHub-ready PACKAGE to Hugging Face for preservation; it does not publish
#   or push a live GitHub repository.
# - The cumulative project state remains the primary intermediate continuation
#   package; oversized artifacts are externalized with immutable HF pointers.
# =============================================================================

from __future__ import annotations

from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import csv
import hashlib
import json
import math
import os
import re
import shutil
import sys
import threading
import time
import traceback
import uuid
import zipfile

# =============================================================================
# USER CONFIG — SET THESE BEFORE RUNNING
# =============================================================================

# Hard-code your Hugging Face WRITE token here. No interactive prompt.
HF_TOKEN = "${IHARQ_HF_TOKEN_P02}"

# Leave blank to auto-create a unique private dataset repository.
HF_REPO_ID = ""

HF_REPO_TYPE = "dataset"
HF_PRIVATE = True

# HARD USER-REQUESTED TRANSPORT LIMIT.
# No locally-created release package may exceed 3 GiB.
MAX_PACKAGE_GIB = 3.0

# We intentionally group only ~2.70 GiB of source bytes per ZIP so ZIP headers
# and filesystem metadata cannot push the completed package above 3 GiB.
PACKAGE_SOURCE_TARGET_GIB = 2.70

# Keep this much free disk AFTER the currently-created package is accounted for.
MIN_FREE_RESERVE_MIB = 700

# Treat model/checkpoint/raw/prediction/large files as the "heavy" package family.
EXTERNALIZE_MIN_MIB = 24

# If an HF repository-commit 429 has a short reset, wait automatically.
# A long (e.g. ~1 hour) reset fails fast so the Kaggle session is not wasted.
AUTO_WAIT_ON_SHORT_429 = True
MAX_AUTOMATIC_429_WAIT_SECONDS = 600

# Start this test in a NEW repository by default rather than continuing the
# many-commit R2.1 repository that already hit the commit quota.
START_FRESH_HF_REPO = True

# P01 cumulative source override. Normally leave blank: the pre-G24 receipt
# already found the P01 ZIP.
P01_CUMULATIVE_SOURCE = ""

# Keep generated release ZIP locally after upload?
# False is the low-disk behavior requested: upload then remove.
KEEP_LOCAL_RELEASE_ZIPS = False

# Preserve the tiny remote-resume state across retries. This avoids creating
# orphan repos or reuploading already-confirmed files after a network failure.
PRESERVE_REMOTE_RESUME_STATE = True

# =============================================================================
# FROZEN PROJECT IDENTITIES
# =============================================================================

EXPECTED_STAGE11_SOURCE_SHA256 = (
    "65a506adb6fa6ce20c2ecd11f9f72cf91341c17565d446b4258344e0701f30c2"
)
EXPECTED_STAGE12_SOURCE_SHA256 = (
    "12c8196a581fee0571508003a90805a936664dc35418055dacb6426f51e75d75"
)

EXPECTED_PRE_G24_ARTIFACT_ID = (
    "P02-PRE-G24-STAGE18S-CONSOLIDATION-READINESS-R1"
)

EXPECTED_S18S = {
    "planned_cells": 162,
    "terminal_counts": {
        "SUCCESS": 135,
        "INPUT_INCOMPATIBLE": 27,
    },
    "member_receipts": 216,
    "member_status_counts": {
        "SUCCESS": 189,
        "INPUT_INCOMPATIBLE": 27,
    },
    "claim_scope": "POST_HOC_SENSITIVITY_DESCRIPTIVE",
}

CHECKPOINT_SUFFIXES = {
    ".pt", ".pth", ".ckpt", ".safetensors", ".onnx",
    ".joblib", ".pkl",
}
HEAVY_BINARY_SUFFIXES = {
    ".npy", ".npz", ".h5", ".hdf5", ".bin",
    ".arrow", ".feather",
}
TEXT_SUFFIXES = {
    ".txt", ".md", ".json", ".jsonl", ".yaml", ".yml",
    ".csv", ".tsv", ".py", ".toml", ".ini", ".cfg",
    ".cff", ".sha256",
}

MiB = 1024 ** 2
GiB = 1024 ** 3
MAX_PACKAGE_BYTES = int(MAX_PACKAGE_GIB * GiB)
PACKAGE_SOURCE_TARGET_BYTES = int(PACKAGE_SOURCE_TARGET_GIB * GiB)
# Keep the old name for downstream helper compatibility, but its meaning is now
# the conservative source-byte target, not the hard completed-package limit.
CHUNK_TARGET_BYTES = PACKAGE_SOURCE_TARGET_BYTES
MIN_FREE_RESERVE_BYTES = int(MIN_FREE_RESERVE_MIB * MiB)
EXTERNALIZE_MIN_BYTES = int(EXTERNALIZE_MIN_MIB * MiB)

# =============================================================================
# PATH RESOLUTION
# =============================================================================

KW = Path("/kaggle/working").resolve()

def _resolve_run_root():
    candidates = []

    if "RUN_ROOT" in globals():
        try:
            candidates.append(Path(RUN_ROOT).resolve())
        except Exception:
            pass

    if "STORE" in globals():
        try:
            sr = Path(STORE.root).resolve()
            candidates.extend([sr.parent, sr])
        except Exception:
            pass

    candidates.extend([
        KW / "iharq_p02_run",
        KW / "IHARQ_P02_RUN",
    ])

    for p in candidates:
        if (
            (p / "stage_ledger").is_dir()
            and (p / "gate_results").is_dir()
        ):
            return p

    raise RuntimeError(
        "STAGE24_R2_RUN_ROOT_NOT_RESOLVED:"
        + json.dumps([str(x) for x in candidates])
    )

RUN_ROOT = _resolve_run_root()

if "STORE" in globals():
    try:
        STORE_ROOT = Path(STORE.root).resolve()
    except Exception:
        STORE_ROOT = RUN_ROOT / "runtime"
else:
    STORE_ROOT = RUN_ROOT / "runtime"

if not STORE_ROOT.is_dir():
    raise RuntimeError(
        f"STAGE24_R2_STORE_ROOT_MISSING:{STORE_ROOT}"
    )

PACKAGE_ROOT_LOCAL = None
if "PACKAGE_ROOT" in globals():
    try:
        PACKAGE_ROOT_LOCAL = Path(PACKAGE_ROOT).resolve()
    except Exception:
        PACKAGE_ROOT_LOCAL = None

if (
    PACKAGE_ROOT_LOCAL is None
    and "SESSION" in globals()
):
    try:
        PACKAGE_ROOT_LOCAL = Path(
            SESSION.package_root
        ).resolve()
    except Exception:
        PACKAGE_ROOT_LOCAL = None

if (
    PACKAGE_ROOT_LOCAL is None
    or not PACKAGE_ROOT_LOCAL.is_dir()
):
    raise RuntimeError(
        "STAGE24_R2_PACKAGE_ROOT_NOT_RESOLVED"
    )

S18S_ROOT = (
    STORE_ROOT
    / "supplements"
    / "stage18S_balanced_sensitivity_R1"
)

PRE_G24_RECEIPT = (
    RUN_ROOT
    / "stage_artifacts"
    / "23B_pre_stage24_release_readiness_R1.json"
)

RELEASE_ROOT = KW / "IHARQ_P02_STAGE24_MAX3G_R2_2"
SCRATCH = RELEASE_ROOT / "scratch"
PROJECT_STAGING = RELEASE_ROOT / "project_state_staging"
MANIFEST_DIR = RELEASE_ROOT / "manifests"

# Small state file only: no token. Kept across retries for remote resume.
REMOTE_STATE_PATH = (
    RUN_ROOT
    / "stage_artifacts"
    / "24_max3g_remote_resume_state_R2_2.json"
)

FINAL_LOCAL_RECEIPT = (
    KW
    / "IHARQ_P02_STAGE24_LOW_DISK_R2_RELEASE_POINTER.json"
)

# =============================================================================
# GENERIC HELPERS
# =============================================================================

def _utc():
    return datetime.now(timezone.utc).isoformat()

def _sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(
            lambda: f.read(8 * 1024 * 1024),
            b"",
        ):
            h.update(block)
    return h.hexdigest()

def _load_json(path: Path):
    return json.loads(
        Path(path).read_text(encoding="utf-8")
    )

def _atomic_json(path: Path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            default=str,
        ) + "\n",
        encoding="utf-8",
    )
    os.replace(tmp, path)

def _atomic_text(path: Path, text: str):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    os.replace(tmp, path)

def _disk():
    du = shutil.disk_usage(KW)
    return {
        "total_gib": round(du.total / GiB, 3),
        "used_gib": round(du.used / GiB, 3),
        "free_gib": round(du.free / GiB, 3),
        "free_bytes": du.free,
    }

def _accepted(stage_id: str):
    if "SESSION" not in globals():
        return None
    try:
        return SESSION.runner.accepted(str(stage_id))
    except Exception:
        return None

def _assert_accepted(stage_id: str):
    obj = _accepted(stage_id)
    if not obj:
        raise RuntimeError(
            f"STAGE24_R2_STAGE_NOT_ACCEPTED:{stage_id}"
        )
    return obj

def _remove_path(path: Path):
    p = Path(path)
    if not p.exists():
        return 0

    size = 0
    try:
        if p.is_file():
            size = p.stat().st_size
            p.unlink()
        elif p.is_dir():
            for q in p.rglob("*"):
                if q.is_file():
                    try:
                        size += q.stat().st_size
                    except Exception:
                        pass
            shutil.rmtree(p)
    except FileNotFoundError:
        pass

    return size

def _is_valid_zip(path: Path):
    p = Path(path)
    if not p.is_file():
        return False
    try:
        with zipfile.ZipFile(p, "r") as z:
            return z.testzip() is None
    except Exception:
        return False

def _safe_rel(path: Path, root: Path):
    rel = path.resolve().relative_to(
        root.resolve()
    ).as_posix()

    if (
        rel.startswith("../")
        or rel == ".."
        or rel.startswith("/")
        or "\x00" in rel
    ):
        raise RuntimeError(
            f"UNSAFE_RELATIVE_PATH:{rel}"
        )
    return rel

def _secret_hits(path: Path):
    p = Path(path)

    if (
        not p.is_file()
        or p.stat().st_size > 20_000_000
        or p.suffix.lower() not in TEXT_SUFFIXES
    ):
        return []

    try:
        text = p.read_text(
            encoding="utf-8",
            errors="ignore",
        )
    except Exception:
        return []

    patterns = {
        "HF_TOKEN": r"\bhf_[A-Za-z0-9_-]{20,}\b",
        "GITHUB_TOKEN": r"\bgh[pousr]_[A-Za-z0-9]{30,}\b",
        "GITHUB_PAT": r"\bgithub_pat_[A-Za-z0-9_]{20,}\b",
        "PRIVATE_KEY":
            r"-----BEGIN (?:RSA |EC |OPENSSH )?PRIVATE KEY-----",
    }

    return [
        k
        for k, pat in patterns.items()
        if re.search(pat, text)
    ]

def _scan_tree_secrets(root: Path):
    hits = []
    root = Path(root)

    if not root.exists():
        return hits

    for p in root.rglob("*"):
        if not p.is_file():
            continue
        for kind in _secret_hits(p):
            hits.append({
                "path": str(
                    p.relative_to(root)
                ),
                "kind": kind,
            })

    return hits

def _safe_extract_zip(src_zip: Path, dst: Path):
    src_zip = Path(src_zip)
    dst = Path(dst)

    with zipfile.ZipFile(src_zip, "r") as z:
        bad = z.testzip()
        if bad is not None:
            raise RuntimeError(
                f"P01_SOURCE_ZIP_CRC_FAIL:{bad}"
            )

        for n in z.namelist():
            pp = Path(n)
            if (
                n.startswith("/")
                or ".." in pp.parts
                or "\x00" in n
            ):
                raise RuntimeError(
                    f"P01_UNSAFE_ZIP_MEMBER:{n}"
                )

        z.extractall(dst)

def _write_yaml(path: Path, obj):
    try:
        import yaml
    except Exception as exc:
        raise RuntimeError(
            "PYYAML_NOT_AVAILABLE:"
            f"{type(exc).__name__}:{exc}"
        )

    _atomic_text(
        path,
        yaml.safe_dump(
            obj,
            sort_keys=False,
            allow_unicode=True,
        ),
    )

def _copy_file(src: Path, dst: Path):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if dst.exists():
        if (
            dst.is_file()
            and dst.stat().st_size
            == src.stat().st_size
            and _sha256_file(dst)
            == _sha256_file(src)
        ):
            return "EXISTING_IDENTICAL"

        if dst.is_dir():
            shutil.rmtree(dst)
        else:
            dst.unlink()

    try:
        os.link(src, dst)
        return "HARDLINK"
    except Exception:
        shutil.copy2(src, dst)
        return "COPY"

def _tree_rows(root: Path):
    root = Path(root)
    rows = []

    for p in sorted(root.rglob("*")):
        if not p.is_file():
            continue

        rows.append({
            "path":
                p.relative_to(root).as_posix(),
            "bytes":
                p.stat().st_size,
            "sha256":
                _sha256_file(p),
        })

    return rows

# =============================================================================
# 1. EMERGENCY CLEANUP — NO WRITES UNTIL SPACE HAS ACTUALLY BEEN RECOVERED
# =============================================================================
#
# The prior R2 cleanup only looked in /kaggle/working itself. The original
# package finalizer writes its temporary bundle to ctx.work_root, which in this
# run is under /kaggle/working/iharq_p02_run. Therefore the multi-GiB failed
# *.tmp bundle was not found.
#
# R2.1 recursively searches ONLY for tightly-scoped Stage24/release transport
# artifacts. It does not delete scientific runtime evidence.
# =============================================================================

print("=" * 120)
print("STAGE24 R2.1 — EMERGENCY LOW-DISK RETRY CLEANUP")
print("=" * 120)

disk_before = _disk()
print(
    "disk_before =",
    json.dumps(
        {
            k: v
            for k, v in disk_before.items()
            if k != "free_bytes"
        },
        indent=2,
    ),
)

# -----------------------------------------------------------------------------
# 1A. Resolve every plausible transport root without creating anything.
# -----------------------------------------------------------------------------
transport_roots = []

def _add_transport_root(p):
    try:
        p = Path(p).resolve()
    except Exception:
        return
    if p.exists() and p not in transport_roots:
        transport_roots.append(p)

_add_transport_root(KW)
_add_transport_root(RUN_ROOT)
_add_transport_root(RUN_ROOT.parent)

if "SESSION" in globals():
    for attr_chain in [
        ("ctx", "work_root"),
        ("ctx", "run_root"),
        ("ctx", "state"),
    ]:
        try:
            obj = SESSION
            for attr in attr_chain:
                obj = getattr(obj, attr)
            if isinstance(obj, (str, os.PathLike, Path)):
                _add_transport_root(obj)
        except Exception:
            pass

# Common state-key variants.
if "SESSION" in globals():
    try:
        st = SESSION.ctx.state
        if isinstance(st, dict):
            for key in (
                "work_root",
                "run_root",
                "output_root",
            ):
                if st.get(key):
                    _add_transport_root(st[key])
    except Exception:
        pass

print(
    "transport_roots =",
    [str(x) for x in transport_roots],
)

# -----------------------------------------------------------------------------
# 1B. Strict transport filename classifiers.
# -----------------------------------------------------------------------------
_transport_tmp_regexes = [
    re.compile(
        r"^IHARQ_P02_L2_Phase_Execution_Bundle_.+\.tmp$"
    ),
    re.compile(
        r"^IHARQ_P02_L2_Phase_Execution_Bundle_.+\.zip\.tmp$"
    ),
    re.compile(
        r"^IHARQ_P02_L2_Partial_Failure_Bundle_.+\.tmp$"
    ),
    re.compile(
        r"^IHARQ_P02_L2_Partial_Failure_Bundle_.+\.zip\.tmp$"
    ),
    re.compile(
        r"^FIXTURE_NON_SCIENTIFIC_RUNTIME_BUNDLE(?:\.zip)?\.tmp$"
    ),
    re.compile(
        r"^FIXTURE_NON_SCIENTIFIC_PARTIAL_FAILURE_BUNDLE(?:\.zip)?\.tmp$"
    ),
    re.compile(
        r"^IHARQ_Project_State_After_Phase_02_R(?:1|2)(?:_.*)?\.zip\.tmp$"
    ),
    re.compile(
        r"^IHARQ_Cumulative_GitHub_Ready_Through_P02_R(?:1|2)(?:_.*)?\.zip\.tmp$"
    ),
]

_transport_final_zip_regexes = [
    re.compile(
        r"^IHARQ_P02_L2_Phase_Execution_Bundle_.+\.zip$"
    ),
    re.compile(
        r"^IHARQ_P02_L2_Partial_Failure_Bundle_.+\.zip$"
    ),
]

# Exact derived directories from failed release wrappers. These are transport
# staging only and are safe to remove; canonical runtime lives elsewhere.
_derived_release_dir_names = {
    "IHARQ_P02_RELEASE_TRANSACTION_R1",
    "IHARQ_P02_STAGE24_LOW_DISK_R2",
    "IHARQ_P02_STAGE24_LOW_DISK_R2_1",
    "IHARQ_P02_STAGE24_MAX3G_R2_2",
}

# -----------------------------------------------------------------------------
# 1C. Discover candidates recursively.
#     We purposely do NOT use a generic "*.tmp" delete.
# -----------------------------------------------------------------------------
candidate_files = {}
candidate_dirs = {}

def _record_file(p: Path, reason: str):
    try:
        rp = p.resolve()
    except Exception:
        rp = p
    if not p.is_file():
        return
    key = str(rp)
    if key not in candidate_files:
        try:
            size = p.stat().st_size
        except Exception:
            size = 0
        candidate_files[key] = {
            "path": p,
            "bytes": size,
            "reason": reason,
        }

def _record_dir(p: Path, reason: str):
    try:
        rp = p.resolve()
    except Exception:
        rp = p
    if p.is_dir():
        candidate_dirs[str(rp)] = {
            "path": p,
            "reason": reason,
        }

# Exact root-level derived release dirs first.
for name in sorted(_derived_release_dir_names):
    p = KW / name
    if p.is_dir():
        _record_dir(
            p,
            "FAILED_DERIVED_RELEASE_STAGING",
        )

# Recursive walk across unique roots. Avoid following symlinks.
walked = set()
for root in transport_roots:
    try:
        rr = root.resolve()
    except Exception:
        rr = root
    if str(rr) in walked:
        continue
    walked.add(str(rr))

    try:
        for dirpath, dirnames, filenames in os.walk(
            rr,
            topdown=True,
            followlinks=False,
        ):
            dpath = Path(dirpath)

            # Never descend into our canonical package source .git/caches.
            dirnames[:] = [
                d for d in dirnames
                if d not in {
                    ".git",
                    "__pycache__",
                    ".ipynb_checkpoints",
                }
            ]

            # Derived release directories may also exist one level deeper.
            for d in list(dirnames):
                if d in _derived_release_dir_names:
                    _record_dir(
                        dpath / d,
                        "FAILED_DERIVED_RELEASE_STAGING",
                    )
                    # Do not descend; the whole derived dir will be removed.
                    dirnames.remove(d)

            for name in filenames:
                p = dpath / name

                if any(
                    rx.match(name)
                    for rx in _transport_tmp_regexes
                ):
                    _record_file(
                        p,
                        "FAILED_STAGE24_TRANSPORT_TMP",
                    )
                    continue

                # Invalid final transport ZIPs from a failed attempt are safe
                # to remove; valid ZIPs are preserved.
                if any(
                    rx.match(name)
                    for rx in _transport_final_zip_regexes
                ):
                    if not _is_valid_zip(p):
                        _record_file(
                            p,
                            "INVALID_STAGE24_TRANSPORT_ZIP",
                        )

    except Exception as exc:
        print(
            "cleanup scan warning:",
            str(root),
            type(exc).__name__,
            str(exc)[:300],
        )

# Also remove old derived P02 release ZIPs created only by the failed wrappers.
for pattern in [
    "IHARQ_Project_State_After_Phase_02_R1.zip",
    "IHARQ_Project_State_After_Phase_02_R2.zip",
    "IHARQ_Cumulative_GitHub_Ready_Through_P02_R1.zip",
    "IHARQ_Cumulative_GitHub_Ready_Through_P02_R2.zip",
]:
    for root in transport_roots:
        try:
            for p in root.rglob(pattern):
                if p.is_file():
                    _record_file(
                        p,
                        "FAILED_DERIVED_RELEASE_ZIP",
                    )
        except Exception:
            pass

# Largest candidates first so we recover usable space immediately.
file_candidates_sorted = sorted(
    candidate_files.values(),
    key=lambda x: x["bytes"],
    reverse=True,
)

print("\ntransport_cleanup_candidates:")
if not file_candidates_sorted and not candidate_dirs:
    print("  NONE FOUND BY STRICT TRANSPORT PATTERNS")
else:
    for rec in file_candidates_sorted:
        print(
            "  FILE",
            f"{rec['bytes'] / MiB:,.1f} MiB",
            rec["reason"],
            rec["path"],
        )
    for rec in candidate_dirs.values():
        print(
            "  DIR ",
            rec["reason"],
            rec["path"],
        )

# -----------------------------------------------------------------------------
# 1D. Delete files first. unlink() itself needs no new output directory.
# -----------------------------------------------------------------------------
freed_logical_bytes = 0
removed_files = 0
removed_dirs = 0

for rec in file_candidates_sorted:
    p = rec["path"]
    try:
        before = p.stat().st_size
    except Exception:
        before = rec["bytes"]

    try:
        p.unlink()
        removed_files += 1
        freed_logical_bytes += before
        print(
            "removed transport file:",
            p,
            f"({before / MiB:,.1f} MiB)",
            rec["reason"],
            flush=True,
        )
    except FileNotFoundError:
        pass

# Remove derived staging dirs after their large transport files.
for rec in sorted(
    candidate_dirs.values(),
    key=lambda x: len(str(x["path"])),
    reverse=True,
):
    p = rec["path"]
    if not p.exists():
        continue

    # Logical bytes are informational only; hardlinks may not free equivalent
    # physical bytes.
    logical = 0
    try:
        for q in p.rglob("*"):
            if q.is_file():
                try:
                    logical += q.stat().st_size
                except Exception:
                    pass
    except Exception:
        pass

    try:
        shutil.rmtree(p)
        removed_dirs += 1
        print(
            "removed transport directory:",
            p,
            f"(logical {logical / MiB:,.1f} MiB)",
            rec["reason"],
            flush=True,
        )
    except FileNotFoundError:
        pass

# Flush filesystem metadata if supported.
try:
    os.sync()
except Exception:
    pass

disk_after_primary_cleanup = _disk()

print(
    "\nprimary_cleanup_result =",
    json.dumps({
        "removed_files": removed_files,
        "removed_dirs": removed_dirs,
        "logical_bytes_unlinked":
            freed_logical_bytes,
        "logical_gib_unlinked":
            round(
                freed_logical_bytes / GiB,
                3,
            ),
        "disk_free_gib":
            disk_after_primary_cleanup[
                "free_gib"
            ],
    }, indent=2),
)

# -----------------------------------------------------------------------------
# 1E. Emergency second pass if the filesystem is STILL critically full.
#
# Delete only LARGE transport-looking *.tmp files. This catches variant names
# while retaining a strict keyword safety boundary. No scientific checkpoint,
# prediction, metric, record, or accepted artifact is selected merely because
# it is large.
# -----------------------------------------------------------------------------
if (
    disk_after_primary_cleanup["free_bytes"]
    < MIN_FREE_RESERVE_BYTES
    + MAX_PACKAGE_BYTES
):
    emergency = []

    safe_transport_keywords = (
        "bundle",
        "release",
        "project_state",
        "github",
        "stage24",
        "finalization",
        "export",
    )

    for root in transport_roots:
        try:
            for p in root.rglob("*.tmp"):
                if not p.is_file():
                    continue

                name_l = p.name.lower()
                path_l = p.as_posix().lower()

                if not any(
                    k in name_l or k in path_l
                    for k in safe_transport_keywords
                ):
                    continue

                try:
                    size = p.stat().st_size
                except Exception:
                    continue

                if size < 32 * MiB:
                    continue

                emergency.append(
                    (size, p)
                )
        except Exception:
            pass

    emergency = sorted(
        {
            str(p.resolve()): (size, p)
            for size, p in emergency
        }.values(),
        key=lambda x: x[0],
        reverse=True,
    )

    if emergency:
        print(
            "\nEmergency transport-temp cleanup:"
        )

    for size, p in emergency:
        try:
            p.unlink()
            freed_logical_bytes += size
            print(
                "removed emergency transport tmp:",
                p,
                f"({size / MiB:,.1f} MiB)",
                flush=True,
            )
        except FileNotFoundError:
            pass

        # Stop as soon as we have enough headroom.
        if (
            shutil.disk_usage(KW).free
            >= MIN_FREE_RESERVE_BYTES
            + 2 * CHUNK_TARGET_BYTES
        ):
            break

    try:
        os.sync()
    except Exception:
        pass

disk_after_cleanup = _disk()

print(
    "\ndisk_after_cleanup =",
    json.dumps(
        {
            k: v
            for k, v in disk_after_cleanup.items()
            if k != "free_bytes"
        },
        indent=2,
    ),
)

# -----------------------------------------------------------------------------
# 1F. If still insufficient, fail BEFORE mkdir and show the largest files.
# -----------------------------------------------------------------------------
required_free = (
    MIN_FREE_RESERVE_BYTES
    + MAX_PACKAGE_BYTES
)

if disk_after_cleanup["free_bytes"] < required_free:
    largest = []

    for root in transport_roots:
        try:
            for p in root.rglob("*"):
                if not p.is_file():
                    continue
                try:
                    size = p.stat().st_size
                except Exception:
                    continue
                if size >= 64 * MiB:
                    largest.append(
                        (size, p)
                    )
        except Exception:
            pass

    largest = sorted(
        {
            str(p.resolve()): (size, p)
            for size, p in largest
        }.values(),
        key=lambda x: x[0],
        reverse=True,
    )[:30]

    print(
        "\nLARGEST FILES REMAINING (diagnostic only; NOT deleted):"
    )
    for size, p in largest:
        print(
            f"  {size / GiB:8.3f} GiB  {p}"
        )

    raise RuntimeError(
        "STAGE24_R2_1_INSUFFICIENT_DISK_AFTER_EMERGENCY_CLEANUP:"
        + json.dumps({
            "free_gib":
                disk_after_cleanup[
                    "free_gib"
                ],
            "required_min_gib":
                round(
                    required_free / GiB,
                    3,
                ),
            "important":
                "NO_SCIENTIFIC_RUNTIME_EVIDENCE_WAS_AUTOMATICALLY_DELETED",
        })
    )

# -----------------------------------------------------------------------------
# 1G. ONLY NOW create fresh low-disk transport directories.
# -----------------------------------------------------------------------------
RELEASE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)
SCRATCH.mkdir(
    parents=True,
    exist_ok=True,
)
MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "\ncleanup PASS — sufficient headroom recovered before any retry staging."
)
print(
    "STAGE24_R2_1_EMERGENCY_TRANSPORT_CLEANUP_PASS"
)


# =============================================================================
# 2. VERIFY PRE-G24 RECEIPT + UPSTREAM CANONICAL STATUS
# =============================================================================

if not PRE_G24_RECEIPT.is_file():
    raise RuntimeError(
        "STAGE24_R2_PRE_G24_RECEIPT_MISSING__"
        "RUN_THE_PREG24_R1_1_CELL_FIRST"
    )

pre = _load_json(PRE_G24_RECEIPT)

if (
    pre.get("artifact_id")
    != EXPECTED_PRE_G24_ARTIFACT_ID
    or pre.get("status") != "PASS"
):
    raise RuntimeError(
        "STAGE24_R2_PRE_G24_RECEIPT_NOT_PASS"
    )

# The corrected Stage11/12 are the accepted canonical upstream stages.
s11_accepted = _assert_accepted("11")
s12_accepted = _assert_accepted("12")

for gid in ["G11", "G12"]:
    gp = RUN_ROOT / "gate_results" / f"{gid}.json"
    if not gp.is_file():
        raise RuntimeError(
            f"STAGE24_R2_GATE_MISSING:{gid}"
        )
    gd = _load_json(gp)
    if gd.get("status") != "PASS":
        raise RuntimeError(
            f"STAGE24_R2_GATE_NOT_PASS:{gid}"
        )

# The full pre-G24 chain must remain accepted.
for sid in [
    "00", "01", "02", "03", "04", "05", "06", "07",
    "08", "09", "10", "11", "12", "13", "14", "15",
    "16", "17", "18", "18U", "19", "20", "21", "22", "23",
]:
    _assert_accepted(sid)

# Stage18S release family from the successful pre-G24 cell.
release_family_path = (
    RUN_ROOT
    / "stage_artifacts"
    / "18_stage18S_consolidated_release_family_R1.json"
)
release_family = _load_json(
    release_family_path
)

if release_family.get("status") != "PASS":
    raise RuntimeError(
        "STAGE24_R2_STAGE18_RELEASE_FAMILY_NOT_PASS"
    )

s18s = (
    release_family.get("stage18S")
    or {}
)

for k, expected in EXPECTED_S18S.items():
    if s18s.get(k) != expected:
        raise RuntimeError(
            "STAGE24_R2_STAGE18S_DRIFT:"
            + json.dumps({
                "field": k,
                "expected": expected,
                "actual": s18s.get(k),
            })
        )

if (
    release_family
    .get("canonical_stage18", {})
    .get("mutated_by_this_cell")
    is not False
):
    raise RuntimeError(
        "STAGE24_R2_CANONICAL_STAGE18_MUTATION_FLAG"
    )

print("\nCanonical upstream check:")
print(json.dumps({
    "Stage11": "ACCEPTED_MAIN",
    "Stage12": "ACCEPTED_MAIN",
    "G11": "PASS",
    "G12": "PASS",
    "accepted_stage11_source_sha256":
        EXPECTED_STAGE11_SOURCE_SHA256,
    "accepted_stage12_source_sha256":
        EXPECTED_STAGE12_SOURCE_SHA256,
    "Stage18": "ACCEPTED",
    "Stage18S":
        "PASS_POST_HOC_SENSITIVITY_DESCRIPTIVE",
    "canonical_G18_modified": False,
}, indent=2))

# =============================================================================
# 3. AUTHENTICATE HF BEFORE DOING MORE LOCAL PACKAGING
# =============================================================================

# Low-disk HF/Xet policy.
# Keep the Xet workspace inside our disposable release directory, disable the
# chunk cache, and sharply bound the upload shard cache so a default multi-GiB
# cache cannot consume Kaggle's remaining disk during the upload.
HF_XET_CACHE_DIR = RELEASE_ROOT / "hf_xet_cache"
HF_XET_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_XET_CACHE"] = str(HF_XET_CACHE_DIR)
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"
os.environ["HF_XET_SHARD_CACHE_SIZE_LIMIT"] = str(128 * MiB)
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

if (
    not HF_TOKEN
    or HF_TOKEN
       == "PASTE_YOUR_HF_WRITE_TOKEN_HERE"
    or not HF_TOKEN.startswith("hf_")
):
    raise RuntimeError(
        "SET_HF_WRITE_TOKEN_AT_TOP_OF_STAGE24_R2_CELL"
    )

try:
    from huggingface_hub import (
        HfApi,
        CommitOperationAdd,
    )
except Exception as exc:
    raise RuntimeError(
        "HUGGINGFACE_HUB_IMPORT_FAILED:"
        f"{type(exc).__name__}:{exc}"
    )

api = HfApi(token=HF_TOKEN)

try:
    who = api.whoami()
except Exception as exc:
    raise RuntimeError(
        "HF_AUTH_FAILED:"
        f"{type(exc).__name__}:{exc}"
    )

hf_username = (
    who.get("name")
    or who.get("fullname")
    or ""
)

if not hf_username:
    raise RuntimeError(
        "HF_USERNAME_NOT_RESOLVED"
    )

# =============================================================================
# 4. VERIFY OR COMPLETE STAGE24 HANDLER — WITHOUT OLD MONOLITHIC ZIP
# =============================================================================

stage24_existing = _accepted("24")

if stage24_existing:
    print(
        "\nStage24 ledger/G24 handler is already accepted. "
        "This is expected after the observed ENOSPC failure: "
        "the handler succeeded and the failure happened later in "
        "NotebookSession.finalize() while building the monolithic ZIP."
    )
    stage24_result = stage24_existing
else:
    print(
        "\nStage24 is not yet accepted. Running the exact registered "
        "Stage24 HANDLER through StageRunner, intentionally bypassing "
        "only NotebookSession's old monolithic ZIP transport."
    )

    try:
        from iharq.layer2_decoders.orchestration import HANDLERS
    except Exception as exc:
        raise RuntimeError(
            "STAGE24_R2_HANDLER_IMPORT_FAILED:"
            f"{type(exc).__name__}:{exc}"
        )

    stage24_result = SESSION.runner.run(
        "24",
        lambda: HANDLERS["24"](SESSION.ctx),
        reuse=True,
    )

    if (
        stage24_result
        not in getattr(
            SESSION,
            "results",
            [],
        )
    ):
        try:
            SESSION.results.append(
                stage24_result
            )
        except Exception:
            pass

stage24_accepted = _assert_accepted("24")

g24_path = (
    RUN_ROOT
    / "gate_results"
    / "G24.json"
)

if not g24_path.is_file():
    raise RuntimeError(
        "STAGE24_R2_G24_MISSING"
    )

g24 = _load_json(g24_path)

if g24.get("status") != "PASS":
    raise RuntimeError(
        "STAGE24_R2_G24_NOT_PASS"
    )

print(
    "Stage24 governed handler = SUCCESS; G24 = PASS"
)

# =============================================================================
# 5. REPRODUCE ORIGINAL FINALIZER PREPARATION, BUT NO MONOLITHIC ZIP
# =============================================================================

try:
    from iharq.layer2_decoders.bundle import (
        checksums as package_checksums,
        verify as package_verify,
    )
    from iharq.layer2_decoders.security import (
        scan_tree as package_scan_tree,
    )
    from iharq.layer2_decoders.writers import (
        atomic_json as package_atomic_json,
        atomic_yaml as package_atomic_yaml,
    )
except Exception as exc:
    raise RuntimeError(
        "STAGE24_R2_FINALIZER_HELPER_IMPORT_FAILED:"
        f"{type(exc).__name__}:{exc}"
    )

# This mirrors orchestration.finalize() through its checksum verification,
# replacing ONLY zip_bundle(st.root, huge_local_zip).
ledger_dir = SESSION.runner.ledger
work = Path(ledger_dir).parent
st = SESSION.ctx.state["store"]

if Path(st.root).resolve() != STORE_ROOT:
    raise RuntimeError(
        "STAGE24_R2_STORE_ROOT_IDENTITY_MISMATCH"
    )

shutil.copytree(
    ledger_dir,
    STORE_ROOT / "manifests" / "stage_ledger",
    dirs_exist_ok=True,
)

for src, dst in [
    (
        work / "gate_results",
        STORE_ROOT / "gate_results",
    ),
    (
        work / "logs",
        STORE_ROOT / "logs",
    ),
    (
        work / "heartbeats",
        STORE_ROOT
        / "diagnostics"
        / "heartbeats",
    ),
    (
        work / "stage_artifacts",
        STORE_ROOT
        / "manifests"
        / "stage_artifacts",
    ),
]:
    if src.exists():
        shutil.copytree(
            src,
            dst,
            dirs_exist_ok=True,
        )

protocol_dir = (
    STORE_ROOT
    / "protocol_change_required"
)
protocol_dir.mkdir(
    parents=True,
    exist_ok=True,
)

for src_rel, dst_name in [
    (
        "contracts/P02_PREEXECUTION_TRAINING_POLICY_AMENDMENT_R2.yaml",
        "P02_PREEXECUTION_TRAINING_POLICY_AMENDMENT_R2.yaml",
    ),
    (
        "docs/P02_FUTURE_PROTOCOL_BUILD_BOOK_SYNC_NOTE_R2.md",
        "P02_FUTURE_PROTOCOL_BUILD_BOOK_SYNC_NOTE_R2.md",
    ),
    (
        "docs/P02_TRAINING_POLICY_EXTERNAL_EVIDENCE_NOTE_R2.md",
        "P02_TRAINING_POLICY_EXTERNAL_EVIDENCE_NOTE_R2.md",
    ),
]:
    src = (
        PACKAGE_ROOT_LOCAL
        / src_rel
    )
    if not src.is_file():
        raise RuntimeError(
            f"STAGE24_R2_FINALIZER_SOURCE_MISSING:{src}"
        )
    shutil.copy2(
        src,
        protocol_dir / dst_name,
    )

package_atomic_yaml(
    STORE_ROOT
    / "manifests"
    / "config_snapshot.yaml",
    SESSION.ctx.state["config"],
)

package_atomic_yaml(
    STORE_ROOT
    / "manifests"
    / "scientific_freeze_snapshot.yaml",
    SESSION.ctx.state["runtime_freeze"],
)

all_cells = SESSION.ctx.state["all_cells"]

package_atomic_json(
    STORE_ROOT
    / "manifests"
    / "input_and_run_cell_manifest.json",
    {
        "run_id":
            SESSION.ctx.state["run_id"],
        "fixture":
            SESSION.ctx.fixture,
        "planned_cells_in_session":
            len(
                SESSION.ctx.state["cells"]
            ),
        "A0_cells":
            len(
                SESSION.ctx.state["a0"]
            ),
        "A4_cells":
            len(
                SESSION.ctx.state[
                    "a4cells"
                ]
            ),
        "training_policy_challenger_cells":
            len(
                SESSION.ctx.state[
                    "challenger_cells"
                ]
            ),
        "official_full_plan_counts": {
            "A0":
                sum(
                    r["ablation_id"] == "A0"
                    for r in all_cells
                ),
            "A4":
                sum(
                    r["ablation_id"] == "A4"
                    for r in all_cells
                ),
            "total":
                len(all_cells),
        },
        "input_pointer_stage":
            "manifests/stage_artifacts/03_pointers.json",
        "input_validation_stage":
            "manifests/stage_artifacts/04_p01_validation.json",
    },
)

# Explicitly record the transport successor rather than pretending the original
# monolithic local ZIP exists.
package_atomic_json(
    STORE_ROOT
    / "manifests"
    / "runtime_manifest.json",
    {
        "status":
            "FINALIZED_FOR_EXTERNALIZED_MULTIPART_EXPORT",
        "fixture":
            SESSION.ctx.fixture,
        "scientific_evidence":
            False
            if SESSION.ctx.fixture
            else True,
        "run_id":
            SESSION.ctx.state["run_id"],
        "config_sha256":
            SESSION.ctx.state[
                "config_hash"
            ],
        "stage_count_expected": 26,
        "bundle_contents_complete": True,
        "partial_failure": False,
        "failure": None,
        "transport_successor":
            "P02-STAGE24-MAX3G-HF-R2.2",
        "transport_reason":
            "KAGGLE_LOCAL_DISK_ENOSPC_DURING_ORIGINAL_MONOLITHIC_ZIP",
        "scientific_configuration_changed":
            False,
    },
)

package_checksums(STORE_ROOT)
verify_result = package_verify(
    STORE_ROOT
)

if verify_result.get("status") != "PASS":
    raise RuntimeError(
        "STAGE24_R2_STORE_CHECKSUM_VERIFY_FAILED:"
        + json.dumps(
            verify_result,
            sort_keys=True,
        )
    )

security_result = package_scan_tree(
    STORE_ROOT
)

if security_result.get("status") != "PASS":
    raise RuntimeError(
        "STAGE24_R2_STORE_SECRET_SCAN_FAILED:"
        + json.dumps(
            security_result,
            sort_keys=True,
        )
    )

print(
    "\nOriginal finalizer preparation reproduced through checksum/security PASS."
)
print(
    "The only replaced operation is the one-huge-local-ZIP transport."
)

# =============================================================================
# 6. CREATE OR RESUME PRIVATE HF RELEASE REPOSITORY
# =============================================================================

remote_state = {}

# This R2.2 test starts clean by default. A failed R2.2 retry can be resumed by
# setting START_FRESH_HF_REPO=False before rerunning.
if START_FRESH_HF_REPO:
    try:
        REMOTE_STATE_PATH.unlink(missing_ok=True)
    except Exception:
        pass
elif (
    PRESERVE_REMOTE_RESUME_STATE
    and REMOTE_STATE_PATH.is_file()
):
    try:
        remote_state = _load_json(
            REMOTE_STATE_PATH
        )
    except Exception:
        remote_state = {}

run_id = str(
    SESSION.ctx.state["run_id"]
)

if remote_state.get("hf_repo_id"):
    hf_repo_id = remote_state[
        "hf_repo_id"
    ]
else:
    if HF_REPO_ID.strip():
        hf_repo_id = (
            HF_REPO_ID.strip()
        )
    else:
        stamp = (
            datetime
            .now(timezone.utc)
            .strftime(
                "%Y%m%dt%H%M%Sz"
            )
            .lower()
        )
        run_slug = re.sub(
            r"[^a-zA-Z0-9-]+",
            "-",
            run_id,
        )[:24].strip("-")

        hf_repo_id = (
            f"{hf_username}/"
            f"iharq-p02-phase02-r2-"
            f"{run_slug}-{stamp}"
        )

    remote_state = {
        "artifact_id":
            "P02-STAGE24-R2.1-REMOTE-RESUME-STATE",
        "created_at_utc":
            _utc(),
        "status":
            "IN_PROGRESS",
        "hf_repo_id":
            hf_repo_id,
        "repo_type":
            HF_REPO_TYPE,
        "private":
            HF_PRIVATE,
        "uploaded": {},
    }
    _atomic_json(
        REMOTE_STATE_PATH,
        remote_state,
    )

api.create_repo(
    repo_id=hf_repo_id,
    repo_type=HF_REPO_TYPE,
    private=HF_PRIVATE,
    exist_ok=True,
)

try:
    remote_files_current = set(
        api.list_repo_files(
            repo_id=hf_repo_id,
            repo_type=HF_REPO_TYPE,
        )
    )
except Exception:
    remote_files_current = set()

print("\nHF release repo:")
print(json.dumps({
    "repo_id": hf_repo_id,
    "private": HF_PRIVATE,
    "resume_state":
        str(REMOTE_STATE_PATH),
}, indent=2))

# =============================================================================
# 7. INVENTORY + CLASSIFY COMPLETE STORE
# =============================================================================

def _externalize_reason(
    p: Path,
    rel: str,
):
    parts = [
        x.lower()
        for x in Path(rel).parts
    ]
    rel_l = rel.lower()
    suffix = p.suffix.lower()

    # Direct model/checkpoint storage.
    if (
        suffix in CHECKPOINT_SUFFIXES
        or "checkpoints" in parts
        or "checkpoint" in rel_l
    ):
        return "CHECKPOINT_OR_MODEL"

    # Raw model/prediction arrays and structured ML payload.
    if (
        "predictions" in parts
        or "raw_outputs" in parts
        or (
            len(parts) >= 2
            and parts[0] == "records"
            and parts[1].lower()
                == "predictionrecord"
        )
    ):
        return "LARGE_RUNTIME_EVIDENCE_NAMESPACE"

    if suffix in HEAVY_BINARY_SUFFIXES:
        return "HEAVY_BINARY_FORMAT"

    if p.stat().st_size >= EXTERNALIZE_MIN_BYTES:
        return "SIZE_THRESHOLD"

    return None

all_store_rows = []
direct_external_rows = []
bundle_light_rows = []

for p in sorted(
    STORE_ROOT.rglob("*")
):
    if (
        not p.is_file()
        or p.is_symlink()
        or p.name.endswith(".tmp")
    ):
        continue

    rel = _safe_rel(
        p,
        STORE_ROOT,
    )

    row = {
        "source_path": str(p),
        "runtime_path": rel,
        "bytes": p.stat().st_size,
        "sha256": _sha256_file(p),
        "suffix": p.suffix.lower(),
    }

    reason = _externalize_reason(
        p,
        rel,
    )

    if reason:
        row[
            "externalize_reason"
        ] = reason
        row[
            "remote_path"
        ] = (
            "phase_02/"
            "external_artifacts/"
            + rel
        )
        direct_external_rows.append(
            row
        )
    else:
        row[
            "bundle_transport"
        ] = "MULTIPART_ZIP"
        bundle_light_rows.append(
            row
        )

    all_store_rows.append(row)

total_store_bytes = sum(
    x["bytes"]
    for x in all_store_rows
)
direct_bytes = sum(
    x["bytes"]
    for x in direct_external_rows
)
light_bytes = sum(
    x["bytes"]
    for x in bundle_light_rows
)

print("\nRelease inventory:")
print(json.dumps({
    "store_files":
        len(all_store_rows),
    "store_gib":
        round(
            total_store_bytes
            / GiB,
            3,
        ),
    "direct_external_files":
        len(direct_external_rows),
    "direct_external_gib":
        round(
            direct_bytes
            / GiB,
            3,
        ),
    "multipart_light_files":
        len(bundle_light_rows),
    "multipart_light_gib_source":
        round(
            light_bytes
            / GiB,
            3,
        ),
    "checkpoint_files_from_pre_g24":
        pre.get(
            "phase_release_inventory",
            {},
        ).get(
            "checkpoint_files"
        ),
    "stage18S_outputs_from_pre_g24":
        pre.get(
            "stage18S",
            {},
        ).get(
            "complete_output_files"
        ),
}, indent=2))

# =============================================================================
# 8. HF UPLOAD HELPERS — LOW-COMMIT MAX-3GiB PACKAGE TRANSPORT
# =============================================================================

def _persist_remote_state():
    remote_state["updated_at_utc"] = _utc()
    _atomic_json(REMOTE_STATE_PATH, remote_state)

def _already_uploaded(remote_path: str, sha256: str):
    rec = (remote_state.get("uploaded", {}) or {}).get(remote_path)
    return bool(
        rec
        and rec.get("sha256") == sha256
        and remote_path in remote_files_current
    )

def _mark_uploaded(
    remote_path: str,
    sha256: str,
    size_bytes: int,
    category: str,
):
    remote_state.setdefault("uploaded", {})[remote_path] = {
        "sha256": sha256,
        "bytes": size_bytes,
        "category": category,
        "confirmed_at_utc": _utc(),
    }
    remote_files_current.add(remote_path)

def _rate_limit_wait_seconds(exc: Exception):
    msg = str(exc)
    m = re.search(r"Retry after\\s+(\\d+)\\s+seconds", msg, re.I)
    if m:
        return int(m.group(1)) + 10
    # Repository-commit granular limit often reports "about 1 hour".
    if re.search(r"repository commits", msg, re.I) and re.search(r"hour", msg, re.I):
        return 3610
    return None

def _upload_one_file(
    local_path: Path,
    remote_path: str,
    *,
    sha256: str | None = None,
    category: str,
):
    """
    Exactly ONE repository commit per completed package.

    This is intentionally the opposite of R2.1's many create_commit batches.
    """
    local_path = Path(local_path)
    actual_sha = _sha256_file(local_path)

    if sha256 is None:
        sha256 = actual_sha
    elif actual_sha != sha256:
        raise RuntimeError(
            "STAGE24_R2_2_LOCAL_PACKAGE_CHANGED_BEFORE_UPLOAD:"
            + json.dumps({
                "path": str(local_path),
                "expected_sha256": sha256,
                "actual_sha256": actual_sha,
            }, sort_keys=True)
        )

    if local_path.stat().st_size > MAX_PACKAGE_BYTES:
        raise RuntimeError(
            "STAGE24_R2_2_PACKAGE_EXCEEDS_3GIB_HARD_LIMIT:"
            + json.dumps({
                "path": str(local_path),
                "bytes": local_path.stat().st_size,
                "max_bytes": MAX_PACKAGE_BYTES,
            }, sort_keys=True)
        )

    if _already_uploaded(remote_path, sha256):
        print("HF reuse:", remote_path)
        return

    attempts = 0
    while True:
        try:
            api.upload_file(
                path_or_fileobj=str(local_path),
                path_in_repo=remote_path,
                repo_id=hf_repo_id,
                repo_type=HF_REPO_TYPE,
                commit_message=(
                    f"P02 R2.2 {category}: "
                    f"{Path(remote_path).name}"
                ),
            )
            break
        except Exception as exc:
            msg = str(exc)
            is_429 = "429" in msg or "Too Many Requests" in msg
            wait_s = _rate_limit_wait_seconds(exc) if is_429 else None

            if (
                is_429
                and AUTO_WAIT_ON_SHORT_429
                and wait_s is not None
                and wait_s <= MAX_AUTOMATIC_429_WAIT_SECONDS
                and attempts < 2
            ):
                attempts += 1
                print(
                    f"HF 429 encountered; waiting {wait_s}s before retrying "
                    f"the SAME package upload ({remote_path}).",
                    flush=True,
                )
                time.sleep(wait_s)
                continue

            if is_429:
                raise RuntimeError(
                    "STAGE24_R2_2_HF_COMMIT_RATE_LIMIT:"
                    + json.dumps({
                        "remote_path": remote_path,
                        "suggested_wait_seconds": wait_s,
                        "note": (
                            "This R2.2 strategy uses only one commit per <=3GiB "
                            "package. If this token belongs to the same currently "
                            "rate-limited account, wait for reset or use a token "
                            "from a different account/organization context."
                        ),
                    }, sort_keys=True)
                ) from exc
            raise

    _mark_uploaded(
        remote_path,
        sha256,
        local_path.stat().st_size,
        category,
    )
    _persist_remote_state()

def _group_rows_target(rows, target_bytes):
    groups = []
    current = []
    current_bytes = 0

    for row in rows:
        # A single source file larger than the conservative target can still be
        # placed alone if it is below the 3GiB hard package limit.
        if current and current_bytes + row["bytes"] > target_bytes:
            groups.append(current)
            current = []
            current_bytes = 0

        current.append(row)
        current_bytes += row["bytes"]

        if current_bytes >= target_bytes:
            groups.append(current)
            current = []
            current_bytes = 0

    if current:
        groups.append(current)

    return groups

def _write_package_zip(
    rows,
    local_zip: Path,
    *,
    archive_root: str,
):
    """
    ZIP_STORED is intentional: it is faster and makes completed size closely
    track source bytes, allowing a reliable <3GiB hard limit.
    """
    tmp = local_zip.with_suffix(".zip.tmp")
    for p in (tmp, local_zip):
        if p.exists():
            p.unlink()

    with zipfile.ZipFile(
        tmp,
        "w",
        compression=zipfile.ZIP_STORED,
        allowZip64=True,
    ) as z:
        info = {
            "artifact_id": "P02-STAGE24-MAX3G-PACKAGE-R2.2",
            "created_at_utc": _utc(),
            "source_files": len(rows),
            "source_bytes": sum(r["bytes"] for r in rows),
            "hard_max_package_bytes": MAX_PACKAGE_BYTES,
            "package_source_target_bytes": PACKAGE_SOURCE_TARGET_BYTES,
        }
        z.writestr(
            f"{archive_root}/PACKAGE_INFO.json",
            json.dumps(info, indent=2, sort_keys=True) + "\\n",
        )

        for row in rows:
            src = Path(row["source_path"])
            arc = (
                Path(archive_root)
                / "runtime"
                / row["runtime_path"]
            ).as_posix()
            z.write(src, arcname=arc)

    os.replace(tmp, local_zip)

    with zipfile.ZipFile(local_zip, "r") as z:
        bad = z.testzip()
        if bad is not None:
            raise RuntimeError(f"STAGE24_R2_2_PACKAGE_CRC_FAIL:{bad}")

    return local_zip.stat().st_size

def _package_rows_to_hf(
    rows,
    *,
    family_name: str,
    remote_prefix: str,
):
    """
    Create <=3GiB packages one at a time:
        create -> CRC/hash -> upload -> confirm state -> DELETE LOCAL PACKAGE.

    Rows are mutated with package_remote_path/archive_member so the final
    per-artifact pointer manifest remains exact.
    """
    if not rows:
        return []

    groups = _group_rows_target(rows, PACKAGE_SOURCE_TARGET_BYTES)
    package_records = []
    package_no = 0

    print(
        f"{family_name}: {len(rows)} source files, "
        f"{sum(r['bytes'] for r in rows)/GiB:.3f} GiB, "
        f"initial groups={len(groups)}"
    )

    # Work queue allows an unexpected >3GiB completed ZIP to split safely.
    queue = list(groups)

    while queue:
        group = queue.pop(0)

        # If one individual source itself is >= the hard package max, do not
        # duplicate it locally. Upload it directly as one artifact/one commit.
        if len(group) == 1 and group[0]["bytes"] >= MAX_PACKAGE_BYTES:
            row = group[0]
            src = Path(row["source_path"])
            remote = (
                f"{remote_prefix.rstrip('/')}/"
                f"oversized_individual/"
                f"{hashlib.sha256(row['runtime_path'].encode()).hexdigest()[:16]}_"
                f"{Path(row['runtime_path']).name}"
            )
            # Direct source upload is not a local package and therefore does
            # not consume extra disk; it is only used for >3GiB source files.
            api.upload_file(
                path_or_fileobj=str(src),
                path_in_repo=remote,
                repo_id=hf_repo_id,
                repo_type=HF_REPO_TYPE,
                commit_message=f"P02 R2.2 oversized individual: {src.name}",
            )
            row["remote_path"] = remote
            row["archive_member"] = None
            row["transport_kind"] = "DIRECT_OVERSIZED_INDIVIDUAL"
            package_records.append({
                "transport_kind": "DIRECT_OVERSIZED_INDIVIDUAL",
                "remote_path": remote,
                "bytes": row["bytes"],
                "sha256": row["sha256"],
                "source_files": 1,
            })
            continue

        package_no += 1
        local_zip = (
            SCRATCH
            / f"{family_name}_package_{package_no:04d}.zip"
        )

        free_now = shutil.disk_usage(KW).free
        expected_source = sum(r["bytes"] for r in group)

        if free_now < MIN_FREE_RESERVE_BYTES + min(
            MAX_PACKAGE_BYTES,
            expected_source + 64 * MiB,
        ):
            raise RuntimeError(
                "STAGE24_R2_2_INSUFFICIENT_DISK_BEFORE_PACKAGE:"
                + json.dumps({
                    "family": family_name,
                    "package_no": package_no,
                    "free_gib": round(free_now/GiB, 3),
                    "source_gib": round(expected_source/GiB, 3),
                    "reserve_gib": round(MIN_FREE_RESERVE_BYTES/GiB, 3),
                }, sort_keys=True)
            )

        completed_size = _write_package_zip(
            group,
            local_zip,
            archive_root=family_name,
        )

        if completed_size > MAX_PACKAGE_BYTES:
            # Unexpected header/metadata overhead. Delete and split group.
            local_zip.unlink(missing_ok=True)

            if len(group) <= 1:
                raise RuntimeError(
                    "STAGE24_R2_2_SINGLE_FILE_PACKAGE_EXCEEDS_3GIB:"
                    + json.dumps({
                        "runtime_path": group[0]["runtime_path"],
                        "source_bytes": group[0]["bytes"],
                        "completed_package_bytes": completed_size,
                    }, sort_keys=True)
                )

            midpoint = max(1, len(group) // 2)
            queue.insert(0, group[midpoint:])
            queue.insert(0, group[:midpoint])
            package_no -= 1
            print(
                f"{family_name}: package exceeded hard 3GiB limit; "
                "split group and retry.",
                flush=True,
            )
            continue

        package_sha = _sha256_file(local_zip)
        remote = (
            f"{remote_prefix.rstrip('/')}/"
            f"{local_zip.name}"
        )

        _upload_one_file(
            local_zip,
            remote,
            sha256=package_sha,
            category=f"{family_name}_PACKAGE",
        )

        # Exact artifact-to-package member mapping.
        for row in group:
            row["remote_path"] = remote
            row["archive_member"] = (
                Path(family_name)
                / "runtime"
                / row["runtime_path"]
            ).as_posix()
            row["transport_kind"] = "ZIP_PACKAGE"

        rec = {
            "transport_kind": "ZIP_PACKAGE",
            "remote_path": remote,
            "sha256": package_sha,
            "bytes": completed_size,
            "source_files": len(group),
            "source_bytes": expected_source,
            "max_3gib_compliant": completed_size <= MAX_PACKAGE_BYTES,
        }
        package_records.append(rec)

        # Requested low-disk lifecycle.
        local_zip.unlink()

        print(
            f"[{family_name}] package {package_no} | "
            f"{completed_size/GiB:.3f} GiB <= 3.000 GiB | "
            f"{len(group)} files | uploaded + deleted | "
            f"disk_free={_disk()['free_gib']:.3f} GiB",
            flush=True,
        )

    return package_records

# =============================================================================
# 9. PACKAGE HEAVY/CHECKPOINT/RAW ARTIFACTS — <=3GiB EACH
# =============================================================================

print("\\n" + "=" * 120)
print("HF HEAVY ARTIFACT PACKAGES — MAX 3 GiB EACH")
print("=" * 120)

heavy_package_rows = _package_rows_to_hf(
    direct_external_rows,
    family_name="heavy_artifacts",
    remote_prefix="phase_02/heavy_packages",
)

# =============================================================================
# 10. PACKAGE REMAINING EXECUTION EVIDENCE — <=3GiB EACH
# =============================================================================

print("\\n" + "=" * 120)
print("HF EXECUTION-BUNDLE PACKAGES — MAX 3 GiB EACH")
print("=" * 120)

part_rows = _package_rows_to_hf(
    bundle_light_rows,
    family_name="execution_bundle",
    remote_prefix="phase_02/execution_bundle_packages",
)

# =============================================================================
# 11. FREEZE HEAVY CONTENT REVISION + WRITE POINTER MANIFESTS
# =============================================================================

# Verify every package/direct-oversized transport object exists remotely.
expected_content_paths = {
    r["remote_path"]
    for r in direct_external_rows
    if r.get("remote_path")
}
expected_content_paths.update(
    p["remote_path"]
    for p in part_rows
)

remote_now = set(
    api.list_repo_files(
        repo_id=hf_repo_id,
        repo_type=HF_REPO_TYPE,
    )
)

missing_content = sorted(
    expected_content_paths
    - remote_now
)

if missing_content:
    raise RuntimeError(
        "STAGE24_R2_HF_CONTENT_MISSING:"
        + json.dumps(
            missing_content[:100]
        )
    )

content_revision = getattr(
    api.repo_info(
        repo_id=hf_repo_id,
        repo_type=HF_REPO_TYPE,
    ),
    "sha",
    None,
)

if not content_revision:
    raise RuntimeError(
        "STAGE24_R2_HF_CONTENT_REVISION_NOT_RESOLVED"
    )

external_pointer_manifest = {
    "manifest_id":
        "P02-EXTERNAL-ARTIFACT-POINTER-MANIFEST-R2.2",
    "created_at_utc":
        _utc(),
    "status": "PASS",
    "phase_id": "P02",
    "provider": "Hugging Face",
    "repository_or_dataset":
        hf_repo_id,
    "repo_type":
        HF_REPO_TYPE,
    "immutable_content_revision":
        content_revision,
    "items": [
        {
            "artifact_id":
                "P02-HF-"
                + hashlib.sha256(
                    row["runtime_path"]
                    .encode("utf-8")
                ).hexdigest()[
                    :16
                ].upper(),
            "provider":
                "Hugging Face",
            "repository_or_dataset":
                hf_repo_id,
            "immutable_revision":
                content_revision,
            "path_or_filename":
                row["remote_path"],
            "archive_member":
                row.get("archive_member"),
            "transport_kind":
                row.get("transport_kind"),
            "source_runtime_path":
                row["runtime_path"],
            "sha256":
                row["sha256"],
            "size_bytes":
                row["bytes"],
            "format":
                (
                    Path(
                        row[
                            "runtime_path"
                        ]
                    )
                    .suffix
                    .lower()
                    .lstrip(".")
                    or "binary"
                ),
            "access_requirements":
                "PRIVATE_HF_DATASET_REPO_AUTHORIZED_TOKEN_REQUIRED",
            "producer_phase":
                "P02",
            "consumer_phases": [
                "P03", "P04", "P05",
                "P06", "P07", "P08",
                "P09", "P10", "P11",
                "P12", "P13", "P14",
                "P15", "REPRODUCTION",
            ],
            "retrieval_instructions":
                (
                    (
                        f"Fetch package {row['remote_path']} from dataset "
                        f"{hf_repo_id} at exact revision {content_revision}; "
                        f"extract member {row.get('archive_member')}; verify "
                        f"the extracted artifact SHA-256 {row['sha256']}."
                    )
                    if row.get("archive_member")
                    else (
                        f"Fetch {row['remote_path']} from dataset {hf_repo_id} "
                        f"at exact revision {content_revision}; verify "
                        f"SHA-256 {row['sha256']}."
                    )
                ),
            "local_copy_status":
                "CANONICAL_KAGGLE_RUNTIME_SOURCE",
            "externalize_reason":
                row.get(
                    "externalize_reason",
                    "PACKAGED_RUNTIME_ARTIFACT",
                ),
        }
        for row in direct_external_rows
    ],
}

execution_bundle_manifest = {
    "manifest_id":
        "P02-STAGE24-MAX3G-PACKAGED-EXECUTION-BUNDLE-R2.2",
    "created_at_utc":
        _utc(),
    "status": "PASS",
    "phase_id": "P02",
    "run_id": run_id,
    "G24": "PASS",
    "scientific_evidence":
        False
        if SESSION.ctx.fixture
        else True,
    "scientific_configuration_changed":
        False,
    "transport_reason":
        "ORIGINAL_MONOLITHIC_ZIP_FAILED_ENOSPC",
    "transport_successor":
        "CREATE_UPLOAD_DELETE_MAX_3GIB_ZIP_PACKAGES",
    "repository_or_dataset":
        hf_repo_id,
    "immutable_content_revision":
        content_revision,
    "store_identity": {
        "files":
            len(all_store_rows),
        "bytes":
            total_store_bytes,
        "checksums_sha256":
            _sha256_file(
                STORE_ROOT
                / "checksums.sha256"
            ),
        "package_verify":
            verify_result,
        "security_scan":
            "PASS",
    },
    "heavy_artifact_packages": {
        "source_files":
            len(direct_external_rows),
        "source_bytes":
            direct_bytes,
        "packages":
            heavy_package_rows,
        "hard_completed_package_limit_bytes":
            MAX_PACKAGE_BYTES,
        "pointer_manifest":
            "release_manifests/"
            "external_artifact_pointer_manifest_R2.json",
    },
    "execution_bundle_packages": {
        "packages":
            part_rows,
        "source_files":
            len(bundle_light_rows),
        "source_bytes":
            light_bytes,
        "hard_completed_package_limit_bytes":
            MAX_PACKAGE_BYTES,
        "reconstruction":
            (
                "Download each package at the exact content revision and "
                "extract it. Per-artifact package/member/hash mappings are "
                "recorded in the release manifests."
            ),
    },
    "stage11": {
        "role":
            "CANONICAL_ACCEPTED_MAIN",
        "accepted_source_sha256":
            EXPECTED_STAGE11_SOURCE_SHA256,
    },
    "stage12": {
        "role":
            "CANONICAL_ACCEPTED_MAIN",
        "accepted_source_sha256":
            EXPECTED_STAGE12_SOURCE_SHA256,
    },
    "stage18": {
        "canonical_G18_unchanged":
            True,
        "stage18S_preserved":
            True,
        "stage18S_claim_scope":
            EXPECTED_S18S[
                "claim_scope"
            ],
    },
}

MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

external_pointer_json = (
    MANIFEST_DIR
    / "external_artifact_pointer_manifest_R2.json"
)
external_pointer_yaml = (
    MANIFEST_DIR
    / "external_artifact_pointer_manifest_R2.yaml"
)
bundle_manifest_json = (
    MANIFEST_DIR
    / "execution_bundle_multipart_manifest_R2.json"
)

_atomic_json(
    external_pointer_json,
    external_pointer_manifest,
)
_write_yaml(
    external_pointer_yaml,
    external_pointer_manifest,
)
_atomic_json(
    bundle_manifest_json,
    execution_bundle_manifest,
)

# Upload pointer/bundle manifests. They intentionally point to the immutable
# CONTENT revision created before these manifest commits.
for local, remote in [
    (
        external_pointer_json,
        "phase_02/"
        "release_manifests/"
        "external_artifact_pointer_manifest_R2.json",
    ),
    (
        external_pointer_yaml,
        "phase_02/"
        "release_manifests/"
        "external_artifact_pointer_manifest_R2.yaml",
    ),
    (
        bundle_manifest_json,
        "phase_02/"
        "release_manifests/"
        "execution_bundle_multipart_manifest_R2.json",
    ),
]:
    _upload_one_file(
        local,
        remote,
        category="RELEASE_MANIFEST",
    )

manifest_revision = getattr(
    api.repo_info(
        repo_id=hf_repo_id,
        repo_type=HF_REPO_TYPE,
    ),
    "sha",
    None,
)

if not manifest_revision:
    raise RuntimeError(
        "STAGE24_R2_MANIFEST_REVISION_NOT_RESOLVED"
    )

# =============================================================================
# 12. LOCATE P01 CUMULATIVE SOURCE
# =============================================================================

def _locate_p01_source():
    if P01_CUMULATIVE_SOURCE.strip():
        p = Path(
            P01_CUMULATIVE_SOURCE.strip()
        ).expanduser().resolve()
        if p.exists():
            return p
        raise RuntimeError(
            f"EXPLICIT_P01_SOURCE_NOT_FOUND:{p}"
        )

    info = (
        pre.get(
            "p01_cumulative_source"
        )
        or {}
    )

    # Prefer the ZIP: it is compact and was explicitly found by pre-G24.
    for raw in (
        info.get("zip_candidates")
        or []
    ):
        p = Path(raw)
        if p.is_file():
            return p.resolve()

    for raw in (
        info.get(
            "directory_candidates"
        )
        or []
    ):
        p = Path(raw)
        if p.is_dir():
            return p.resolve()

    # Recursive fallback.
    for pat in [
        "*Cumulative*P01*.zip",
        "*Through_P01*.zip",
    ]:
        hits = sorted(
            p
            for p in KW.rglob(pat)
            if p.is_file()
        )
        if hits:
            return hits[0].resolve()

    raise RuntimeError(
        "STAGE24_R2_P01_CUMULATIVE_SOURCE_NOT_FOUND"
    )

p01_source = _locate_p01_source()

print(
    "\nP01 cumulative source =",
    p01_source,
)

# =============================================================================
# 13. BUILD ONE LIGHTWEIGHT CUMULATIVE/GITHUB-READY PROJECT TREE
# =============================================================================

if PROJECT_STAGING.exists():
    shutil.rmtree(
        PROJECT_STAGING
    )
PROJECT_STAGING.mkdir(
    parents=True,
    exist_ok=True,
)

if p01_source.is_file():
    _safe_extract_zip(
        p01_source,
        PROJECT_STAGING,
    )
else:
    # Copy previous GitHub-ready/cumulative source without caches.
    for p in sorted(
        p01_source.rglob("*")
    ):
        if (
            not p.is_file()
            or p.is_symlink()
        ):
            continue
        rel = p.relative_to(
            p01_source
        )
        if any(
            x in {
                ".git",
                "__pycache__",
                ".ipynb_checkpoints",
            }
            for x in rel.parts
        ):
            continue
        _copy_file(
            p,
            PROJECT_STAGING / rel,
        )

# If the ZIP extracted into a single wrapper folder, use that folder as the
# actual project root to preserve the P01 structure exactly.
children = list(
    PROJECT_STAGING.iterdir()
)

project_root = PROJECT_STAGING

if (
    len(children) == 1
    and children[0].is_dir()
    and (
        (
            children[0]
            / "CURRENT_PROJECT_STATUS.json"
        ).is_file()
        or (
            children[0]
            / "CURRENT_CUMULATIVE_REPOSITORY_MANIFEST.json"
        ).is_file()
    )
):
    project_root = children[0]

phase2_current = (
    project_root
    / "current"
    / "phase_02"
)
phase2_current.mkdir(
    parents=True,
    exist_ok=True,
)

# -----------------------------------------------------------------------------
# 13A. P02 implementation source — code/configs/schemas/tests/docs/contracts.
# -----------------------------------------------------------------------------

impl_out = (
    phase2_current
    / "implementation"
)

for name in [
    "src",
    "configs",
    "schemas",
    "tests",
    "scripts",
    "validation",
    "contracts",
    "docs",
]:
    src = (
        PACKAGE_ROOT_LOCAL
        / name
    )
    if not src.is_dir():
        continue

    for p in sorted(
        src.rglob("*")
    ):
        if (
            not p.is_file()
            or p.is_symlink()
        ):
            continue

        rel = p.relative_to(
            src
        )

        if any(
            x in {
                "__pycache__",
                ".ipynb_checkpoints",
            }
            for x in rel.parts
        ):
            continue

        _copy_file(
            p,
            impl_out
            / name
            / rel,
        )

for name in [
    "README.md",
    "pyproject.toml",
    "requirements.txt",
    "requirements-lock.txt",
    "requirements-lock.sha256",
]:
    p = (
        PACKAGE_ROOT_LOCAL
        / name
    )
    if p.is_file():
        _copy_file(
            p,
            impl_out / name,
        )

# -----------------------------------------------------------------------------
# 13B. P02 lightweight evidence surfaces.
# -----------------------------------------------------------------------------

runtime_out = (
    phase2_current
    / "runtime_evidence"
)

# Keep summary/analysis/handoff/manifest surfaces directly in the cumulative
# and GitHub-ready package. Heavy raw evidence is already on HF.
light_namespaces = [
    "analysis_inputs",
    "figure_source_data",
    "table_source_data",
    "handoffs",
    "manifests",
    "gate_results",
    "logs",
    "protocol_change_required",
    "diagnostics",
]

for namespace in light_namespaces:
    src_root = (
        STORE_ROOT
        / namespace
    )
    if not src_root.exists():
        continue

    for p in sorted(
        src_root.rglob("*")
    ):
        if (
            not p.is_file()
            or p.is_symlink()
            or p.name.endswith(".tmp")
        ):
            continue

        # Do not copy unexpectedly large files into GitHub-ready control plane.
        reason = _externalize_reason(
            p,
            _safe_rel(
                p,
                STORE_ROOT,
            ),
        )

        if reason:
            continue

        _copy_file(
            p,
            runtime_out
            / namespace
            / p.relative_to(
                src_root
            ),
        )

# Small readiness/failure records explicitly useful downstream.
for record_family in [
    "Layer2ReadinessReport",
    "FailureCaseIndex",
]:
    src_root = (
        STORE_ROOT
        / "records"
        / record_family
    )
    if not src_root.exists():
        continue

    for p in sorted(
        src_root.rglob("*")
    ):
        if (
            p.is_file()
            and not p.is_symlink()
            and p.stat().st_size
                < EXTERNALIZE_MIN_BYTES
        ):
            _copy_file(
                p,
                runtime_out
                / "records"
                / record_family
                / p.relative_to(
                    src_root
                ),
            )

# Orchestration ledgers/gates/stage artifacts.
for src_root, dst_name in [
    (
        RUN_ROOT / "stage_ledger",
        "stage_ledger",
    ),
    (
        RUN_ROOT / "gate_results",
        "gate_results",
    ),
    (
        RUN_ROOT / "stage_artifacts",
        "stage_artifacts",
    ),
]:
    if not src_root.exists():
        continue
    for p in sorted(
        src_root.rglob("*")
    ):
        if p.is_file():
            _copy_file(
                p,
                runtime_out
                / dst_name
                / p.relative_to(
                    src_root
                ),
            )

# Release manifests/pointers.
release_meta = (
    phase2_current
    / "release_metadata"
)
release_meta.mkdir(
    parents=True,
    exist_ok=True,
)

for p in [
    PRE_G24_RECEIPT,
    release_family_path,
    external_pointer_json,
    external_pointer_yaml,
    bundle_manifest_json,
    g24_path,
    RUN_ROOT
    / "stage_ledger"
    / "stage_24.json",
]:
    _copy_file(
        p,
        release_meta / p.name,
    )

# -----------------------------------------------------------------------------
# 13C. Root-level V6.1 external pointer + current status/handoff.
# -----------------------------------------------------------------------------

# Preserve any prior cumulative external-pointer authority before installing
# the P02 composite pointer surface.
prior_pointer_yaml = (
    project_root
    / "external_artifact_pointer_manifest.yaml"
)
prior_pointer_json = (
    project_root
    / "external_artifact_pointer_manifest.json"
)

prior_pointer_history = []

if prior_pointer_yaml.is_file():
    hist = (
        project_root
        / "history"
        / "external_pointers"
        / "through_P01_external_artifact_pointer_manifest.yaml"
    )
    _copy_file(
        prior_pointer_yaml,
        hist,
    )
    prior_pointer_history.append(
        hist.relative_to(project_root).as_posix()
    )

if prior_pointer_json.is_file():
    hist = (
        project_root
        / "history"
        / "external_pointers"
        / "through_P01_external_artifact_pointer_manifest.json"
    )
    _copy_file(
        prior_pointer_json,
        hist,
    )
    prior_pointer_history.append(
        hist.relative_to(project_root).as_posix()
    )

composite_external_pointer = {
    "manifest_id":
        "IHARQ-EXTERNAL-ARTIFACT-POINTERS-THROUGH-P02-R2",
    "created_at_utc":
        _utc(),
    "status":
        "PASS",
    "prior_pointer_manifests":
        prior_pointer_history,
    "phase_02":
        external_pointer_manifest,
}

_write_yaml(
    project_root
    / "external_artifact_pointer_manifest.yaml",
    composite_external_pointer,
)
_atomic_json(
    project_root
    / "external_artifact_pointer_manifest.json",
    composite_external_pointer,
)

documentary_note = {
    "P02_execution_G24": "PASS",
    "P02_scientific_execution_complete": True,
    "P02_stage11": {
        "status":
            "CANONICAL_ACCEPTED_MAIN",
        "source_sha256":
            EXPECTED_STAGE11_SOURCE_SHA256,
    },
    "P02_stage12": {
        "status":
            "CANONICAL_ACCEPTED_MAIN",
        "source_sha256":
            EXPECTED_STAGE12_SOURCE_SHA256,
    },
    "P02_stage18": {
        "canonical_G18":
            "PASS_UNCHANGED",
        "stage18S":
            "PASS_POST_HOC_SENSITIVITY_DESCRIPTIVE",
    },
    "P02_heavy_artifact_store": {
        "provider":
            "Hugging Face",
        "repository":
            hf_repo_id,
        "immutable_content_revision":
            content_revision,
        "manifest_revision":
            manifest_revision,
    },
    "final_GitHub_publication":
        "DEFERRED_UNTIL_WHOLE_PROJECT_COMPLETION_PER_V6_1",
}

_atomic_json(
    phase2_current
    / "P02_EXECUTION_STATUS.json",
    documentary_note,
)

# Preserve previous current status but augment P02 truthfully.
status_path = (
    project_root
    / "CURRENT_PROJECT_STATUS.json"
)

if status_path.is_file():
    try:
        current_status = _load_json(
            status_path
        )
    except Exception:
        current_status = {}
else:
    current_status = {}

current_status.update({
    "project_state_id":
        "IHARQ-CUMULATIVE-THROUGH-P02-R2",
    "p02_execution_complete":
        True,
    "p02_G24":
        "PASS",
    "p02_stage11":
        "CANONICAL_ACCEPTED_MAIN",
    "p02_stage12":
        "CANONICAL_ACCEPTED_MAIN",
    "p02_stage18":
        "CANONICAL_G18_PASS",
    "p02_stage18S":
        "PASS_POST_HOC_SENSITIVITY_DESCRIPTIVE",
    "p02_heavy_artifact_pointer": {
        "repo_id":
            hf_repo_id,
        "immutable_content_revision":
            content_revision,
        "pointer_manifest":
            "external_artifact_pointer_manifest.yaml",
    },
    "final_github_migration":
        "DEFERRED_UNTIL_WHOLE_PROJECT_COMPLETION",
})

_atomic_json(
    status_path,
    current_status,
)

phase_handoff = {
    "phase_handoff": {
        "phase_id": "P02",
        "execution_status":
            "G24_PASS",
        "scientific_evidence":
            True,
        "stage11": {
            "canonical":
                True,
            "source_sha256":
                EXPECTED_STAGE11_SOURCE_SHA256,
        },
        "stage12": {
            "canonical":
                True,
            "source_sha256":
                EXPECTED_STAGE12_SOURCE_SHA256,
        },
        "stage18S": {
            "preserved":
                True,
            "claim_scope":
                EXPECTED_S18S[
                    "claim_scope"
                ],
            "canonical_G18_unchanged":
                True,
        },
        "external_artifacts": {
            "provider":
                "Hugging Face",
            "repository":
                hf_repo_id,
            "immutable_content_revision":
                content_revision,
            "pointer_manifest":
                "external_artifact_pointer_manifest.yaml",
        },
        "release_transport":
            "LOW_DISK_EXTERNALIZED_MULTIPART_R2",
    }
}

_write_yaml(
    project_root
    / "phase_handoff.yaml",
    phase_handoff,
)

# Artifact index: local light files + every external item.
artifact_index = (
    project_root
    / "current_artifact_index.csv"
)

artifact_fields = [
    "scope",
    "path",
    "bytes",
    "sha256",
    "storage",
    "immutable_revision",
]

prior_artifact_rows = []
if artifact_index.is_file():
    try:
        with artifact_index.open(
            "r",
            newline="",
            encoding="utf-8",
        ) as f:
            reader = csv.DictReader(f)
            for row in reader:
                prior_artifact_rows.append({
                    k: row.get(k, "")
                    for k in artifact_fields
                })
    except Exception:
        prior_artifact_rows = []

with artifact_index.open(
    "w",
    newline="",
    encoding="utf-8",
) as f:
    writer = csv.DictWriter(
        f,
        fieldnames=artifact_fields,
    )
    writer.writeheader()

    # Preserve P00/P01 current artifact index rows first.
    for row in prior_artifact_rows:
        writer.writerow(row)

    for p in sorted(
        phase2_current.rglob("*")
    ):
        if not p.is_file():
            continue

        writer.writerow({
            "scope":
                "P02_LIGHT_LOCAL",
            "path":
                p.relative_to(
                    project_root
                ).as_posix(),
            "bytes":
                p.stat().st_size,
            "sha256":
                _sha256_file(p),
            "storage":
                "CUMULATIVE_GITHUB_READY_PACKAGE",
            "immutable_revision":
                "",
        })

    for row in direct_external_rows:
        writer.writerow({
            "scope":
                "P02_EXTERNAL_HF",
            "path":
                (
                    row["remote_path"]
                    + (
                        "::"
                        + row["archive_member"]
                        if row.get("archive_member")
                        else ""
                    )
                ),
            "bytes":
                row["bytes"],
            "sha256":
                row["sha256"],
            "storage":
                f"HF:{hf_repo_id}",
            "immutable_revision":
                content_revision,
        })

# Preserve the P00/P01 document index and append the new P02 current section.
doc_index_path = (
    project_root
    / "current_document_index.md"
)
prior_doc_index = ""
if doc_index_path.is_file():
    try:
        prior_doc_index = doc_index_path.read_text(
            encoding="utf-8"
        ).rstrip()
    except Exception:
        prior_doc_index = ""

p02_doc_section = (
    "## Phase 02 current execution/release status\n\n"
    "- P02 G24: **PASS**\n"
    "- Corrected Stage11: **canonical accepted main**\n"
    "- Corrected Stage12: **canonical accepted main**\n"
    "- Stage18/G18: **canonical PASS, unchanged**\n"
    "- Stage18S: **preserved post-hoc descriptive sensitivity**\n"
    f"- HF heavy repo: `{hf_repo_id}`\n"
    f"- HF immutable content revision: `{content_revision}`\n"
    "- Final GitHub publication: **deferred by V6.1**\n"
)

doc_index = (
    (prior_doc_index + "\n\n" if prior_doc_index else
     "# IHARQ current document index — through P02 execution\n\n")
    + p02_doc_section
)

_atomic_text(
    doc_index_path,
    doc_index,
)

# Project-state manifest.
project_state_manifest = {
    "manifest_id":
        "IHARQ-PROJECT-STATE-AFTER-P02-R2",
    "created_at_utc":
        _utc(),
    "phase_02": {
        "G24": "PASS",
        "stage11":
            "CANONICAL_ACCEPTED_MAIN",
        "stage12":
            "CANONICAL_ACCEPTED_MAIN",
        "stage18":
            "CANONICAL_G18_UNCHANGED",
        "stage18S":
            "POST_HOC_SENSITIVITY_DESCRIPTIVE_PASS",
        "external_artifact_repository":
            hf_repo_id,
        "immutable_content_revision":
            content_revision,
    },
    "primary_intermediate_transport":
        "CUMULATIVE_PROJECT_STATE_PACKAGE",
    "github_ready_derivative":
        "SEPARATE_LIGHTWEIGHT_PACKAGE_WITH_HF_POINTERS",
    "external_artifact_pointer_manifest":
        "external_artifact_pointer_manifest.yaml",
}

_write_yaml(
    project_root
    / "project_state_manifest.yaml",
    project_state_manifest,
)

# Secret scan BEFORE creating any project-state package.
project_secret_hits = (
    _scan_tree_secrets(
        project_root
    )
)

if project_secret_hits:
    raise RuntimeError(
        "STAGE24_R2_PROJECT_TREE_SECRET_SCAN_FAILED:"
        + json.dumps(
            project_secret_hits[:50],
            sort_keys=True,
        )
    )

# Repository manifest/checksums.
project_rows = _tree_rows(
    project_root
)

_atomic_json(
    project_root
    / "CURRENT_CUMULATIVE_REPOSITORY_MANIFEST.json",
    {
        "manifest_type":
            "RepositoryManifest",
        "manifest_id":
            "IHARQ-CUMULATIVE-THROUGH-P02-R2",
        "status":
            "P02_EXECUTION_G24_PASS",
        "files_count":
            len(project_rows),
        "files":
            project_rows,
    },
)

def _write_tree_checksums(
    root: Path,
    out_name: str,
):
    root = Path(root)
    out_path = (
        root / out_name
    )
    rows = []

    for p in sorted(
        root.rglob("*")
    ):
        if (
            not p.is_file()
            or p == out_path
        ):
            continue

        rows.append(
            f"{_sha256_file(p)}  "
            f"{p.relative_to(root).as_posix()}"
        )

    _atomic_text(
        out_path,
        "\n".join(rows)
        + "\n",
    )

_write_tree_checksums(
    project_root,
    "project_state_checksums.sha256",
)
_write_tree_checksums(
    project_root,
    "REPOSITORY_CHECKSUMS.sha256",
)

# =============================================================================
# 14. PACKAGE HELPER — SINGLE ZIP IF IT FITS, ELSE MULTIPART
#     EVERY LOCAL ZIP/PART IS UPLOADED THEN DELETED.
# =============================================================================

def _zip_tree_single(
    root: Path,
    out_path: Path,
):
    tmp = out_path.with_suffix(
        out_path.suffix
        + ".tmp"
    )

    for p in [
        tmp,
        out_path,
    ]:
        if p.exists():
            p.unlink()

    with zipfile.ZipFile(
        tmp,
        "w",
        compression=
            zipfile.ZIP_DEFLATED,
        allowZip64=True,
    ) as z:
        for p in sorted(
            root.rglob("*")
        ):
            if not p.is_file():
                continue

            arc = p.relative_to(
                root
            ).as_posix()

            if (
                arc.startswith("/")
                or ".." in Path(
                    arc
                ).parts
            ):
                raise RuntimeError(
                    f"UNSAFE_PACKAGE_PATH:{arc}"
                )

            z.write(
                p,
                arcname=arc,
            )

    os.replace(
        tmp,
        out_path,
    )

    with zipfile.ZipFile(
        out_path,
        "r",
    ) as z:
        bad = z.testzip()
        if bad is not None:
            raise RuntimeError(
                f"PACKAGE_ZIP_CRC_FAIL:{bad}"
            )

def _package_tree_to_hf(
    root: Path,
    *,
    package_basename: str,
    remote_prefix: str,
    category: str,
):
    """
    Prefer one ZIP. If local disk still cannot hold it, automatically switch
    to bounded independent ZIP parts. Every created ZIP is uploaded and deleted.
    """
    root = Path(root)
    rows = [
        {
            "source_path": str(p),
            "relative_path":
                p.relative_to(
                    root
                ).as_posix(),
            "bytes":
                p.stat().st_size,
            "sha256":
                _sha256_file(p),
        }
        for p in sorted(
            root.rglob("*")
        )
        if p.is_file()
    ]

    source_bytes = sum(
        r["bytes"]
        for r in rows
    )

    local_zip = (
        SCRATCH
        / f"{package_basename}.zip"
    )

    # Single ZIP is attempted only when source bytes are below the
    # conservative 2.70GiB target. This guarantees the completed archive stays
    # below the user's hard 3GiB package limit.
    try_single = (
        source_bytes <= PACKAGE_SOURCE_TARGET_BYTES
        and shutil.disk_usage(KW).free
            > MIN_FREE_RESERVE_BYTES
              + source_bytes
              + 64 * MiB
    )

    if try_single:
        try:
            _zip_tree_single(
                root,
                local_zip,
            )

            zip_bytes = local_zip.stat().st_size
            if zip_bytes > MAX_PACKAGE_BYTES:
                local_zip.unlink(missing_ok=True)
                raise OSError(
                    28,
                    "Completed release ZIP exceeded the 3GiB hard limit; "
                    "switching to multipart packaging.",
                )

            zip_sha = _sha256_file(local_zip)
            remote_path = (
                remote_prefix.rstrip("/")
                + "/"
                + local_zip.name
            )

            _upload_one_file(
                local_zip,
                remote_path,
                sha256=zip_sha,
                category=category,
            )

            if not KEEP_LOCAL_RELEASE_ZIPS:
                local_zip.unlink()

            return {
                "transport":
                    "SINGLE_ZIP",
                "remote_path":
                    remote_path,
                "sha256":
                    zip_sha,
                "bytes":
                    zip_bytes,
                "source_files":
                    len(rows),
                "source_bytes":
                    source_bytes,
                "local_deleted_after_upload":
                    not KEEP_LOCAL_RELEASE_ZIPS,
            }

        except OSError as exc:
            try:
                if local_zip.exists():
                    local_zip.unlink()
                tmp = (
                    local_zip
                    .with_suffix(
                        local_zip.suffix
                        + ".tmp"
                    )
                )
                if tmp.exists():
                    tmp.unlink()
            except Exception:
                pass

            if getattr(
                exc,
                "errno",
                None,
            ) != 28:
                raise

            print(
                f"{category}: single ZIP hit ENOSPC; "
                "switching to bounded multipart package."
            )

    # Multipart fallback.
    groups = []
    cur = []
    cur_b = 0

    for row in rows:
        if (
            cur
            and cur_b
                + row["bytes"]
                > CHUNK_TARGET_BYTES
        ):
            groups.append(cur)
            cur = []
            cur_b = 0

        cur.append(row)
        cur_b += row[
            "bytes"
        ]

    if cur:
        groups.append(cur)

    parts = []

    for i, group in enumerate(
        groups,
        start=1,
    ):
        part = (
            SCRATCH
            / (
                f"{package_basename}"
                f"_part_{i:04d}_of_"
                f"{len(groups):04d}.zip"
            )
        )
        tmp = part.with_suffix(
            ".zip.tmp"
        )

        for q in [
            part,
            tmp,
        ]:
            if q.exists():
                q.unlink()

        with zipfile.ZipFile(
            tmp,
            "w",
            compression=
                zipfile.ZIP_DEFLATED,
            allowZip64=True,
        ) as z:
            for row in group:
                z.write(
                    row[
                        "source_path"
                    ],
                    arcname=
                        row[
                            "relative_path"
                        ],
                )

        os.replace(
            tmp,
            part,
        )

        with zipfile.ZipFile(
            part,
            "r",
        ) as z:
            bad = z.testzip()
            if bad is not None:
                raise RuntimeError(
                    f"{category}_PART_CRC_FAIL:{bad}"
                )

        size = part.stat().st_size
        if size > MAX_PACKAGE_BYTES:
            part.unlink(missing_ok=True)
            raise RuntimeError(
                "STAGE24_R2_2_RELEASE_PACKAGE_EXCEEDS_3GIB:"
                + json.dumps({
                    "package": str(part),
                    "bytes": size,
                    "max_bytes": MAX_PACKAGE_BYTES,
                    "action": "RERUN_WITH_LOWER_PACKAGE_SOURCE_TARGET_GIB",
                }, sort_keys=True)
            )

        sha = _sha256_file(part)
        remote = (
            remote_prefix.rstrip("/")
            + "/"
            + part.name
        )

        _upload_one_file(
            part,
            remote,
            sha256=sha,
            category=category
                + "_PART",
        )

        part.unlink()

        parts.append({
            "part_index": i,
            "part_count":
                len(groups),
            "remote_path":
                remote,
            "sha256":
                sha,
            "bytes":
                size,
            "source_files":
                len(group),
        })

        print(
            f"[{category}] part "
            f"{i}/{len(groups)} "
            "uploaded+deleted | "
            f"disk_free="
            f"{_disk()['free_gib']:.3f} GiB",
            flush=True,
        )

    return {
        "transport":
            "MULTIPART_ZIP",
        "parts":
            parts,
        "source_files":
            len(rows),
        "source_bytes":
            source_bytes,
        "local_deleted_after_upload":
            True,
    }

# =============================================================================
# 15. UPLOAD PRIMARY CUMULATIVE PACKAGE
# =============================================================================

print("\n" + "=" * 120)
print(
    "PRIMARY CUMULATIVE PROJECT STATE "
    "— PACKAGE -> HF -> DELETE LOCAL"
)
print("=" * 120)

cumulative_package = _package_tree_to_hf(
    project_root,
    package_basename=
        "IHARQ_Project_State_After_Phase_02_R2",
    remote_prefix=
        "phase_02/release_packages/cumulative_project_state",
    category=
        "CUMULATIVE_PROJECT_STATE",
)

# =============================================================================
# 16. GITHUB-READY DERIVATIVE + UPLOAD TO HF TOO
# =============================================================================

# Add GitHub-specific boundary and exact HF pointer. The tree already excludes
# P02 checkpoints/predictions/heavy payloads by construction.
github_boundary = f"""# IHARQ P02 intermediate GitHub-ready derivative

This package is a clean intermediate GitHub-ready derivative. It is not pushed
to a live GitHub repository by this notebook because project Governance V6.1
defers final GitHub migration until whole-project completion.

P02 large runtime evidence is stored externally on Hugging Face.

HF dataset:
{hf_repo_id}

Immutable heavy-content revision:
{content_revision}

External pointer manifest:
external_artifact_pointer_manifest.yaml

Canonical upstream:
- Stage11: corrected accepted main ({EXPECTED_STAGE11_SOURCE_SHA256})
- Stage12: corrected accepted main ({EXPECTED_STAGE12_SOURCE_SHA256})

Stage18:
- canonical G18 remains unchanged
- Stage18S is preserved as post-hoc descriptive sensitivity evidence
"""

_atomic_text(
    project_root
    / "GITHUB_RELEASE_BOUNDARY.md",
    github_boundary,
)

# Root convenience pointer.
_atomic_json(
    project_root
    / "HF_ARTIFACT_POINTER.json",
    {
        "provider":
            "Hugging Face",
        "repo_id":
            hf_repo_id,
        "repo_type":
            HF_REPO_TYPE,
        "immutable_content_revision":
            content_revision,
        "pointer_manifest":
            "external_artifact_pointer_manifest.yaml",
        "execution_bundle_manifest_remote":
            "phase_02/release_manifests/"
            "execution_bundle_multipart_manifest_R2.json",
    },
)

# Re-check secrets after GitHub boundary files.
github_secret_hits = (
    _scan_tree_secrets(
        project_root
    )
)

if github_secret_hits:
    raise RuntimeError(
        "STAGE24_R2_GITHUB_READY_SECRET_SCAN_FAILED:"
        + json.dumps(
            github_secret_hits[:50],
            sort_keys=True,
        )
    )

_write_tree_checksums(
    project_root,
    "REPOSITORY_CHECKSUMS.sha256",
)

print("\n" + "=" * 120)
print(
    "GITHUB-READY DERIVATIVE "
    "— PACKAGE -> HF -> DELETE LOCAL"
)
print("=" * 120)

github_package = _package_tree_to_hf(
    project_root,
    package_basename=
        "IHARQ_Cumulative_GitHub_Ready_Through_P02_R2",
    remote_prefix=
        "phase_02/release_packages/github_ready",
    category=
        "GITHUB_READY_DERIVATIVE",
)

# =============================================================================
# 17. CLEAN PROJECT STAGING AFTER BOTH PACKAGES ARE SAFELY ON HF
# =============================================================================

project_staging_bytes = 0
if PROJECT_STAGING.exists():
    for p in PROJECT_STAGING.rglob(
        "*"
    ):
        if p.is_file():
            try:
                project_staging_bytes += (
                    p.stat().st_size
                )
            except Exception:
                pass

    shutil.rmtree(
        PROJECT_STAGING
    )

print(
    "\nproject staging removed after upload:",
    round(
        project_staging_bytes
        / GiB,
        3,
    ),
    "GiB logical bytes",
)

# =============================================================================
# 18. FINAL HF REMOTE VERIFICATION
# =============================================================================

release_revision = getattr(
    api.repo_info(
        repo_id=hf_repo_id,
        repo_type=HF_REPO_TYPE,
    ),
    "sha",
    None,
)

if not release_revision:
    raise RuntimeError(
        "STAGE24_R2_FINAL_HF_REVISION_NOT_RESOLVED"
    )

remote_final = set(
    api.list_repo_files(
        repo_id=hf_repo_id,
        repo_type=HF_REPO_TYPE,
        revision=release_revision,
    )
)

# Required release packages can be single ZIP or multipart.
required_remote = set(
    expected_content_paths
)

required_remote.update([
    "phase_02/release_manifests/"
    "external_artifact_pointer_manifest_R2.json",
    "phase_02/release_manifests/"
    "external_artifact_pointer_manifest_R2.yaml",
    "phase_02/release_manifests/"
    "execution_bundle_multipart_manifest_R2.json",
])

def _collect_package_paths(obj):
    if obj.get("transport") == "SINGLE_ZIP":
        return {
            obj["remote_path"]
        }
    return {
        x["remote_path"]
        for x in obj.get(
            "parts",
            [],
        )
    }

required_remote.update(
    _collect_package_paths(
        cumulative_package
    )
)
required_remote.update(
    _collect_package_paths(
        github_package
    )
)

missing_final = sorted(
    required_remote
    - remote_final
)

if missing_final:
    raise RuntimeError(
        "STAGE24_R2_FINAL_HF_COMPLETENESS_FAIL:"
        + json.dumps(
            missing_final[:100]
        )
    )

# =============================================================================
# 19. FINAL SMALL LOCAL RECEIPT + SESSION FINALIZATION SUCCESSOR
# =============================================================================

release_receipt = {
    "artifact_id":
        "P02-STAGE24-MAX3G-HF-RELEASE-R2.2",
    "created_at_utc":
        _utc(),
    "status":
        "PASS",
    "reason_for_successor":
        "ORIGINAL_STAGE24_MONOLITHIC_ZIP_FAILED_ENOSPC",
    "scientific_configuration_changed":
        False,
    "governed_stage24": {
        "accepted":
            bool(_accepted("24")),
        "G24":
            g24.get("status"),
        "attempt_id":
            stage24_accepted.get(
                "attempt_id"
            ),
    },
    "canonical_upstream": {
        "stage11": {
            "status":
                "CANONICAL_ACCEPTED_MAIN",
            "source_sha256":
                EXPECTED_STAGE11_SOURCE_SHA256,
        },
        "stage12": {
            "status":
                "CANONICAL_ACCEPTED_MAIN",
            "source_sha256":
                EXPECTED_STAGE12_SOURCE_SHA256,
        },
        "stage18": {
            "G18":
                "PASS_UNCHANGED",
            "stage18S":
                "PASS_POST_HOC_SENSITIVITY_DESCRIPTIVE",
        },
    },
    "hugging_face": {
        "repo_id":
            hf_repo_id,
        "repo_type":
            HF_REPO_TYPE,
        "private":
            HF_PRIVATE,
        "content_revision":
            content_revision,
        "manifest_revision":
            manifest_revision,
        "final_release_revision":
            release_revision,
        "heavy_source_files":
            len(direct_external_rows),
        "heavy_source_bytes":
            direct_bytes,
        "heavy_packages":
            len(heavy_package_rows),
        "execution_bundle_packages":
            len(part_rows),
        "hard_max_package_bytes":
            MAX_PACKAGE_BYTES,
        "remote_completeness":
            "PASS",
    },
    "execution_bundle": {
        "transport":
            "EXTERNALIZED_MULTIPART",
        "parts":
            part_rows,
        "manifest_remote":
            "phase_02/release_manifests/"
            "execution_bundle_multipart_manifest_R2.json",
        "external_pointer_remote":
            "phase_02/release_manifests/"
            "external_artifact_pointer_manifest_R2.yaml",
    },
    "cumulative_project_state":
        cumulative_package,
    "github_ready_derivative":
        github_package,
    "github_ready_uploaded_to_hf":
        True,
    "local_release_zip_policy":
        (
            "DELETED_AFTER_CONFIRMED_HF_UPLOAD"
            if not KEEP_LOCAL_RELEASE_ZIPS
            else "PRESERVED"
        ),
    "disk_after_release":
        {
            k: v
            for k, v in _disk().items()
            if k != "free_bytes"
        },
    "pre_g24_receipt":
        str(PRE_G24_RECEIPT),
}

_atomic_json(
    FINAL_LOCAL_RECEIPT,
    release_receipt,
)

# Keep a copy under stage_artifacts for provenance.
stage24_transport_receipt = (
    RUN_ROOT
    / "stage_artifacts"
    / "24_low_disk_chunked_hf_release_R2.json"
)

_atomic_json(
    stage24_transport_receipt,
    release_receipt,
)

remote_state.update({
    "status": "PASS",
    "completed_at_utc":
        _utc(),
    "content_revision":
        content_revision,
    "manifest_revision":
        manifest_revision,
    "release_revision":
        release_revision,
    "final_local_receipt":
        str(
            FINAL_LOCAL_RECEIPT
        ),
})
_persist_remote_state()

# Provide a NotebookSession finalization successor object for any later local
# code that expects SESSION.finalization to be non-None.
try:
    SESSION.finalization = {
        "status":
            "PASS_EXTERNALIZED_MULTIPART",
        "run_id":
            run_id,
        "scientific_evidence":
            False
            if SESSION.ctx.fixture
            else True,
        "transport_successor":
            "P02-STAGE24-MAX3G-HF-R2.2",
        "hugging_face_repo":
            hf_repo_id,
        "content_revision":
            content_revision,
        "release_revision":
            release_revision,
        "execution_bundle_manifest":
            execution_bundle_manifest,
        "cumulative_project_state":
            cumulative_package,
        "github_ready_derivative":
            github_package,
    }
except Exception:
    pass

# Remove only scratch/cache generated by this retry. Tiny manifests + receipts stay.
if SCRATCH.exists():
    shutil.rmtree(
        SCRATCH
    )

if HF_XET_CACHE_DIR.exists():
    shutil.rmtree(
        HF_XET_CACHE_DIR,
        ignore_errors=True,
    )

print("\n" + "=" * 120)
print(
    "STAGE24 R2.2 — MAX-3GiB HF / GITHUB-READY RELEASE PASS"
)
print("=" * 120)

print(json.dumps({
    "G24":
        "PASS",
    "Stage11":
        "CANONICAL_ACCEPTED_MAIN",
    "Stage12":
        "CANONICAL_ACCEPTED_MAIN",
    "canonical_G18_modified":
        False,
    "Stage18S":
        "PASS_POST_HOC_SENSITIVITY_DESCRIPTIVE",
    "HF_repo":
        hf_repo_id,
    "HF_content_revision":
        content_revision,
    "HF_final_release_revision":
        release_revision,
    "heavy_source_files":
        len(direct_external_rows),
    "heavy_packages":
        len(heavy_package_rows),
    "execution_bundle_packages":
        len(part_rows),
    "hard_max_package_gib":
        MAX_PACKAGE_GIB,
    "cumulative_project_state":
        cumulative_package,
    "github_ready_derivative":
        github_package,
    "github_ready_uploaded_to_HF":
        True,
    "disk_free_gib":
        _disk()["free_gib"],
    "local_pointer_receipt":
        str(
            FINAL_LOCAL_RECEIPT
        ),
}, indent=2, default=str))

print("\nTRUSTED MARKER:")
print(
    "IHARQ_P02_STAGE24_MAX3G_HF_GITHUB_RELEASE_R2_2_PASS"
)


STAGE24 R2.1 — EMERGENCY LOW-DISK RETRY CLEANUP
disk_before = {
  "total_gib": 19.518,
  "used_gib": 15.786,
  "free_gib": 3.717
}
transport_roots = ['/kaggle/working', '/kaggle/working/iharq_p02_run']

transport_cleanup_candidates:
  DIR  FAILED_DERIVED_RELEASE_STAGING /kaggle/working/IHARQ_P02_STAGE24_LOW_DISK_R2
removed transport directory: /kaggle/working/IHARQ_P02_STAGE24_LOW_DISK_R2 (logical 10.5 MiB) FAILED_DERIVED_RELEASE_STAGING

primary_cleanup_result = {
  "removed_files": 0,
  "removed_dirs": 1,
  "logical_bytes_unlinked": 0,
  "logical_gib_unlinked": 0.0,
  "disk_free_gib": 3.722
}

disk_after_cleanup = {
  "total_gib": 19.518,
  "used_gib": 15.781,
  "free_gib": 3.722
}

cleanup PASS — sufficient headroom recovered before any retry staging.
STAGE24_R2_1_EMERGENCY_TRANSPORT_CLEANUP_PASS

Canonical upstream check:
{
  "Stage11": "ACCEPTED_MAIN",
  "Stage12": "ACCEPTED_MAIN",
  "G11": "PASS",
  "G12": "PASS",
  "accepted_stage11_source_sha256": "65a506adb6fa6ce20c2ecd11f9f72

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[heavy_artifacts] package 1 | 2.696 GiB <= 3.000 GiB | 1714 files | uploaded + deleted | disk_free=3.717 GiB


Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            


GITHUB-READY DERIVATIVE — PACKAGE -> HF -> DELETE LOCAL


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


project staging removed after upload: 0.77 GiB logical bytes

STAGE24 R2.2 — MAX-3GiB HF / GITHUB-READY RELEASE PASS
{
  "G24": "PASS",
  "Stage11": "CANONICAL_ACCEPTED_MAIN",
  "Stage12": "CANONICAL_ACCEPTED_MAIN",
  "canonical_G18_modified": false,
  "Stage18S": "PASS_POST_HOC_SENSITIVITY_DESCRIPTIVE",
  "HF_repo": "Csthv/iharq-p02-phase02-r2-P02-L2-OFFICIAL-RUN-R4DY-20260814t051352z",
  "HF_content_revision": "bc88dccd6518aa7b172697be1ef98344997f27ff",
  "HF_final_release_revision": "257a407b5c7ea37b6c620863ee261c010c8f197c",
  "heavy_source_files": 3968,
  "heavy_packages": 4,
  "execution_bundle_packages": 1,
  "hard_max_package_gib": 3.0,
  "cumulative_project_state": {
    "transport": "SINGLE_ZIP",
    "remote_path": "phase_02/release_packages/cumulative_project_state/IHARQ_Project_State_After_Phase_02_R2.zip",
    "sha256": "b4f1d12b0fb37faf908a3eadd2d6a43d52efbe7a2601220a030ffd6858b9b8cc",
    "bytes": 118117660,
    "source_files": 20743,
    "source_bytes": 826871216,
    

In [ ]:
result = RUN_STAGE("24")

# Stable Kaggle-output copy for retrieval; this is the same
# checksummed bundle, not a recomputation.
if SESSION.finalization is None:
    # NotebookSession.run finalizes only when sid == 24.
    raise RuntimeError("FINALIZATION_OBJECT_NOT_AVAILABLE_AFTER_STAGE24")

final_zip = Path(SESSION.finalization["zip"]["path"])
export_zip = KAGGLE_WORKING / final_zip.name

if final_zip.resolve() != export_zip.resolve():
    shutil.copy2(final_zip, export_zip)

print(json.dumps({
    "final_runtime_bundle": str(export_zip),
    "run_id": SESSION.finalization["run_id"],
    "scientific_evidence": SESSION.finalization["scientific_evidence"],
    "runtime_successor_id": P02_RUNTIME_SUCCESSOR_ID,
}, indent=2))

## Final post-Stage24 Hugging Face recovery / release checkpoint

# Final whole-working Hugging Face preservation
Run this **after Stage 24 succeeds** and the kernel is idle. It creates a new unique private Hugging Face dataset repository and uploads the complete meaningful `/kaggle/working` tree with a SHA-256 manifest and remote-path completeness verification. Save the printed `REPO_ID`, `REVISION`, and `EXPECTED_MANIFEST_SHA256`; those are the values for the first restore cell of the next fresh session.


In [24]:
# =============================================================================
# IHARQ — WHOLE /kaggle/working SNAPSHOT -> NEW UNIQUE PRIVATE HF DATASET REPO
# R1 — one new repo per invocation + byte manifest + remote path verification
#
# RUN ONLY AFTER:
#   1) Stage 24 has completed successfully.
#   2) The Kaggle kernel is idle.
#
# IMPORTANT:
#   - Every run creates a NEW repo name. It does not reuse the old continuation repo.
#   - The scientific /kaggle/working payload is preserved.
#   - Only transient transport/editor metadata is excluded:
#       .virtual_documents/
#       .ipynb_checkpoints/
#       any .cache/huggingface/
#   - Hugging Face upload_folder itself ignores .git metadata.
#   - No scientific stage is rerun or modified.
# =============================================================================

from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import re
import subprocess
import sys
import time
import uuid

KAGGLE_WORKING = Path("/kaggle/working").resolve()
REPO_TYPE = "dataset"
PRIVATE = True

# Change only this human-readable label if desired.
SNAPSHOT_LABEL = "p02-phase02-final-whole-working"

# =============================================================================
# HUGGING FACE WRITE TOKEN — SET ONCE BEFORE THE REAL RUN
# =============================================================================
# Paste your Hugging Face WRITE token between the quotes BEFORE running this
# cell. The cell will NOT prompt for a token during execution.
#
# IMPORTANT:
#   - Do not commit/share a notebook containing the real token.
#   - The token is used only in memory for HfApi authentication.
#   - The token is not written into the snapshot metadata or manifest.
#
# Example format only:
# INLINE_HF_TOKEN = "${IHARQ_HF_TOKEN_P02}"
INLINE_HF_TOKEN = "${IHARQ_HF_TOKEN_P02}"

if (
    not INLINE_HF_TOKEN
    or INLINE_HF_TOKEN == "PASTE_YOUR_HF_WRITE_TOKEN_HERE"
    or not INLINE_HF_TOKEN.startswith("hf_")
):
    raise RuntimeError(
        "SET_INLINE_HF_WRITE_TOKEN_BEFORE_RUNNING_THIS_CELL"
    )

STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
NONCE = uuid.uuid4().hex[:8]
LOCAL_META = Path("/kaggle/temp") / f"iharq_working_snapshot_meta_{STAMP}_{NONCE}"
LOCAL_META.mkdir(parents=True, exist_ok=False)

MANIFEST_PATH = LOCAL_META / "WORKING_TREE_MANIFEST.jsonl"
METADATA_PATH = LOCAL_META / "SNAPSHOT_METADATA.json"

REMOTE_META_DIR = "__IHARQ_SNAPSHOT__"
REMOTE_MANIFEST = f"{REMOTE_META_DIR}/WORKING_TREE_MANIFEST.jsonl"
REMOTE_METADATA = f"{REMOTE_META_DIR}/SNAPSHOT_METADATA.json"


def fail(code, detail=None):
    raise RuntimeError(str(code) + (f": {detail}" if detail is not None else ""))


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def is_transient(rel: str) -> bool:
    rel = rel.replace("\\", "/")
    parts = Path(rel).parts

    if rel == ".virtual_documents" or rel.startswith(".virtual_documents/"):
        return True
    if rel == ".ipynb_checkpoints" or rel.startswith(".ipynb_checkpoints/"):
        return True

    # Exclude any Hugging Face local transfer cache anywhere in the tree.
    for i in range(len(parts) - 1):
        if parts[i] == ".cache" and parts[i + 1] == "huggingface":
            return True

    return False


def get_hf_token():
    """
    Non-interactive token resolution for the final snapshot run.

    The token MUST already be hard-coded in INLINE_HF_TOKEN above.
    No getpass prompt, environment lookup, Kaggle secret lookup,
    or Hugging Face cached-token lookup is performed.
    """
    token = (INLINE_HF_TOKEN or "").strip()

    if (
        not token
        or token == "PASTE_YOUR_HF_WRITE_TOKEN_HERE"
        or not token.startswith("hf_")
    ):
        fail("HF_INLINE_WRITE_TOKEN_NOT_CONFIGURED")

    return token


if not KAGGLE_WORKING.is_dir():
    fail("KAGGLE_WORKING_NOT_FOUND", KAGGLE_WORKING)

try:
    from huggingface_hub import HfApi, get_token
except Exception:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--no-input", "-U",
        "huggingface_hub[hf_xet]",
    ])
    from huggingface_hub import HfApi, get_token

token = get_hf_token()
if not token:
    fail("HF_TOKEN_REQUIRED")

api = HfApi(token=token)
print("HF authentication mode = INLINE_HARDCODED_TOKEN (non-interactive)")
who = api.whoami()
username = str(who.get("name") or "").strip()
if not username:
    fail("HF_USERNAME_RESOLUTION_FAILED")

# -------------------------------------------------------------------------
# 1. Build a NEW, unique repo name every time.
# -------------------------------------------------------------------------
slug = re.sub(r"[^a-z0-9._-]+", "-", SNAPSHOT_LABEL.lower()).strip("-")
repo_name = f"{slug}-{STAMP.lower()}-{NONCE}"
repo_id = f"{username}/{repo_name}"

print("=" * 118)
print("IHARQ — WHOLE KAGGLE WORKING SNAPSHOT")
print("=" * 118)
print("source      =", KAGGLE_WORKING)
print("new repo    =", repo_id)
print("repo type   =", REPO_TYPE)
print("private     =", PRIVATE)
print("token       = [PRESENT — NOT DISPLAYED]")

api.create_repo(
    repo_id=repo_id,
    repo_type=REPO_TYPE,
    private=PRIVATE,
    exist_ok=False,
)

# -------------------------------------------------------------------------
# 2. Secret scan of files that would actually be uploaded.
# -------------------------------------------------------------------------
hf_token_pattern = re.compile(rb"hf_[A-Za-z0-9]{20,}")
text_ext = {
    ".py", ".ipynb", ".json", ".jsonl", ".md", ".txt",
    ".yaml", ".yml", ".toml", ".ini", ".cfg", ".sh",
}
secret_hits = []
scanned = 0

print("\nScanning upload-eligible text files for embedded HF tokens...")
for p in KAGGLE_WORKING.rglob("*"):
    if not p.is_file():
        continue
    rel = p.relative_to(KAGGLE_WORKING).as_posix()
    if is_transient(rel):
        continue
    if p.suffix.lower() not in text_ext:
        continue
    if p.stat().st_size > 8 * 1024 * 1024:
        continue
    scanned += 1
    try:
        data = p.read_bytes()
    except Exception:
        continue
    real_token_found = False
    for m in hf_token_pattern.finditer(data):
        candidate = m.group(0)
        body = candidate[3:]
        # Ignore documentation placeholders such as hf_xxxxxxxxx....
        if body and set(body.lower()) == {ord("x")}:
            continue
        real_token_found = True
        break
    if real_token_found:
        secret_hits.append(rel)

if secret_hits:
    fail(
        "EMBEDDED_HF_TOKEN_FOUND_IN_UPLOAD_PAYLOAD",
        {
            "count": len(secret_hits),
            "examples": secret_hits[:20],
            "action":
                "Remove/replace the embedded token in those files before uploading. "
                "Do not disable this guard.",
        },
    )

print(f"Secret scan: PASS ({scanned} eligible text files scanned)")

# -------------------------------------------------------------------------
# 3. Byte-level manifest of the meaningful /kaggle/working payload.
# -------------------------------------------------------------------------
print("\nBuilding SHA-256 working-tree manifest...")
rows = []
total_bytes = 0
payload_files = []

for p in sorted(KAGGLE_WORKING.rglob("*")):
    if not p.is_file():
        continue

    rel = p.relative_to(KAGGLE_WORKING).as_posix()
    if is_transient(rel):
        continue

    # upload_folder ignores .git directories. Exclude them from the manifest too.
    if ".git" in Path(rel).parts:
        continue

    size = int(p.stat().st_size)
    sha = sha256_file(p)
    rows.append({"path": rel, "bytes": size, "sha256": sha})
    payload_files.append(rel)
    total_bytes += size

    n = len(rows)
    if n == 1 or n % 250 == 0:
        print(f"[MANIFEST {n:6d}] payload_GiB={total_bytes/1024**3:.3f}")

if not rows:
    fail("WORKING_TREE_PAYLOAD_EMPTY")

with MANIFEST_PATH.open("w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, sort_keys=True, separators=(",", ":")) + "\n")

manifest_sha = sha256_file(MANIFEST_PATH)

metadata = {
    "artifact_id": "IHARQ-WHOLE-KAGGLE-WORKING-HF-SNAPSHOT-R1",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "repo_id": repo_id,
    "repo_type": REPO_TYPE,
    "private": PRIVATE,
    "source_root": str(KAGGLE_WORKING),
    "manifest_file": REMOTE_MANIFEST,
    "manifest_sha256": manifest_sha,
    "manifested_files": len(rows),
    "manifested_bytes": total_bytes,
    "manifested_gib": total_bytes / 1024**3,
    "excluded_transient_paths": [
        ".virtual_documents/**",
        ".ipynb_checkpoints/**",
        "**/.cache/huggingface/**",
        "**/.git/**",
    ],
    "scientific_stage_rerun": False,
    "scientific_configuration_changed": False,
}
METADATA_PATH.write_text(
    json.dumps(metadata, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print(json.dumps({
    "manifested_files": len(rows),
    "manifested_gib": round(total_bytes / 1024**3, 6),
    "manifest_sha256": manifest_sha,
}, indent=2))

# -------------------------------------------------------------------------
# 4. Upload the entire working payload to the ROOT of the NEW repo.
# -------------------------------------------------------------------------
ignore_patterns = [
    ".virtual_documents/**",
    ".ipynb_checkpoints/**",
    "**/.cache/huggingface/**",
    "**/.git/**",
]

last_exc = None
commit_info = None

for attempt in range(1, 4):
    try:
        print(f"\n[FULL UPLOAD ATTEMPT {attempt}/3]")
        commit_info = api.upload_folder(
            repo_id=repo_id,
            repo_type=REPO_TYPE,
            folder_path=str(KAGGLE_WORKING),
            path_in_repo=".",
            ignore_patterns=ignore_patterns,
            commit_message=f"IHARQ whole /kaggle/working snapshot {STAMP}",
        )
        break
    except Exception as exc:
        last_exc = exc
        print(f"Upload attempt {attempt} failed: {type(exc).__name__}: {exc}")
        if attempt < 3:
            time.sleep(15 * attempt)

if commit_info is None:
    fail("FULL_WORKING_TREE_UPLOAD_FAILED", repr(last_exc))

# -------------------------------------------------------------------------
# 5. Upload manifest + metadata after the payload.
# -------------------------------------------------------------------------
api.upload_file(
    repo_id=repo_id,
    repo_type=REPO_TYPE,
    path_or_fileobj=str(MANIFEST_PATH),
    path_in_repo=REMOTE_MANIFEST,
    commit_message=f"IHARQ snapshot manifest {STAMP}",
)
api.upload_file(
    repo_id=repo_id,
    repo_type=REPO_TYPE,
    path_or_fileobj=str(METADATA_PATH),
    path_in_repo=REMOTE_METADATA,
    commit_message=f"IHARQ snapshot metadata {STAMP}",
)

final_revision = str(api.repo_info(
    repo_id=repo_id,
    repo_type=REPO_TYPE,
).sha)

if not final_revision or len(final_revision) < 20:
    fail("FINAL_HF_REVISION_NOT_RESOLVED", final_revision)

# -------------------------------------------------------------------------
# 6. Remote path completeness check at the immutable final revision.
# -------------------------------------------------------------------------
print("\nChecking remote path completeness at immutable revision...")
remote_files = set(api.list_repo_files(
    repo_id=repo_id,
    repo_type=REPO_TYPE,
    revision=final_revision,
))

missing = [rel for rel in payload_files if rel not in remote_files]
for required in (REMOTE_MANIFEST, REMOTE_METADATA):
    if required not in remote_files:
        missing.append(required)

if missing:
    fail(
        "REMOTE_SNAPSHOT_PATH_COMPLETENESS_FAILED",
        {"missing_count": len(missing), "examples": missing[:25]},
    )

print("\n" + "=" * 118)
print("WHOLE WORKING SNAPSHOT UPLOAD COMPLETE")
print("=" * 118)
print("COPY THESE THREE VALUES INTO THE FRESH-SESSION RESTORE CELL:")
print()
print(f'REPO_ID = "{repo_id}"')
print(f'REVISION = "{final_revision}"')
print(f'EXPECTED_MANIFEST_SHA256 = "{manifest_sha}"')
print()
print(f"manifested_files = {len(rows)}")
print(f"manifested_GiB   = {total_bytes/1024**3:.6f}")
print()
print("TRUSTED SUCCESS MARKER:")
print("IHARQ_WHOLE_KAGGLE_WORKING_HF_SNAPSHOT_COMPLETE")


HF authentication mode = INLINE_HARDCODED_TOKEN (non-interactive)
IHARQ — WHOLE KAGGLE WORKING SNAPSHOT
source      = /kaggle/working
new repo    = Csthv/p02-phase02-final-whole-working-20260814t052120z-b890ff92
repo type   = dataset
private     = True
token       = [PRESENT — NOT DISPLAYED]

Scanning upload-eligible text files for embedded HF tokens...
Secret scan: PASS (21352 eligible text files scanned)

Building SHA-256 working-tree manifest...
[MANIFEST      1] payload_GiB=0.000
[MANIFEST    250] payload_GiB=0.004
[MANIFEST    500] payload_GiB=0.004
[MANIFEST    750] payload_GiB=1.982
[MANIFEST   1000] payload_GiB=1.982
[MANIFEST   1250] payload_GiB=2.067
[MANIFEST   1500] payload_GiB=2.068
[MANIFEST   1750] payload_GiB=2.075
[MANIFEST   2000] payload_GiB=2.081
[MANIFEST   2250] payload_GiB=3.481
[MANIFEST   2500] payload_GiB=4.742
[MANIFEST   2750] payload_GiB=4.745
[MANIFEST   3000] payload_GiB=4.746
[MANIFEST   3250] payload_GiB=6.096
[MANIFEST   3500] payload_GiB=6.106
[MANIFE